In [ ]:
import json
import os
import re
from tqdm import tqdm
from openai import OpenAI

OPENAI_API_KEY = "REDACTED-SET-OPENAI_API_KEY-ENV-VAR"
MODEL_NAME = "gpt-4o"
INPUT_FILE = "scraped_results.json"
OUTPUT_FILE = "ccpa_combined_output.json"

client = OpenAI(api_key=OPENAI_API_KEY)

def build_prompt(domain, policy_text):
    rubric_schema = {
        "completeness_score": "integer between 0 and 3",
        "usability_score": "integer between 0 and 3",
        "accuracy_score": "integer between 0 and 3",
        "policy_contradiction": "boolean",
        "disclosure_map": {
            "data_collected": "boolean",
            "data_shared": "boolean",
            "purpose_of_collection": "boolean",
            "retention_period": "boolean",
            "right_to_access": "boolean",
            "right_to_delete": "boolean",
            "opt_out": "boolean"
        }
    }

    behavior_claims_schema = {
        "honors_gpc": "true | false | 'unspecified'",
        "respects_dnt": "true | false | 'unspecified'",
        "sets_cookies_before_consent": "true | false | 'unspecified'",
        "sets_cookies_after_rejecting_consent": "true | false | 'unspecified'",
        "deletes_cookies_on_rejection": "true | false | 'unspecified'",
        "uses_tracking_only_after_consent": "true | false | 'unspecified'",
        "sells_data": "true | false | 'unspecified'",
        "shares_with_third_parties": "true | false | 'unspecified'",
        "justifications": {
            "honors_gpc": "...",
            "respects_dnt": "...",
            "sets_cookies_before_consent": "...",
            "sets_cookies_after_rejecting_consent": "...",
            "deletes_cookies_on_rejection": "...",
            "uses_tracking_only_after_consent": "...",
            "sells_data": "...",
            "shares_with_third_parties": "..."
        }
    }

    prompt = f"""
You are a privacy compliance analyst working under the California Consumer Privacy Act (CCPA).

Your task is to analyze the privacy policy for the domain: {domain}. Use ONLY the policy text below.
Return your output as a JSON object that includes:

1. **Policy Quality Assessment** (Scored Rubric):
 - completeness_score (0–3): Mentions of access, deletion, and opt-out rights.
 - usability_score (0–3): Are mechanisms like email/forms/links actionable?
 - accuracy_score (0–3): Proper legal terms used (e.g., "sale", "verified consumer")?
 - policy_contradiction (boolean): Whether stated policy claims are suspicious or contradictory
 - disclosure_map: Discloses {list(rubric_schema['disclosure_map'].keys())}?

2. **Semantic Behavior Claims** (for contradiction validation):
Return true, false, or "unspecified" for each:
 - honors_gpc: Does the policy explicitly mention honoring the Global Privacy Control (GPC)?
 - respects_dnt: Mentions honoring Do Not Track?
 - sets_cookies_before_consent: Is tracking initiated before user consent?
 - sets_cookies_after_rejecting_consent: Is tracking persistent even after opt-out?
 - deletes_cookies_on_rejection: Do cookies get removed if user opts out?
 - uses_tracking_only_after_consent: Explicitly states that tracking only happens after opt-in?
 - sells_data: Does it mention selling user data?
 - shares_with_third_parties: Mentions sharing with external parties?
 - justifications: 1–2 sentence textual rationale for each claim based only on policy content

If something is not mentioned clearly, respond with "unspecified".

Return only a JSON object like this (schemas shown):

## Rubric Schema:
{json.dumps(rubric_schema, indent=2)}

## Behavioral Claim Schema:
{json.dumps(behavior_claims_schema, indent=2)}

---

Privacy Policy Text:
{policy_text.strip()[:6000]}

---
"""
    return prompt

def extract_policy_text(policy_dict):
    all_texts = []
    urls_data = policy_dict.get("urls_with_text_versions", [])

    for item in urls_data:
        if not isinstance(item, dict):
            continue
        for _, sublinks in item.items():
            if isinstance(sublinks, list):
                for snippet in sublinks:
                    context = snippet.get("context")
                    if context and context not in all_texts:
                        all_texts.append(context.strip())
            elif isinstance(sublinks, dict):
                for _, passages in sublinks.items():
                    for snippet in passages:
                        context = snippet.get("context")
                        if context and context not in all_texts:
                            all_texts.append(context.strip())

    return "\n\n".join(all_texts)

def run_audit():
    with open(INPUT_FILE, "r") as f:
        raw_data = json.load(f)

    print(f"🔍 Processing all {len(raw_data)} domains from {INPUT_FILE}")

    results = {}

    for domain_entry in tqdm(raw_data):
        domain = list(domain_entry.keys())[0]
        policy_dict = domain_entry[domain]
        policy_text = extract_policy_text(policy_dict)

        if not policy_text:
            results[domain] = {"error": "No policy text found."}
            with open(OUTPUT_FILE, "w") as f:
                json.dump(results, f, indent=2)
            continue

        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "user", "content": build_prompt(domain, policy_text)}
                ],
                temperature=0,
                max_tokens=2048
            )
            content = response.choices[0].message.content

            json_blocks = re.findall(r"```json\s*(\{.*?\})\s*```", content, re.DOTALL)

            if len(json_blocks) == 1:
                results[domain] = json.loads(json_blocks[0])
            elif len(json_blocks) == 2:
                rubric = json.loads(json_blocks[0])
                behaviors = json.loads(json_blocks[1])
                results[domain] = {**rubric, **behaviors}
            else:
                results[domain] = {
                    "error": f"Unexpected number of JSON blocks: {len(json_blocks)}",
                    "raw_response": content
                }

        except Exception as e:
            results[domain] = {
                "error": str(e),
                "raw_response": content if 'content' in locals() else "No content"
            }

        with open(OUTPUT_FILE, "w") as f:
            json.dump(results, f, indent=2)

    print(f"✅ Done! Results saved to: {OUTPUT_FILE}")

if __name__ == "__main__":
    run_audit()

In [ ]:
import json
import os
import re
from tqdm import tqdm
from openai import OpenAI

OPENAI_API_KEY = "REDACTED-SET-OPENAI_API_KEY-ENV-VAR"
MODEL_NAME = "gpt-4o"
INPUT_FILE = "scraped_results.json"
OUTPUT_FILE = "new_ccpa_combined_output.json"

client = OpenAI(api_key=OPENAI_API_KEY)

def build_prompt(domain, policy_text):
    rubric_schema = {
        "completeness_score": "integer between 0 and 3",
        "completeness_reason": "string explaining the score decision",
        "usability_score": "integer between 0 and 3",
        "usability_reason": "string explaining the score decision",
        "accuracy_score": "integer between 0 and 3",
        "accuracy_reason": "string explaining the score decision",
        "policy_contradiction": "boolean",
        "disclosure_map": {
            "data_collected": "boolean",
            "data_shared": "boolean",
            "purpose_of_collection": "boolean",
            "retention_period": "boolean",
            "right_to_access": "boolean",
            "right_to_delete": "boolean",
            "opt_out": "boolean"
        }
    }

    behavior_claims_schema = {
        "honors_gpc": "true | false | 'unspecified'",
        "respects_dnt": "true | false | 'unspecified'",
        "sets_cookies_before_consent": "true | false | 'unspecified'",
        "sets_cookies_after_rejecting_consent": "true | false | 'unspecified'",
        "deletes_cookies_on_rejection": "true | false | 'unspecified'",
        "uses_tracking_only_after_consent": "true | false | 'unspecified'",
        "sells_data": "true | false | 'unspecified'",
        "shares_with_third_parties": "true | false | 'unspecified'",
        "justifications": {
            "honors_gpc": "...",
            "respects_dnt": "...",
            "sets_cookies_before_consent": "...",
            "sets_cookies_after_rejecting_consent": "...",
            "deletes_cookies_on_rejection": "...",
            "uses_tracking_only_after_consent": "...",
            "sells_data": "...",
            "shares_with_third_parties": "..."
        }
    }

    prompt = f"""
You are a privacy compliance analyst working under the California Consumer Privacy Act (CCPA).

Your task is to analyze the privacy policy for the domain: {domain}. Use ONLY the policy text provided below.

Your output should include:

1. **Policy Scope Categorization**
    - online_data_practices: A brief summary of sections discussing data collection or tracking via websites, apps, cookies, analytics, or online forms.
    - offline_data_practices: A brief summary of sections discussing offline data collection like physical stores, phone calls, or in-person forms.
    - If a section is not covered in the policy, state: "Not mentioned."

2. **Policy Quality Assessment (Rubric with Justifications)**
    - completeness_score (0–3): Does the policy mention access, deletion, and opt-out rights?
    - completeness_reason: Why was this score assigned? If low, explain what is missing.
    - usability_score (0–3): Are mechanisms like email/forms/links clearly actionable?
    - usability_reason: Why was this score assigned? If low, explain usability gaps.
    - accuracy_score (0–3): Does the policy use correct legal terms (e.g., "sale", "verified consumer")?
    - accuracy_reason: Why was this score assigned? If low, point out vague or incorrect language.
    - policy_contradiction: Is there any suspicious or self-contradictory claim?
    - disclosure_map: Boolean flags indicating if the policy discloses:
      {list(rubric_schema['disclosure_map'].keys())}

3. **Semantic Behavior Claims (for contradiction validation)**
    Return true, false, or "unspecified" for each:
    - honors_gpc
    - respects_dnt
    - sets_cookies_before_consent
    - sets_cookies_after_rejecting_consent
    - deletes_cookies_on_rejection
    - uses_tracking_only_after_consent
    - sells_data
    - shares_with_third_parties
    - justifications: For each claim, provide 1–2 sentence rationale based only on policy content.

If the policy does not explicitly mention a claim, respond with "unspecified".

Your final output must be a single JSON object with the following structure:

## JSON Schema:
{{
  "online_data_practices": "...",
  "offline_data_practices": "...",
  "rubric_assessment": {json.dumps(rubric_schema, indent=2)},
  "behavioral_claims": {json.dumps(behavior_claims_schema, indent=2)}
}}

---

Privacy Policy Text:
{policy_text.strip()[:6000]}

---
"""
    return prompt

def extract_policy_text(policy_dict):
    all_texts = []
    urls_data = policy_dict.get("urls_with_text_versions", [])

    for item in urls_data:
        if not isinstance(item, dict):
            continue
        for _, sublinks in item.items():
            if isinstance(sublinks, list):
                for snippet in sublinks:
                    context = snippet.get("context")
                    if context and context not in all_texts:
                        all_texts.append(context.strip())
            elif isinstance(sublinks, dict):
                for _, passages in sublinks.items():
                    for snippet in passages:
                        context = snippet.get("context")
                        if context and context not in all_texts:
                            all_texts.append(context.strip())

    return "\n\n".join(all_texts)

def run_audit():
    with open(INPUT_FILE, "r") as f:
        raw_data = json.load(f)

    print(f"🔍 Processing all {len(raw_data)} domains from {INPUT_FILE}")

    results = {}

    for domain_entry in tqdm(raw_data):
        domain = list(domain_entry.keys())[0]
        policy_dict = domain_entry[domain]
        policy_text = extract_policy_text(policy_dict)

        if not policy_text:
            results[domain] = {"error": "No policy text found."}
            with open(OUTPUT_FILE, "w") as f:
                json.dump(results, f, indent=2)
            continue

        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "user", "content": build_prompt(domain, policy_text)}
                ],
                temperature=0,
                max_tokens=4096
            )
            content = response.choices[0].message.content

            json_blocks = re.findall(r"```json\s*(\{.*?\})\s*```", content, re.DOTALL)

            if json_blocks:
                results[domain] = json.loads(json_blocks[0])
            else:

                match = re.search(r"(\{.*\})", content, re.DOTALL)
                if match:
                    results[domain] = json.loads(match.group(1))
                else:
                    results[domain] = {
                        "error": "No JSON found.",
                        "raw_response": content
                    }

        except Exception as e:
            results[domain] = {
                "error": str(e),
                "raw_response": content if 'content' in locals() else "No content"
            }

        with open(OUTPUT_FILE, "w") as f:
            json.dump(results, f, indent=2)

    print(f"✅ Done! Results saved to: {OUTPUT_FILE}")

if __name__ == "__main__":
    run_audit()

In [ ]:
import json
import os
import re
from tqdm import tqdm
from openai import OpenAI

OPENAI_API_KEY = "REDACTED-SET-OPENAI_API_KEY-ENV-VAR"
MODEL_NAME = "gpt-4o"
INPUT_FILE = "scraped_results.json"
OUTPUT_FILE = "ccpa_combined_output.json"

client = OpenAI(api_key=OPENAI_API_KEY)

def build_prompt(domain, policy_text):
    rubric_schema = {
        "completeness_score": "integer between 0 and 3",
        "usability_score": "integer between 0 and 3",
        "accuracy_score": "integer between 0 and 3",
        "policy_contradiction": "boolean",
        "disclosure_map": {
            "data_collected": "boolean",
            "data_shared": "boolean",
            "purpose_of_collection": "boolean",
            "retention_period": "boolean",
            "right_to_access": "boolean",
            "right_to_delete": "boolean",
            "opt_out": "boolean"
        }
    }

    behavior_claims_schema = {
        "honors_gpc": "true | false | 'unspecified'",
        "respects_dnt": "true | false | 'unspecified'",
        "sets_cookies_before_consent": "true | false | 'unspecified'",
        "sets_cookies_after_rejecting_consent": "true | false | 'unspecified'",
        "deletes_cookies_on_rejection": "true | false | 'unspecified'",
        "uses_tracking_only_after_consent": "true | false | 'unspecified'",
        "sells_data": "true | false | 'unspecified'",
        "shares_with_third_parties": "true | false | 'unspecified'",
        "justifications": {
            "honors_gpc": "...",
            "respects_dnt": "...",
            "sets_cookies_before_consent": "...",
            "sets_cookies_after_rejecting_consent": "...",
            "deletes_cookies_on_rejection": "...",
            "uses_tracking_only_after_consent": "...",
            "sells_data": "...",
            "shares_with_third_parties": "..."
        }
    }

    prompt = f"""
You are a privacy compliance analyst working under the California Consumer Privacy Act (CCPA).

Your task is to analyze the privacy policy for the domain: {domain}. Use ONLY the policy text below.
Return your output as a JSON object that includes:

1. **Policy Quality Assessment** (Scored Rubric):
 - completeness_score (0–3): Mentions of access, deletion, and opt-out rights.
 - usability_score (0–3): Are mechanisms like email/forms/links actionable?
 - accuracy_score (0–3): Proper legal terms used (e.g., "sale", "verified consumer")?
 - policy_contradiction (boolean): Whether stated policy claims are suspicious or contradictory
 - disclosure_map: Discloses {list(rubric_schema['disclosure_map'].keys())}?

2. **Semantic Behavior Claims** (for contradiction validation):
Return true, false, or "unspecified" for each:
 - honors_gpc: Does the policy explicitly mention honoring the Global Privacy Control (GPC)?
 - respects_dnt: Mentions honoring Do Not Track?
 - sets_cookies_before_consent: Is tracking initiated before user consent?
 - sets_cookies_after_rejecting_consent: Is tracking persistent even after opt-out?
 - deletes_cookies_on_rejection: Do cookies get removed if user opts out?
 - uses_tracking_only_after_consent: Explicitly states that tracking only happens after opt-in?
 - sells_data: Does it mention selling user data?
 - shares_with_third_parties: Mentions sharing with external parties?
 - justifications: 1–2 sentence textual rationale for each claim based only on policy content

If something is not mentioned clearly, respond with "unspecified".

Return only a JSON object like this (schemas shown):

## Rubric Schema:
{json.dumps(rubric_schema, indent=2)}

## Behavioral Claim Schema:
{json.dumps(behavior_claims_schema, indent=2)}

---

Privacy Policy Text:
{policy_text.strip()[:6000]}

---
"""
    return prompt

def extract_policy_text(policy_dict):
    all_texts = []
    urls_data = policy_dict.get("urls_with_text_versions", [])

    for item in urls_data:
        if not isinstance(item, dict):
            continue
        for _, sublinks in item.items():
            if isinstance(sublinks, list):
                for snippet in sublinks:
                    context = snippet.get("context")
                    if context and context not in all_texts:
                        all_texts.append(context.strip())
            elif isinstance(sublinks, dict):
                for _, passages in sublinks.items():
                    for snippet in passages:
                        context = snippet.get("context")
                        if context and context not in all_texts:
                            all_texts.append(context.strip())

    return "\n\n".join(all_texts)

def run_audit(start_index=850, output_file="ccpa_results_from_850.json"):
    with open(INPUT_FILE, "r") as f:
        raw_data = json.load(f)

    raw_data = raw_data[start_index:]
    num_domains = len(raw_data)
    print(f"🔍 Processing {num_domains} domains starting from index {start_index}...")

    results = {}

    for domain_entry in tqdm(raw_data):
        domain = list(domain_entry.keys())[0]
        policy_dict = domain_entry[domain]
        policy_text = extract_policy_text(policy_dict)

        if not policy_text:
            results[domain] = {"error": "No policy text found."}
            continue

        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "user", "content": build_prompt(domain, policy_text)}
                ],
                temperature=0,
                max_tokens=2048
            )
            content = response.choices[0].message.content

            json_blocks = re.findall(r"```json\s*(\{.*?\})\s*```", content, re.DOTALL)

            if len(json_blocks) == 1:
                results[domain] = json.loads(json_blocks[0])
            elif len(json_blocks) == 2:
                rubric = json.loads(json_blocks[0])
                behaviors = json.loads(json_blocks[1])
                results[domain] = {**rubric, **behaviors}
            else:
                results[domain] = {
                    "error": f"Unexpected number of JSON blocks: {len(json_blocks)}",
                    "raw_response": content
                }

        except Exception as e:
            results[domain] = {
                "error": str(e),
                "raw_response": content if 'content' in locals() else "No content"
            }

    with open(output_file, "w") as f:
        json.dump(results, f, indent=2)

    print(f"✅ Done! Results saved to: {output_file}")

if __name__ == "__main__":
    run_audit(start_index=850, output_file="ccpa_results_from_850.json")

In [ ]:
import json

file1 = "ccpa_combined_output.json"
file2 = "ccpa_results_from_850.json"
output_file = "ccpa_merged_output.json"

with open(file1, "r") as f1, open(file2, "r") as f2:
    data1 = json.load(f1)
    data2 = json.load(f2)

merged_data = {**data1, **data2}

with open(output_file, "w") as out:
    json.dump(merged_data, out, indent=2)

print(f"✅ Merged {len(data1)} + {len(data2)} domains → {len(merged_data)} total")
print(f"📁 Saved to: {output_file}")

In [ ]:
import pandas as pd
import json

merged_path = "cookie_data_final - merged.csv"
df = pd.read_csv(merged_path)

tracking_keywords = ["Targeting", "Performance", "Marketing", "Analytics"]
df["description"] = df["description"].fillna("")

tracking_cookies = df[df["description"].str.contains('|'.join(tracking_keywords), case=False)]

tracking_counts = tracking_cookies.groupby("domain")["cookie_name"].count().reset_index()
tracking_counts.columns = ["domain", "pre_consent_tracking_cookie_count"]

tracking_counts["domain"] = tracking_counts["domain"].str.replace("^\\.", "", regex=True).str.strip()

with open("ccpa_merged_output.json", "r") as f:
    llm_data = json.load(f)

llm_claims = []
for domain_url, results in llm_data.items():
    domain = domain_url.replace("https://", "").replace("http://", "").split("/")[0].strip()
    claim = results.get("uses_tracking_only_after_consent", "unspecified")
    justification = results.get("justifications", {}).get("uses_tracking_only_after_consent", "")
    llm_claims.append({
        "domain": domain,
        "uses_tracking_only_after_consent": claim,
        "llm_justification": justification
    })

df_llm = pd.DataFrame(llm_claims)

merged_df = pd.merge(df_llm, tracking_counts, on="domain", how="left")
merged_df["pre_consent_tracking_cookie_count"] = merged_df["pre_consent_tracking_cookie_count"].fillna(0).astype(int)

def is_silent_tracking(row):
    claim_str = str(row["uses_tracking_only_after_consent"]).strip().lower()
    return claim_str == "true" and row["pre_consent_tracking_cookie_count"] > 0

merged_df["silent_tracking"] = merged_df.apply(is_silent_tracking, axis=1)

merged_df["contradiction_reason"] = merged_df.apply(
    lambda row: "Silent Tracking: Policy claims tracking occurs only after consent, but tracking cookies were observed on initial page load."
    if row["silent_tracking"] else "", axis=1
)

merged_df.to_csv("silent_tracking_contradictions.csv", index=False)

contradictions = merged_df[merged_df["silent_tracking"] == True]
print(f"⚠️  Found {len(contradictions)} domains with Silent Tracking contradictions.")
print(contradictions[["domain", "pre_consent_tracking_cookie_count", "contradiction_reason"]].head(10))

In [ ]:
import pandas as pd
import json

df_banner = pd.read_csv("cookie_data_final - Banner_present.csv")

df_banner["domain"] = df_banner["domain"].astype(str).str.strip().str.lower()
df_banner["website"] = df_banner["website"].astype(str).str.strip().str.lower()

with open("ccpa_merged_output.json", "r") as f:
    llm_results = json.load(f)

llm_records = []
for domain_url, values in llm_results.items():
    domain_cleaned = domain_url.split("//")[-1].split("/")[0].replace("www.", "").strip().lower()
    llm_records.append({
        "domain": domain_cleaned,
        "sets_cookies_after_rejecting_consent": values.get("sets_cookies_after_rejecting_consent", "unspecified"),
        "justification": values.get("justifications", {}).get("sets_cookies_after_rejecting_consent", "")
    })

df_llm = pd.DataFrame(llm_records)

df_llm_filtered = df_llm[df_llm["sets_cookies_after_rejecting_consent"] == "false"]

df_banner["consent_reject"] = df_banner["consent_reject"].astype(str)
df_rejected = df_banner[df_banner["consent_reject"].str.strip() != ""]

reject_counts = df_rejected.groupby("domain")["cookie_name"].count().reset_index()
reject_counts = reject_counts.rename(columns={"cookie_name": "rejected_cookie_count"})

merged = df_llm_filtered.merge(reject_counts, on="domain", how="left")
merged["rejected_cookie_count"] = merged["rejected_cookie_count"].fillna(0).astype(int)

merged["contradiction"] = merged["rejected_cookie_count"] > 0
merged["contradiction_reason"] = merged.apply(
    lambda row: "Cookies were set after rejection despite policy claiming otherwise."
    if row["contradiction"] else "", axis=1
)

contradictions = merged[merged["contradiction"] == True][
    ["domain", "rejected_cookie_count", "justification", "contradiction_reason"]
]

contradictions.to_csv("post_rejection_tracking_contradictions.csv", index=False)

print("✅ Post-Rejection Tracking Analysis Complete.")
print(f"Contradictions found: {len(contradictions)}")
print(contradictions.head(10))

In [ ]:
import pandas as pd
import json

df = pd.read_csv("cookie_data_final - merged.csv")

df["domain"] = df["domain"].astype(str).str.strip().str.lower()

with open("ccpa_merged_output.json", "r") as f:
    llm_data = json.load(f)

llm_rows = []
for domain_url, values in llm_data.items():
    domain = domain_url.split("//")[-1].split("/")[0].replace("www.", "").strip().lower()
    claim = values.get("respects_dnt", "unspecified")
    justification = values.get("justifications", {}).get("respects_dnt", "")
    llm_rows.append({
        "domain": domain,
        "respects_dnt": claim,
        "llm_justification": justification
    })

df_llm = pd.DataFrame(llm_rows)
df_llm_filtered = df_llm[df_llm["respects_dnt"] == "true"]

def count_cookies(row):
    try:
        cookies = eval(row["initial_cookies"])
        dnt_cookies = eval(row["dnt"])
        return pd.Series({
            "initial_cookie_count": len(cookies),
            "dnt_cookie_count": len(dnt_cookies)
        })
    except:
        return pd.Series({"initial_cookie_count": 0, "dnt_cookie_count": 0})

cookie_counts = df.apply(count_cookies, axis=1)
df = pd.concat([df, cookie_counts], axis=1)

agg = df.groupby("domain")[["initial_cookie_count", "dnt_cookie_count"]].mean().reset_index()

merged = df_llm_filtered.merge(agg, on="domain", how="left")
merged = merged.fillna(0)

merged["cookie_reduction_pct"] = (
    1 - (merged["dnt_cookie_count"] / merged["initial_cookie_count"].replace(0, 1))
) * 100

merged["contradiction"] = merged["cookie_reduction_pct"] < 10
merged["contradiction_reason"] = merged.apply(
    lambda row: "DNT signal was ignored: Policy claims DNT is respected, but cookie count remained nearly unchanged."
    if row["contradiction"] else "", axis=1
)

contradictions = merged[merged["contradiction"] == True][
    ["domain", "initial_cookie_count", "dnt_cookie_count", "cookie_reduction_pct", "llm_justification", "contradiction_reason"]
]

contradictions.to_csv("dnt_tracking_contradictions.csv", index=False)

print("✅ DNT contradiction analysis complete.")
print(f"Contradictions found: {len(contradictions)}")
print(contradictions.head(10))

In [ ]:
import json

with open("ccpa_merged_output.json", "r") as f:
    llm_data = json.load(f)

dnt_true = []
dnt_false = []

for domain_url, values in llm_data.items():

    dnt_raw = values.get("respects_dnt", "unspecified")
    dnt_claim = str(dnt_raw).strip().lower()

    domain = domain_url.split("//")[-1].split("/")[0].replace("www.", "").strip().lower()

    if dnt_claim == "true":
        dnt_true.append(domain)
    elif dnt_claim == "false":
        dnt_false.append(domain)

print(f"✅ Domains that explicitly respect DNT: {len(dnt_true)}")
print(dnt_true[:10])

print(f"\n❌ Domains that explicitly deny DNT: {len(dnt_false)}")
print(dnt_false[:10])

## 🍪 Cookie Type Analysis Across Privacy Configurations

This notebook analyzes cookie data across different browser privacy configurations. It quantifies and visualizes how cookie usage—especially tracking-related cookies—changes under each configuration.

### 🔍 Dataset
The dataset (`cookie_data_final - merged.csv`) includes cookie entries extracted under five privacy configurations:
- **Default** (no protection)
- **Do Not Track**
- **Block 3rd Party**
- **Allow 3rd Party**
- **uBlock Origin**

Each configuration has a corresponding column in the dataset indicating the presence of cookies.

---

### 📊 Analysis Workflow

1. **Load and Parse Data**
   - Load the merged CSV file containing cookie dumps.
   - Define mappings for configurations and cookie categories.

2. **Count Cookie Types**
   - For each configuration, count the total number of cookies.
   - Extract counts by category:
     - `Targeting Cookies`
     - `Performance Cookies`
     - `Functional Cookies`
     - `Strictly Necessary Cookies`
   - Compute `Tracking Cookies` as the sum of Targeting and Performance cookies.

3. **Calculate Reductions**
   - Compare each configuration to the Default baseline.
   - Compute the **percentage reduction** in each cookie type.
   - Output:
     - Absolute cookie count table.
     - Reduction percentage table.
     - Combined summary table.

4. **Save Tables**
   - `cookie_counts_by_configuration.csv`: Raw cookie counts per configuration.
   - `cookie_reduction_percentages.csv`: Percentage reductions from Default.
   - `cookie_combined_summary.csv`: Merged table of both absolute and % reductions.

5. **Visualization 1: Stacked Bar Chart**
   - Visualizes the **absolute number of cookies** per type across configurations.
   - Categories color-coded:
     - 🔴 Targeting
     - 🟠 Performance
     - 🔵 Functional
     - 🟢 Strictly Necessary

6. **Visualization 2: Percentage Reduction Bar Chart**
   - Shows reduction (%) in:
     - Targeting Cookies
     - Performance Cookies
     - Functional Cookies
     - Combined Tracking Cookies
   - Useful for understanding the effectiveness of each configuration.

---

### 🎯 Objective

This analysis supports research on cookie compliance, tracking reduction, and user privacy. It reveals how aggressive configurations like **uBlock** drastically reduce tracking technologies, while softer options like **Do Not Track** or **Block 3rd Party** offer limited protection.

The figures and tables can be used in papers, presentations, or audits to show real-world impact of privacy-enhancing technologies.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("cookie_data_final - merged.csv")

configs = {
    "Default": "initial_cookies",
    "Do Not Track": "do_not_track",
    "Block 3rd Party": "block_3rd_party",
    "Allow 3rd Party": "allow_3rd_party",
    "uBlock": "ublock"
}

category_col = "Unnamed: 4"

summary = []

for label, col in configs.items():
    if col in df.columns:
        filtered = df[df[col].notna()]
        total_count = filtered.shape[0]
        counts = filtered[category_col].value_counts()

        targeting = counts.get("Targeting Cookies", 0)
        performance = counts.get("Performance Cookies", 0)
        functional = counts.get("Functional Cookies", 0)
        necessary = counts.get("Strictly Necessary Cookies", 0)

        summary.append({
            "Configuration": label,
            "Total Cookies": total_count,
            "Targeting Cookies": targeting,
            "Performance Cookies": performance,
            "Functional Cookies": functional,
            "Strictly Necessary Cookies": necessary,
            "Tracking Cookies": targeting + performance
        })

summary_df = pd.DataFrame(summary).set_index("Configuration")

default_row = summary_df.loc["Default"]
reduction_df = summary_df.copy()

for col in summary_df.columns:
    if col == "Strictly Necessary Cookies":
        continue
    reduction_df[col + " % Reduction"] = (
        100 * (default_row[col] - summary_df[col]) / default_row[col]
    ).round(2)

combined_df = summary_df.copy()
for col in summary_df.columns:
    if col == "Strictly Necessary Cookies":
        continue
    combined_df[col + " % Reduction"] = reduction_df[col + " % Reduction"]

summary_df.to_csv("cookie_counts_by_configuration.csv")
reduction_df.to_csv("cookie_reduction_percentages.csv")
combined_df.to_csv("cookie_combined_summary.csv")

print("\n📋 Combined Cookie Summary (for LaTeX):")
print(combined_df)

colors = {
    "Targeting Cookies": "#FF9999",
    "Performance Cookies": "#FFCC99",
    "Functional Cookies": "#99CCFF",
    "Strictly Necessary Cookies": "#99FF99"
}

plot_df = summary_df[
    ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]
]

ax = plot_df.plot(
    kind='bar',
    stacked=True,
    color=[colors[c] for c in plot_df.columns],
    figsize=(12, 6),
    edgecolor='black'
)

plt.title("Cookie Type Distribution by Privacy Configuration", fontsize=16, fontweight='bold')
plt.xlabel("Privacy Configuration", fontsize=13)
plt.ylabel("Number of Cookies", fontsize=13)
plt.xticks(rotation=15, ha='center', fontsize=11)
plt.yticks(fontsize=11)
plt.legend(title="Cookie Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("pets_cookie_counts_absolute.png", dpi=300)
plt.savefig("pets_cookie_counts_absolute.pdf")
plt.show()

reduction_plot_df = combined_df[
    ["Targeting Cookies % Reduction", "Performance Cookies % Reduction",
     "Functional Cookies % Reduction", "Tracking Cookies % Reduction"]
].drop("Default")

reduction_plot_df.plot(
    kind="bar",
    figsize=(10, 6),
    color=["#E74C3C", "#F39C12", "#3498DB", "#9B59B6"],
    edgecolor='black'
)

plt.title("Percentage Reduction in Cookie Types Compared to Default", fontsize=15, fontweight='bold')
plt.ylabel("Reduction (%)", fontsize=13)
plt.xlabel("Privacy Configuration", fontsize=13)
plt.xticks(rotation=15, ha='center')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.legend(title="Cookie Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("pets_cookie_reduction_percentages.png", dpi=300)
plt.savefig("pets_cookie_reduction_percentages.pdf")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv("cookie_data_final - merged.csv")
df['expiry_days'] = (df['remaining_expiry_time'] / (3600 * 24)).round(2)
df = df[df['expiry_days'].notna()]

configs = {
    "Default": "initial_cookies",
    "Do Not Track": "do_not_track",
    "Block 3rd Party": "block_3rd_party",
    "Allow 3rd Party": "allow_3rd_party",
    "uBlock": "ublock"
}
category_col = "Unnamed: 4"
cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]

def build_survival_curve(data, expiry_col='expiry_days', max_days=365):
    total = len(data)
    curve = {}
    for day in range(0, max_days + 1):
        survived = (data[expiry_col] > day).sum()
        curve[day] = round((survived / total) * 100, 2) if total else 0.0
    return pd.Series(curve)

survival_all_configs = pd.DataFrame()
for label, col in configs.items():
    filtered = df[df[col].notna()]
    curve = build_survival_curve(filtered)
    survival_all_configs[label] = curve
survival_all_configs.to_csv("survival_curves_all_configs.csv")

survival_by_type = pd.DataFrame()
for ctype in cookie_types:
    filtered = df[(df['initial_cookies'].notna()) & (df[category_col] == ctype)]
    curve = build_survival_curve(filtered)
    survival_by_type[ctype] = curve
survival_by_type.to_csv("survival_curves_by_cookie_type.csv")

summary_stats = []
for label, col in configs.items():
    filtered = df[df[col].notna()]
    summary_stats.append({
        "Configuration": label,
        "Total Cookies": len(filtered),
        "Mean Expiry (days)": round(filtered['expiry_days'].mean(), 2),
        "Median Expiry (days)": round(filtered['expiry_days'].median(), 2),
        "Max Expiry (days)": round(filtered['expiry_days'].max(), 2),
        "90th Percentile": round(np.percentile(filtered['expiry_days'], 90), 2)
    })
pd.DataFrame(summary_stats).to_csv("cookie_expiry_summary_stats.csv", index=False)

plt.figure(figsize=(10, 6))
for label in survival_all_configs.columns:
    plt.plot(survival_all_configs.index, survival_all_configs[label], label=label)
plt.title("Survival Curve by Privacy Configuration", fontsize=14, fontweight='bold')
plt.xlabel("Days Since Set", fontsize=12)
plt.ylabel("Cookies Remaining (%)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig("cookie_survival_curve_by_config.png", dpi=300)
plt.savefig("cookie_survival_curve_by_config.pdf")
plt.show()

plt.figure(figsize=(10, 6))
for label in survival_by_type.columns:
    plt.plot(survival_by_type.index, survival_by_type[label], label=label)
plt.title("Survival Curve by Cookie Type (Default)", fontsize=14, fontweight='bold')
plt.xlabel("Days Since Set", fontsize=12)
plt.ylabel("Cookies Remaining (%)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig("cookie_survival_curve_by_type.png", dpi=300)
plt.savefig("cookie_survival_curve_by_type.pdf")
plt.show()

plt.figure(figsize=(12, 6))
for label, col in configs.items():
    filtered = df[df[col].notna()]
    sns.kdeplot(
        filtered['expiry_days'], label=label, fill=True, alpha=0.3, bw_adjust=0.5
    )
plt.title("Cookie Expiration Distribution by Configuration", fontsize=14, fontweight='bold')
plt.xlabel("Expiry Days", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig("cookie_expiry_distribution_by_config.png", dpi=300)
plt.savefig("cookie_expiry_distribution_by_config.pdf")
plt.show()

plt.figure(figsize=(10, 6))
for label, col in configs.items():
    filtered = df[df[col].notna()]
    sorted_days = np.sort(filtered['expiry_days'])
    cdf = np.arange(len(sorted_days)) / float(len(sorted_days))
    plt.plot(sorted_days, cdf, label=label)
plt.title("CDF of Cookie Expiry by Configuration", fontsize=14, fontweight='bold')
plt.xlabel("Expiry Days", fontsize=12)
plt.ylabel("CDF", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig("cookie_expiry_cdf_by_config.png", dpi=300)
plt.savefig("cookie_expiry_cdf_by_config.pdf")
plt.show()

plt.figure(figsize=(10, 6))
for ctype in cookie_types:
    filtered = df[(df['initial_cookies'].notna()) & (df[category_col] == ctype)]
    sorted_days = np.sort(filtered['expiry_days'])
    cdf = np.arange(1, len(sorted_days) + 1) / float(len(sorted_days))
    plt.plot(sorted_days, cdf, label=ctype)

plt.title("CDF of Cookie Expiry by Cookie Type (Default Configuration)", fontsize=14, fontweight='bold')
plt.xlabel("Expiry Days", fontsize=12)
plt.ylabel("Cumulative Fraction of Cookies", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(title="Cookie Type")
plt.tight_layout()
plt.savefig("cookie_expiry_cdf_by_type_default.png", dpi=300)
plt.savefig("cookie_expiry_cdf_by_type_default.pdf")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

present_df = pd.read_csv("cookie_data_final - Banner_present.csv")
not_present_df = pd.read_csv("cookie_data_final - Banner_not_present.csv")

category_col = "Unnamed: 4" if "Unnamed: 4" in present_df.columns else "category"
expiry_col = "remaining_expiry_time"
cookie_types = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies"
]

def compute_summary(df):
    summary = {
        "Total Cookies": df.shape[0],
        "Tracking Cookies": df[category_col].isin(["Targeting Cookies", "Performance Cookies"]).sum(),
        "Functional Cookies": (df[category_col] == "Functional Cookies").sum(),
        "Strictly Necessary Cookies": (df[category_col] == "Strictly Necessary Cookies").sum(),
        "Targeting Cookies": (df[category_col] == "Targeting Cookies").sum(),
        "Performance Cookies": (df[category_col] == "Performance Cookies").sum(),
        "Median Expiry": df[expiry_col].median(),
        "Mean Expiry": df[expiry_col].mean()
    }
    return summary

present_stats = compute_summary(present_df)
not_present_stats = compute_summary(not_present_df)

summary_df = pd.DataFrame([present_stats, not_present_stats], index=["Banner Present", "Banner Not Present"])
summary_df.to_csv("cookie_banner_summary_stats.csv")
print("\n📋 Banner Presence Summary:\n", summary_df)

counts = pd.DataFrame({
    "Banner Present": present_df[category_col].value_counts(),
    "Banner Not Present": not_present_df[category_col].value_counts()
}).fillna(0)

counts = counts.loc[cookie_types]

counts.plot(kind="bar", figsize=(10, 6), edgecolor='black')
plt.title("Cookie Category Distribution by Banner Presence", fontsize=15, fontweight='bold')
plt.ylabel("Number of Cookies", fontsize=13)
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("banner_cookie_category_distribution.png", dpi=300)
plt.savefig("banner_cookie_category_distribution.pdf")
plt.show()

plt.figure(figsize=(10, 6))
sns.kdeplot(present_df[expiry_col], label="Banner Present", fill=True, alpha=0.3, bw_adjust=0.7)
sns.kdeplot(not_present_df[expiry_col], label="Banner Not Present", fill=True, alpha=0.3, bw_adjust=0.7)
plt.title("Cookie Expiry Distribution by Banner Presence", fontsize=15, fontweight='bold')
plt.xlabel("Expiry Days", fontsize=13)
plt.ylabel("Density", fontsize=13)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("banner_cookie_expiry_distribution.png", dpi=300)
plt.savefig("banner_cookie_expiry_distribution.pdf")
plt.show()

def plot_tracking_pie(df, label):
    tracking = df[category_col].isin(["Targeting Cookies", "Performance Cookies"]).sum()
    non_tracking = df.shape[0] - tracking
    plt.figure(figsize=(5.5, 5.5))
    plt.pie(
        [tracking, non_tracking],
        labels=["Tracking", "Other"],
        colors=["#E74C3C", "#3498DB"],
        autopct='%1.1f%%',
        startangle=140
    )
    plt.title(f"Tracking Cookies Proportion - {label}")
    plt.tight_layout()
    plt.savefig(f"banner_tracking_pie_{label.replace(' ', '_').lower()}.png", dpi=300)
    plt.savefig(f"banner_tracking_pie_{label.replace(' ', '_').lower()}.pdf")
    plt.show()

plot_tracking_pie(present_df, "Banner Present")
plot_tracking_pie(not_present_df, "Banner Not Present")

counts.T.to_csv("cookie_category_counts_by_banner.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("cookie_data_final - Banner_present.csv")

df['set_before_consent'] = df['initial_cookies'].astype(str) != '0'
pre_consent_cookies = df[df['set_before_consent']]

df['present_after_reject'] = df['consent_reject'].astype(str) != '0'
df['deleted_after_reject'] = df['set_before_consent'] & (~df['present_after_reject'])

df['new_after_reject'] = (~df['set_before_consent']) & (df['present_after_reject'])

df['is_tracking'] = df['category'].isin(["Targeting Cookies", "Performance Cookies"])
df['tracking_after_reject'] = df['is_tracking'] & df['present_after_reject']

website_stats = df.groupby('website').agg({
    'set_before_consent': 'sum',
    'deleted_after_reject': 'sum',
    'new_after_reject': 'sum',
    'tracking_after_reject': 'sum',
    'cookie_name': 'count'
}).rename(columns={'cookie_name': 'total_cookies'}).reset_index()

total_websites = website_stats.shape[0]
websites_with_preconsent = (website_stats['set_before_consent'] > 0).sum()
websites_with_deletion = (website_stats['deleted_after_reject'] > 0).sum()
websites_with_mimicry = (website_stats['new_after_reject'] > 0).sum()
websites_with_tracking_after_reject = (website_stats['tracking_after_reject'] > 0).sum()
websites_respecting_rejection = (website_stats['tracking_after_reject'] == 0).sum()

print(f"Total websites: {total_websites}")
print(f"Websites with cookies set before any action: {websites_with_preconsent} ({100 * websites_with_preconsent / total_websites:.1f}%)")
print(f"Websites that deleted cookies after rejection: {websites_with_deletion} ({100 * websites_with_deletion / total_websites:.1f}%)")
print(f"Websites that set new cookies after rejection (mimicry): {websites_with_mimicry} ({100 * websites_with_mimicry / total_websites:.1f}%)")
print(f"Websites that still set tracking cookies after rejection: {websites_with_tracking_after_reject} ({100 * websites_with_tracking_after_reject / total_websites:.1f}%)")
print(f"Websites respecting rejection (no tracking after reject): {websites_respecting_rejection} ({100 * websites_respecting_rejection / total_websites:.1f}%)")

counts = {
    "Cookies Set Pre-Consent": websites_with_preconsent,
    "Deleted After Rejection": websites_with_deletion,
    "New After Rejection": websites_with_mimicry,
    "Tracking After Rejection": websites_with_tracking_after_reject,
    "Respecting Rejection": websites_respecting_rejection
}
plt.figure(figsize=(10, 6))
sns.barplot(x=list(counts.keys()), y=list(counts.values()), palette="muted")
plt.title("Cookie Behavior by Website", fontsize=14, fontweight='bold')
plt.ylabel("Number of Websites")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("cookie_behavior_consent_actions.png", dpi=300)
plt.savefig("cookie_behavior_consent_actions.pdf")
plt.show()

website_stats.to_csv("cookie_behavior_site_summary.csv", index=False)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

input_path = "ccpa_merged_output.json"
output_dir = "semantic_outputs"
os.makedirs(output_dir, exist_ok=True)

with open(input_path, "r") as f:
    raw_data = json.load(f)

flat_data = []
for domain, info in raw_data.items():
    if "error" in info:
        continue

    entry = {
        "domain": domain,
        "completeness_score": info.get("completeness_score", None),
        "usability_score": info.get("usability_score", None),
        "accuracy_score": info.get("accuracy_score", None),
        "policy_contradiction": info.get("policy_contradiction", None),
    }

    semantic_fields = [
        "honors_gpc", "respects_dnt", "sets_cookies_before_consent",
        "sets_cookies_after_rejecting_consent", "deletes_cookies_on_rejection",
        "uses_tracking_only_after_consent", "sells_data", "shares_with_third_parties"
    ]

    for field in semantic_fields:
        entry[field] = str(info.get(field, "unspecified")).lower()

    flat_data.append(entry)

df = pd.DataFrame(flat_data)

score_cols = ["completeness_score", "usability_score", "accuracy_score"]
df[score_cols] = df[score_cols].apply(pd.to_numeric, errors="coerce")

summary = df[score_cols].describe()
summary.to_csv(os.path.join(output_dir, "policy_quality_summary.csv"))

plt.figure(figsize=(8, 6))
sns.boxplot(data=df[score_cols], palette="Set2")
plt.title("Distribution of Policy Quality Scores", fontsize=14, fontweight='bold')
plt.ylabel("Score (0–3)", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "policy_quality_boxplot.png"), dpi=300)
plt.savefig(os.path.join(output_dir, "policy_quality_boxplot.pdf"))
plt.show()

semantic_cols = [
    "honors_gpc", "respects_dnt", "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent", "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent", "sells_data", "shares_with_third_parties"
]

df[semantic_cols] = df[semantic_cols].fillna("unspecified")
claim_counts = df[semantic_cols].apply(lambda col: col.value_counts()).fillna(0).astype(int)
claim_counts.to_csv(os.path.join(output_dir, "semantic_claim_counts.csv"))

plt.figure(figsize=(10, 6))
claim_counts.T[["true", "false", "unspecified"]].plot(
    kind="bar", stacked=True, figsize=(10, 6),
    color=["#2ecc71", "#e74c3c", "#95a5a6"]
)
plt.title("Prevalence of Semantic Policy Claims", fontsize=14, fontweight='bold')
plt.ylabel("Number of Policies", fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.legend(title="Claim Status")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "semantic_claim_prevalence.png"), dpi=300)
plt.savefig(os.path.join(output_dir, "semantic_claim_prevalence.pdf"))
plt.show()

latex_summary = claim_counts.T[["true", "false", "unspecified"]]
latex_summary.columns = ["True", "False", "Unspecified"]
latex_summary.index.name = "Semantic Claim"
latex_summary.to_latex(os.path.join(output_dir, "semantic_claims_table.tex"), index=True)

print("✅ All outputs saved to:", output_dir)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

heatmap_df = df.copy()

semantic_map = {"true": 1, "false": 0, "unspecified": -1}
semantic_cols = [
    "honors_gpc", "respects_dnt", "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent", "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent", "sells_data", "shares_with_third_parties"
]

for col in semantic_cols:
    heatmap_df[col] = heatmap_df[col].map(semantic_map)

plot_cols = ["completeness_score", "usability_score", "accuracy_score"] + semantic_cols
plot_df = heatmap_df.set_index("domain")[plot_cols]

plot_df = plot_df.sort_values("completeness_score", ascending=False)

plt.figure(figsize=(14, max(8, 0.3 * len(plot_df))))
sns.heatmap(
    plot_df,
    cmap=sns.color_palette(["#e74c3c", "#bdc3c7", "#2ecc71"], n_colors=3),
    cbar_kws={"label": "Score / Claim"},
    linewidths=0.5,
    linecolor='lightgray'
)
plt.title("Policy Quality and Semantic Claim Heatmap by Domain", fontsize=14, fontweight='bold')
plt.xlabel("Policy Feature")
plt.ylabel("Domain")
plt.tight_layout()
plt.savefig("semantic_heatmap_per_domain.png", dpi=300)
plt.savefig("semantic_heatmap_per_domain.pdf")
plt.show()

In [ ]:
import json
import pandas as pd
import plotly.express as px

with open("ccpa_merged_output.json", "r") as f:
    data = json.load(f)

records = []
for domain, details in data.items():
    try:
        record = {
            "domain": domain,
            "completeness_score": details["completeness_score"],
            "usability_score": details["usability_score"],
            "accuracy_score": details["accuracy_score"],
        }
        records.append(record)
    except KeyError:
        continue

df = pd.DataFrame(records)

df = df.dropna()

df[["completeness_score", "usability_score", "accuracy_score"]] = df[
    ["completeness_score", "usability_score", "accuracy_score"]
].astype(int)

fig = px.scatter_3d(
    df,
    x="completeness_score",
    y="usability_score",
    z="accuracy_score",
    hover_name="domain",
    color="completeness_score",
    title="3D Scatter Plot of CCPA Scores by Domain",
)
fig.update_layout(margin=dict(l=0, r=0, b=0, t=30))
fig.write_html("ccpa_scores_3d.html")
fig.show()

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

with open("ccpa_merged_output.json", "r") as f:
    data = json.load(f)

records = []
for domain, info in data.items():
    try:
        records.append({
            "domain": domain,
            "completeness_score": info["completeness_score"],
            "usability_score": info["usability_score"],
            "accuracy_score": info["accuracy_score"]
        })
    except KeyError:
        continue

df = pd.DataFrame(records)

df = df.dropna()
df[["completeness_score", "usability_score", "accuracy_score"]] = df[
    ["completeness_score", "usability_score", "accuracy_score"]
].astype(int)

df_long = df.melt(id_vars="domain",
                  value_vars=["completeness_score", "usability_score", "accuracy_score"],
                  var_name="Score Type",
                  value_name="Score")

df_long["x_jitter"] = df_long["Score Type"].map({"completeness_score": 0, "usability_score": 1, "accuracy_score": 2})
df_long["x_jitter"] += np.random.normal(scale=0.05, size=len(df_long))

plt.figure(figsize=(10, 6))
sns.stripplot(data=df_long, x="x_jitter", y="Score", hue="Score Type", jitter=False, dodge=False, palette="Set2", alpha=0.8)
plt.xticks([0, 1, 2], ["Completeness", "Usability", "Accuracy"], fontsize=12)
plt.yticks(range(0, 4), ["0", "1", "2", "3"])
plt.title("CCPA Policy Quality Scores per Domain", fontsize=14, fontweight="bold")
plt.xlabel("")
plt.ylabel("Score (0–3)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(title="", loc="upper right")
plt.tight_layout()
plt.savefig("ccpa_policy_scores_2d.png", dpi=300)
plt.savefig("ccpa_policy_scores_2d.pdf")
plt.show()

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

with open("ccpa_merged_output.json", "r") as f:
    raw_data = json.load(f)

records = []
for domain, result in raw_data.items():
    if isinstance(result, dict) and all(score in result for score in ["completeness_score", "usability_score", "accuracy_score"]):
        records.append({
            "domain": domain,
            "completeness_score": result["completeness_score"],
            "usability_score": result["usability_score"],
            "accuracy_score": result["accuracy_score"]
        })

df = pd.DataFrame(records)
df = df.sort_values(by="completeness_score", ascending=False)

df["short_domain"] = [f"Domain {i+1}" for i in range(len(df))]

df_melted = pd.melt(
    df,
    id_vars=["short_domain"],
    value_vars=["completeness_score", "usability_score", "accuracy_score"],
    var_name="Score Type",
    value_name="Score"
)

plt.figure(figsize=(10, len(df) * 0.35))
sns.set(style="whitegrid")
sns.barplot(
    data=df_melted,
    y="short_domain",
    x="Score",
    hue="Score Type",
    palette="Set2"
)
plt.title("CCPA Policy Quality Scores by Domain", fontsize=15, fontweight="bold")
plt.xlabel("Score (0–3)", fontsize=13)
plt.ylabel("Domain", fontsize=13)
plt.xlim(0, 3.2)
plt.legend(title="Score Type", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("ccpa_scores_horizontal.png", dpi=300)
plt.savefig("ccpa_scores_horizontal.pdf")
plt.show()

In [ ]:
import json
import pandas as pd

with open("ccpa_merged_output.json", "r") as f:
    raw_data = json.load(f)

records = []
for domain, info in raw_data.items():
    if "error" in info:
        continue
    record = {
        "domain": domain,
        "completeness_score": info.get("completeness_score"),
        "usability_score": info.get("usability_score"),
        "accuracy_score": info.get("accuracy_score")
    }
    records.append(record)

df = pd.DataFrame(records)

score_cols = ["completeness_score", "usability_score", "accuracy_score"]
df[score_cols] = df[score_cols].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=score_cols)

summary = {}
for score in score_cols:
    counts = df[score].value_counts().sort_index()
    percentages = counts / len(df) * 100
    summary[score] = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages.round(2)
    })

for score, table in summary.items():
    print(f"\n=== {score.upper()} ===")
    print(table)
    table.to_csv(f"{score}_distribution.csv")

stats_summary = df[score_cols].describe().T
stats_summary.to_csv("ccpa_score_summary_stats.csv")
print("\nSaved detailed stats to 'ccpa_score_summary_stats.csv'")

In [ ]:
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

with open("ccpa_merged_output.json", "r") as f:
    raw_data = json.load(f)

semantic_fields = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties"
]

records = []
domains = []
for domain, result in raw_data.items():
    if not isinstance(result, dict):
        continue
    if any(k not in result for k in semantic_fields):
        continue
    row = []
    for field in semantic_fields:
        value = result[field]
        if isinstance(value, str):
            value = value.lower()
        row.append(value)
    records.append(row)
    domains.append(domain)

df = pd.DataFrame(records, columns=semantic_fields, index=domains)

df.index = [f"D{i+1}" for i in range(len(df))]

value_map = {"true": 1, "false": 0, "unspecified": -1}
df_mapped = df.applymap(lambda x: value_map.get(str(x).lower(), -1))

plt.figure(figsize=(18, 6))
cmap = sns.color_palette(["gray", "red", "green"])
sns.heatmap(
    df_mapped.T,
    cmap=cmap,
    cbar_kws={"ticks": [-1, 0, 1], "label": "Policy Claim"},
    linewidths=0.3,
    linecolor="white"
)

plt.yticks(rotation=0)
plt.xticks(rotation=90)
plt.title("Semantic Behavior Claims in Privacy Policies", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("semantic_policy_heatmap.png", dpi=300)
plt.savefig("semantic_policy_heatmap.pdf")
plt.show()

In [ ]:
import json
import pandas as pd

with open("ccpa_merged_output.json", "r") as f:
    raw_data = json.load(f)

semantic_fields = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties"
]

records = []
for domain, result in raw_data.items():
    if not isinstance(result, dict):
        continue
    if all(field in result for field in semantic_fields):
        row = {field: str(result[field]).lower() for field in semantic_fields}
        records.append(row)

df = pd.DataFrame(records)

summary_stats = {}

for field in semantic_fields:
    counts = df[field].value_counts(dropna=False).reindex(['true', 'false', 'unspecified'], fill_value=0)
    total = counts.sum()
    percentages = (counts / total * 100).round(2)
    summary_stats[field] = pd.DataFrame({
        'Count': counts,
        'Percentage': percentages
    })

for field, stats in summary_stats.items():
    print(f"\n=== {field.upper()} ===")
    print(stats)

In [ ]:
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

with open("ccpa_merged_output.json", "r") as f:
    raw_data = json.load(f)

disclosure_fields = [
    "data_collected",
    "data_shared",
    "purpose_of_collection",
    "retention_period",
    "right_to_access",
    "right_to_delete",
    "opt_out"
]

records = []
for domain, result in raw_data.items():
    if isinstance(result, dict) and isinstance(result.get("disclosure_map"), dict):
        record = {"domain": domain}
        for field in disclosure_fields:
            val = result["disclosure_map"].get(field, "missing")
            record[field] = val
        records.append(record)

df = pd.DataFrame(records)
df.set_index("domain", inplace=True)

df = df.applymap(lambda x: str(x).lower() if pd.notnull(x) else "missing")
df.replace({"true": "True", "false": "False", "missing": "Missing"}, inplace=True)

summary = {}
total = len(df)
for field in disclosure_fields:
    counts = df[field].value_counts()
    percents = (counts / total * 100).round(2)
    field_summary = pd.DataFrame({"Count": counts, "Percentage": percents})
    summary[field] = field_summary

for field, stats in summary.items():
    print(f"\n=== {field.upper()} ===")
    print(stats)

value_map = {"True": 1, "False": 0, "Missing": -1}
df_heatmap = df.applymap(lambda x: value_map.get(x, -1))

df_heatmap.index = [f"D{i+1}" for i in range(len(df_heatmap))]

plt.figure(figsize=(18, 6))
cmap = sns.color_palette(["gray", "red", "green"])
sns.heatmap(
    df_heatmap.T,
    cmap=cmap,
    cbar_kws={"ticks": [-1, 0, 1], "label": "Disclosure Status"},
    linewidths=0.3,
    linecolor="white"
)
plt.yticks(rotation=0)
plt.xticks(rotation=90)
plt.title("CCPA Disclosure Map Across Domains", fontsize=14, fontweight="bold")
plt.tight_layout()

plt.savefig("disclosure_map_heatmap.png", dpi=300)
plt.savefig("disclosure_map_heatmap.pdf")
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("cookie_data_final - merged.csv")

configs = {
    "Default": "initial_cookies",
    "Do Not Track": "do_not_track",
    "Block 3rd Party": "block_3rd_party",
    "Allow 3rd Party": "allow_3rd_party",
    "uBlock": "ublock"
}
category_col = "Unnamed: 4"

summary = []

for label, col in configs.items():
    if col in df.columns:
        filtered = df[df[col].notna()]

        total_count = filtered.shape[0]

        counts = filtered[category_col].value_counts()
        targeting = counts.get("Targeting Cookies", 0)
        performance = counts.get("Performance Cookies", 0)
        functional = counts.get("Functional Cookies", 0)
        necessary = counts.get("Strictly Necessary Cookies", 0)

        site_medians = filtered.groupby("domain")[col].count()
        median_count = site_medians.median()

        summary.append({
            "Configuration": label,
            "Total Cookies": total_count,
            "Targeting Cookies": targeting,
            "Performance Cookies": performance,
            "Functional Cookies": functional,
            "Strictly Necessary Cookies": necessary,
            "Median Cookies per Website": round(median_count, 2)
        })

summary_df = pd.DataFrame(summary)
summary_df = summary_df.set_index("Configuration")
summary_df.to_csv("final_cookie_counts_median_table.csv")

print("\n✅ Final Cookie Summary Table (with Median):")
print(summary_df)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("cookie_data_final - merged.csv")

configs = {
    "Default": "initial_cookies",
    "Do Not Track": "do_not_track",
    "Block 3rd Party": "block_3rd_party",
    "Allow 3rd Party": "allow_3rd_party",
    "uBlock": "ublock"
}

plt.figure(figsize=(8, 6))

for label, col in configs.items():
    if col in df.columns:

        counts = pd.to_numeric(df[col], errors='coerce').dropna()
        counts = counts[counts >= 0].sort_values()

        cdf = np.arange(1, len(counts) + 1) / len(counts)
        median = int(np.median(counts))

        plt.plot(counts, cdf, label=f"{label} (median={median})", linewidth=2)

        plt.axvline(median, linestyle='--', color=plt.gca().lines[-1].get_color(), alpha=0.4)

plt.xscale("log")
plt.xlabel("Number of Cookies per Website", fontsize=13)
plt.ylabel("Cumulative Fraction of Websites", fontsize=13)
plt.title("CDF of Cookies per Website Across Privacy Configurations", fontsize=14, fontweight='bold')
plt.legend(title="Configuration", loc="lower right", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.savefig("pets_cookie_cdf_plot.pdf", dpi=300)
plt.savefig("pets_cookie_cdf_plot.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("cookie_data_final - merged.csv")

config_cols = {
    "Default": "initial_cookies",
    "Do Not Track": "do_not_track",
    "Block 3rd Party": "block_3rd_party",
    "Allow 3rd Party": "allow_3rd_party",
    "uBlock": "ublock"
}

website_cookie_counts = pd.DataFrame()

website_cookie_counts["website"] = df["website"].unique()

for label, col in config_cols.items():
    per_site = df[df[col].notna()].groupby("website").size()
    website_cookie_counts[label] = website_cookie_counts["website"].map(per_site).fillna(0).astype(int)

for label in config_cols.keys():
    if label == "Default":
        continue
    website_cookie_counts[f"{label} Reduction"] = website_cookie_counts["Default"] - website_cookie_counts[label]

website_cookie_counts.to_csv("cookie_counts_per_website.csv", index=False)

print("\n📊 Median Cookies Per Website:")
for label in config_cols.keys():
    median = website_cookie_counts[label].median()
    print(f"- {label}: {median:.0f} cookies")

plt.figure(figsize=(8, 6))

for label in config_cols.keys():
    values = website_cookie_counts[label].sort_values()
    cdf = np.arange(1, len(values) + 1) / len(values)
    median = np.median(values)
    plt.plot(values, cdf, label=f"{label} (median={int(median)})")
    plt.axvline(median, linestyle='--', color=plt.gca().lines[-1].get_color(), alpha=0.4)

plt.xscale("log")
plt.xlabel("Number of Cookies per Website", fontsize=13)
plt.ylabel("Cumulative Fraction of Websites", fontsize=13)
plt.title("CDF of Cookies per Website Across Privacy Configurations", fontsize=14, fontweight='bold')
plt.legend(title="Configuration", loc="lower right", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("pets_cookie_cdf_plot.pdf", dpi=300)
plt.savefig("pets_cookie_cdf_plot.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd

df_banner = pd.read_csv("cookie_data_final - Banner_present.csv")
df_merged = pd.read_csv("cookie_data_final - merged_updated.csv")

target_domain = "dsw.com"
df_banner_dsw = df_banner[df_banner["website"].str.contains(target_domain, na=False)]
df_merged_dsw = df_merged[df_merged["website"].str.contains(target_domain, na=False)]

reject_cookies = df_banner_dsw[df_banner_dsw["consent_reject"].notna()]
initial_cookies = df_merged_dsw.copy()

tracking_categories = ["Targeting Cookies", "Performance Cookies"]

def count_tracking_cookies(df):
    return df[df["category"].fillna("").str.strip().isin(tracking_categories)]

reject_tracking = count_tracking_cookies(reject_cookies)
initial_tracking = count_tracking_cookies(initial_cookies)

print("=== DSW Cookie Contradiction Check ===")
print(f"Total cookies set during consent rejection: {len(reject_cookies)}")
print(f"Tracking cookies set during consent rejection: {len(reject_tracking)}\n")

print(f"Total cookies set during initial load: {len(initial_cookies)}")
print(f"Tracking cookies set during initial load: {len(initial_tracking)}\n")

if len(reject_tracking) > 0:
    print("⚠️ CONTRADICTION: Tracking cookies were set after consent rejection.")
else:
    print("✅ No tracking cookies found after consent rejection.")

reject_tracking.to_csv("tracking_cookies_reject.csv", index=False)
initial_tracking.to_csv("tracking_cookies_initial.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("cookie_data_final - merged.csv")

df.rename(columns={"Unnamed: 4": "category"}, inplace=True)

df.to_csv("cookie_data_final - merged_updated.csv", index=False)

print("✅ Column 'Unnamed: 4' successfully renamed to 'category'.")

In [ ]:
import pandas as pd

df = pd.read_csv("cookie_data_final - merged_updated.csv")

target_domain = "dsw.com"
df_dsw = df[df["website"].str.contains(target_domain, na=False, case=False)].copy()

tracking_categories = ["Targeting Cookies", "Performance Cookies"]

total_initial = df_dsw["initial_cookies"].notna().sum()
total_dnt = df_dsw["do_not_track"].notna().sum()

tracking_dsw = df_dsw[df_dsw["category"].isin(tracking_categories)]
tracking_initial = tracking_dsw["initial_cookies"].notna().sum()
tracking_dnt = tracking_dsw["do_not_track"].notna().sum()

print("=== DNT Contradiction Check: dsw.com ===")
print(f"Total cookies - Initial: {total_initial}")
print(f"Total cookies - DNT:     {total_dnt}\n")

print(f"Tracking cookies - Initial: {tracking_initial}")
print(f"Tracking cookies - DNT:     {tracking_dnt}\n")

tracking_dsw[tracking_dsw["initial_cookies"].notna()].to_csv("tracking_cookies_initial.csv", index=False)
tracking_dsw[tracking_dsw["do_not_track"].notna()].to_csv("tracking_cookies_dnt.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("cookie_data_final - merged_updated.csv")

target_domain = "dsw.com"
df_dsw = df[df["website"].str.contains(target_domain, na=False)]

tracking_categories = ["Targeting Cookies", "Performance Cookies"]

df_initial = df_dsw[df_dsw["initial_cookies"].notna()]

total_initial_cookies = len(df_initial)

tracking_initial_cookies = df_initial[df_initial["category"].isin(tracking_categories)]
num_tracking_cookies = len(tracking_initial_cookies)

print("=== Silent Tracking Contradiction Check: dsw.com ===")
print(f"Total cookies set on initial page load: {total_initial_cookies}")
print(f"Tracking cookies set on initial page load: {num_tracking_cookies}")

tracking_initial_cookies.to_csv("tracking_cookies_initial.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("cookie_data_final - merged.csv")

config_cols = {
    "Default": "initial_cookies",
    "Do Not Track": "do_not_track",
    "Block 3rd Party": "block_3rd_party",
    "Allow 3rd Party": "allow_3rd_party",
    "uBlock": "ublock"
}

website_cookie_counts = pd.DataFrame()
website_cookie_counts["website"] = df["website"].unique()

for label, col in config_cols.items():
    per_site = df[df[col].notna()].groupby("website").size()
    website_cookie_counts[label] = website_cookie_counts["website"].map(per_site).fillna(0).astype(int)

for label in config_cols.keys():
    if label == "Default":
        continue
    website_cookie_counts[f"{label} Reduction"] = website_cookie_counts["Default"] - website_cookie_counts[label]

website_cookie_counts.to_csv("cookie_counts_per_website.csv", index=False)

median_summary = {label: website_cookie_counts[label].median() for label in config_cols.keys()}

print("\n📊 Table 4: Median Cookies Per Website by Configuration")
print("{:<20} {:>10}".format("Configuration", "Median"))
print("-" * 30)
for label, median in median_summary.items():
    print("{:<20} {:>10.0f}".format(label, median))

plt.figure(figsize=(8, 6))

for label in config_cols.keys():
    values = website_cookie_counts[label].sort_values()
    cdf = np.arange(1, len(values) + 1) / len(values)
    median = np.median(values)
    plt.plot(values, cdf, label=f"{label} (median={int(median)})")
    plt.axvline(median, linestyle='--', color=plt.gca().lines[-1].get_color(), alpha=0.4)

plt.xlabel("Number of Cookies per Website", fontsize=13)
plt.ylabel("Cumulative Fraction of Websites", fontsize=13)
plt.title("CDF of Cookies per Website Across Privacy Configurations", fontsize=14, fontweight='bold')
plt.legend(title="Configuration", loc="lower right", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("pets_cookie_cdf_plot_linear.pdf", dpi=300)
plt.savefig("pets_cookie_cdf_plot_linear.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("cookie_data_final - merged_updated.csv")

config_cols = ["allow_3rd_party", "initial_cookies", "block_3rd_party", "do_not_track", "ublock"]
config_labels = ["Allow 3rd Party", "Default", "Block 3rd Party", "Do Not Track", "uBlock"]

cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]

print("\n📊 Median Cookies Per Cookie Type per Configuration:\n")
header_row = "{:<30}".format("Cookie Type") + "".join([f"{label:<20}" for label in config_labels])
print(header_row)
print("-" * len(header_row))

median_table = []

for ctype in cookie_types:
    row = {"Cookie Type": ctype}
    df_type = df[df["category"] == ctype]
    row_str = "{:<30}".format(ctype)
    for col in config_cols:
        median_val = df_type[col].notnull().sum()
        row[col] = median_val
        row_str += f"{median_val:<20}"
    median_table.append(row)
    print(row_str)

website_cookie_counts = df.groupby('website')[config_cols].apply(lambda x: x.notnull().sum()).reset_index()

plt.figure(figsize=(8, 6))
for col, label in zip(config_cols, config_labels):
    values = website_cookie_counts[col].sort_values()
    cdf = np.arange(1, len(values) + 1) / len(values)
    plt.plot(values, cdf, label=f"{label}")

plt.xlabel("Number of Cookies per Website", fontsize=13)
plt.ylabel("Cumulative Fraction of Websites", fontsize=13)
plt.title("CDF of Cookies per Website Across Privacy Configurations", fontsize=14, fontweight='bold')
plt.legend(title="Configuration", loc="lower right", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("pets_cookie_cdf_plot_linear_no_dashed.pdf", dpi=300)
plt.savefig("pets_cookie_cdf_plot_linear_no_dashed.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("cookie_data_final - merged_updated.csv")

config_cols = ["allow_3rd_party", "initial_cookies", "block_3rd_party", "do_not_track", "ublock"]
config_labels = ["Allow 3rd Party", "Default", "Block 3rd Party", "Do Not Track", "uBlock"]

cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]

grouped = df.groupby(['website', 'category'])[config_cols].apply(lambda x: x.notnull().sum()).reset_index()

print("\n📊 Median Cookies Per Website Per Cookie Type per Configuration:\n")
header_row = "{:<30}".format("Cookie Type") + "".join([f"{label:<20}" for label in config_labels])
print(header_row)
print("-" * len(header_row))

for ctype in cookie_types:
    row_str = "{:<30}".format(ctype)
    df_type = grouped[grouped["category"] == ctype]
    for col in config_cols:
        median_val = int(df_type[col].median())
        row_str += f"{median_val:<20}"
    print(row_str)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

present_df = pd.read_csv("cookie_data_final - Banner_present.csv")
not_present_df = pd.read_csv("cookie_data_final - Banner_not_present.csv")

category_col = "category"
website_col = "website"
expiry_col = "remaining_expiry_time"
initial_col = "initial_cookies"
accept_col = "consent_accept"
reject_col = "consent_reject"

cookie_types = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies"
]

def compute_banner_summary(df):
    summary = {
        "Total Cookies": df.shape[0],
        "Targeting Cookies": (df[category_col] == "Targeting Cookies").sum(),
        "Performance Cookies": (df[category_col] == "Performance Cookies").sum(),
        "Functional Cookies": (df[category_col] == "Functional Cookies").sum(),
        "Strictly Necessary Cookies": (df[category_col] == "Strictly Necessary Cookies").sum(),
        "Tracking Cookies": df[category_col].isin(["Targeting Cookies", "Performance Cookies"]).sum(),
        "Median Expiry": df[expiry_col].median(),
        "Mean Expiry": df[expiry_col].mean()
    }
    return summary

banner_present_stats = compute_banner_summary(present_df)
banner_not_present_stats = compute_banner_summary(not_present_df)

banner_summary_df = pd.DataFrame([banner_present_stats, banner_not_present_stats], index=["Banner Present", "Banner Not Present"])
banner_summary_df.to_csv("banner_presence_summary.csv")
print("\n📊 Banner Presence Summary:\n", banner_summary_df)

def consent_flow_breakdown(df):
    breakdown = []
    for ctype in cookie_types:
        df_type = df[df[category_col] == ctype]
        initial_count = df_type[df_type[initial_col].notnull()].shape[0]
        accept_count = df_type[df_type[accept_col].notnull()].shape[0]
        reject_count = df_type[df_type[reject_col].notnull()].shape[0]
        total = initial_count + accept_count + reject_count
        breakdown.append({
            "Cookie Type": ctype,
            "Initial Load": initial_count,
            "Consent Given": accept_count,
            "Consent Rejected": reject_count,
            "Total": total
        })
    return pd.DataFrame(breakdown)

consent_flow_df = consent_flow_breakdown(present_df)
consent_flow_df.to_csv("consent_flow_cookie_counts.csv", index=False)
print("\n📊 Consent Flow Cookie Breakdown:\n", consent_flow_df)

website_breakdown = []
for ctype in cookie_types:
    df_type = present_df[present_df[category_col] == ctype]
    initial_sites = df_type[df_type[initial_col].notnull()][website_col].nunique()
    accept_sites = df_type[df_type[accept_col].notnull()][website_col].nunique()
    reject_sites = df_type[df_type[reject_col].notnull()][website_col].nunique()
    website_breakdown.append({
        "Cookie Type": ctype,
        "Websites (Initial Load)": initial_sites,
        "Websites (Consent Given)": accept_sites,
        "Websites (Consent Rejected)": reject_sites
    })

website_df = pd.DataFrame(website_breakdown)
website_df.to_csv("consent_flow_website_counts.csv", index=False)
print("\n📊 Website-Level Breakdown per Consent Flow:\n", website_df)

cdf_data = []
for stage, col in [("Initial Load", initial_col), ("Consent Given", accept_col), ("Consent Rejected", reject_col)]:
    counts = present_df[present_df[col].notnull()].groupby(website_col).size()
    counts = counts.value_counts().sort_index().cumsum()
    total_websites = counts.max()
    counts_cdf = counts / total_websites
    cdf_data.append((stage, counts_cdf))

plt.figure(figsize=(8, 6))
for stage, cdf in cdf_data:
    plt.step(cdf.index, cdf.values, label=stage)

plt.xlabel("Number of Cookies per Website", fontsize=13)
plt.ylabel("Cumulative Fraction of Websites", fontsize=13)
plt.title("CDF of Cookies per Website Across Consent Stages", fontsize=14, fontweight='bold')
plt.legend(title="Consent Stage", loc="lower right", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("consent_flow_cdf_plot.pdf", dpi=300)
plt.savefig("consent_flow_cdf_plot.png", dpi=300)
plt.show()

plt.figure(figsize=(10, 6))
sns.kdeplot(present_df[expiry_col], label="Banner Present", fill=True, alpha=0.3, bw_adjust=0.7)
sns.kdeplot(not_present_df[expiry_col], label="Banner Not Present", fill=True, alpha=0.3, bw_adjust=0.7)
plt.title("Cookie Expiry Distribution by Banner Presence", fontsize=15, fontweight='bold')
plt.xlabel("Expiry Days", fontsize=13)
plt.ylabel("Density", fontsize=13)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("banner_cookie_expiry_distribution.png", dpi=300)
plt.savefig("banner_cookie_expiry_distribution.pdf")
plt.show()

def plot_tracking_pie(df, label):
    tracking = df[category_col].isin(["Targeting Cookies", "Performance Cookies"]).sum()
    non_tracking = df.shape[0] - tracking
    plt.figure(figsize=(5.5, 5.5))
    plt.pie(
        [tracking, non_tracking],
        labels=["Tracking", "Other"],
        colors=["#E74C3C", "#3498DB"],
        autopct='%1.1f%%',
        startangle=140
    )
    plt.title(f"Tracking Cookies Proportion - {label}")
    plt.tight_layout()
    plt.savefig(f"tracking_pie_{label.replace(' ', '_').lower()}.pdf", dpi=300)
    plt.savefig(f"tracking_pie_{label.replace(' ', '_').lower()}.png", dpi=300)
    plt.show()

plot_tracking_pie(present_df, "Banner Present")
plot_tracking_pie(not_present_df, "Banner Not Present")

category_counts = pd.DataFrame({
    "Banner Present": present_df[category_col].value_counts(),
    "Banner Not Present": not_present_df[category_col].value_counts()
}).fillna(0).loc[cookie_types]

category_counts.to_csv("cookie_category_counts_by_banner.csv")
print("\n📊 Category Counts by Banner Presence:\n", category_counts)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("cookie_data_final - merged_updated.csv")

config_cols = ["allow_3rd_party", "initial_cookies", "block_3rd_party", "do_not_track", "ublock"]
config_labels = ["Allow 3rd Party", "Default", "Block 3rd Party", "Do Not Track", "uBlock"]

cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]

ccpa_groups = {
    "CCPA-Subjected": df[df["ccpa"] == "subjected"],
    "Non-Subjected": df[df["ccpa"] != "subjected"]
}

for group_label, group_df in ccpa_groups.items():
    print(f"\n=== {group_label} ===\n")

    print("\n📊 Median Cookies Per Cookie Type per Configuration:\n")
    header_row = "{:<30}".format("Cookie Type") + "".join([f"{label:<20}" for label in config_labels])
    print(header_row)
    print("-" * len(header_row))

    median_table = []
    for ctype in cookie_types:
        row = {"Cookie Type": ctype}
        df_type = group_df[group_df["category"] == ctype]
        row_str = "{:<30}".format(ctype)
        for col in config_cols:
            median_val = df_type[col].notnull().sum()
            row[col] = median_val
            row_str += f"{median_val:<20}"
        median_table.append(row)
        print(row_str)

    median_df = pd.DataFrame(median_table)
    median_df.to_csv(f"median_cookie_counts_{group_label.replace(' ', '_')}.csv", index=False)

    website_cookie_counts = group_df.groupby('website')[config_cols].apply(lambda x: x.notnull().sum()).reset_index()

    plt.figure(figsize=(8, 6))
    for col, label in zip(config_cols, config_labels):
        values = website_cookie_counts[col].sort_values()
        cdf = np.arange(1, len(values) + 1) / len(values)
        plt.plot(values, cdf, label=f"{label}")

    plt.xlabel("Number of Cookies per Website", fontsize=13)
    plt.ylabel("Cumulative Fraction of Websites", fontsize=13)
    plt.title(f"CDF of Cookies per Website - {group_label}", fontsize=14, fontweight='bold')
    plt.legend(title="Configuration", loc="lower right", fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"pets_cookie_cdf_{group_label.replace(' ', '_').lower()}.pdf", dpi=300)
    plt.savefig(f"pets_cookie_cdf_{group_label.replace(' ', '_').lower()}.png", dpi=300)
    plt.show()

print("\n✅ CCPA vs Non-Subjected Configuration Analysis Completed.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

present_df = pd.read_csv("cookie_data_final - Banner_present.csv")
not_present_df = pd.read_csv("cookie_data_final - Banner_not_present.csv")

category_col = "category"
ccpa_col = "ccpa"
website_col = "website"
expiry_col = "remaining_expiry_time"
initial_col = "initial_cookies"
accept_col = "consent_accept"
reject_col = "consent_reject"

cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]

def compute_banner_summary(df):
    summary = {
        "Total Cookies": df.shape[0],
        "Targeting Cookies": (df[category_col] == "Targeting Cookies").sum(),
        "Performance Cookies": (df[category_col] == "Performance Cookies").sum(),
        "Functional Cookies": (df[category_col] == "Functional Cookies").sum(),
        "Strictly Necessary Cookies": (df[category_col] == "Strictly Necessary Cookies").sum(),
        "Median Expiry": df[expiry_col].median(),
        "Mean Expiry": df[expiry_col].mean()
    }
    return summary

def consent_flow_breakdown(df):
    breakdown = []
    for ctype in cookie_types:
        df_type = df[df[category_col] == ctype]
        initial_count = df_type[df_type[initial_col].notnull()].shape[0]
        accept_count = df_type[df_type[accept_col].notnull()].shape[0]
        reject_count = df_type[df_type[reject_col].notnull()].shape[0]
        total = initial_count + accept_count + reject_count
        breakdown.append({
            "Cookie Type": ctype,
            "Initial Load": initial_count,
            "Consent Given": accept_count,
            "Consent Rejected": reject_count,
            "Total": total
        })
    return pd.DataFrame(breakdown)

def website_level_breakdown(df):
    breakdown = []
    for ctype in cookie_types:
        df_type = df[df[category_col] == ctype]
        initial_sites = df_type[df_type[initial_col].notnull()][website_col].nunique()
        accept_sites = df_type[df_type[accept_col].notnull()][website_col].nunique()
        reject_sites = df_type[df_type[reject_col].notnull()][website_col].nunique()
        breakdown.append({
            "Cookie Type": ctype,
            "Websites (Initial Load)": initial_sites,
            "Websites (Consent Given)": accept_sites,
            "Websites (Consent Rejected)": reject_sites
        })
    return pd.DataFrame(breakdown)

def plot_cdf(df, label_suffix):
    cdf_data = []
    for stage, col in [("Initial Load", initial_col), ("Consent Given", accept_col), ("Consent Rejected", reject_col)]:
        counts = df[df[col].notnull()].groupby(website_col).size()
        counts = counts.value_counts().sort_index().cumsum()
        total_websites = counts.max()
        counts_cdf = counts / total_websites
        cdf_data.append((stage, counts_cdf))

    plt.figure(figsize=(8, 6))
    for stage, cdf in cdf_data:
        plt.step(cdf.index, cdf.values, label=stage)

    plt.xlabel("Number of Cookies per Website", fontsize=13)
    plt.ylabel("Cumulative Fraction of Websites", fontsize=13)
    plt.title(f"CDF of Cookies per Website ({label_suffix})", fontsize=14, fontweight='bold')
    plt.legend(title="Consent Stage", loc="lower right", fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"consent_flow_cdf_{label_suffix.replace(' ', '_').lower()}.pdf", dpi=300)
    plt.savefig(f"consent_flow_cdf_{label_suffix.replace(' ', '_').lower()}.png", dpi=300)
    plt.show()

ccpa_groups = {
    "CCPA-Subjected": "subjected",
    "Non-Subjected": "non-subjected"
}

for label, ccpa_value in ccpa_groups.items():
    if ccpa_value == "subjected":
        present_subset = present_df[present_df[ccpa_col] == "subjected"]
        not_present_subset = not_present_df[not_present_df[ccpa_col] == "subjected"]
    else:
        present_subset = present_df[present_df[ccpa_col] != "subjected"]
        not_present_subset = not_present_df[not_present_df[ccpa_col] != "subjected"]

    print(f"\n=== {label} Analysis ===")

    banner_present_stats = compute_banner_summary(present_subset)
    banner_not_present_stats = compute_banner_summary(not_present_subset)

    banner_summary_df = pd.DataFrame([banner_present_stats, banner_not_present_stats], index=["Banner Present", "Banner Not Present"])
    banner_summary_df.to_csv(f"banner_presence_summary_{label}.csv")
    print(f"\n📊 {label} Banner Presence Summary:\n", banner_summary_df)

    consent_flow_df = consent_flow_breakdown(present_subset)
    consent_flow_df.to_csv(f"consent_flow_cookie_counts_{label}.csv", index=False)
    print(f"\n📊 {label} Consent Flow Breakdown:\n", consent_flow_df)

    website_df = website_level_breakdown(present_subset)
    website_df.to_csv(f"consent_flow_website_counts_{label}.csv", index=False)
    print(f"\n📊 {label} Website-Level Breakdown:\n", website_df)

    plot_cdf(present_subset, label)

print("\n✅ CCPA-Subjected vs Non-Subjected Full Analysis Completed.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

df = pd.read_csv("cookie_data_final - merged_updated.csv")

cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]
industry_col = "Industry"
website_col = "website"
category_col = "category"

industry_cookie_counts = df[df[category_col].isin(cookie_types)].groupby([industry_col, category_col]).size().unstack(fill_value=0)

for ctype in cookie_types:
    if ctype not in industry_cookie_counts.columns:
        industry_cookie_counts[ctype] = 0

industry_cookie_counts["Total"] = industry_cookie_counts.sum(axis=1)
industry_cookie_counts = industry_cookie_counts.sort_values(by="Total", ascending=False).drop(columns="Total")

bar_width = 0.2
industries = industry_cookie_counts.index.tolist()
x = np.arange(len(industries))

plt.figure(figsize=(14, 6))
for i, ctype in enumerate(cookie_types):
    plt.bar(x + (i - 1.5) * bar_width, industry_cookie_counts[ctype], width=bar_width, label=ctype)

plt.xlabel("Industry Sector", fontsize=13)
plt.ylabel("Number of Cookies", fontsize=13)
plt.title("Cookie Type Breakdown per Industry", fontsize=15, fontweight='bold')
plt.xticks(x, industries, rotation=45, ha='right')
plt.legend(title="Cookie Type")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("industry_cookie_breakdown.png", dpi=300)
plt.savefig("industry_cookie_breakdown.pdf")
plt.show()

output_dir = "industry_cdf_plots"
os.makedirs(output_dir, exist_ok=True)

for industry in industries:
    industry_df = df[df[industry_col] == industry]
    website_cookie_counts = industry_df.groupby(website_col).size()
    website_cookie_counts = website_cookie_counts.sort_values()

    if len(website_cookie_counts) < 5:
        continue

    cdf = np.arange(1, len(website_cookie_counts) + 1) / len(website_cookie_counts)

    plt.figure(figsize=(8, 5))
    plt.step(website_cookie_counts.values, cdf, where='post', linewidth=2)
    plt.xlabel("Number of Cookies per Website", fontsize=12)
    plt.ylabel("Cumulative Fraction of Websites", fontsize=12)
    plt.title(f"CDF of Cookies per Website - {industry}", fontsize=13, fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/cdf_{industry.replace(' ', '_').lower()}.png", dpi=300)
    plt.savefig(f"{output_dir}/cdf_{industry.replace(' ', '_').lower()}.pdf")
    plt.close()

print("\n✅ Industry Breakdown Bar Chart + CDF Plots per Industry Generated.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

df = pd.read_csv("cookie_data_final - merged_updated.csv")

cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]
industry_col = "Industry"
website_col = "website"
category_col = "category"

industry_cookie_counts = df[df[category_col].isin(cookie_types)].groupby([industry_col, category_col]).size().unstack(fill_value=0)

for ctype in cookie_types:
    if ctype not in industry_cookie_counts.columns:
        industry_cookie_counts[ctype] = 0

industry_cookie_counts["Total"] = industry_cookie_counts.sum(axis=1)
industry_cookie_counts = industry_cookie_counts.sort_values(by="Total", ascending=False).drop(columns="Total")

bar_width = 0.2
industries = industry_cookie_counts.index.tolist()
x = np.arange(len(industries))

plt.figure(figsize=(14, 6))
for i, ctype in enumerate(cookie_types):
    plt.bar(x + (i - 1.5) * bar_width, industry_cookie_counts[ctype], width=bar_width, label=ctype)

plt.xlabel("Industry Sector", fontsize=13)
plt.ylabel("Number of Cookies", fontsize=13)
plt.title("Cookie Type Breakdown per Industry", fontsize=15, fontweight='bold')
plt.xticks(x, industries, rotation=45, ha='right')
plt.legend(title="Cookie Type")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("industry_cookie_breakdown.png", dpi=300)
plt.savefig("industry_cookie_breakdown.pdf")
plt.show()

print("\n📊 Cookie Type Breakdown per Industry:")
print(industry_cookie_counts)

industry_cookie_counts.to_csv("industry_cookie_breakdown_counts.csv")

industry_website_medians = []

for industry in industries:
    industry_df = df[df[industry_col] == industry]
    website_counts = industry_df.groupby(website_col).size()
    median_cookies = website_counts.median()
    mean_cookies = website_counts.mean()
    p75 = website_counts.quantile(0.75)
    p90 = website_counts.quantile(0.90)

    industry_website_medians.append({
        "Industry": industry,
        "Median Cookies per Website": median_cookies,
        "Mean Cookies per Website": mean_cookies,
        "75th Percentile": p75,
        "90th Percentile": p90,
        "Total Websites": website_counts.nunique()
    })

industry_medians_df = pd.DataFrame(industry_website_medians)
print("\n📊 Website-Level Cookie Statistics per Industry:")
print(industry_medians_df)

industry_medians_df.to_csv("industry_website_cookie_medians.csv", index=False)

output_dir = "industry_cdf_plots"
os.makedirs(output_dir, exist_ok=True)

cdf_summary = []

for industry in industries:
    industry_df = df[df[industry_col] == industry]
    website_cookie_counts = industry_df.groupby(website_col).size()
    website_cookie_counts = website_cookie_counts.sort_values()

    if len(website_cookie_counts) < 5:
        continue

    cdf = np.arange(1, len(website_cookie_counts) + 1) / len(website_cookie_counts)

    plt.figure(figsize=(8, 5))
    plt.step(website_cookie_counts.values, cdf, where='post', linewidth=2)
    plt.xlabel("Number of Cookies per Website", fontsize=12)
    plt.ylabel("Cumulative Fraction of Websites", fontsize=12)
    plt.title(f"CDF of Cookies per Website - {industry}", fontsize=13, fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/cdf_{industry.replace(' ', '_').lower()}.png", dpi=300)
    plt.savefig(f"{output_dir}/cdf_{industry.replace(' ', '_').lower()}.pdf")
    plt.close()

    cdf_summary.append({
        "Industry": industry,
        "50th Percentile (Median)": website_cookie_counts.quantile(0.5),
        "75th Percentile": website_cookie_counts.quantile(0.75),
        "90th Percentile": website_cookie_counts.quantile(0.90),
        "Max Cookies per Website": website_cookie_counts.max()
    })

cdf_summary_df = pd.DataFrame(cdf_summary)
print("\n📊 CDF Quantile Summary per Industry:")
print(cdf_summary_df)

cdf_summary_df.to_csv("industry_cdf_summary_stats.csv", index=False)

print("\n✅ Full Industry Analysis Completed: Counts, Medians, CDFs.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("cookie_data_final - merged_updated.csv")

industry_col = "Industry"
website_col = "website"

industries = df[industry_col].unique()
cdf_data = {}

for industry in industries:
    industry_df = df[df[industry_col] == industry]
    website_cookie_counts = industry_df.groupby(website_col).size()

    if len(website_cookie_counts) < 5:
        continue

    sorted_counts = website_cookie_counts.sort_values()
    cdf = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)
    cdf_data[industry] = (sorted_counts.values, cdf)

plt.figure(figsize=(10, 7))

for industry, (counts, cdf) in cdf_data.items():
    plt.step(counts, cdf, where='post', label=industry)

plt.xlabel("Number of Cookies per Website", fontsize=13)
plt.ylabel("Cumulative Fraction of Websites", fontsize=13)
plt.title("CDF of Cookies per Website Across Industry Sectors", fontsize=14, fontweight='bold')
plt.legend(fontsize=9, loc='lower right', ncol=2)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("industry_cdf_all_sectors.pdf", dpi=300)
plt.savefig("industry_cdf_all_sectors.png", dpi=300)
plt.show()

print("\n✅ CDF Plot for All Industry Sectors Generated.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("cookie_data_final - merged_updated.csv")

industry_col = "Industry"
config_cols = ["allow_3rd_party", "initial_cookies", "block_3rd_party", "do_not_track", "ublock"]
config_labels = ["Allow 3rd Party", "Default", "Block 3rd Party", "Do Not Track", "uBlock"]

industry_config_counts = df.groupby(industry_col)[config_cols].apply(lambda x: x.notnull().sum())

industry_config_counts = industry_config_counts.sort_values(by="initial_cookies", ascending=False)

industries = industry_config_counts.index.tolist()
x = np.arange(len(industries))
bar_width = 0.15

plt.figure(figsize=(14, 7))

for i, (col, label) in enumerate(zip(config_cols, config_labels)):
    plt.bar(x + (i - 2) * bar_width, industry_config_counts[col], width=bar_width, label=label)

plt.xlabel("Industry Sector", fontsize=13)
plt.ylabel("Number of Cookies", fontsize=13)
plt.title("Cookie Counts per Industry Sector Across Privacy Configurations", fontsize=14, fontweight='bold')
plt.xticks(x, industries, rotation=45, ha='right')
plt.legend(title="Privacy Configuration", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("industry_privacy_config_barplot.pdf", dpi=300)
plt.savefig("industry_privacy_config_barplot.png", dpi=300)
plt.show()

industry_config_counts.to_csv("industry_privacy_config_cookie_counts.csv")

print("\n✅ Bar Plot Comparing Industry Sectors per Privacy Configurations Generated.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

df = pd.read_csv("cookie_data_final - merged_updated.csv")

industry_col = "Industry"
category_col = "category"
cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]
config_cols = ["allow_3rd_party", "initial_cookies", "block_3rd_party", "do_not_track", "ublock"]
config_labels = ["Allow 3rd Party", "Default", "Block 3rd Party", "Do Not Track", "uBlock"]

results = []
industries = df[industry_col].unique()

for ctype in cookie_types:
    for industry in industries:
        df_subset = df[(df[category_col] == ctype) & (df[industry_col] == industry)]
        counts = {label: df_subset[col].notnull().sum() for col, label in zip(config_cols, config_labels)}
        counts["Industry"] = industry
        counts["Cookie Type"] = ctype
        results.append(counts)

counts_df = pd.DataFrame(results)

print("\n📊 Cookie Counts per Industry × Privacy Configuration × Cookie Type:\n")
for ctype in cookie_types:
    print(f"\n--- {ctype} ---\n")
    print(counts_df[counts_df["Cookie Type"] == ctype][["Industry"] + config_labels].to_string(index=False))

counts_df.to_csv("industry_config_cookie_type_counts.csv", index=False)

output_dir = "industry_privacy_config_barplots"
os.makedirs(output_dir, exist_ok=True)

for ctype in cookie_types:
    ctype_df = counts_df[counts_df["Cookie Type"] == ctype]
    ctype_df = ctype_df.sort_values(by="Default", ascending=False)
    industries = ctype_df["Industry"].tolist()
    x = np.arange(len(industries))
    bar_width = 0.15

    plt.figure(figsize=(14, 7))
    for i, label in enumerate(config_labels):
        plt.bar(x + (i - 2) * bar_width, ctype_df[label], width=bar_width, label=label)

    plt.xlabel("Industry Sector", fontsize=13)
    plt.ylabel("Number of Cookies", fontsize=13)
    plt.title(f"{ctype} per Industry Sector Across Privacy Configurations", fontsize=14, fontweight='bold')
    plt.xticks(x, industries, rotation=45, ha='right')
    plt.legend(title="Privacy Configuration", fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{ctype.replace(' ', '_').lower()}_barplot.pdf", dpi=300)
    plt.savefig(f"{output_dir}/{ctype.replace(' ', '_').lower()}_barplot.png", dpi=300)
    plt.show()

print("\n✅ Grouped Bar Plots and CSV Results Generated for Each Cookie Type per Industry × Privacy Configurations.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

df = pd.read_csv("cookie_data_final - merged_updated.csv")

industry_col = "Industry"
category_col = "category"
cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]
config_cols = ["allow_3rd_party", "initial_cookies", "block_3rd_party", "do_not_track", "ublock"]
config_labels = ["Allow 3rd Party", "Default", "Block 3rd Party", "Do Not Track", "uBlock"]

results = []
industries = df[industry_col].unique()

for industry in industries:
    df_industry = df[df[industry_col] == industry]
    industry_result = {"Industry": industry}
    for config_col, config_label in zip(config_cols, config_labels):
        for ctype in cookie_types:
            key = f"{config_label} - {ctype}"
            count = df_industry[(df_industry[category_col] == ctype)][config_col].notnull().sum()
            industry_result[key] = count
    results.append(industry_result)

stacked_df = pd.DataFrame(results)

stacked_df = stacked_df.set_index("Industry")
industries = stacked_df.index.tolist()

plot_layers = {}
for ctype in cookie_types:
    layer = []
    for config_label in config_labels:
        col_name = f"{config_label} - {ctype}"
        layer.append(stacked_df[col_name].values)
    plot_layers[ctype] = np.array(layer).T

x = np.arange(len(industries))
bar_width = 0.15

plt.figure(figsize=(16, 8))

colors = {
    "Targeting Cookies": "#E74C3C",
    "Performance Cookies": "#3498DB",
    "Functional Cookies": "#2ECC71",
    "Strictly Necessary Cookies": "#F1C40F"
}

for i, config_label in enumerate(config_labels):
    bottom = np.zeros(len(industries))
    for ctype in cookie_types:
        plt.bar(x + (i - 2) * bar_width, plot_layers[ctype][:, i],
                width=bar_width, bottom=bottom, label=ctype if i == 0 else "",
                color=colors[ctype], edgecolor='black')
        bottom += plot_layers[ctype][:, i]

plt.xlabel("Industry Sector", fontsize=13)
plt.ylabel("Number of Cookies", fontsize=13)
plt.title("Stacked Cookie Type Breakdown per Industry across Privacy Configurations", fontsize=14, fontweight='bold')
plt.xticks(x, industries, rotation=45, ha='right')
plt.legend(title="Cookie Type", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("industry_privacy_config_stacked_barplot.pdf", dpi=300)
plt.savefig("industry_privacy_config_stacked_barplot.png", dpi=300)
plt.show()

print("\n✅ Stacked Bar Plot (Industry × Configurations × Cookie Types) Generated.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

df = pd.read_csv("cookie_data_final - merged_updated.csv")

cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]
industry_col = "Industry"
website_col = "website"
category_col = "category"

industry_cookie_counts = df[df[category_col].isin(cookie_types)].groupby([industry_col, category_col]).size().unstack(fill_value=0)

for ctype in cookie_types:
    if ctype not in industry_cookie_counts.columns:
        industry_cookie_counts[ctype] = 0

industry_cookie_counts["Total"] = industry_cookie_counts.sum(axis=1)
industry_cookie_counts = industry_cookie_counts.sort_values(by="Total", ascending=False).drop(columns="Total")

industries = industry_cookie_counts.index.tolist()
x = np.arange(len(industries))

plt.figure(figsize=(14, 7))

bottom = np.zeros(len(industries))
colors = {
    "Targeting Cookies": "#E74C3C",
    "Performance Cookies": "#3498DB",
    "Functional Cookies": "#2ECC71",
    "Strictly Necessary Cookies": "#F1C40F"
}

for ctype in cookie_types:
    plt.bar(x, industry_cookie_counts[ctype], bottom=bottom, label=ctype, color=colors[ctype], edgecolor='black')
    bottom += industry_cookie_counts[ctype].values

plt.xlabel("Industry Sector", fontsize=13)
plt.ylabel("Number of Cookies", fontsize=13)
plt.title("Stacked Cookie Type Breakdown per Industry", fontsize=15, fontweight='bold')
plt.xticks(x, industries, rotation=45, ha='right')
plt.legend(title="Cookie Type")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("industry_cookie_stacked_barplot.png", dpi=300)
plt.savefig("industry_cookie_stacked_barplot.pdf")
plt.show()

print("\n📊 Cookie Type Breakdown per Industry:")
print(industry_cookie_counts)

industry_cookie_counts.to_csv("industry_cookie_breakdown_counts.csv")

industry_website_medians = []

for industry in industries:
    industry_df = df[df[industry_col] == industry]
    website_counts = industry_df.groupby(website_col).size()
    median_cookies = website_counts.median()
    mean_cookies = website_counts.mean()
    p75 = website_counts.quantile(0.75)
    p90 = website_counts.quantile(0.90)

    industry_website_medians.append({
        "Industry": industry,
        "Median Cookies per Website": median_cookies,
        "Mean Cookies per Website": mean_cookies,
        "75th Percentile": p75,
        "90th Percentile": p90,
        "Total Websites": website_counts.nunique()
    })

industry_medians_df = pd.DataFrame(industry_website_medians)
print("\n📊 Website-Level Cookie Statistics per Industry:")
print(industry_medians_df)

industry_medians_df.to_csv("industry_website_cookie_medians.csv", index=False)

output_dir = "industry_cdf_plots"
os.makedirs(output_dir, exist_ok=True)

cdf_summary = []

for industry in industries:
    industry_df = df[df[industry_col] == industry]
    website_cookie_counts = industry_df.groupby(website_col).size()
    website_cookie_counts = website_cookie_counts.sort_values()

    if len(website_cookie_counts) < 5:
        continue

    cdf = np.arange(1, len(website_cookie_counts) + 1) / len(website_cookie_counts)

    plt.figure(figsize=(8, 5))
    plt.step(website_cookie_counts.values, cdf, where='post', linewidth=2)
    plt.xlabel("Number of Cookies per Website", fontsize=12)
    plt.ylabel("Cumulative Fraction of Websites", fontsize=12)
    plt.title(f"CDF of Cookies per Website - {industry}", fontsize=13, fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/cdf_{industry.replace(' ', '_').lower()}.png", dpi=300)
    plt.savefig(f"{output_dir}/cdf_{industry.replace(' ', '_').lower()}.pdf")
    plt.close()

    cdf_summary.append({
        "Industry": industry,
        "50th Percentile (Median)": website_cookie_counts.quantile(0.5),
        "75th Percentile": website_cookie_counts.quantile(0.75),
        "90th Percentile": website_cookie_counts.quantile(0.90),
        "Max Cookies per Website": website_cookie_counts.max()
    })

cdf_summary_df = pd.DataFrame(cdf_summary)
print("\n📊 CDF Quantile Summary per Industry:")
print(cdf_summary_df)

cdf_summary_df.to_csv("industry_cdf_summary_stats.csv", index=False)

print("\n✅ Full Industry Stacked Barplot + Detailed Analysis Completed.")

In [ ]:
import pandas as pd

df = pd.read_csv("cookie_data_final - merged_updated.csv")

unique_websites = df['website'].nunique()

print(f"Total Unique Websites: {unique_websites}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

df = pd.read_csv("cookie_data_final - merged_updated.csv")

cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]
industry_col = "Industry"
website_col = "website"
category_col = "category"

industry_cookie_counts = df[df[category_col].isin(cookie_types)].groupby([industry_col, category_col]).size().unstack(fill_value=0)

for ctype in cookie_types:
    if ctype not in industry_cookie_counts.columns:
        industry_cookie_counts[ctype] = 0

total_websites_per_industry = df.groupby(industry_col)[website_col].nunique()

industry_cookie_counts = industry_cookie_counts.merge(
    total_websites_per_industry.rename("Total Websites"),
    left_index=True, right_index=True, how="left"
)

industries = industry_cookie_counts.index.tolist()
x = np.arange(len(industries))

plt.figure(figsize=(14, 7))

bottom = np.zeros(len(industries))
colors = {
    "Targeting Cookies": "#E74C3C",
    "Performance Cookies": "#3498DB",
    "Functional Cookies": "#2ECC71",
    "Strictly Necessary Cookies": "#F1C40F"
}

for ctype in cookie_types:
    plt.bar(x, industry_cookie_counts[ctype], bottom=bottom, label=ctype, color=colors[ctype], edgecolor='black')
    bottom += industry_cookie_counts[ctype].values

plt.xlabel("Industry Sector", fontsize=13)
plt.ylabel("Number of Cookies", fontsize=13)
plt.title("Stacked Cookie Type Breakdown per Industry", fontsize=15, fontweight='bold')
plt.xticks(x, industries, rotation=45, ha='right')
plt.legend(title="Cookie Type")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("industry_cookie_stacked_barplot_fixed_websites.png", dpi=300)
plt.savefig("industry_cookie_stacked_barplot_fixed_websites.pdf")
plt.show()

print("\n📊 Cookie Type Breakdown per Industry (with Total Websites):")
print(industry_cookie_counts)

industry_cookie_counts.to_csv("industry_cookie_breakdown_with_total_websites.csv")

print(f"\n✅ Total Unique Websites Counted Across All Industries: {total_websites_per_industry.sum()}")

In [ ]:
import json
from collections import Counter

with open("new_ccpa_combined_output.json", "r") as f:
    data = json.load(f)

coverage_counter = Counter()
examples_per_category = {
    "Online Only": [],
    "Offline Only": [],
    "Both Online & Offline": [],
    "Neither": []
}

for domain, result in data.items():
    if 'error' in result:
        continue

    online_text = result.get("online_data_practices", "").strip().lower()
    offline_text = result.get("offline_data_practices", "").strip().lower()

    has_online = online_text and online_text != "not mentioned."
    has_offline = offline_text and offline_text != "not mentioned."

    if has_online and has_offline:
        category = "Both Online & Offline"
    elif has_online:
        category = "Online Only"
    elif has_offline:
        category = "Offline Only"
    else:
        category = "Neither"

    coverage_counter[category] += 1

    if len(examples_per_category[category]) < 5:
        examples_per_category[category].append({
            "domain": domain,
            "online": result.get("online_data_practices", ""),
            "offline": result.get("offline_data_practices", "")
        })

print("\n📊 Online/Offline Coverage Breakdown (excluding error domains):")
total = sum(coverage_counter.values())
for key, count in coverage_counter.items():
    print(f"{key}: {count} ({(count/total)*100:.1f}%)")
print(f"\n✅ Total Domains Analyzed: {total}")

for category, examples in examples_per_category.items():
    print(f"\n--- {category} ---")
    for example in examples:
        print(f"Domain: {example['domain']}")
        print(f"Online: {example['online']}")
        print(f"Offline: {example['offline']}")
        print("-" * 80)

In [ ]:
import json
from collections import Counter, defaultdict

with open("new_ccpa_combined_output.json", "r") as f:
    data = json.load(f)

scores_summary = defaultdict(list)
reasons_summary = defaultdict(list)

for domain, result in data.items():

    if 'error' in result:
        continue

    rubric = result.get('rubric_assessment')
    if not rubric:
        continue

    for score_type in ["completeness_score", "usability_score", "accuracy_score"]:
        score = rubric.get(score_type)
        reason_key = f"{score_type.replace('_score', '')}_reason"
        reason = rubric.get(reason_key, "No reason provided.")
        if score is not None:
            scores_summary[score_type].append(score)
            reasons_summary[score_type].append((score, reason))

print("\n📊 Rubric Scores Distribution:")
for score_type in scores_summary:
    counter = Counter(scores_summary[score_type])
    total = sum(counter.values())
    print(f"\n--- {score_type.replace('_', ' ').title()} ---")
    for score_val in range(0, 4):
        count = counter.get(score_val, 0)
        print(f"Score {score_val}: {count} ({(count/total)*100:.1f}%)")

print("\n📋 Sample Reasons for Low Scores (0 or 1):")
for score_type in reasons_summary:
    print(f"\n--- {score_type.replace('_', ' ').title()} ---")
    shown = 0
    for score, reason in reasons_summary[score_type]:
        if score <= 1:
            print(f"Score {score}: {reason}")
            shown += 1
        if shown >= 5:
            break

print("\n✅ Rubric Scores Breakdown Completed.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rubric_scores = {
    "Completeness": {0: 139, 1: 118, 2: 191, 3: 52},
    "Usability": {0: 122, 1: 49, 2: 209, 3: 120},
    "Accuracy": {0: 120, 1: 47, 2: 169, 3: 164}
}

score_labels = [0, 1, 2, 3]
categories = list(rubric_scores.keys())
bar_width = 0.5

score_counts_per_category = []
for cat in categories:
    counts = [rubric_scores[cat].get(s, 0) for s in score_labels]
    score_counts_per_category.append(counts)

score_counts_per_category = np.array(score_counts_per_category)

fig, ax = plt.subplots(figsize=(8, 6))

bottom = np.zeros(len(categories))
colors = ['#E74C3C', '#F39C12', '#3498DB', '#2ECC71']

for idx, score_val in enumerate(score_labels):
    ax.bar(categories, score_counts_per_category[:, idx], bottom=bottom, label=f"Score {score_val}", color=colors[idx], edgecolor='black')
    bottom += score_counts_per_category[:, idx]

ax.set_ylabel('Number of Websites', fontsize=12)
ax.set_title('Rubric Scores Distribution (Completeness, Usability, Accuracy)', fontsize=14, fontweight='bold')
ax.legend(title="Rubric Score")
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.savefig("rubric_scores_barplot.pdf", dpi=300)
plt.savefig("rubric_scores_barplot.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

with open("new_ccpa_combined_output.json", "r") as f:
    data = json.load(f)

disclosure_fields = ["data_collected", "data_shared", "purpose_of_collection",
                     "retention_period", "right_to_access", "right_to_delete", "opt_out"]

disclosure_counts = {field: 0 for field in disclosure_fields}
total_domains = 0

disclosure_matrix = []
for domain, result in data.items():
    rubric = result.get('rubric_assessment')
    if not rubric:
        continue

    disclosures = rubric.get('disclosure_map', {})
    if not disclosures:
        continue

    row = []
    for field in disclosure_fields:
        value = disclosures.get(field, False)
        if value:
            disclosure_counts[field] += 1
        row.append(1 if value else 0)

    disclosure_matrix.append(row)
    total_domains += 1

disclosure_df = pd.DataFrame(disclosure_matrix, columns=disclosure_fields)

plt.figure(figsize=(10, 6))
sns.heatmap(disclosure_df.T, cmap="Blues", cbar=False, linewidths=0.5)
plt.title("Disclosure Coverage Heatmap Across Policies", fontsize=14, fontweight='bold')
plt.xlabel("Policy Index", fontsize=12)
plt.ylabel("Disclosure Field", fontsize=12)
plt.tight_layout()
plt.savefig("disclosure_heatmap.png", dpi=300)
plt.show()

coverage_percentages = [disclosure_counts[field]/total_domains*100 for field in disclosure_fields]
missing_percentages = [100 - val for val in coverage_percentages]

df_bar = pd.DataFrame({
    'Disclosure': disclosure_fields,
    'Covered': coverage_percentages,
    'Missing': missing_percentages
})

df_bar.set_index('Disclosure')[['Covered', 'Missing']].plot(kind='bar', stacked=True, figsize=(10,6), color=["#2ECC71", "#E74C3C"], edgecolor='black')
plt.title("Disclosure Coverage Stacked Bar Chart", fontsize=14, fontweight='bold')
plt.ylabel("Percentage of Policies", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("disclosure_stacked_bar.png", dpi=300)
plt.show()

print("\n📊 Disclosure Fulfillment Summary:")
for field in disclosure_fields:
    count = disclosure_counts[field]
    print(f"{field}: {count}/{total_domains} ({(count/total_domains)*100:.1f}%)")

In [ ]:
import json
from collections import Counter

with open("new_ccpa_combined_output.json", "r") as f:
    data = json.load(f)

behavior_fields = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties"
]

field_coverage = {field: Counter() for field in behavior_fields}

for domain, result in data.items():
    if 'error' in result:
        continue

    behavioral_claims = result.get('behavioral_claims', {})
    for field in behavior_fields:
        value = behavioral_claims.get(field, "unspecified")
        if isinstance(value, bool):
            value = str(value).lower()
        elif not isinstance(value, str):
            value = "unspecified"
        else:
            value = value.lower()
        field_coverage[field][value] += 1

print("\n📊 Behavioral Claims Semantic Coverage Summary:")
total_domains = sum(1 for domain in data if 'error' not in data[domain])
for field in behavior_fields:
    unspecified = field_coverage[field].get("unspecified", 0)
    true_count = field_coverage[field].get("true", 0)
    false_count = field_coverage[field].get("false", 0)
    print(f"\n--- {field} ---")
    print(f"Unspecified: {unspecified} ({(unspecified/total_domains)*100:.1f}%)")
    print(f"True: {true_count} ({(true_count/total_domains)*100:.1f}%)")
    print(f"False: {false_count} ({(false_count/total_domains)*100:.1f}%)")

print(f"\n✅ Total Domains Analyzed: {total_domains}")

In [ ]:
import json
from collections import defaultdict

with open("new_ccpa_combined_output.json", "r") as f:
    data = json.load(f)

behavior_fields = [
    "honors_gpc", "respects_dnt", "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent", "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent", "sells_data", "shares_with_third_parties"
]

vagueness_patterns = [
    "does not mention", "does not specify", "not specified",
    "unspecified", "no mention", "no reference"
]

justification_stats = defaultdict(lambda: {"total": 0, "vague": 0})
justification_examples = defaultdict(list)

for domain, result in data.items():
    behavioral_claims = result.get("behavioral_claims", {})
    justifications = behavioral_claims.get("justifications", {})

    for field in behavior_fields:
        justification = justifications.get(field, "").strip()
        if justification:
            justification_stats[field]["total"] += 1

            if any(pat in justification.lower() for pat in vagueness_patterns):
                justification_stats[field]["vague"] += 1

            if len(justification_examples[field]) < 3:
                justification_examples[field].append({
                    "domain": domain,
                    "justification": justification
                })

print("\n📊 Justifications Quality Summary (Vagueness Coverage):\n")
for field in behavior_fields:
    total = justification_stats[field]["total"]
    vague = justification_stats[field]["vague"]
    vague_percent = (vague / total) * 100 if total else 0
    print(f"--- {field} ---")
    print(f"Total Policies with Claim: {total}")
    print(f"Vague Justifications: {vague} ({vague_percent:.1f}%)")
    print("Sample Justifications:")
    for ex in justification_examples[field]:
        print(f"Domain: {ex['domain']}\nJustification: {ex['justification']}\n")
    print("-" * 70)

print("\n✅ Justifications Quality Analysis Completed.")

In [ ]:
import json

with open("new_ccpa_combined_output.json", "r") as f:
    data = json.load(f)

behavior_fields = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties"
]

examples = {field: {'true': None, 'false': None, 'unspecified': None} for field in behavior_fields}

for domain, result in data.items():
    behavioral_claims = result.get("behavioral_claims", {})
    justifications = behavioral_claims.get("justifications", {})

    for field in behavior_fields:
        claim_value = str(behavioral_claims.get(field, "unspecified")).lower()
        justification_text = justifications.get(field, None)

        if claim_value in examples[field] and examples[field][claim_value] is None and justification_text:
            examples[field][claim_value] = {
                "domain": domain,
                "justification": justification_text.strip()
            }

for field in behavior_fields:
    print(f"\n--- {field} ---")
    for case in ['true', 'false', 'unspecified']:
        example = examples[field][case]
        if example:
            print(f"\nCase: {case.capitalize()}")
            print(f"Domain: {example['domain']}")
            print(f"Justification: {example['justification']}")
        else:
            print(f"\nCase: {case.capitalize()} - No example found.")

print("\n✅ Example Justifications Extracted.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("cookie_data_final - merged_updated.csv")

cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]
website_col = "website"
category_col = "category"
initial_cookies_col = "initial_cookies"

df_filtered = df[df[category_col].isin(cookie_types)].dropna(subset=[initial_cookies_col])

website_cookie_counts = df_filtered.groupby([website_col, category_col]).size().unstack(fill_value=0)

plt.figure(figsize=(10, 6))

for ctype in cookie_types:
    counts = website_cookie_counts[ctype].values
    counts = np.clip(counts, 0, 30)
    counts_sorted = np.sort(counts)
    cdf = np.arange(1, len(counts_sorted) + 1) / len(counts_sorted)
    plt.step(counts_sorted, cdf * 100, where='post', label=ctype)

plt.xlim(0, 30)
plt.ylim(0, 100)
plt.xlabel("Number of Cookies per Website", fontsize=12)
plt.ylabel("Cumulative Percentage of Websites", fontsize=12)
plt.title("CDF of Cookie Counts per Website by Cookie Type (Default Configuration)", fontsize=14, fontweight='bold')
plt.legend(title="Cookie Type")
plt.grid(axis='both', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.savefig("cookie_type_cdf_shaped_sketch.pdf", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("cookie_data_final - merged_updated.csv")

df_targeting = df[df['category'] == 'Targeting Cookies']

config_cols = {
    'Default': 'initial_cookies',
    'Do Not Track': 'do_not_track',
    'Block 3rd Party': 'block_3rd_party',
    'Allow 3rd Party': 'allow_3rd_party',
    'uBlock': 'ublock'
}

website_cookie_counts = {config: [] for config in config_cols}

for website, group in df_targeting.groupby('website'):
    for config, col in config_cols.items():
        cookie_set = group[col].dropna().unique()
        count = len(cookie_set)
        website_cookie_counts[config].append(count)

plt.figure(figsize=(10, 6))

for config, counts in website_cookie_counts.items():
    counts = np.array(counts)
    sorted_counts = np.sort(counts)
    cdf = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)
    plt.step(sorted_counts, cdf * 100, where='post', label=config)

plt.xlim(0, 30)
plt.ylim(0, 100)
plt.xlabel("Number of Targeting Cookies per Website", fontsize=12)
plt.ylabel("Cumulative Percentage of Websites", fontsize=12)
plt.title("CDF of Targeting Cookies per Website across Privacy Configurations", fontsize=14, fontweight='bold')
plt.legend(title="Configuration")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.savefig("targeting_cookies_cdf_comparison_sketch.pdf", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("cookie_data_final - merged_updated.csv")

cookie_types = ["Targeting Cookies", "Performance Cookies", "Functional Cookies", "Strictly Necessary Cookies"]
website_col = "website"
category_col = "category"
initial_cookies_col = "initial_cookies"

df_filtered = df[df[category_col].isin(cookie_types)].dropna(subset=[initial_cookies_col])

website_cookie_counts = df_filtered.groupby([website_col, category_col]).size().unstack(fill_value=0)

plt.figure(figsize=(10, 6))

for ctype in cookie_types:
    counts = website_cookie_counts[ctype].values
    counts = np.clip(counts, 0, 30)
    counts_sorted = np.sort(counts)
    total_websites = len(counts_sorted)
    cdf = np.arange(1, total_websites + 1)
    plt.step(counts_sorted, cdf, where='post', label=ctype)

plt.xlim(0, 30)
plt.xlabel("Number of Cookies per Website", fontsize=12)
plt.ylabel("Cumulative Number of Websites", fontsize=12)
plt.title("CDF of Cookie Counts per Website by Cookie Type (Absolute Numbers)", fontsize=14, fontweight='bold')
plt.legend(title="Cookie Type")
plt.grid(axis='both', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.savefig("cookie_type_cdf_absolute.pdf", dpi=300)
plt.savefig("cookie_type_cdf_absolute.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

CSV_PATH = "cookie_data_final - Banner_present.csv"
WEBSITE_COL = "website"
CANDIDATE_CATEGORY_COLS = ["category", "Unnamed: 4"]
STAGE_COLS = ["initial_cookies", "consent_accept", "consent_reject"]
TRACKING_CATEGORIES = {"Targeting Cookies", "Performance Cookies"}

OUTDIR = Path("figs_cdf_consent_absolute")
OUTDIR.mkdir(parents=True, exist_ok=True)

FIGSIZE = (9, 6)
X_LIMIT_ALL = 100
X_LIMIT_TRACKING = 60

def load_and_normalize(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    cat_col = None
    for c in CANDIDATE_CATEGORY_COLS:
        if c in df.columns:
            cat_col = c
            break
    if cat_col is None:
        raise ValueError(f"Could not find a category column in: {path}")
    if cat_col != "category":
        df = df.rename(columns={cat_col: "category"})

    for col in STAGE_COLS:
        if col not in df.columns:
            raise ValueError(f"Missing required column '{col}' in: {path}")
        df[col] = df[col].astype(str)
    return df

def per_site_count(df: pd.DataFrame, stage_col: str, categories=None) -> pd.Series:
    d = df.copy()
    if categories is not None:
        d = d[d["category"].isin(categories)]
    mask = d[stage_col].notna() & (d[stage_col].astype(str).str.len() > 0) & (d[stage_col].astype(str).str.lower() != "nan")
    d = d[mask]
    counts = d.groupby(WEBSITE_COL).size()
    all_sites = df[WEBSITE_COL].dropna().unique()
    counts = counts.reindex(all_sites, fill_value=0).astype(int)
    return counts

def plot_cdf(series_dict, x_limit, title, x_label, outfile_prefix):
    plt.figure(figsize=FIGSIZE)
    for label, s in series_dict.items():
        vals = np.sort(s.values.astype(float))
        n = len(vals) if len(vals) > 0 else 1
        y = np.arange(1, n + 1)
        plt.step(vals, y, where="post", label=label)

    plt.xlim(0, x_limit)
    plt.ylim(0, max(y))
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel("Number of Websites", fontsize=12)
    plt.title(title, fontsize=14, fontweight="bold")
    plt.grid(linestyle="--", alpha=0.5)
    plt.legend(title="Consent Stage")
    plt.tight_layout()
    plt.savefig(OUTDIR / f"{outfile_prefix}.png", dpi=300)
    plt.savefig(OUTDIR / f"{outfile_prefix}.pdf")
    plt.close()

def main():
    df = load_and_normalize(CSV_PATH)

    all_counts = {
        "Initial": per_site_count(df, "initial_cookies"),
        "Consent Accepted": per_site_count(df, "consent_accept"),
        "Consent Rejected": per_site_count(df, "consent_reject"),
    }
    plot_cdf(
        all_counts,
        x_limit=X_LIMIT_ALL,
        title="CDF of Total Cookies per Website (Absolute Counts, Banner Present)",
        x_label="Number of Cookies per Website",
        outfile_prefix="cdf_all_cookies_banner_present_absolute",
    )

    tracking_counts = {
        "Initial": per_site_count(df, "initial_cookies", categories=TRACKING_CATEGORIES),
        "Consent Accepted": per_site_count(df, "consent_accept", categories=TRACKING_CATEGORIES),
        "Consent Rejected": per_site_count(df, "consent_reject", categories=TRACKING_CATEGORIES),
    }
    plot_cdf(
        tracking_counts,
        x_limit=X_LIMIT_TRACKING,
        title="CDF of Tracking Cookies per Website (Absolute Counts, Banner Present)",
        x_label="Number of Tracking Cookies per Website",
        outfile_prefix="cdf_tracking_cookies_banner_present_absolute",
    )

    print(f"✅ Saved figures to: {OUTDIR.resolve()}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

CSV_PATH = "cookie_data_final - Banner_present.csv"
MAX_X = 30
TRACKING_SET = {"Targeting Cookies", "Performance Cookies"}
STAGES = ["initial_cookies", "consent_accept", "consent_reject"]

def detect_columns(df):

    if "category" in df.columns:
        cat_col = "category"
    elif "Unnamed: 4" in df.columns:
        cat_col = "Unnamed: 4"
    else:
        raise ValueError("Could not find cookie category column (expected 'category' or 'Unnamed: 4').")

    for col in ["website"]:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
    missing_stages = [s for s in STAGES if s not in df.columns]
    if missing_stages:
        raise ValueError(f"Missing stage columns: {missing_stages}")
    return cat_col

def stage_counts_per_site(df, stage_col, all_sites):
    """Return a Series (index=website) of counts of cookies present at this stage."""
    d = df.copy()

    mask = d[stage_col].astype(str).str.strip().str.lower()
    present = d[~mask.isin(["", "nan", "none"])]
    counts = present.groupby("website").size()

    counts = counts.reindex(all_sites, fill_value=0).astype(int)
    return counts

def build_histogram(counts, max_x=30):
    """Return dict: exact counts for 0..max_x and a 'ge_max' bucket."""
    hist = {}
    for k in range(0, max_x + 1):
        hist[k] = int((counts == k).sum())
    hist[f">={max_x}"] = int((counts >= max_x).sum())
    return hist

def build_cdf_abs(counts, max_x=30):
    """Return dict: CDF (absolute # websites) for <= 0..max_x."""
    cdf = {}
    for k in range(0, max_x + 1):
        cdf[k] = int((counts <= k).sum())
    return cdf

def summary_stats(counts, total_rows_for_stage):
    arr = counts.to_numpy()
    n_sites = len(arr)
    return {
        "Websites (N)": n_sites,
        "Total Cookies (rows)": int(total_rows_for_stage),
        "Mean per Site": round(float(np.mean(arr)), 2),
        "Median per Site": int(np.median(arr)),
        "P90 per Site": int(np.percentile(arr, 90)),
        "P95 per Site": int(np.percentile(arr, 95)),
        "Max per Site": int(np.max(arr)),
        "Sites with 0 Cookies": int((arr == 0).sum())
    }

def print_latex_block(title, data_dict, col_order=None, as_int=True):
    print(f"\n% === {title} ===")
    print("\\begin{tabular}{l" + "r" * len(next(iter(data_dict.values()))) + "}")
    print("\\toprule")
    header = [""] + list(next(iter(data_dict.values())).keys())
    print(" & ".join(header) + " \\\\")
    print("\\midrule")
    for stage, stats in data_dict.items():
        row = [stage]
        keys = col_order if col_order else stats.keys()
        for k in keys:
            v = stats[k]
            if as_int and isinstance(v, (float, int)) and not isinstance(v, bool):
                row.append(str(int(v)))
            else:
                row.append(str(v))
        print(" & ".join(row) + " \\\\")
    print("\\bottomrule")
    print("\\end{tabular}")

def main():
    df = pd.read_csv(CSV_PATH)
    cat_col = detect_columns(df)

    all_websites = sorted(df["website"].dropna().unique().tolist())

    outdir = Path("cookie_counts_outputs")
    outdir.mkdir(exist_ok=True)

    print("\n==============================")
    print("ALL COOKIES — Per-Website Counts & CDFs")
    print("==============================")

    all_counts_by_stage = {}
    all_hist_by_stage = {}
    all_cdf_by_stage = {}
    all_summary = {}

    for stage in STAGES:
        counts = stage_counts_per_site(df, stage, all_websites)
        total_rows = (~df[stage].astype(str).str.strip().str.lower().isin(["", "nan", "none"])).sum()

        all_counts_by_stage[stage] = counts
        all_hist_by_stage[stage] = build_histogram(counts, MAX_X)
        all_cdf_by_stage[stage] = build_cdf_abs(counts, MAX_X)
        all_summary[stage] = summary_stats(counts, total_rows)

        counts.to_csv(outdir / f"allcookies_per_site_counts_{stage}.csv")
        pd.Series(all_hist_by_stage[stage]).to_csv(outdir / f"allcookies_hist_{stage}.csv")
        pd.Series(all_cdf_by_stage[stage]).to_csv(outdir / f"allcookies_cdf_abs_{stage}.csv")

    print("\n% LaTeX: Summary stats (All Cookies)")
    print_latex_block(
        title="Summary (All Cookies)",
        data_dict=all_summary,
        col_order=["Websites (N)", "Total Cookies (rows)", "Mean per Site", "Median per Site",
                   "P90 per Site", "P95 per Site", "Max per Site", "Sites with 0 Cookies"],
        as_int=False
    )

    print("\n% LaTeX: Histogram (All Cookies) — # websites with exactly k cookies")
    print("\\begin{tabular}{l" + "r" * (MAX_X + 2) + "}")
    print("\\toprule")
    header = ["Stage"] + [str(k) for k in range(0, MAX_X + 1)] + [f"$\\geq {MAX_X}$"]
    print(" & ".join(header) + " \\\\")
    print("\\midrule")
    for stage, hist in all_hist_by_stage.items():
        row = [stage] + [str(hist[k]) for k in range(0, MAX_X + 1)] + [str(hist[f">={MAX_X}"])]
        print(" & ".join(row) + " \\\\")
    print("\\bottomrule")
    print("\\end{tabular}")

    print("\n% LaTeX: CDF (All Cookies) — # websites with \\leq k cookies")
    print("\\begin{tabular}{l" + "r" * (MAX_X + 1) + "}")
    print("\\toprule")
    header = ["Stage"] + [f"$\\leq {k}$" for k in range(0, MAX_X + 1)]
    print(" & ".join(header) + " \\\\")
    print("\\midrule")
    for stage, cdf in all_cdf_by_stage.items():
        row = [stage] + [str(cdf[k]) for k in range(0, MAX_X + 1)]
        print(" & ".join(row) + " \\\\")
    print("\\bottomrule")
    print("\\end{tabular}")

    print("\n==============================")
    print("TRACKING COOKIES ONLY — Per-Website Counts & CDFs")
    print("==============================")

    df_tracking = df[df[cat_col].isin(TRACKING_SET)].copy()

    tr_counts_by_stage = {}
    tr_hist_by_stage = {}
    tr_cdf_by_stage = {}
    tr_summary = {}

    for stage in STAGES:
        counts = stage_counts_per_site(df_tracking, stage, all_websites)
        total_rows = (~df_tracking[stage].astype(str).str.strip().str.lower().isin(["", "nan", "none"])).sum()

        tr_counts_by_stage[stage] = counts
        tr_hist_by_stage[stage] = build_histogram(counts, MAX_X)
        tr_cdf_by_stage[stage] = build_cdf_abs(counts, MAX_X)
        tr_summary[stage] = summary_stats(counts, total_rows)

        counts.to_csv(outdir / f"tracking_per_site_counts_{stage}.csv")
        pd.Series(tr_hist_by_stage[stage]).to_csv(outdir / f"tracking_hist_{stage}.csv")
        pd.Series(tr_cdf_by_stage[stage]).to_csv(outdir / f"tracking_cdf_abs_{stage}.csv")

    print("\n% LaTeX: Summary stats (Tracking Only)")
    print_latex_block(
        title="Summary (Tracking Only)",
        data_dict=tr_summary,
        col_order=["Websites (N)", "Total Cookies (rows)", "Mean per Site", "Median per Site",
                   "P90 per Site", "P95 per Site", "Max per Site", "Sites with 0 Cookies"],
        as_int=False
    )

    print("\n% LaTeX: Histogram (Tracking Only) — # websites with exactly k cookies")
    print("\\begin{tabular}{l" + "r" * (MAX_X + 2) + "}")
    print("\\toprule")
    header = ["Stage"] + [str(k) for k in range(0, MAX_X + 1)] + [f"$\\geq {MAX_X}$"]
    print(" & ".join(header) + " \\\\")
    print("\\midrule")
    for stage, hist in tr_hist_by_stage.items():
        row = [stage] + [str(hist[k]) for k in range(0, MAX_X + 1)] + [str(hist[f">={MAX_X}"])]
        print(" & ".join(row) + " \\\\")
    print("\\bottomrule")
    print("\\end{tabular}")

    print("\n% LaTeX: CDF (Tracking Only) — # websites with \\leq k cookies")
    print("\\begin{tabular}{l" + "r" * (MAX_X + 1) + "}")
    print("\\toprule")
    header = ["Stage"] + [f"$\\leq {k}$" for k in range(0, MAX_X + 1)]
    print(" & ".join(header) + " \\\\")
    print("\\midrule")
    for stage, cdf in tr_cdf_by_stage.items():
        row = [stage] + [str(cdf[k]) for k in range(0, MAX_X + 1)]
        print(" & ".join(row) + " \\\\")
    print("\\bottomrule")
    print("\\end{tabular}")

    print(f"\n✅ Done. CSV outputs saved to: {outdir.resolve()}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV = "cookie_data_final - Banner_present.csv"

WEBSITE_COL = "website"
CATEGORY_COL = "category"
COL_INITIAL = "initial_cookies"
COL_ACCEPT  = "consent_accept"
COL_REJECT  = "consent_reject"

TRACKING_ONLY = False

TRACKING_SET = {"Targeting Cookies", "Performance Cookies"}

df = pd.read_csv(CSV)

if CATEGORY_COL not in df.columns and "Unnamed: 4" in df.columns:
    CATEGORY_COL = "Unnamed: 4"

if TRACKING_ONLY:
    df = df[df[CATEGORY_COL].isin(TRACKING_SET)].copy()

def count_stage_per_site(stage_col: str) -> pd.Series:

    return df.groupby(WEBSITE_COL)[stage_col].apply(lambda s: s.notna().sum())

counts_init   = count_stage_per_site(COL_INITIAL)
counts_accept = count_stage_per_site(COL_ACCEPT)
counts_reject = count_stage_per_site(COL_REJECT)

sites = sorted(set(counts_init.index) | set(counts_accept.index) | set(counts_reject.index))
counts_init   = counts_init.reindex(sites, fill_value=0)
counts_accept = counts_accept.reindex(sites, fill_value=0)
counts_reject = counts_reject.reindex(sites, fill_value=0)

order = np.argsort(counts_init.values)
sites_sorted = np.array(sites)[order]
y_init   = counts_init.values[order]
y_accept = counts_accept.values[order]
y_reject = counts_reject.values[order]

x = np.arange(len(sites_sorted))

plt.figure(figsize=(12, 6))

plt.scatter(x, y_init, s=12, color="black", label="Initial")

plt.quiver(
    x, y_init,
    np.zeros_like(x),
    y_accept - y_init,
    angles='xy', scale_units='xy', scale=1,
    width=0.002, headwidth=6, headlength=6, headaxislength=5,
    color="tab:blue", alpha=0.65, label="After Accept"
)

plt.quiver(
    x, y_init,
    np.zeros_like(x),
    y_reject - y_init,
    angles='xy', scale_units='xy', scale=1,
    width=0.002, headwidth=6, headlength=6, headaxislength=5,
    color="tab:red", alpha=0.65, label="After Reject"
)

title_suffix = "Tracking Cookies Only" if TRACKING_ONLY else "All Cookie Types"
plt.title(f"Per-Website Cookie Counts Across Consent Stages ({title_suffix})", fontsize=14, fontweight="bold")
plt.xlabel("Websites (sorted by Initial count)", fontsize=12)
plt.ylabel("Number of Cookies", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.margins(x=0.01)

plt.xticks([], [])

handles, labels = plt.gca().get_legend_handles_labels()

uniq = dict(zip(labels, handles))
plt.legend(uniq.values(), uniq.keys(), frameon=True)

plt.tight_layout()

out_base = "per_site_arrows_tracking" if TRACKING_ONLY else "per_site_arrows_all"
plt.savefig(f"{out_base}.png", dpi=300)
plt.savefig(f"{out_base}.pdf")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV = "cookie_data_final - Banner_present.csv"

WEBSITE_COL = "website"
CATEGORY_COL = "category"
COL_INITIAL = "initial_cookies"
COL_ACCEPT  = "consent_accept"
COL_REJECT  = "consent_reject"

TRACKING_ONLY = False
TRACKING_SET = {"Targeting Cookies", "Performance Cookies"}

X_MAX = 60

df = pd.read_csv(CSV)

if CATEGORY_COL not in df.columns and "Unnamed: 4" in df.columns:
    CATEGORY_COL = "Unnamed: 4"

if TRACKING_ONLY:
    df = df[df[CATEGORY_COL].isin(TRACKING_SET)].copy()

def counts_per_site(stage_col: str) -> pd.Series:

    return df.groupby(WEBSITE_COL)[stage_col].apply(lambda s: s.notna().sum())

c_init   = counts_per_site(COL_INITIAL)
c_accept = counts_per_site(COL_ACCEPT)
c_reject = counts_per_site(COL_REJECT)

sites = sorted(set(c_init.index) | set(c_accept.index) | set(c_reject.index))
c_init   = c_init.reindex(sites, fill_value=0)
c_accept = c_accept.reindex(sites, fill_value=0)
c_reject = c_reject.reindex(sites, fill_value=0)

order = np.argsort(c_init.values)[::-1]
sites_sorted = np.array(sites)[order]
x_init   = c_init.values[order]
x_accept = c_accept.values[order]
x_reject = c_reject.values[order]

y = np.arange(len(sites_sorted))

plt.figure(figsize=(11, max(5, len(sites_sorted) * 0.015)))

for xi, xa, xr, yi in zip(x_init, x_accept, x_reject, y):

    plt.plot([xi, xa], [yi, yi], color="tab:blue", linewidth=0.8, alpha=0.65)

    plt.plot([xi, xr], [yi, yi], color="tab:red", linewidth=0.8, alpha=0.65)

plt.scatter(x_init,   y, s=14, color="black",      label="Initial")
plt.scatter(x_accept, y, s=14, color="tab:blue",   label="After Accept")
plt.scatter(x_reject, y, s=14, color="tab:red",    label="After Reject")

title_suffix = "Tracking Cookies Only" if TRACKING_ONLY else "All Cookie Types"
plt.title(f"Per-Website Cookie Counts Across Consent Stages ({title_suffix})",
          fontsize=14, fontweight="bold")

plt.xlabel("Number of Cookies", fontsize=12)
plt.ylabel("Websites (sorted by Initial)", fontsize=12)

n = len(sites_sorted)
tick_pos = np.linspace(0, n-1, num=min(10, n)).round().astype(int)
plt.yticks(tick_pos, [sites_sorted[i] for i in tick_pos], fontsize=8)

if X_MAX is not None:
    plt.xlim(0, X_MAX)

plt.grid(axis="x", linestyle="--", alpha=0.35)
plt.margins(y=0.01)

handles, labels = plt.gca().get_legend_handles_labels()
uniq = dict(zip(labels, handles))
plt.legend(uniq.values(), uniq.keys(), frameon=True)

plt.tight_layout()

base = "per_site_dumbbell_tracking" if TRACKING_ONLY else "per_site_dumbbell_all"
plt.savefig(f"{base}.png", dpi=300)
plt.savefig(f"{base}.pdf")
plt.show()

delta_accept = (x_accept - x_init)
delta_reject = (x_reject - x_init)

print("Δ Accept – Initial:  mean=", round(delta_accept.mean(),2),
      " median=", np.median(delta_accept),
      " ↑sites=", int((delta_accept>0).sum()),
      " ↓sites=", int((delta_accept<0).sum()),
      " =sites=", int((delta_accept==0).sum()))

print("Δ Reject – Initial:  mean=", round(delta_reject.mean(),2),
      " median=", np.median(delta_reject),
      " ↑sites=", int((delta_reject>0).sum()),
      " ↓sites=", int((delta_reject<0).sum()),
      " =sites=", int((delta_reject==0).sum()))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV = "cookie_data_final - Banner_present.csv"

WEBSITE_COL = "website"
CATEGORY_COL = "category"
COL_INITIAL = "initial_cookies"
COL_ACCEPT  = "consent_accept"
COL_REJECT  = "consent_reject"

TRACKING_ONLY = False
TRACKING_SET = {"Targeting Cookies", "Performance Cookies"}

X_MAX = 60
INSET = True
INSET_POS = [0.67, 0.60, 0.30, 0.32]
BINS = np.arange(-30, 31, 1)

df = pd.read_csv(CSV)

if CATEGORY_COL not in df.columns and "Unnamed: 4" in df.columns:
    CATEGORY_COL = "Unnamed: 4"

if TRACKING_ONLY:
    df = df[df[CATEGORY_COL].isin(TRACKING_SET)].copy()

def counts_per_site(stage_col: str) -> pd.Series:

    return df.groupby(WEBSITE_COL)[stage_col].apply(lambda s: s.notna().sum())

c_init   = counts_per_site(COL_INITIAL)
c_accept = counts_per_site(COL_ACCEPT)
c_reject = counts_per_site(COL_REJECT)

sites = sorted(set(c_init.index) | set(c_accept.index) | set(c_reject.index))
c_init   = c_init.reindex(sites, fill_value=0)
c_accept = c_accept.reindex(sites, fill_value=0)
c_reject = c_reject.reindex(sites, fill_value=0)

order = np.argsort(c_init.values)[::-1]
sites_sorted = np.array(sites)[order]
x_init   = c_init.values[order]
x_accept = c_accept.values[order]
x_reject = c_reject.values[order]
y = np.arange(len(sites_sorted))

delta_accept = x_accept - x_init
delta_reject = x_reject - x_init

fig = plt.figure(figsize=(11, max(5, len(sites_sorted) * 0.015)))
ax = fig.add_subplot(111)

for xi, xa, xr, yi in zip(x_init, x_accept, x_reject, y):
    ax.plot([xi, xa], [yi, yi], color="tab:blue", linewidth=0.8, alpha=0.65)
    ax.plot([xi, xr], [yi, yi], color="tab:red", linewidth=0.8, alpha=0.65)

ax.scatter(x_init,   y, s=14, color="black",    label="Initial")
ax.scatter(x_accept, y, s=14, color="tab:blue", label="After Accept")
ax.scatter(x_reject, y, s=14, color="tab:red",  label="After Reject")

title_suffix = "Tracking Cookies Only" if TRACKING_ONLY else "All Cookie Types"
ax.set_title(f"Per-Website Cookie Counts Across Consent Stages ({title_suffix})",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Cookies", fontsize=12)
ax.set_ylabel("Websites (sorted by Initial)", fontsize=12)

n = len(sites_sorted)
tick_pos = np.linspace(0, n-1, num=min(10, n)).round().astype(int)
ax.set_yticks(tick_pos)
ax.set_yticklabels([sites_sorted[i] for i in tick_pos], fontsize=8)

if X_MAX is not None:
    ax.set_xlim(0, X_MAX)

ax.grid(axis="x", linestyle="--", alpha=0.35)
handles, labels = ax.get_legend_handles_labels()
uniq = dict(zip(labels, handles))
ax.legend(uniq.values(), uniq.keys(), frameon=True, fontsize=10)

if INSET:
    inset = fig.add_axes(INSET_POS)

    inset.hist(delta_accept, bins=BINS, histtype="stepfilled", alpha=0.35, label="Accept − Initial")
    inset.hist(delta_reject, bins=BINS, histtype="stepfilled", alpha=0.35, label="Reject − Initial")
    inset.axvline(0, color="k", linewidth=0.8, linestyle="--", alpha=0.7)
    inset.set_title("Per-Site Change (Δ cookies)", fontsize=10)
    inset.set_xlabel("Δ cookies", fontsize=9)
    inset.set_ylabel("# websites", fontsize=9)
    inset.grid(axis="y", linestyle="--", alpha=0.3)
    inset.legend(fontsize=8, frameon=True, loc="upper left")

plt.tight_layout()

base = "per_site_dumbbell_tracking" if TRACKING_ONLY else "per_site_dumbbell_all"
plt.savefig(f"{base}.png", dpi=300)
plt.savefig(f"{base}.pdf")
plt.show()

def delta_stats(name, d):
    up = int((d > 0).sum())
    down = int((d < 0).sum())
    same = int((d == 0).sum())
    print(f"{name}: mean={d.mean():.2f}, median={np.median(d)}, "
          f"sites↑={up}, sites↓={down}, sites=={same}, "
          f"min={d.min()}, max={d.max()}")

delta_stats("Accept − Initial", delta_accept)
delta_stats("Reject − Initial", delta_reject)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

df = pd.read_csv("cookie_data_final - Banner_present.csv")

WEBSITE = "website"
INIT = "initial_cookies"
ACPT = "consent_accept"
REJT = "consent_reject"
CATEGORY = "category"

mask = df[[INIT, ACPT, REJT]].notna().any(axis=1)
df = df[mask].copy()

counts = (df
          .groupby(WEBSITE)[[INIT, ACPT, REJT]]
          .apply(lambda g: pd.Series({
              "init_n": g[INIT].notna().sum(),
              "acc_n":  g[ACPT].notna().sum(),
              "rej_n":  g[REJT].notna().sum()
          }))
          .reset_index())

counts = counts.sort_values("init_n", ascending=False).reset_index(drop=True)
counts["idx"] = np.arange(1, len(counts) + 1)

plt.rcParams.update({"figure.dpi": 150})
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(counts["idx"], counts["init_n"], color="#E0E0E0", width=0.85, label="Initial")

ax.scatter(counts["idx"], counts["acc_n"], s=12, color="#1f77b4", label="After Accept")
ax.scatter(counts["idx"], counts["rej_n"], s=12, color="#d62728", label="After Reject")

for x, y0, y1, y2 in counts[["idx", "init_n", "acc_n", "rej_n"]].itertuples(index=False):
    ax.plot([x, x], [y0, y1], color="#1f77b4", alpha=0.35, linewidth=0.8)
    ax.plot([x, x], [y0, y2], color="#d62728", alpha=0.35, linewidth=0.8)

ax.set_xlabel("Websites (sorted by initial cookie count)")
ax.set_ylabel("Number of Cookies")
ax.set_title("Per-Website Cookie Counts: Initial vs After Accept/Reject (Banner-Present Sites)")
ax.set_xlim(0, len(counts) + 1)

leg = ax.legend(frameon=False, ncols=3, loc="upper right", bbox_to_anchor=(1, 1.05))

ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.5, zorder=0)

inset = inset_axes(ax, width="32%", height="38%", loc="upper left",
                   bbox_to_anchor=(0.03, 0.6, 0.36, 0.38),
                   bbox_transform=ax.transAxes, borderpad=0.0)

delta_acc = counts["acc_n"] - counts["init_n"]
delta_rej = counts["rej_n"] - counts["init_n"]

bins = np.arange(min(delta_acc.min(), delta_rej.min()) - 0.5,
                 max(delta_acc.max(), delta_rej.max()) + 1.5, 1)

inset.hist(delta_acc, bins=bins, alpha=0.55, label="Accept - Initial")
inset.hist(delta_rej, bins=bins, alpha=0.55, label="Reject - Initial")
inset.axvline(0, color="k", linewidth=1, alpha=0.7)
inset.set_title("Δ Cookies per Site", fontsize=9)
inset.set_xlabel("Change", fontsize=8)
inset.set_ylabel("# Sites", fontsize=8)
inset.tick_params(axis='both', which='major', labelsize=8)
inset.legend(frameon=False, fontsize=8, loc="upper left")

plt.tight_layout()
plt.savefig("per_site_cookie_arrows_with_inset.png", dpi=300)
plt.savefig("per_site_cookie_arrows_with_inset.pdf")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("cookie_data_final - Banner_present.csv")

WEBSITE = "website"
INIT = "initial_cookies"
ACPT = "consent_accept"
REJT = "consent_reject"

mask = df[[INIT, ACPT, REJT]].notna().any(axis=1)
df = df[mask].copy()

counts = (df
          .groupby(WEBSITE)[[INIT, ACPT, REJT]]
          .apply(lambda g: pd.Series({
              "init_n": g[INIT].notna().sum(),
              "acc_n":  g[ACPT].notna().sum(),
              "rej_n":  g[REJT].notna().sum()
          }))
          .reset_index())

counts = counts.sort_values("init_n", ascending=False).reset_index(drop=True)
counts["idx"] = np.arange(1, len(counts) + 1)

plt.rcParams.update({"figure.dpi": 150})
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(counts["idx"], counts["init_n"], color="#E0E0E0", width=0.85, label="Initial")

ax.scatter(counts["idx"], counts["acc_n"], s=12, color="#1f77b4", label="After Accept")
ax.scatter(counts["idx"], counts["rej_n"], s=12, color="#d62728", label="After Reject")

for x, y0, y1, y2 in counts[["idx", "init_n", "acc_n", "rej_n"]].itertuples(index=False):
    ax.plot([x, x], [y0, y1], color="#1f77b4", alpha=0.35, linewidth=0.8)
    ax.plot([x, x], [y0, y2], color="#d62728", alpha=0.35, linewidth=0.8)

ax.set_xlabel("Websites (sorted by initial cookie count)")
ax.set_ylabel("Number of Cookies")
ax.set_title("Per-Website Cookie Counts: Initial vs After Accept/Reject (Banner-Present Sites)")
ax.set_xlim(0, len(counts) + 1)

ax.legend(frameon=False, ncols=3, loc="upper right", bbox_to_anchor=(1, 1.05))

ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.5, zorder=0)

plt.tight_layout()
plt.savefig("per_site_cookie_arrows.png", dpi=300)
plt.savefig("per_site_cookie_arrows.pdf")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("cookie_data_final - Banner_present.csv")

website_col = "website" if "website" in df.columns else ("domain" if "domain" in df.columns else None)
if website_col is None:
    raise ValueError("Couldn't find a website/domain column. Expected one of ['website', 'domain'].")

category_col = None
for cand in ["category", "Unnamed: 4"]:
    if cand in df.columns:
        category_col = cand
        break
if category_col is None:
    raise ValueError("Couldn't find a category column. Expected one of ['category', 'Unnamed: 4'].")

stage_cols = [c for c in ["initial_cookies", "consent_accept", "consent_reject"] if c in df.columns]
if len(stage_cols) < 1:
    raise ValueError("No stage columns found. Need at least one of: initial_cookies, consent_accept, consent_reject")

cookie_types = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies",
]

all_sites = pd.Index(sorted(df[website_col].astype(str).unique()))

def counts_per_site_for_type(stage_col: str, ctype: str) -> pd.Series:
    sub = df[df[category_col] == ctype]
    if sub.empty:
        return pd.Series(0, index=all_sites, dtype=int)
    present = sub[stage_col].notna() & sub[stage_col].astype(str).str.strip().ne("")
    by_site = present.groupby(sub[website_col].astype(str)).sum().astype(int)

    return by_site.reindex(all_sites, fill_value=0)

counts = {
    ctype: {stage: counts_per_site_for_type(stage, ctype) for stage in stage_cols}
    for ctype in cookie_types
}

def compute_cdf(series: pd.Series):
    x = np.sort(series.values)
    y = np.arange(1, len(x) + 1)
    return x, y

stage_labels = {
    "initial_cookies": "Initial",
    "consent_accept": "Consent Accepted",
    "consent_reject": "Consent Rejected",
}
stage_linestyles = {
    "initial_cookies": "-",
    "consent_accept": "--",
    "consent_reject": ":",
}

plt.figure(figsize=(9, 7))

for ctype in cookie_types:
    for stage in stage_cols:
        x, y = compute_cdf(counts[ctype][stage])

        label = f"{ctype} — {stage_labels.get(stage, stage)}"
        plt.step(x, y, where="post", linewidth=1.8, label=label)

plt.xlabel("Cookies per Website", fontsize=12)
plt.ylabel("Number of Websites (CDF)", fontsize=12)
plt.title("CDF of Cookies per Website by Type across Consent Stages (Banner Present)", fontsize=13)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig("cdf_cookies_by_type_all_stages_singlepanel.pdf")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
ax_grid = axes.ravel()

for ax, ctype in zip(ax_grid, cookie_types):
    for stage in stage_cols:
        x, y = compute_cdf(counts[ctype][stage])
        ax.step(x, y, where="post", linewidth=2.0,
                linestyle=stage_linestyles.get(stage, "-"),
                label=stage_labels.get(stage, stage))
    ax.set_title(ctype, fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.4)

fig.text(0.5, 0.04, "Cookies per Website", ha="center", fontsize=12)
fig.text(0.04, 0.5, "Number of Websites (CDF)", va="center", rotation="vertical", fontsize=12)

handles, labels = ax_grid[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(stage_cols), frameon=False)
plt.tight_layout(rect=[0.03, 0.07, 1, 1])
plt.savefig("cdf_cookies_by_type_all_stages_faceted.pdf")
plt.show()

print("\nWebsites (N):", len(all_sites))
for ctype in cookie_types:
    print(f"\n[{ctype}]")
    for stage in stage_cols:
        s = counts[ctype][stage]
        print(f"  {stage_labels.get(stage, stage)}: mean={s.mean():.2f}, median={s.median()}, p90={np.percentile(s, 90):.0f}, max={s.max()}, zeros={(s==0).sum()}")

In [ ]:
import pandas as pd
import numpy as np

INPUT = "cookie_data_final - Banner_present.csv"
OUTPUT_SUMMARY = "reject_spike_sites_summary.csv"
OUTPUT_BREAKDOWN = "reject_spike_sites_breakdown_by_category.csv"

MIN_ABS_DELTA = 1
MIN_RATIO = 1.5
TOP_N = 25

df = pd.read_csv(INPUT)

website_col = "website" if "website" in df.columns else ("domain" if "domain" in df.columns else None)
if website_col is None:
    raise ValueError("Couldn't find a website/domain column. Expected one of ['website', 'domain'].")

category_col = None
for cand in ["category", "Unnamed: 4"]:
    if cand in df.columns:
        category_col = cand
        break
if category_col is None:
    raise ValueError("Couldn't find a category column. Expected one of ['category', 'Unnamed: 4'].")

stage_cols = [c for c in ["initial_cookies", "consent_accept", "consent_reject"] if c in df.columns]
required = {"initial_cookies", "consent_accept", "consent_reject"}
if not required.issubset(set(stage_cols)):
    raise ValueError("Missing one of the required stage columns: initial_cookies, consent_accept, consent_reject")

def present(s: pd.Series) -> pd.Series:
    return s.notna() & s.astype(str).str.strip().ne("")

for c in stage_cols:
    df[f"__has_{c}"] = present(df[c]).astype(int)

site_counts = (
    df.groupby(website_col)[[f"__has_{c}" for c in stage_cols]]
      .sum()
      .rename(columns={f"__has_{c}": c for c in stage_cols})
      .astype(int)
)
site_counts = site_counts.reset_index()

site_counts["max_init_accept"] = site_counts[["initial_cookies", "consent_accept"]].max(axis=1)
site_counts["delta_reject_vs_max"] = site_counts["consent_reject"] - site_counts["max_init_accept"]
site_counts["ratio_reject_vs_max"] = np.where(
    site_counts["max_init_accept"] > 0,
    site_counts["consent_reject"] / site_counts["max_init_accept"],
    np.inf
)

rej = site_counts["consent_reject"].astype(float)
site_counts["reject_zscore"] = (rej - rej.mean()) / (rej.std(ddof=1) if rej.std(ddof=1) else 1.0)

site_counts["flag_reject_spike"] = (
    (site_counts["delta_reject_vs_max"] >= MIN_ABS_DELTA) &
    (site_counts["ratio_reject_vs_max"] >= MIN_RATIO)
)

offenders = (
    site_counts[site_counts["flag_reject_spike"]]
    .sort_values(["delta_reject_vs_max", "ratio_reject_vs_max", "consent_reject"], ascending=[False, False, False])
)

site_counts.to_csv(OUTPUT_SUMMARY, index=False)
print(f"\nSaved site-level summary to: {OUTPUT_SUMMARY}")
print(f"Total websites: {len(site_counts)} | Flagged (reject spike): {offenders.shape[0]}")

print("\nTop offenders (consent_reject much higher than initial/accept):")
cols_to_show = [
    website_col, "initial_cookies", "consent_accept", "consent_reject",
    "delta_reject_vs_max", "ratio_reject_vs_max", "reject_zscore"
]
print(offenders.head(TOP_N)[cols_to_show].to_string(index=False))

def per_site_category_counts(stage_col: str) -> pd.DataFrame:
    sub = df[[website_col, category_col, f"__has_{stage_col}"]].copy()
    sub = sub.groupby([website_col, category_col])[f"__has_{stage_col}"].sum().reset_index()
    sub = sub.rename(columns={f"__has_{stage_col}": f"count_{stage_col}"})
    return sub

cat_init = per_site_category_counts("initial_cookies")
cat_acc  = per_site_category_counts("consent_accept")
cat_rej  = per_site_category_counts("consent_reject")

cat_all = (
    cat_init.merge(cat_acc, on=[website_col, category_col], how="outer")
            .merge(cat_rej, on=[website_col, category_col], how="outer")
            .fillna(0)
)

flagged_sites = set(offenders[website_col].astype(str))
cat_flagged = cat_all[cat_all[website_col].astype(str).isin(flagged_sites)].copy()

cat_flagged["total_all_stages"] = (
    cat_flagged["count_initial_cookies"] +
    cat_flagged["count_consent_accept"] +
    cat_flagged["count_consent_reject"]
).astype(int)

cat_flagged = cat_flagged.sort_values([website_col, "total_all_stages", category_col], ascending=[True, False, True])

cat_flagged.to_csv(OUTPUT_BREAKDOWN, index=False)
print(f"\nSaved per-category breakdown for flagged sites to: {OUTPUT_BREAKDOWN}")

print("\nSample breakdown (first 10 rows):")
print(cat_flagged.head(10).to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np

INPUT = "cookie_data_final - Banner_present.csv"
OUTPUT_SITE_SUMMARY = "reject_gt_max_summary.csv"
OUTPUT_CATEGORY_BREAKDOWN = "reject_gt_max_breakdown_by_category.csv"
TOP_N_PRINT = 25

df = pd.read_csv(INPUT)

website_col = "website" if "website" in df.columns else ("domain" if "domain" in df.columns else None)
if website_col is None:
    raise ValueError("Couldn't find a website/domain column. Expected one of ['website', 'domain'].")

category_col = None
for cand in ["category", "Unnamed: 4"]:
    if cand in df.columns:
        category_col = cand
        break
if category_col is None:
    raise ValueError("Couldn't find a category column. Expected one of ['category', 'Unnamed: 4'].")

required_stage_cols = ["initial_cookies", "consent_accept", "consent_reject"]
missing = [c for c in required_stage_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required stage columns: {missing}. Need {required_stage_cols}")

def present(s: pd.Series) -> pd.Series:

    return s.notna() & s.astype(str).str.strip().ne("")

for c in required_stage_cols:
    df[f"__has_{c}"] = present(df[c]).astype(int)

site_counts = (
    df.groupby(website_col)[[f"__has_{c}" for c in required_stage_cols]]
      .sum()
      .rename(columns={f"__has_{c}": c for c in required_stage_cols})
      .astype(int)
      .reset_index()
)

site_counts["max_init_accept"] = site_counts[["initial_cookies", "consent_accept"]].max(axis=1)
site_counts["delta_reject_vs_max"] = site_counts["consent_reject"] - site_counts["max_init_accept"]
site_counts["ratio_reject_vs_max"] = np.where(
    site_counts["max_init_accept"] > 0,
    site_counts["consent_reject"] / site_counts["max_init_accept"],
    np.inf
)

site_counts["flag_reject_gt_max"] = site_counts["consent_reject"] > site_counts["max_init_accept"]

rej = site_counts["consent_reject"].astype(float)
std = rej.std(ddof=1)
site_counts["reject_zscore"] = (rej - rej.mean()) / (std if std else 1.0)

site_counts.to_csv(OUTPUT_SITE_SUMMARY, index=False)
print(f"Saved site-level summary to: {OUTPUT_SITE_SUMMARY}")
print(f"Total websites: {len(site_counts)} | Flagged (reject > max(init, accept)): {site_counts['flag_reject_gt_max'].sum()}")

offenders = (
    site_counts[site_counts["flag_reject_gt_max"]]
    .sort_values(["delta_reject_vs_max", "ratio_reject_vs_max", "consent_reject"], ascending=[False, False, False])
)
if not offenders.empty:
    cols_to_show = [
        website_col, "initial_cookies", "consent_accept", "consent_reject",
        "delta_reject_vs_max", "ratio_reject_vs_max", "reject_zscore"
    ]
    print("\nTop offenders (reject strictly greater than both initial & accept):")
    print(offenders.head(TOP_N_PRINT)[cols_to_show].to_string(index=False))
else:
    print("\nNo offenders found under strict reject > max(initial, accept) condition.")

def per_site_category_counts(stage_col: str) -> pd.DataFrame:
    sub = df[[website_col, category_col, f"__has_{stage_col}"]].copy()
    sub = sub.groupby([website_col, category_col])[f"__has_{stage_col}"].sum().reset_index()
    sub = sub.rename(columns={f"__has_{stage_col}": f"count_{stage_col}"})
    return sub

cat_init = per_site_category_counts("initial_cookies")
cat_acc  = per_site_category_counts("consent_accept")
cat_rej  = per_site_category_counts("consent_reject")

cat_all = (
    cat_init.merge(cat_acc, on=[website_col, category_col], how="outer")
            .merge(cat_rej, on=[website_col, category_col], how="outer")
            .fillna(0)
)

cat_all["max_init_accept_cat"] = cat_all[["count_initial_cookies", "count_consent_accept"]].max(axis=1)
cat_all["delta_reject_vs_max_cat"] = cat_all["count_consent_reject"] - cat_all["max_init_accept_cat"]
cat_all["flag_reject_gt_max_cat"] = cat_all["count_consent_reject"] > cat_all["max_init_accept_cat"]

flagged_sites = set(offenders[website_col].astype(str))
cat_flagged = cat_all[cat_all[website_col].astype(str).isin(flagged_sites)].copy()

cat_flagged["total_all_stages"] = (
    cat_flagged["count_initial_cookies"] +
    cat_flagged["count_consent_accept"] +
    cat_flagged["count_consent_reject"]
).astype(int)

cat_flagged = cat_flagged.sort_values(
    [website_col, "flag_reject_gt_max_cat", "delta_reject_vs_max_cat", "total_all_stages", category_col],
    ascending=[True, False, False, False, True]
)

cat_flagged.to_csv(OUTPUT_CATEGORY_BREAKDOWN, index=False)
print(f"Saved per-category breakdown to: {OUTPUT_CATEGORY_BREAKDOWN}")

print("\nSample category breakdown (first 10 rows):")
peek_cols = [
    website_col, category_col,
    "count_initial_cookies", "count_consent_accept", "count_consent_reject",
    "delta_reject_vs_max_cat", "flag_reject_gt_max_cat"
]
print(cat_flagged.head(10)[peek_cols].to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT = "cookie_data_final - merged_updated.csv"
COOKIE_TYPES = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies",
]
DEFAULT_COL_CANDIDATES = ["initial_cookies", "default", "Default"]
WEBSITE_COL_CANDIDATES = ["website", "domain"]

df = pd.read_csv(INPUT)

default_col = next((c for c in DEFAULT_COL_CANDIDATES if c in df.columns), None)
if default_col is None:
    raise ValueError(f"Couldn't find a default-state column. Tried: {DEFAULT_COL_CANDIDATES}")

website_col = next((c for c in WEBSITE_COL_CANDIDATES if c in df.columns), None)
if website_col is None:
    raise ValueError(f"Couldn't find a website/domain column. Tried: {WEBSITE_COL_CANDIDATES}")

for col in ["category", "ccpa"]:
    if col not in df.columns:
        raise ValueError(f"Expected '{col}' column in the dataset.")

groups = {
    "Subjected": df[df["ccpa"] == "subjected"],
    "Non-Subjected": df[df["ccpa"] != "subjected"],
}
linestyle_for = {"Subjected": "-", "Non-Subjected": "--"}

per_group_type_counts = {}
group_sizes = {}
for gname, gdf in groups.items():
    all_websites = gdf[website_col].dropna().astype(str).unique()
    group_sizes[gname] = len(all_websites)
    per_group_type_counts[gname] = {}
    for ctype in COOKIE_TYPES:
        sub = gdf[gdf["category"] == ctype]
        counts = (
            sub.groupby(website_col)[default_col]
               .apply(lambda s: s.notnull().sum())
               .astype(int)
               .reindex(all_websites, fill_value=0)
        )
        per_group_type_counts[gname][ctype] = counts

plt.figure(figsize=(9, 6))

for gname in ["Subjected", "Non-Subjected"]:
    for ctype in COOKIE_TYPES:
        values = per_group_type_counts[gname][ctype].sort_values().to_numpy()
        if len(values) == 0:
            continue
        y_abs = np.arange(1, len(values) + 1)
        plt.plot(values, y_abs, linestyle=linestyle_for[gname], label=f"{ctype} — {gname} (N={len(values)})")

plt.xlabel("Number of Cookies per Website (Default)")
plt.ylabel("Websites with ≤ X Cookies (absolute count)")
plt.title("ECDF by Cookie Type (Default) — CCPA Subjected vs Non-Subjected", fontweight="bold")
plt.legend(title="Cookie Type × Group", loc="lower right", fontsize=9)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("ecdf_counts_cookie_types_default_subjected_vs_nonsubjected.pdf", dpi=300)
plt.savefig("ecdf_counts_cookie_types_default_subjected_vs_nonsubjected.png", dpi=300)
plt.show()

print("✅ Wrote: ecdf_counts_cookie_types_default_subjected_vs_nonsubjected.pdf / .png")

In [ ]:
import pandas as pd
import numpy as np

INPUT = "cookie_data_final - merged_updated.csv"
COOKIE_TYPES = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies",
]
DEFAULT_COL_CANDIDATES = ["initial_cookies", "default", "Default"]
WEBSITE_COL_CANDIDATES = ["website", "domain"]

try:
    from scipy.stats import ks_2samp
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

def detect_column(candidates, cols, what):
    for c in candidates:
        if c in cols:
            return c
    raise ValueError(f"Couldn't find {what}. Tried: {candidates}. Available: {list(cols)}")

def normalize_ccpa(val: str) -> str:
    """Return 'subjected' if value == 'subjected' (case-insensitive), else 'non-subjected'."""
    if pd.isna(val):
        return "non-subjected"
    return "subjected" if str(val).strip().lower() == "subjected" else "non-subjected"

def build_site_partitions(df: pd.DataFrame, website_col: str) -> tuple[dict, pd.DataFrame]:
    """
    Partition unique websites into Subjected vs Non-Subjected.
    If a site has mixed labels across rows, classify as 'subjected' (conservative) and mark conflict.
    Returns:
      site_lists: {"Subjected": [...], "Non-Subjected": [...]}
      site_status_df: DataFrame with columns [website, label, conflict, raw_labels]
    """
    raw = df[[website_col, "ccpa"]].dropna(subset=[website_col]).copy()
    raw["norm"] = raw["ccpa"].apply(normalize_ccpa)

    agg = (raw.groupby(website_col)["norm"]
               .agg(lambda s: sorted(set(s)))
               .reset_index()
               .rename(columns={"norm": "raw_labels"}))

    def choose_label(labels):

        return "subjected" if "subjected" in labels else "non-subjected"

    agg["label"] = agg["raw_labels"].apply(choose_label)
    agg["conflict"] = agg["raw_labels"].apply(lambda lab: ("subjected" in lab and "non-subjected" in lab))

    site_lists = {
        "Subjected": agg.loc[agg["label"] == "subjected", website_col].astype(str).tolist(),
        "Non-Subjected": agg.loc[agg["label"] == "non-subjected", website_col].astype(str).tolist(),
    }
    return site_lists, agg.rename(columns={website_col: "website"})

def per_website_counts(df_group, website_col, default_col, cookie_type):
    """Count rows (non-null at default state) per website for a given cookie type."""
    sub = df_group[df_group["category"] == cookie_type]
    counts = (
        sub.groupby(website_col)[default_col]
           .apply(lambda s: s.notnull().sum())
           .astype(int)
    )

    all_sites = df_group[website_col].dropna().astype(str).unique()
    return counts.reindex(all_sites, fill_value=0)

def summarize(arr):
    arr = np.asarray(arr, dtype=float)
    n = len(arr)
    if n == 0:
        return dict(N=0, mean=np.nan, median=np.nan, p90=np.nan, p95=np.nan,
                    max=np.nan, zeros=0, zeros_pct=np.nan)
    return dict(
        N=int(n),
        mean=float(np.mean(arr)),
        median=float(np.percentile(arr, 50)),
        p90=float(np.percentile(arr, 90)),
        p95=float(np.percentile(arr, 95)),
        max=float(np.max(arr)),
        zeros=int(np.sum(arr == 0)),
        zeros_pct=float(np.sum(arr == 0) / n * 100.0),
    )

def fmt_num(x, dec=2):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
        return "NA"
    if abs(x - round(x)) < 1e-9:
        return str(int(round(x)))
    return f"{x:.{dec}f}"

df = pd.read_csv(INPUT)

default_col = detect_column(DEFAULT_COL_CANDIDATES, df.columns, "default-state column")
website_col = detect_column(WEBSITE_COL_CANDIDATES, df.columns, "website/domain column")
if "category" not in df.columns:
    raise ValueError("Expected 'category' column in the dataset.")
if "ccpa" not in df.columns:
    raise ValueError("Expected 'ccpa' column in the dataset.")

site_lists, site_status_df = build_site_partitions(df, website_col)

N_total_sites = df[website_col].dropna().astype(str).nunique()
N_subj = len(site_lists["Subjected"])
N_non  = len(site_lists["Non-Subjected"])
pct_subj = 100.0 * N_subj / N_total_sites if N_total_sites else float("nan")
pct_non  = 100.0 * N_non  / N_total_sites if N_total_sites else float("nan")
N_conflicts = int(site_status_df["conflict"].sum())

print("=== Website counts by CCPA subjectivity (unique websites) ===")
print(f"File: {INPUT}")
print(f"Default column used: {default_col}")
print(f"Website column used: {website_col}")
print(f"Total unique websites      : {N_total_sites}")
print(f"Subjected websites         : {N_subj} ({pct_subj:.1f}%)")
print(f"Non-Subjected websites     : {N_non} ({pct_non:.1f}%)")
print(f"Websites with mixed labels : {N_conflicts}  (classified as 'subjected')\n")

groups = {
    "Subjected": df[df[website_col].astype(str).isin(site_lists["Subjected"])].copy(),
    "Non-Subjected": df[df[website_col].astype(str).isin(site_lists["Non-Subjected"])].copy(),
}

print("=== Default-state per-website cookie counts by cookie type ===")
print(f"SciPy KS available: {HAVE_SCIPY}\n")

for gname, gdf in groups.items():
    n_sites = gdf[website_col].dropna().astype(str).nunique()
    print(f"--- Group: {gname} (websites N={n_sites}) ---")
    for ctype in COOKIE_TYPES:
        vals = per_website_counts(gdf, website_col, default_col, ctype).to_numpy()
        s = summarize(vals)
        print(f"\n{ctype}")
        print(f"  N websites        : {fmt_num(s['N'],0)}")
        print(f"  Median / P90 / P95: {fmt_num(s['median'],0)} / {fmt_num(s['p90'],0)} / {fmt_num(s['p95'],0)}")
        print(f"  Mean / Max        : {fmt_num(s['mean'])} / {fmt_num(s['max'],0)}")
        print(f"  Zeros (# / %)     : {fmt_num(s['zeros'],0)} / {fmt_num(s['zeros_pct'])}%")
    print()

print("=== Contrasts (Subjected minus Non-Subjected) by cookie type ===")
for ctype in COOKIE_TYPES:
    subj_vals = per_website_counts(groups["Subjected"], website_col, default_col, ctype).to_numpy()
    nons_vals = per_website_counts(groups["Non-Subjected"], website_col, default_col, ctype).to_numpy()
    s = summarize(subj_vals)
    n = summarize(nons_vals)

    d_median = s["median"] - n["median"] if s["N"] and n["N"] else np.nan
    d_p90    = s["p90"] - n["p90"]       if s["N"] and n["N"] else np.nan
    d_mean   = s["mean"] - n["mean"]     if s["N"] and n["N"] else np.nan

    if HAVE_SCIPY and s["N"] and n["N"]:
        ks = ks_2samp(subj_vals, nons_vals, alternative="two-sided", mode="auto")
        ks_stat, ks_p = ks.statistic, ks.pvalue
    else:
        ks_stat = ks_p = np.nan

    print(f"\n{ctype}")
    print(f"  ΔMedian / ΔP90 / ΔMean : {fmt_num(d_median,2)} / {fmt_num(d_p90,2)} / {fmt_num(d_mean,2)}")
    print(f"  KS (stat, p)           : ({fmt_num(ks_stat,3)}, {fmt_num(ks_p,3)})")

print("\n✅ Done. Paste these counts/stats here and I’ll format them into LaTeX.")

In [ ]:
import pandas as pd

MERGED = "cookie_data_final - merged_updated.csv"
BANNER_PRESENT = "cookie_data_final - Banner_present.csv"
BANNER_NOT_PRESENT = "cookie_data_final - Banner_not_present.csv"

WEBSITE_COL = "website"
CCPA_COL = "ccpa"

def ccpa_strict(val) -> str:
    if pd.isna(val):
        return "non-subjected"
    return "subjected" if str(val).strip().lower() == "subjected" else "non-subjected"

def ccpa_inclusive(val) -> str:
    if pd.isna(val):
        return "non-subjected"
    return "subjected" if "subjected" in str(val).strip().lower() else "non-subjected"

CCPA_NORMALIZER = ccpa_strict

df_all = pd.read_csv(MERGED, usecols=[WEBSITE_COL, CCPA_COL])
df_bp  = pd.read_csv(BANNER_PRESENT, usecols=[WEBSITE_COL])
df_bn  = pd.read_csv(BANNER_NOT_PRESENT, usecols=[WEBSITE_COL])

df_all[WEBSITE_COL] = df_all[WEBSITE_COL].astype(str).str.strip()
df_bp[WEBSITE_COL]  = df_bp[WEBSITE_COL].astype(str).str.strip()
df_bn[WEBSITE_COL]  = df_bn[WEBSITE_COL].astype(str).str.strip()

ccpa_map = (df_all
    .assign(ccpa_norm=lambda d: d[CCPA_COL].apply(CCPA_NORMALIZER))
    .drop_duplicates(subset=[WEBSITE_COL])
    .set_index(WEBSITE_COL)["ccpa_norm"]
    .to_dict()
)

all_sites = set(df_all[WEBSITE_COL].unique())
sites_with_banner = set(df_bp[WEBSITE_COL].unique())
sites_without_banner = set(df_bn[WEBSITE_COL].unique())

has_banner = {}
for w in all_sites:
    if w in sites_with_banner:
        has_banner[w] = True
    elif w in sites_without_banner:
        has_banner[w] = False
    else:

        has_banner[w] = False

subj_sites = {w for w in all_sites if ccpa_map.get(w, "non-subjected") == "subjected"}
nons_sites = all_sites - subj_sites

def pct(a, b):
    return (100.0 * a / b) if b else float("nan")

subj_total = len(subj_sites)
subj_with  = sum(1 for w in subj_sites if has_banner[w])
subj_without = subj_total - subj_with

nons_total = len(nons_sites)
nons_with  = sum(1 for w in nons_sites if has_banner[w])
nons_without = nons_total - nons_with

print("=== Banner presence by CCPA subjectivity (unique websites) ===")
print(f"Total unique websites      : {len(all_sites)}")
print(f"Subjected websites         : {subj_total}")
print(f"  With banner              : {subj_with} ({pct(subj_with, subj_total):.1f}%)")
print(f"  Without banner           : {subj_without} ({pct(subj_without, subj_total):.1f}%)")
print(f"Non-Subjected websites     : {nons_total}")
print(f"  With banner              : {nons_with} ({pct(nons_with, nons_total):.1f}%)")
print(f"  Without banner           : {nons_without} ({pct(nons_without, nons_total):.1f}%)")

summary = pd.DataFrame({
    "Group": ["Subjected", "Non-Subjected"],
    "With Banner (N)": [subj_with, nons_with],
    "Without Banner (N)": [subj_without, nons_without],
    "Total (N)": [subj_total, nons_total],
    "With Banner (%)": [pct(subj_with, subj_total), pct(nons_with, nons_total)],
    "Without Banner (%)": [pct(subj_without, subj_total), pct(nons_without, nons_total)],
})

print("\nSummary table:")
print(summary.to_string(index=False))

summary.to_csv("banner_by_ccpa_summary.csv", index=False)
print("\n✅ Wrote banner_by_ccpa_summary.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT = "cookie_data_final - merged_updated.csv"
WEBSITE_COL = "website"
COOKIE_TYPE_COL = "category"
CCPA_COL = "ccpa"
DEFAULT_COL = "initial_cookies"

COOKIE_TYPES = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies",
]

def ccpa_strict(val) -> str:
    if pd.isna(val):
        return "non-subjected"
    return "subjected" if str(val).strip().lower() == "subjected" else "non-subjected"

df = pd.read_csv(INPUT)
for c in [WEBSITE_COL, COOKIE_TYPE_COL, CCPA_COL, DEFAULT_COL]:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

df = df.copy()
df[WEBSITE_COL] = df[WEBSITE_COL].astype(str).str.strip()
df["ccpa_norm"] = df[CCPA_COL].apply(ccpa_strict)

def per_website_counts(group_df: pd.DataFrame, cookie_type: str) -> np.ndarray:
    sub = group_df[group_df[COOKIE_TYPE_COL] == cookie_type]
    counts = (
        sub.groupby(WEBSITE_COL)[DEFAULT_COL]
           .apply(lambda s: s.notnull().sum())
           .astype(int)
    )
    all_sites = group_df[WEBSITE_COL].dropna().unique()
    counts = counts.reindex(all_sites, fill_value=0)
    return counts.to_numpy()

g_subj = df[df["ccpa_norm"] == "subjected"]
g_non  = df[df["ccpa_norm"] != "subjected"]

data = []
for ctype in COOKIE_TYPES:
    vals_subj = per_website_counts(g_subj, ctype)
    vals_non  = per_website_counts(g_non,  ctype)
    data.append((ctype, vals_subj, vals_non))

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey=True)
axes = axes.ravel()

for ax, (ctype, vals_subj, vals_non) in zip(axes, data):

    bp = ax.boxplot(
        [vals_subj, vals_non],
        widths=0.6,
        showfliers=False,
        notch=True,
        showmeans=True,
        meanline=True,
        manage_ticks=False,
    )

    ax.set_xticks([1, 2])
    ax.set_xticklabels([
        f"Subjected (N={len(vals_subj)})",
        f"Non-Subjected (N={len(vals_non)})"
    ], rotation=0)
    ax.set_title(ctype, fontsize=11)
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)

fig.text(0.04, 0.5, "Cookies per Website (Default)", va="center", rotation="vertical")
fig.suptitle("Per-Website Cookie Counts by Type — CCPA Subjected vs Non-Subjected", fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0.06, 0.03, 1, 0.94])

plt.savefig("boxplot_default_cookie_types_ccpa_split_grid.pdf", dpi=300)
plt.savefig("boxplot_default_cookie_types_ccpa_split_grid.png", dpi=300)
plt.show()

print("✅ Wrote: boxplot_default_cookie_types_ccpa_split_grid.pdf / .png")

In [ ]:
import pandas as pd
import numpy as np

INPUT = "cookie_data_final - merged_updated.csv"
WEBSITE_COL = "website"
COOKIE_TYPE_COL = "category"
CCPA_COL = "ccpa"
DEFAULT_COL = "initial_cookies"

COOKIE_TYPES = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies",
]

def ccpa_strict(val) -> str:
    if pd.isna(val):
        return "non-subjected"
    return "subjected" if str(val).strip().lower() == "subjected" else "non-subjected"

def per_website_counts(group_df: pd.DataFrame, cookie_type: str) -> np.ndarray:
    """Count non-null default-state rows per website for a given cookie type."""
    sub = group_df[group_df[COOKIE_TYPE_COL] == cookie_type]
    counts = (
        sub.groupby(WEBSITE_COL)[DEFAULT_COL]
           .apply(lambda s: s.notnull().sum())
           .astype(int)
    )
    all_sites = group_df[WEBSITE_COL].dropna().astype(str).unique()
    return counts.reindex(all_sites, fill_value=0).to_numpy()

def tukey_whiskers(values: np.ndarray) -> tuple[float, float]:
    """Observed Tukey whiskers used by boxplots."""
    v = np.asarray(values, dtype=float)
    if v.size == 0:
        return (np.nan, np.nan)
    q1, q3 = np.percentile(v, [25, 75])
    iqr = q3 - q1
    low_lim, high_lim = q1 - 1.5*iqr, q3 + 1.5*iqr
    low_obs = float(np.min(v[v >= low_lim])) if np.any(v >= low_lim) else float(np.min(v))
    high_obs = float(np.max(v[v <= high_lim])) if np.any(v <= high_lim) else float(np.max(v))
    return low_obs, high_obs

def summarize(values: np.ndarray) -> dict:
    v = np.asarray(values, dtype=float)
    n = len(v)
    if n == 0:
        return dict(N=0, median=np.nan, q1=np.nan, q3=np.nan, iqr=np.nan, mean=np.nan,
                    std=np.nan, p90=np.nan, p95=np.nan, vmin=np.nan, vmax=np.nan,
                    zeros=0, zeros_pct=np.nan, whisker_low=np.nan, whisker_high=np.nan)
    q1, median, q3 = np.percentile(v, [25, 50, 75])
    iqr = q3 - q1
    wl, wh = tukey_whiskers(v)
    return dict(
        N=int(n),
        median=float(median),
        q1=float(q1),
        q3=float(q3),
        iqr=float(iqr),
        mean=float(np.mean(v)),
        std=float(np.std(v, ddof=1) if n > 1 else 0.0),
        p90=float(np.percentile(v, 90)),
        p95=float(np.percentile(v, 95)),
        vmin=float(np.min(v)),
        vmax=float(np.max(v)),
        zeros=int(np.sum(v == 0)),
        zeros_pct=float(np.sum(v == 0) / n * 100.0),
        whisker_low=float(wl),
        whisker_high=float(wh),
    )

def fmt(x, digits=2):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
        return "NA"
    if abs(x - round(x)) < 1e-9:
        return f"{int(round(x))}"
    return f"{x:.{digits}f}"

df = pd.read_csv(INPUT)
for c in [WEBSITE_COL, COOKIE_TYPE_COL, CCPA_COL, DEFAULT_COL]:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

df[WEBSITE_COL] = df[WEBSITE_COL].astype(str).str.strip()
df["ccpa_norm"] = df[CCPA_COL].apply(ccpa_strict)

g_subj = df[df["ccpa_norm"] == "subjected"].copy()
g_non  = df[df["ccpa_norm"] != "subjected"].copy()

N_subj = g_subj[WEBSITE_COL].nunique()
N_non  = g_non[WEBSITE_COL].nunique()

print("=== Website counts (unique) ===")
print(f"CCPA-Subjected   : {N_subj}")
print(f"Non-Subjected    : {N_non}\n")

rows = []
for ctype in COOKIE_TYPES:
    vals_subj = per_website_counts(g_subj, ctype)
    vals_non  = per_website_counts(g_non,  ctype)

    s = summarize(vals_subj)
    n = summarize(vals_non)

    rows.append({
        "Cookie Type": ctype, "Group": "Subjected", **s
    })
    rows.append({
        "Cookie Type": ctype, "Group": "Non-Subjected", **n
    })

stats_df = pd.DataFrame(rows)

print("=== Default-state per-website cookie stats (by cookie type × group) ===")
for ctype in COOKIE_TYPES:
    block = stats_df[stats_df["Cookie Type"] == ctype].copy()
    print(f"\n{ctype}")
    for _, r in block.iterrows():
        print(f"  {r['Group']}: N={fmt(r['N'],0)}, "
              f"Median={fmt(r['median'],0)} (Q1={fmt(r['q1'],0)}, Q3={fmt(r['q3'],0)}, IQR={fmt(r['iqr'],0)}), "
              f"Mean={fmt(r['mean'])}±{fmt(r['std'])}, "
              f"P90={fmt(r['p90'],0)}, P95={fmt(r['p95'],0)}, "
              f"Min/Max={fmt(r['vmin'],0)}/{fmt(r['vmax'],0)}, "
              f"Zeros={fmt(r['zeros'],0)} ({fmt(r['zeros_pct'])}%), "
              f"Whiskers={fmt(r['whisker_low'],0)}–{fmt(r['whisker_high'],0)}")

print("\n=== Contrasts (Subjected − Non-Subjected) ===")
contrast_rows = []
for ctype in COOKIE_TYPES:
    s = stats_df[(stats_df["Cookie Type"] == ctype) & (stats_df["Group"] == "Subjected")].iloc[0]
    n = stats_df[(stats_df["Cookie Type"] == ctype) & (stats_df["Group"] == "Non-Subjected")].iloc[0]
    contrast_rows.append({
        "Cookie Type": ctype,
        "ΔMedian": s["median"] - n["median"],
        "ΔMean": s["mean"] - n["mean"],
        "ΔP90": s["p90"] - n["p90"],
        "ΔP95": s["p95"] - n["p95"],
    })

contrasts_df = pd.DataFrame(contrast_rows)
print(contrasts_df.to_string(index=False))

stats_df.to_csv("boxplot_stats_default_ccpa_split.csv", index=False)
contrasts_df.to_csv("boxplot_stats_default_ccpa_contrasts.csv", index=False)
print("\n✅ Wrote boxplot_stats_default_ccpa_split.csv and boxplot_stats_default_ccpa_contrasts.csv")

In [ ]:
import pandas as pd
import numpy as np

MERGED = "cookie_data_final - merged_updated.csv"
BANNER_PRESENT = "cookie_data_final - Banner_present.csv"

WEBSITE_COL = "website"
CCPA_COL = "ccpa"
CATEGORY_COL = "category"
STAGE_COLS = {
    "Initial": "initial_cookies",
    "Accept": "consent_accept",
    "Reject": "consent_reject",
}

COOKIE_TYPES = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies",
]

def ccpa_strict(val) -> str:
    if pd.isna(val):
        return "non-subjected"
    return "subjected" if str(val).strip().lower() == "subjected" else "non-subjected"

def per_site_counts(df: pd.DataFrame, stage_col: str, cookie_type: str) -> pd.Series:
    """Per-website counts at a given stage for one cookie type (count non-null rows)."""
    sub = df[df[CATEGORY_COL] == cookie_type]
    counts = (
        sub.groupby(WEBSITE_COL)[stage_col]
           .apply(lambda s: s.notnull().sum())
           .astype(int)
    )

    all_sites = df[WEBSITE_COL].dropna().astype(str).unique()
    return counts.reindex(all_sites, fill_value=0)

def summarize(vec: pd.Series) -> dict:
    v = vec.to_numpy()
    if len(v) == 0:
        return {"Median": np.nan, "P90": np.nan, "Mean": np.nan}
    return {
        "Median": float(np.percentile(v, 50)),
        "P90": float(np.percentile(v, 90)),
        "Mean": float(np.mean(v)),
    }

def fmt(x, d=2):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
        return "NA"
    if abs(x - round(x)) < 1e-9:
        return f"{int(round(x))}"
    return f"{x:.{d}f}"

df_ccpa = pd.read_csv(MERGED, usecols=[WEBSITE_COL, CCPA_COL]).copy()
df_flow = pd.read_csv(BANNER_PRESENT).copy()

for c in [WEBSITE_COL, CATEGORY_COL, *STAGE_COLS.values()]:
    if c not in df_flow.columns:
        raise ValueError(f"Missing required column in Banner_present: {c}")
if WEBSITE_COL not in df_ccpa.columns or CCPA_COL not in df_ccpa.columns:
    raise ValueError("MERGED must contain website and ccpa columns.")

df_ccpa[WEBSITE_COL] = df_ccpa[WEBSITE_COL].astype(str).str.strip()
df_ccpa["ccpa_norm"] = df_ccpa[CCPA_COL].apply(ccpa_strict)
ccpa_map = df_ccpa.drop_duplicates(subset=[WEBSITE_COL]).set_index(WEBSITE_COL)["ccpa_norm"].to_dict()

df_flow[WEBSITE_COL] = df_flow[WEBSITE_COL].astype(str).str.strip()
df_flow["ccpa_norm"] = df_flow[WEBSITE_COL].map(ccpa_map).fillna("non-subjected")

df_flow = df_flow[df_flow[WEBSITE_COL].isin(df_ccpa[WEBSITE_COL])].copy()

groups = {
    "CCPA-Subjected": df_flow[df_flow["ccpa_norm"] == "subjected"].copy(),
    "Non-Subjected":  df_flow[df_flow["ccpa_norm"] != "subjected"].copy(),
}

all_rows = []
print("=== Consent Flow (per-website) — Median / P90 / Mean ===\n")
for gname, gdf in groups.items():
    n_sites = gdf[WEBSITE_COL].nunique()
    print(f"-- {gname} (websites N={n_sites}) --")
    group_rows = []
    for ctype in COOKIE_TYPES:
        row = {"Group": gname, "Cookie Type": ctype}
        for stage_label, stage_col in STAGE_COLS.items():
            counts = per_site_counts(gdf, stage_col, ctype)
            s = summarize(counts)

            row[f"{stage_label} Median"] = s["Median"]
            row[f"{stage_label} P90"]   = s["P90"]
            row[f"{stage_label} Mean"]  = s["Mean"]
        group_rows.append(row)

        pretty = (
            f"{ctype}: "
            f"Init Med/P90/Mean={fmt(row['Initial Median'],0)}/{fmt(row['Initial P90'],0)}/{fmt(row['Initial Mean'])}, "
            f"Accept Med/P90/Mean={fmt(row['Accept Median'],0)}/{fmt(row['Accept P90'],0)}/{fmt(row['Accept Mean'])}, "
            f"Reject Med/P90/Mean={fmt(row['Reject Median'],0)}/{fmt(row['Reject P90'],0)}/{fmt(row['Reject Mean'])}"
        )
        print("  " + pretty)
    print()
    all_rows.extend(group_rows)

summary_df = pd.DataFrame(all_rows)

cols = ["Group", "Cookie Type"] + \
       [f"{st} {m}" for st in ["Initial","Accept","Reject"] for m in ["Median","P90","Mean"]]
summary_df = summary_df[cols]

summary_df.to_csv("consent_flow_per_website_summary.csv", index=False)
print("✅ Wrote consent_flow_per_website_summary.csv")

delta_rows = []
for gname, gdf in groups.items():
    for ctype in COOKIE_TYPES:
        init = per_site_counts(gdf, STAGE_COLS["Initial"], ctype)
        acc  = per_site_counts(gdf, STAGE_COLS["Accept"],  ctype)
        rej  = per_site_counts(gdf, STAGE_COLS["Reject"],  ctype)
        delta_rows.append({
            "Group": gname, "Cookie Type": ctype,
            "Δ(Accept-Initial) Median": float(np.percentile((acc - init).to_numpy(), 50)),
            "Δ(Reject-Initial) Median": float(np.percentile((rej - init).to_numpy(), 50)),
        })
delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv("consent_flow_per_website_deltas.csv", index=False)
print("✅ Wrote consent_flow_per_website_deltas.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT = "cookie_data_final - merged_updated.csv"
WEBSITE_COL_CANDS = ["website", "domain"]
COOKIE_TYPE_COL_CANDS = ["category", "cookie_type", "type"]
DEFAULT_COL_CANDS = ["initial_cookies", "default", "Default"]
INDUSTRY_COL_CANDS = ["industry", "Industry", "sector", "industry_sector", "site_category", "vertical"]

COOKIE_TYPES = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies",
]

def detect(cols, cands, what):
    for c in cands:
        if c in cols:
            return c
    raise ValueError(f"Couldn't find {what}. Tried: {cands}. Available: {list(cols)}")

def per_site_counts(df, website_col, cookie_type_col, default_col, cookie_type):
    """Per-website counts of non-null default entries for a given cookie type."""
    sub = df[df[cookie_type_col] == cookie_type]
    counts = (
        sub.groupby(website_col)[default_col]
           .apply(lambda s: s.notnull().sum())
           .astype(int)
    )
    all_sites = df[website_col].dropna().astype(str).unique()
    return counts.reindex(all_sites, fill_value=0)

def summarize_counts(series):
    v = series.to_numpy()
    if len(v) == 0:
        return {"Median": np.nan, "P90": np.nan, "Mean": np.nan, "N_sites": 0}
    return {
        "Median": float(np.percentile(v, 50)),
        "P90": float(np.percentile(v, 90)),
        "Mean": float(np.mean(v)),
        "N_sites": int(len(v)),
    }

def fmt(x, d=2):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
        return "NA"
    if abs(x - round(x)) < 1e-9:
        return str(int(round(x)))
    return f"{x:.{d}f}"

df = pd.read_csv(INPUT)

website_col = detect(df.columns, WEBSITE_COL_CANDS, "website column")
cookie_type_col = detect(df.columns, COOKIE_TYPE_COL_CANDS, "cookie type column")
default_col = detect(df.columns, DEFAULT_COL_CANDS, "default-state column")
industry_col = detect(df.columns, INDUSTRY_COL_CANDS, "industry/sector column")

df = df.copy()
df[website_col] = df[website_col].astype(str).str.strip()
df[industry_col] = df[industry_col].astype(str).str.strip()

df = df[df[cookie_type_col].isin(COOKIE_TYPES)].copy()

rows = []
for industry, g in df.groupby(industry_col):

    for ctype in COOKIE_TYPES:
        counts = per_site_counts(g, website_col, cookie_type_col, default_col, ctype)
        stats = summarize_counts(counts)
        rows.append({
            "Industry": industry,
            "Cookie Type": ctype,
            "N websites": stats["N_sites"],
            "Median": stats["Median"],
            "P90": stats["P90"],
            "Mean": stats["Mean"],
        })

summary = pd.DataFrame(rows)

order = (summary.pivot(index="Industry", columns="Cookie Type", values="Mean")
               .fillna(0)
               .assign(_total=lambda d: d.sum(axis=1))
               .sort_values("_total", ascending=False)
               .index.tolist())
summary["Industry"] = pd.Categorical(summary["Industry"], ordered=True, categories=order)
summary = summary.sort_values(["Industry", "Cookie Type"]).reset_index(drop=True)

print("=== Industry × Cookie Type (per-website Default counts) — Median / P90 / Mean ===")
for industry in order:
    block = summary[summary["Industry"] == industry]
    n_sites = int(block["N websites"].max()) if not block.empty else 0
    print(f"\n-- {industry} (websites N={n_sites}) --")
    for _, r in block.iterrows():
        print(f"  {r['Cookie Type']}: "
              f"Median/P90/Mean = {fmt(r['Median'],0)}/{fmt(r['P90'],0)}/{fmt(r['Mean'])}")

summary.to_csv("industry_per_website_summary_long.csv", index=False)

wide_mean = summary.pivot_table(index="Industry", columns="Cookie Type", values="Mean", aggfunc="first").fillna(0)
wide_mean.to_csv("industry_per_website_means_wide.csv")
print("\n✅ Wrote industry_per_website_summary_long.csv and industry_per_website_means_wide.csv")

plt.figure(figsize=(12, 7))

wide_mean = wide_mean.loc[order]

bottom = np.zeros(len(wide_mean))
x = np.arange(len(wide_mean))
labels = wide_mean.index.tolist()

for ctype in COOKIE_TYPES:
    vals = wide_mean[ctype].to_numpy()
    plt.bar(x, vals, bottom=bottom, label=ctype)
    bottom += vals

plt.xticks(x, labels, rotation=60, ha="right")
plt.ylabel("Mean Cookies per Website (Default)")
plt.title("Industry-Specific Cookie Distribution (Per-Website Means, Default)")
plt.legend(title="Cookie Type", ncols=2, fontsize=9)
plt.tight_layout()
plt.savefig("industry_cookie_stacked_barplot_per_website.pdf", dpi=300)
plt.savefig("industry_cookie_stacked_barplot_per_website.png", dpi=300)
plt.show()

print("✅ Wrote industry_cookie_stacked_barplot_per_website.pdf / .png")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

INPUT = "cookie_data_final - merged_updated.csv"

WEBSITE_COL_CANDS = ["website", "domain"]
COOKIE_TYPE_COL_CANDS = ["category", "cookie_type", "type"]
DEFAULT_COL_CANDS = ["initial_cookies", "default", "Default"]
INDUSTRY_COL_CANDS = ["industry", "Industry", "sector", "industry_sector", "site_category", "vertical"]

COOKIE_TYPES = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies",
]

def detect(cols, cands, what):
    for c in cands:
        if c in cols:
            return c
    raise ValueError(f"Couldn't find {what}. Tried: {cands}. Available: {list(cols)}")

def per_site_counts(df, website_col, cookie_type_col, default_col, cookie_type):
    """Return per-website counts (Default state) for one cookie type within df (one industry)."""
    sub = df[df[cookie_type_col] == cookie_type]
    counts = (
        sub.groupby(website_col)[default_col]
           .apply(lambda s: s.notnull().sum())
           .astype(int)
    )

    all_sites = df[website_col].dropna().astype(str).unique()
    return counts.reindex(all_sites, fill_value=0)

def summarize(series):
    v = series.to_numpy()
    if len(v) == 0:
        return {"Median": np.nan, "P90": np.nan, "Mean": np.nan, "N_sites": 0}
    return {
        "Median": float(np.percentile(v, 50)),
        "P90": float(np.percentile(v, 90)),
        "Mean": float(np.mean(v)),
        "N_sites": int(len(v)),
    }

def fmt(x, d=2):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
        return "NA"
    if abs(x - round(x)) < 1e-9:
        return str(int(round(x)))
    return f"{x:.{d}f}"

df = pd.read_csv(INPUT)
website_col = detect(df.columns, WEBSITE_COL_CANDS, "website column")
cookie_type_col = detect(df.columns, COOKIE_TYPE_COL_CANDS, "cookie type column")
default_col = detect(df.columns, DEFAULT_COL_CANDS, "default-state column")
industry_col = detect(df.columns, INDUSTRY_COL_CANDS, "industry/sector column")

df = df.copy()
df[website_col] = df[website_col].astype(str).str.strip()
df[industry_col] = df[industry_col].astype(str).str.strip()

df = df[df[cookie_type_col].isin(COOKIE_TYPES)].copy()

rows = []
industry_order = []
data_by_type = {t: [] for t in COOKIE_TYPES}
n_by_industry = {}

tmp_mean = (df.groupby([industry_col, cookie_type_col])[default_col]
              .apply(lambda s: s.notnull().groupby(df.loc[s.index, website_col]).sum().mean())
              .reset_index(name="mean_per_website"))
order_df = (tmp_mean.pivot(index=industry_col, columns=cookie_type_col, values="mean_per_website")
                    .fillna(0.0))
order_df["_total"] = order_df.sum(axis=1)
industry_order = list(order_df.sort_values("_total", ascending=False).index)

for ind in industry_order:
    g = df[df[industry_col] == ind]
    n_sites = g[website_col].nunique()
    n_by_industry[ind] = n_sites
    for ctype in COOKIE_TYPES:
        counts = per_site_counts(g, website_col, cookie_type_col, default_col, ctype)
        stats = summarize(counts)
        rows.append({
            "Industry": ind,
            "Cookie Type": ctype,
            "N websites": stats["N_sites"],
            "Median": stats["Median"],
            "P90": stats["P90"],
            "Mean": stats["Mean"],
        })
        data_by_type[ctype].append(counts.to_numpy())

summary = pd.DataFrame(rows)
summary.to_csv("industry_per_website_summary_long.csv", index=False)

wide_means = summary.pivot_table(index="Industry", columns="Cookie Type", values="Mean", aggfunc="first").fillna(0.0)
wide_means = wide_means.loc[industry_order]
wide_means.to_csv("industry_per_website_means_wide.csv")

print("=== Industry × Cookie Type (per-website Default counts) — Median / P90 / Mean ===")
total_sites_union = df[[industry_col, website_col]].drop_duplicates()[website_col].nunique()
print(f"(Unique websites represented across industries: {total_sites_union})")

for ind in industry_order:
    block = summary[summary["Industry"] == ind]
    n_sites = n_by_industry[ind]
    print(f"\n-- {ind} (websites N={n_sites}) --")
    for _, r in block.iterrows():
        print(f"  {r['Cookie Type']}: "
              f"Median/P90/Mean = {fmt(r['Median'],0)}/{fmt(r['P90'],0)}/{fmt(r['Mean'])}")

print("\n✅ Wrote industry_per_website_summary_long.csv and industry_per_website_means_wide.csv")

fig, ax = plt.subplots(figsize=(18, 7))

num_industries = len(industry_order)
x = np.arange(num_industries, dtype=float)
box_width = 0.16
offsets = {
    "Targeting Cookies": -1.5 * box_width,
    "Performance Cookies": -0.5 * box_width,
    "Functional Cookies":  +0.5 * box_width,
    "Strictly Necessary Cookies": +1.5 * box_width,
}

palette = {
    "Targeting Cookies": "#4E79A7",
    "Performance Cookies": "#F28E2B",
    "Functional Cookies": "#59A14F",
    "Strictly Necessary Cookies": "#E15759",
}

legend_patches = []
for ctype in COOKIE_TYPES:
    pos = x + offsets[ctype]
    bp = ax.boxplot(
        data_by_type[ctype],
        positions=pos,
        widths=box_width*0.95,
        patch_artist=True,
        showfliers=False,
        notch=True,
        manage_ticks=False,
    )

    for box in bp['boxes']:
        box.set(facecolor=palette[ctype], edgecolor="black", alpha=0.75)
    for median in bp['medians']:
        median.set(color="black", linewidth=1.2)
    for whisk in bp['whiskers']:
        whisk.set(color="black", linewidth=1.0)
    for cap in bp['caps']:
        cap.set(color="black", linewidth=1.0)

    legend_patches.append(Patch(facecolor=palette[ctype], edgecolor="black", alpha=0.75, label=ctype))

ax.set_xticks(x)
ax.set_xticklabels(industry_order, rotation=60, ha="right")
ax.set_ylabel("Cookies per Website (Default)")
ax.set_title("Per-Website Cookie Counts by Industry and Type (Default)")
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
ax.legend(handles=legend_patches, title="Cookie Type", ncols=2, fontsize=9)

plt.tight_layout()
plt.savefig("industry_boxplot_per_website_by_type.pdf", dpi=300)
plt.savefig("industry_boxplot_per_website_by_type.png", dpi=300)
plt.show()

print("✅ Wrote industry_boxplot_per_website_by_type.pdf / .png")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT = "cookie_data_final - merged_updated.csv"

WEBSITE_COL_CANDS = ["website", "domain"]
COOKIE_TYPE_COL_CANDS = ["category", "cookie_type", "type"]
INDUSTRY_COL_CANDS = ["industry", "Industry", "sector", "industry_sector", "site_category", "vertical"]

CONFIG_COLS = {
    "Allow 3rd Party": "allow_3rd_party",
    "Default": "initial_cookies",
    "Block 3rd Party": "block_3rd_party",
    "Do Not Track": "do_not_track",
    "uBlock Origin": "ublock",
}

COOKIE_TYPES = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies",
]

def detect(cols, cands, what):
    for c in cands:
        if c in cols: return c
    raise ValueError(f"Couldn't find {what}. Tried: {cands}. Available: {list(cols)}")

def per_site_counts_for(df_industry, website_col, cookie_type_col, config_col, cookie_type):
    """
    Within a single industry subset (df_industry), return per-website counts for a given cookie type
    in a given configuration (count non-null rows in that config column).
    """
    sub = df_industry[df_industry[cookie_type_col] == cookie_type]
    counts = (
        sub.groupby(website_col)[config_col]
           .apply(lambda s: s.notnull().sum())
           .astype(int)
    )

    all_sites = df_industry[website_col].dropna().astype(str).unique()
    return counts.reindex(all_sites, fill_value=0)

def summarize_counts(arr_like):
    v = np.asarray(arr_like, dtype=float)
    if v.size == 0:
        return {"Median": np.nan, "P90": np.nan, "Mean": np.nan, "N_sites": 0}
    return {
        "Median": float(np.percentile(v, 50)),
        "P90": float(np.percentile(v, 90)),
        "Mean": float(np.mean(v)),
        "N_sites": int(v.size),
    }

def fmt(x, d=2):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))): return "NA"
    if abs(x - round(x)) < 1e-9: return str(int(round(x)))
    return f"{x:.{d}f}"

df = pd.read_csv(INPUT)

website_col = detect(df.columns, WEBSITE_COL_CANDS, "website/domain column")
cookie_type_col = detect(df.columns, COOKIE_TYPE_COL_CANDS, "cookie type column")
industry_col = detect(df.columns, INDUSTRY_COL_CANDS, "industry/sector column")

missing_cfg = [lab for lab, col in CONFIG_COLS.items() if col not in df.columns]
if missing_cfg:
    raise ValueError(f"Missing configuration columns: {missing_cfg}. "
                     f"Available columns: {list(df.columns)}")

df = df.copy()
df[website_col] = df[website_col].astype(str).str.strip()
df[industry_col] = df[industry_col].astype(str).str.strip()

df = df[df[cookie_type_col].isin(COOKIE_TYPES)].copy()

rows = []
industries = sorted(df[industry_col].dropna().unique().tolist())
for ind in industries:
    g = df[df[industry_col] == ind]
    n_sites = g[website_col].nunique()
    for ctype in COOKIE_TYPES:
        for cfg_label, cfg_col in CONFIG_COLS.items():
            counts = per_site_counts_for(g, website_col, cookie_type_col, cfg_col, ctype)
            stats = summarize_counts(counts.to_numpy())
            rows.append({
                "Industry": ind,
                "Cookie Type": ctype,
                "Config": cfg_label,
                "N websites": n_sites,
                "Median": stats["Median"],
                "P90": stats["P90"],
                "Mean": stats["Mean"],
            })

summary_long = pd.DataFrame(rows)
summary_long.to_csv("sector_config_type_perwebsite_summary_long.csv", index=False)

summary_wide = (summary_long
    .pivot_table(index=["Industry", "Cookie Type"],
                 columns="Config",
                 values=["Median", "P90", "Mean"],
                 aggfunc="first")
    .sort_index()
)
summary_wide.to_csv("sector_config_type_perwebsite_summary_wide.csv")

print("=== Preview: Industry × Config × Type (per-website counts) — Median / P90 / Mean ===")
for ind in industries[:5]:
    print(f"\n-- {ind} (websites N={int(summary_long[summary_long['Industry']==ind]['N websites'].max())}) --")
    for ctype in COOKIE_TYPES:
        line = [f"{cfg}: {fmt(summary_long.loc[(summary_long['Industry']==ind) & (summary_long['Cookie Type']==ctype) & (summary_long['Config']==cfg), 'Median'].values[0],0)}/"
                      f"{fmt(summary_long.loc[(summary_long['Industry']==ind) & (summary_long['Cookie Type']==ctype) & (summary_long['Config']==cfg), 'P90'].values[0],0)}/"
                      f"{fmt(summary_long.loc[(summary_long['Industry']==ind) & (summary_long['Cookie Type']==ctype) & (summary_long['Config']==cfg), 'Mean'].values[0])}"
                for cfg in CONFIG_COLS.keys()]
        print(f"  {ctype}: " + " | ".join(line))

print("\n✅ Wrote sector_config_type_perwebsite_summary_long.csv and sector_config_type_perwebsite_summary_wide.csv")

fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.5), sharey=False)
axes = axes.ravel()

for ax, ctype in zip(axes, COOKIE_TYPES):

    data_by_cfg = []
    labels = []
    for cfg_label, cfg_col in CONFIG_COLS.items():
        sub = df[df[cookie_type_col] == ctype]

        counts = (sub.groupby(website_col)[cfg_col].apply(lambda s: s.notnull().sum()).astype(int))
        all_sites = df[website_col].dropna().astype(str).unique()
        counts = counts.reindex(all_sites, fill_value=0)
        data_by_cfg.append(counts.to_numpy())
        labels.append(cfg_label)

    bp = ax.boxplot(
        data_by_cfg,
        widths=0.6,
        showfliers=False,
        notch=True,
        showmeans=True,
        meanline=True,
        manage_ticks=False,
    )
    ax.set_xticks(range(1, len(labels)+1))
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_title(ctype, fontsize=11)
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)
    ax.set_ylabel("Cookies per Website")

fig.suptitle("Per-Website Cookie Counts by Configuration (All Industries) — One Panel per Cookie Type",
             fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0, 0.02, 1, 0.95])
plt.savefig("sector_config_type_perwebsite_boxplots_by_type.pdf", dpi=300)
plt.savefig("sector_config_type_perwebsite_boxplots_by_type.png", dpi=300)
plt.show()
print("✅ Wrote sector_config_type_perwebsite_boxplots_by_type.pdf / .png")

def plot_industry(industry_name: str):
    g = df[df[industry_col] == industry_name]
    if g.empty:
        print(f"[warn] No rows for industry={industry_name!r}.")
        return
    fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.5), sharey=False)
    axes = axes.ravel()
    for ax, ctype in zip(axes, COOKIE_TYPES):
        data_by_cfg, labels = [], []
        for cfg_label, cfg_col in CONFIG_COLS.items():
            counts = per_site_counts_for(g, website_col, cookie_type_col, cfg_col, ctype)
            data_by_cfg.append(counts.to_numpy())
            labels.append(cfg_label)
        ax.boxplot(
            data_by_cfg,
            widths=0.6,
            showfliers=False,
            notch=True,
            showmeans=True,
            meanline=True,
            manage_ticks=False,
        )
        ax.set_xticks(range(1, len(labels)+1))
        ax.set_xticklabels(labels, rotation=20, ha="right")
        ax.set_title(f"{ctype}", fontsize=11)
        ax.grid(True, axis="y", linestyle="--", alpha=0.5)
        ax.set_ylabel("Cookies per Website")
    fig.suptitle(f"{industry_name}: Per-Website Cookie Counts by Configuration (2×2 by Type)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout(rect=[0, 0.02, 1, 0.95])
    safe = industry_name.replace("/", "_").replace(" ", "_")
    out_pdf = f"sector_{safe}_perwebsite_boxplots_by_type.pdf"
    out_png = f"sector_{safe}_perwebsite_boxplots_by_type.png"
    plt.savefig(out_pdf, dpi=300)
    plt.savefig(out_png, dpi=300)
    plt.show()
    print(f"✅ Wrote {out_pdf} / {out_png}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import logrank_test
    LIFELINES_OK = True
except Exception:
    LIFELINES_OK = False
    print("[warn] lifelines not installed; survival analysis will be skipped.")

MAIN = "cookie_data_final - merged_updated.csv"
BANNER_PRESENT = "cookie_data_final - Banner_present.csv"
BANNER_ABSENT  = "cookie_data_final - Banner_not_present.csv"

WEBSITE_COL = "website"
CCPA_COL    = "ccpa"
INDUSTRY_COL= "Industry"
TYPE_COL    = "category"
EXPIRY_SEC_COL = "remaining_expiry_time"
SESSION_COL = "session"

CONFIG_COLS = {
    "Allow 3rd Party": "allow_3rd_party",
    "Default":         "initial_cookies",
    "Block 3rd Party": "block_3rd_party",
    "Do Not Track":    "do_not_track",
    "uBlock Origin":   "ublock",
}

COOKIE_TYPES = [
    "Targeting Cookies",
    "Performance Cookies",
    "Functional Cookies",
    "Strictly Necessary Cookies",
]
TRACKING_COMBINED = ("Targeting Cookies", "Performance Cookies")

THRESHOLDS = [5, 10, 15]

df = pd.read_csv(MAIN, low_memory=False)

df[WEBSITE_COL] = df[WEBSITE_COL].astype(str).str.strip()
df[INDUSTRY_COL]= df[INDUSTRY_COL].astype(str).str.strip()

df["ccpa_group"] = df[CCPA_COL].astype(str).str.strip().str.lower().eq("subjected")\
                    .map({True:"Subjected", False:"Non-Subjected"})

df = df[df[TYPE_COL].isin(COOKIE_TYPES)].copy()

bp = pd.read_csv(BANNER_PRESENT, low_memory=False)
bn = pd.read_csv(BANNER_ABSENT, low_memory=False)

bp[WEBSITE_COL] = bp[WEBSITE_COL].astype(str).str.strip()
bn[WEBSITE_COL] = bn[WEBSITE_COL].astype(str).str.strip()

present_sites = set(bp[WEBSITE_COL])
absent_sites  = set(bn[WEBSITE_COL])

absent_sites -= (present_sites & absent_sites)

banner_map = pd.DataFrame({WEBSITE_COL: list(present_sites | absent_sites)})
banner_map["banner_present"] = banner_map[WEBSITE_COL].isin(present_sites)
df = df.merge(banner_map, on=WEBSITE_COL, how="left")
df["banner_group"] = df["banner_present"].map({True:"Banner Present", False:"No Banner"})

def to_days(x):

    x = pd.to_numeric(x, errors="coerce")
    return x / 86400.0

def clean_durations_days(sub):

    secs = to_days(sub[EXPIRY_SEC_COL])
    if SESSION_COL in sub.columns:
        mask_session = sub[SESSION_COL].astype(str).str.lower().isin(["true", "1", "yes"])
        secs = secs[~mask_session]
    secs = secs[(~secs.isna()) & (secs > 0)]
    return secs.values

def km_plot(ax, t1, t2, label1, label2, title):
    km1 = KaplanMeierFitter()
    km2 = KaplanMeierFitter()
    e1 = np.ones_like(t1, dtype=int)
    e2 = np.ones_like(t2, dtype=int)
    if t1.size:
        km1.fit(t1, event_observed=e1, label=label1).plot(ax=ax, ci_show=False)
    if t2.size:
        km2.fit(t2, event_observed=e2, label=label2).plot(ax=ax, ci_show=False)
    ax.set_title(title)
    ax.set_xlabel("Days until expiry")
    ax.set_ylabel("Survival probability")
    ax.grid(True, linestyle="--", alpha=0.4)
    return km1 if t1.size else None, km2 if t2.size else None

def summarize(arr):
    if len(arr)==0:
        return dict(N=0, Median=np.nan, P25=np.nan, P75=np.nan, IQR=np.nan, Mean=np.nan)
    q25, med, q75 = np.percentile(arr, [25,50,75])
    return dict(N=int(len(arr)), Median=float(med), P25=float(q25), P75=float(q75),
                IQR=float(q75-q25), Mean=float(np.mean(arr)))

if LIFELINES_OK:
    rows_ccpa, rows_banner = [], []

    for ctype in COOKIE_TYPES:
        sub = df[df[TYPE_COL]==ctype]

        t_subj = clean_durations_days(sub[sub["ccpa_group"]=="Subjected"])
        t_non  = clean_durations_days(sub[sub["ccpa_group"]=="Non-Subjected"])

        fig, ax = plt.subplots(figsize=(6,4))
        kmA, kmB = km_plot(ax, t_subj, t_non, "Subjected", "Non-Subjected",
                           f"Cookie Survival by CCPA — {ctype}")
        plt.tight_layout()
        fname = ctype.replace(" ", "_").lower()
        plt.savefig(f"survival_by_ccpa_{fname}.pdf", dpi=300)
        plt.savefig(f"survival_by_ccpa_{fname}.png", dpi=300)
        plt.close(fig)

        if len(t_subj) and len(t_non):
            res = logrank_test(t_subj, t_non, event_observed_A=np.ones_like(t_subj, int),
                               event_observed_B=np.ones_like(t_non, int))
            pval = float(res.p_value)
        else:
            pval = np.nan

        sA = summarize(t_subj); sB = summarize(t_non)
        km_med_A = float(getattr(kmA, "median_survival_time_", np.nan)) if kmA else np.nan
        km_med_B = float(getattr(kmB, "median_survival_time_", np.nan)) if kmB else np.nan

        rows_ccpa += [
            {"Cookie Type": ctype, "Group":"Subjected", "N":sA["N"],
             "KM_Median_Days": km_med_A, "Median_Days": sA["Median"],
             "IQR_Days": sA["IQR"], "Mean_Days": sA["Mean"], "Logrank_p": pval},
            {"Cookie Type": ctype, "Group":"Non-Subjected", "N":sB["N"],
             "KM_Median_Days": km_med_B, "Median_Days": sB["Median"],
             "IQR_Days": sB["IQR"], "Mean_Days": sB["Mean"], "Logrank_p": pval},
        ]

        t_pres = clean_durations_days(sub[sub["banner_group"]=="Banner Present"])
        t_abs  = clean_durations_days(sub[sub["banner_group"]=="No Banner"])

        fig, ax = plt.subplots(figsize=(6,4))
        kmP, kmN = km_plot(ax, t_pres, t_abs, "Banner Present", "No Banner",
                           f"Cookie Survival by Banner — {ctype}")
        plt.tight_layout()
        plt.savefig(f"survival_by_banner_{fname}.pdf", dpi=300)
        plt.savefig(f"survival_by_banner_{fname}.png", dpi=300)
        plt.close(fig)

        if len(t_pres) and len(t_abs):
            res = logrank_test(t_pres, t_abs, event_observed_A=np.ones_like(t_pres, int),
                               event_observed_B=np.ones_like(t_abs, int))
            pval2 = float(res.p_value)
        else:
            pval2 = np.nan

        sP = summarize(t_pres); sN = summarize(t_abs)
        km_med_P = float(getattr(kmP, "median_survival_time_", np.nan)) if kmP else np.nan
        km_med_N = float(getattr(kmN, "median_survival_time_", np.nan)) if kmN else np.nan

        rows_banner += [
            {"Cookie Type": ctype, "Group":"Banner Present", "N":sP["N"],
             "KM_Median_Days": km_med_P, "Median_Days": sP["Median"],
             "IQR_Days": sP["IQR"], "Mean_Days": sP["Mean"], "Logrank_p": pval2},
            {"Cookie Type": ctype, "Group":"No Banner", "N":sN["N"],
             "KM_Median_Days": km_med_N, "Median_Days": sN["Median"],
             "IQR_Days": sN["IQR"], "Mean_Days": sN["Mean"], "Logrank_p": pval2},
        ]

    pd.DataFrame(rows_ccpa).to_csv("survival_summary_by_ccpa.csv", index=False)
    pd.DataFrame(rows_banner).to_csv("survival_summary_by_banner.csv", index=False)
    print("✅ Wrote survival_by_* PDFs/PNGs and survival_summary CSVs")
else:
    print("⚠️ Skipping survival analysis (install lifelines).")

def per_site_counts(df, cookie_type, cfg_col):
    sub = df[df[TYPE_COL] == cookie_type]
    counts = (sub.groupby(WEBSITE_COL)[cfg_col]
                .apply(lambda s: s.notnull().sum())
                .astype(int))

    all_sites = df[WEBSITE_COL].dropna().astype(str).unique()
    return counts.reindex(all_sites, fill_value=0)

def tail_shares(sites, counts, thresholds):
    s = counts.reindex(sites, fill_value=0)
    N = len(sites)
    out = []
    for th in thresholds:
        n_ge = int((s >= th).sum())
        out.append((th, N, n_ge, (n_ge / N * 100.0 if N else np.nan)))
    return out

all_sites = df[WEBSITE_COL].dropna().unique().tolist()
sites_by_ccpa = {
    "Subjected": df.loc[df["ccpa_group"]=="Subjected", WEBSITE_COL].unique().tolist(),
    "Non-Subjected": df.loc[df["ccpa_group"]=="Non-Subjected", WEBSITE_COL].unique().tolist(),
}
sites_by_industry = {ind: g[WEBSITE_COL].unique().tolist()
                     for ind, g in df.groupby(INDUSTRY_COL)}

rows_ccpa = []
for ctype in list(COOKIE_TYPES) + ["Tracking (Targeting+Performance)"]:
    for cfg_label, cfg_col in CONFIG_COLS.items():
        if ctype == "Tracking (Targeting+Performance)":
            c1 = per_site_counts(df, TRACKING_COMBINED[0], cfg_col)
            c2 = per_site_counts(df, TRACKING_COMBINED[1], cfg_col)
            counts = c1 + c2
        else:
            counts = per_site_counts(df, ctype, cfg_col)

        for grp, sites in sites_by_ccpa.items():
            for th, N, n_ge, pct in tail_shares(sites, counts, THRESHOLDS):
                rows_ccpa.append({
                    "Dimension": "CCPA", "Group": grp,
                    "Cookie Type": ctype, "Config": cfg_label,
                    "Threshold": th, "Websites (N)": N,
                    "N >= Threshold": n_ge, "Share >= Threshold (%)": pct
                })
pd.DataFrame(rows_ccpa).to_csv("tail_risk_shares_by_ccpa.csv", index=False)
print("✅ Wrote tail_risk_shares_by_ccpa.csv")

rows_ind = []
for ctype in list(COOKIE_TYPES) + ["Tracking (Targeting+Performance)"]:
    for cfg_label, cfg_col in CONFIG_COLS.items():
        if ctype == "Tracking (Targeting+Performance)":
            c1 = per_site_counts(df, TRACKING_COMBINED[0], cfg_col)
            c2 = per_site_counts(df, TRACKING_COMBINED[1], cfg_col)
            counts = c1 + c2
        else:
            counts = per_site_counts(df, ctype, cfg_col)

        for ind, sites in sites_by_industry.items():
            for th, N, n_ge, pct in tail_shares(sites, counts, THRESHOLDS):
                rows_ind.append({
                    "Dimension": "Industry", "Group": ind,
                    "Cookie Type": ctype, "Config": cfg_label,
                    "Threshold": th, "Websites (N)": N,
                    "N >= Threshold": n_ge, "Share >= Threshold (%)": pct
                })
pd.DataFrame(rows_ind).to_csv("tail_risk_shares_by_industry.csv", index=False)
print("✅ Wrote tail_risk_shares_by_industry.csv")

rows_all = []
for ctype in list(COOKIE_TYPES) + ["Tracking (Targeting+Performance)"]:
    for cfg_label, cfg_col in CONFIG_COLS.items():
        if ctype == "Tracking (Targeting+Performance)":
            c1 = per_site_counts(df, TRACKING_COMBINED[0], cfg_col)
            c2 = per_site_counts(df, TRACKING_COMBINED[1], cfg_col)
            counts = c1 + c2
        else:
            counts = per_site_counts(df, ctype, cfg_col)

        for th, N, n_ge, pct in tail_shares(all_sites, counts, THRESHOLDS):
            rows_all.append({
                "Dimension": "All", "Group": "All websites",
                "Cookie Type": ctype, "Config": cfg_label,
                "Threshold": th, "Websites (N)": N,
                "N >= Threshold": n_ge, "Share >= Threshold (%)": pct
            })
pd.DataFrame(rows_all).to_csv("tail_risk_shares_all.csv", index=False)
print("✅ Wrote tail_risk_shares_all.csv")

In [ ]:
import os
import glob
import json
from pathlib import Path
from typing import Dict, Any, List, Tuple

from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

POLICIES_DIR = "policies_text"
OUTPUT_FILE  = "ccpa_policy_audit_trial.json"
MODEL        = "gpt-4o"
TEMPERATURE  = 0
MAX_OUTPUT_TOKENS = 1500
MAX_SAMPLES  = 5

ESTIMATED_TOKEN_SOFT_CAP = 100_000

load_dotenv()
from openai import OpenAI

API_KEY = "REDACTED-SET-OPENAI_API_KEY-ENV-VAR"
client = OpenAI(api_key=API_KEY)

def iter_policy_files(limit: int) -> List[Tuple[str, Path]]:
    """
    Returns up to `limit` (domain, path) pairs from POLICIES_DIR/*.txt.
    Assumes names like 'www_example_com__combined.txt'.
    """
    files = sorted(glob.glob(os.path.join(POLICIES_DIR, "*.txt")))
    pairs = []
    for p in files[:limit]:
        name = Path(p).stem
        domain = name.split("__combined", 1)[0].replace("_", ".")
        pairs.append((domain, Path(p)))
    return pairs

def load_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

def est_tokens_from_chars(s: str) -> int:

    return max(1, len(s) // 4)

def build_prompt(domain: str, policy_text: str) -> str:
    """
    Returns a detailed instruction prompt for the LLM with clear scoring rubrics,
    definition of each metric/flag, and a strict JSON schema to follow.
    """
    return f"""
You are a privacy compliance analyst evaluating a privacy policy strictly under the California Consumer Privacy Act (CCPA) and related U.S. state privacy concepts (CPRA updates included). You must rely ONLY on the policy text provided below—no external knowledge or assumptions.

Analyze the policy for: {domain}

# Output Requirements
Return **one** JSON object matching this exact schema and types:
{{
  "online_data_practices": "string (brief summary or 'Not mentioned.')",
  "offline_data_practices": "string (brief summary or 'Not mentioned.')",
  "rubric_assessment": {{
    "completeness_score": 0-3,
    "completeness_reason": "string",
    "usability_score": 0-3,
    "usability_reason": "string",
    "accuracy_score": 0-3,
    "accuracy_reason": "string",
    "policy_contradiction": true/false,
    "disclosure_map": {{
      "data_collected": true/false,
      "data_shared": true/false,
      "purpose_of_collection": true/false,
      "retention_period": true/false,
      "right_to_access": true/false,
      "right_to_delete": true/false,
      "opt_out": true/false
    }}
  }},
  "behavioral_claims": {{
    "honors_gpc": true/false/"unspecified",
    "respects_dnt": true/false/"unspecified",
    "sets_cookies_before_consent": true/false/"unspecified",
    "sets_cookies_after_rejecting_consent": true/false/"unspecified",
    "deletes_cookies_on_rejection": true/false/"unspecified",
    "uses_tracking_only_after_consent": true/false/"unspecified",
    "sells_data": true/false/"unspecified",
    "shares_with_third_parties": true/false/"unspecified",
    "justifications": {{
      "honors_gpc": "1–2 sentences or 'unspecified'.",
      "respects_dnt": " ... ",
      "sets_cookies_before_consent": " ... ",
      "sets_cookies_after_rejecting_consent": " ... ",
      "deletes_cookies_on_rejection": " ... ",
      "uses_tracking_only_after_consent": " ... ",
      "sells_data": " ... ",
      "shares_with_third_parties": " ... "
    }}
  }}
}}

## Section A — Policy Scope Categorization
- **online_data_practices**: Briefly summarize policy text on web/app tracking and collection (e.g., cookies, pixels, SDKs, analytics, ad tech, web forms, account signup). If not present, return "Not mentioned."
- **offline_data_practices**: Briefly summarize policy text on in-store collection, call centers, paper forms, events, or other in-person channels. If not present, return "Not mentioned."

## Section B — Policy Quality Assessment (Scored 0–3)
Score using only what the policy explicitly states.

### 1) completeness_score (0–3)
Assess whether the policy **mentions** and **explains** consumer rights central to CCPA and how to exercise them.
- **What to look for**: Access/know; deletion; opt-out of sale/share/targeted advertising; (nice-to-have but not required: correction, portability, non-discrimination, appeal).
- **Scoring guide**:
  - **0**: No meaningful mention of rights or how to exercise them.
  - **1**: Vague rights mention, minimal specifics, unclear processes.
  - **2**: Rights identified with at least some concrete instructions (e.g., link/email/form/phone), but gaps remain (e.g., missing one core right or unclear verification).
  - **3**: Rights clearly listed (access/know, delete, opt-out of sale/share/targeted advertising), with actionable mechanisms (webform/email/phone link) and basics on verification, timelines, and scope.

### 2) usability_score (0–3)
Evaluate how **actionable** the mechanisms are.
- **What to look for**: Dedicated links/buttons, DSAR portals, emails, phone numbers, clear steps, eligibility/verification guidance, response timelines, required information.
- **Scoring guide**:
  - **0**: No method to act; purely informational.
  - **1**: A method exists but is hard to find or ambiguous (e.g., generic support email, broken/missing link).
  - **2**: Clear method(s) with some guidance; minor friction or missing details.
  - **3**: Multiple clear methods + step-by-step clarity (verification, timelines, exceptions, appeal) and unambiguous labels (e.g., “Do Not Sell or Share My Personal Information”).

### 3) accuracy_score (0–3)
Evaluate legal/terminology correctness and internal consistency.
- **What to look for**: Correct use of terms (e.g., “sale” includes broad consideration; “share” for targeted advertising; “verified consumer”; “sensitive personal information” limits; “household”/“personal information” definitions), avoidance of outdated/incorrect claims, correct opt-out signal references (e.g., GPC).
- **Scoring guide**:
  - **0**: Clearly incorrect or misleading legal statements.
  - **1**: Several inaccuracies or outdated framing; ambiguous on critical terms.
  - **2**: Mostly accurate with minor issues/ambiguities.
  - **3**: Accurate, current, and aligned with CCPA/CPRA language; exceptions and scope are described correctly.

### policy_contradiction (boolean)
True if the text contains **self-contradictory** or suspicious claims (e.g., says “we do not sell data” but also describes selling, or claims to honor GPC while simultaneously stating universal signals are ignored).

### disclosure_map (booleans)
Mark **true** only if the policy explicitly discloses the item.
- **data_collected**: Lists categories collected (e.g., identifiers, geolocation, internet activity).
- **data_shared**: States data is shared with third parties (including for targeted advertising/cross-context behavioral advertising).
- **purpose_of_collection**: Explains why data is collected (e.g., service provision, security, marketing, analytics).
- **retention_period**: States a duration/criteria for retention.
- **right_to_access**: Explains the right to know/access and how to exercise it.
- **right_to_delete**: Explains the right to delete and how to exercise it; may list exceptions.
- **opt_out**: Provides a mechanism to opt out of **sale** or **share/targeted advertising** (e.g., DNSMPI page, “Your Privacy Choices” link, opt-out signal).

## Section C — Semantic Behavior Claims (True/False/"unspecified")
Use **true/false** ONLY if the policy explicitly supports the claim. Otherwise use **"unspecified"**.
Provide a short justification for each item.

- **honors_gpc**: States it honors the **Global Privacy Control** (GPC) or similar universal opt-out signals.
- **respects_dnt**: States it honors **Do Not Track (DNT)** signals (rare; many policies reject DNT explicitly).
- **sets_cookies_before_consent**: Indicates non-essential cookies/trackers are set **before** consent or without consent where the policy requires it.
- **sets_cookies_after_rejecting_consent**: Indicates trackers still fire **after** a user rejects/opts-out (e.g., cookie banner behavior described).
- **deletes_cookies_on_rejection**: States cookies are removed/disabled upon rejection or opt-out.
- **uses_tracking_only_after_consent**: States tracking begins only **after** affirmative consent.
- **sells_data**: States that the company **sells** personal information (including broad “valuable consideration”).
- **shares_with_third_parties**: States data is **shared** with third parties (e.g., targeted advertising, analytics, affiliates).

### Justifications
For each behavioral flag, provide **1–2 sentences** quoting/paraphrasing the relevant policy line(s). If the policy doesn’t say, set the flag to **"unspecified"** and the justification to **"unspecified"**.

## Important Rules
- Base everything **only** on the provided policy text.
- Do **not** assume. If unclear or absent, use **false** (for disclosure_map booleans) or **"unspecified"** (for behavioral flags).
- Keep summaries concise, specific, and neutral.
- Return **only** the JSON object—no prose before or after.

================== BEGIN POLICY TEXT ==================
{policy_text}
=================== END POLICY TEXT ===================
""".strip()

def main():
    pairs = iter_policy_files(MAX_SAMPLES)
    if not pairs:
        raise RuntimeError(f"No .txt files found in {POLICIES_DIR}")

    sizes = []
    for _, path in pairs:
        txt = load_text(path)
        sizes.append((path.name, len(txt), est_tokens_from_chars(txt)))
    print("Size report for selected files (name | chars | ~tokens):")
    for name, chars, toks in sizes:
        print(f"- {name} | {chars} | ~{toks}")

    results: Dict[str, Any] = {}
    print(f"\n🔍 Running trial on {len(pairs)} policies (no chunking)...\n")

    for domain, path in tqdm(pairs):
        txt = load_text(path)
        if not txt.strip():
            results[domain] = {"error": "Empty policy file"}

            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            continue

        approx_tokens = est_tokens_from_chars(txt)
        if approx_tokens > ESTIMATED_TOKEN_SOFT_CAP:
            results[domain] = {
                "error": f"Estimated tokens {approx_tokens} exceed soft cap {ESTIMATED_TOKEN_SOFT_CAP}. "
                         f"Skipped to honor no-chunking."
            }
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            continue

        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_OUTPUT_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "Return only one strict JSON object. No commentary."},
                    {"role": "user", "content": build_prompt(domain, txt)},
                ],
            )
            content = resp.choices[0].message.content or ""
            data = json.loads(content)
            results[domain] = data
        except Exception as e:
            results[domain] = {"error": str(e)}

        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Done. Trial results saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import glob
import json
import re
from pathlib import Path
from typing import Dict, Any, List, Tuple

from openai import OpenAI
from tqdm import tqdm

POLICIES_DIR = "policies_text"
OUTPUT_FILE  = "ccpa_policy_audit_all.json"
MODEL        = "gpt-4o"
TEMPERATURE  = 0
MAX_OUTPUT_TOKENS = 1500

ESTIMATED_TOKEN_SOFT_CAP = 120_000

API_KEY = "REDACTED-SET-OPENAI_API_KEY-ENV-VAR"
client = OpenAI(api_key=API_KEY)

def iter_policy_files() -> List[Tuple[str, Path]]:
    """
    Returns (domain, path) pairs from POLICIES_DIR/*.txt.
    Assumes names like 'www_example_com__combined.txt'.
    """
    files = sorted(glob.glob(os.path.join(POLICIES_DIR, "*.txt")))
    pairs = []
    for p in files:
        name = Path(p).stem
        domain = name.split("__combined", 1)[0].replace("_", ".")
        pairs.append((domain, Path(p)))
    return pairs

def load_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

def est_tokens_from_chars(s: str) -> int:

    return max(1, len(s) // 4)

def build_prompt(domain: str, policy_text: str) -> str:
    """
    Returns a detailed instruction prompt for the LLM with clear scoring rubrics,
    definition of each metric/flag, and a strict JSON schema to follow.
    Also instructs to always provide non-empty justifications.
    """
    return f"""
You are a privacy compliance analyst evaluating a privacy policy strictly under the California Consumer Privacy Act (CCPA) and related U.S. state privacy concepts (CPRA updates included). You must rely ONLY on the policy text provided below—no external knowledge or assumptions.

Analyze the policy for: {domain}

# Output Requirements
Return **one** JSON object matching this exact schema and types:
{{
  "online_data_practices": "string (brief summary or 'Not mentioned.')",
  "offline_data_practices": "string (brief summary or 'Not mentioned.')",
  "rubric_assessment": {{
    "completeness_score": 0-3,
    "completeness_reason": "string",
    "usability_score": 0-3,
    "usability_reason": "string",
    "accuracy_score": 0-3,
    "accuracy_reason": "string",
    "policy_contradiction": true/false,
    "disclosure_map": {{
      "data_collected": true/false,
      "data_shared": true/false,
      "purpose_of_collection": true/false,
      "retention_period": true/false,
      "right_to_access": true/false,
      "right_to_delete": true/false,
      "opt_out": true/false
    }}
  }},
  "behavioral_claims": {{
    "honors_gpc": true/false/"unspecified",
    "respects_dnt": true/false/"unspecified",
    "sets_cookies_before_consent": true/false/"unspecified",
    "sets_cookies_after_rejecting_consent": true/false/"unspecified",
    "deletes_cookies_on_rejection": true/false/"unspecified",
    "uses_tracking_only_after_consent": true/false/"unspecified",
    "sells_data": true/false/"unspecified",
    "shares_with_third_parties": true/false/"unspecified",
    "justifications": {{
      "honors_gpc": "1–2 sentences. NEVER 'unspecified'. If flag is 'unspecified', explicitly say 'No explicit statement found; flag set to unspecified.' and cite the nearest relevant line(s).",
      "respects_dnt": " ... ",
      "sets_cookies_before_consent": " ... ",
      "sets_cookies_after_rejecting_consent": " ... ",
      "deletes_cookies_on_rejection": " ... ",
      "uses_tracking_only_after_consent": " ... ",
      "sells_data": " ... ",
      "shares_with_third_parties": " ... "
    }}
  }}
}}

## Section A — Policy Scope Categorization
- **online_data_practices**: Briefly summarize policy text on web/app tracking and collection (e.g., cookies, pixels, SDKs, analytics, ad tech, web forms, account signup). If not present, return "Not mentioned."
- **offline_data_practices**: Briefly summarize policy text on in-store collection, call centers, paper forms, events, or other in-person channels. If not present, return "Not mentioned."

## Section B — Policy Quality Assessment (Scored 0–3)
Score using only what the policy explicitly states.

### 1) completeness_score (0–3)
Assess whether the policy **mentions** and **explains** consumer rights central to CCPA and how to exercise them.
- **What to look for**: Access/know; deletion; opt-out of sale/share/targeted advertising; (nice-to-have but not required: correction, portability, non-discrimination, appeal).
- **Scoring guide**:
  - **0**: No meaningful mention of rights or how to exercise them.
  - **1**: Vague rights mention, minimal specifics, unclear processes.
  - **2**: Rights identified with at least some concrete instructions (e.g., link/email/form/phone), but gaps remain (e.g., missing one core right or unclear verification).
  - **3**: Rights clearly listed (access/know, delete, opt-out of sale/share/targeted advertising), with actionable mechanisms (webform/email/phone link) and basics on verification, timelines, and scope.

### 2) usability_score (0–3)
Evaluate how **actionable** the mechanisms are.
- **What to look for**: Dedicated links/buttons, DSAR portals, emails, phone numbers, clear steps, eligibility/verification guidance, response timelines, required information.
- **Scoring guide**:
  - **0**: No method to act; purely informational.
  - **1**: A method exists but is hard to find or ambiguous (e.g., generic support email, broken/missing link).
  - **2**: Clear method(s) with some guidance; minor friction or missing details.
  - **3**: Multiple clear methods + step-by-step clarity (verification, timelines, exceptions, appeal) and unambiguous labels (e.g., “Do Not Sell or Share My Personal Information”).

### 3) accuracy_score (0–3)
Evaluate legal/terminology correctness and internal consistency.
- **What to look for**: Correct use of terms (e.g., “sale” includes broad consideration; “share” for targeted advertising; “verified consumer”; “sensitive personal information” limits; “household”/“personal information” definitions), avoidance of outdated/incorrect claims, correct opt-out signal references (e.g., GPC).
- **Scoring guide**:
  - **0**: Clearly incorrect or misleading legal statements.
  - **1**: Several inaccuracies or outdated framing; ambiguous on critical terms.
  - **2**: Mostly accurate with minor issues/ambiguities.
  - **3**: Accurate, current, and aligned with CCPA/CPRA language; exceptions and scope are described correctly.

### policy_contradiction (boolean)
True if the text contains **self-contradictory** or suspicious claims (e.g., says “we do not sell data” but also describes selling, or claims to honor GPC while simultaneously stating universal signals are ignored).

### disclosure_map (booleans)
Mark **true** only if the policy explicitly discloses the item.
- **data_collected**: Lists categories collected (e.g., identifiers, geolocation, internet activity).
- **data_shared**: States data is shared with third parties (including for targeted advertising/cross-context behavioral advertising).
- **purpose_of_collection**: Explains why data is collected (e.g., service provision, security, marketing, analytics).
- **retention_period**: States a duration/criteria for retention.
- **right_to_access**: Explains the right to know/access and how to exercise it.
- **right_to_delete**: Explains the right to delete and how to exercise it; may list exceptions.
- **opt_out**: Provides a mechanism to opt out of **sale** or **share/targeted advertising** (e.g., DNSMPI page, “Your Privacy Choices” link, opt-out signal).

## Section C — Semantic Behavior Claims (True/False/"unspecified")
Use **true/false** ONLY if the policy explicitly supports the claim. Otherwise use **"unspecified"**.
For **every** behavioral flag, provide a **non-empty justification**:
- If explicit: quote/paraphrase the line(s) and cite the section title if available.
- If not explicit: write, “No explicit statement found; flag set to unspecified.” and include the nearest relevant line(s) that discuss cookies/tracking/opt-out signals/third-party sharing, if any.

## Important Rules
- Base everything **only** on the provided policy text.
- Do **not** assume. If unclear or absent, use **false** (for disclosure_map booleans) or **'unspecified'** (for behavioral flags).
- Keep summaries concise, specific, and neutral.
- Return **only** the JSON object—no prose before or after.

================== BEGIN POLICY TEXT ==================
{policy_text}
=================== END POLICY TEXT ===================
""".strip()

RUBRIC_KEYS = [
    "data_collected", "data_shared", "purpose_of_collection", "retention_period",
    "right_to_access", "right_to_delete", "opt_out"
]
BEHAVIOR_KEYS = [
    "honors_gpc", "respects_dnt", "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent", "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent", "sells_data", "shares_with_third_parties"
]

KEYWORDS_MAP = {
    "honors_gpc": ["global privacy control", "gpc", "universal opt-out", "opt-out signal", "ccpa signal"],
    "respects_dnt": ["do not track", "dnt"],
    "sets_cookies_before_consent": ["before consent", "prior to consent", "without consent", "implied consent"],
    "sets_cookies_after_rejecting_consent": ["after rejecting", "after refusal", "even if you reject", "decline"],
    "deletes_cookies_on_rejection": ["delete cookies", "remove cookies", "disable cookies", "withdraw consent"],
    "uses_tracking_only_after_consent": ["only after consent", "after you consent", "upon consent"],
    "sells_data": ["sell", "sold", "sale of personal information", "sale of personal data"],
    "shares_with_third_parties": ["share", "shared", "third parties", "cross-context behavioral advertising", "targeted advertising"]
}

def _tri(v):

    if isinstance(v, bool):
        return True if v else False
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}: return True
    if s in {"false", "no", "0"}: return False
    return "unspecified"

def _bool(v):
    if isinstance(v, bool): return v
    s = str(v).strip().lower()
    return s in {"true", "yes", "1"}

def _text(v, default="Not mentioned."):
    return v.strip() if isinstance(v, str) and v.strip() else default

def _nearest_snippet(policy_text: str, keywords: List[str], window_chars: int = 280) -> str:
    """Return a short context window around the first keyword hit, if any."""
    text = policy_text or ""
    low = text.lower()
    for kw in keywords:
        i = low.find(kw)
        if i != -1:
            start = max(0, i - window_chars // 2)
            end = min(len(text), i + len(kw) + window_chars // 2)
            snippet = text[start:end]

            snippet = re.sub(r"\s+", " ", snippet).strip()
            return snippet
    return ""

def normalize_result(raw: dict, policy_text: str) -> dict:

    out = {
        "online_data_practices": _text(raw.get("online_data_practices", "")),
        "offline_data_practices": _text(raw.get("offline_data_practices", "")),
        "rubric_assessment": {
            "completeness_score": int(raw.get("rubric_assessment", {}).get("completeness_score", 0) or 0),
            "completeness_reason": str(raw.get("rubric_assessment", {}).get("completeness_reason", ""))[:2000],
            "usability_score": int(raw.get("rubric_assessment", {}).get("usability_score", 0) or 0),
            "usability_reason": str(raw.get("rubric_assessment", {}).get("usability_reason", ""))[:2000],
            "accuracy_score": int(raw.get("rubric_assessment", {}).get("accuracy_score", 0) or 0),
            "accuracy_reason": str(raw.get("rubric_assessment", {}).get("accuracy_reason", ""))[:2000],
            "policy_contradiction": _bool(raw.get("rubric_assessment", {}).get("policy_contradiction", False)),
            "disclosure_map": {},
        },
        "behavioral_claims": {
            k: "unspecified" for k in BEHAVIOR_KEYS
        }
    }

    dm = raw.get("rubric_assessment", {}).get("disclosure_map", {}) or {}
    if "right_to_acces" in dm and "right_to_access" not in dm:
        dm["right_to_access"] = dm.pop("right_to_acces")

    for k in RUBRIC_KEYS:
        out["rubric_assessment"]["disclosure_map"][k] = _bool(dm.get(k, False))

    bc = raw.get("behavioral_claims", {}) or {}
    just = bc.get("justifications", {}) or {}

    for k in BEHAVIOR_KEYS:
        out["behavioral_claims"][k] = _tri(bc.get(k, "unspecified"))

    norm_just = {}
    for k in BEHAVIOR_KEYS:
        j = just.get(k, "")
        j_clean = j.strip() if isinstance(j, str) else ""
        if not j_clean or j_clean.lower() == "unspecified":

            snippet = _nearest_snippet(policy_text, KEYWORDS_MAP.get(k, []))
            if snippet:
                norm_just[k] = f"No explicit statement found; flag set to 'unspecified'. Nearest relevant excerpt: \"{snippet}\""
            else:
                norm_just[k] = "No explicit statement found; flag set to 'unspecified'. Policy does not discuss this item in clear terms."
        else:
            norm_just[k] = j_clean[:1000]
    out["behavioral_claims"]["justifications"] = norm_just

    return out

def main():
    pairs = iter_policy_files()
    if not pairs:
        raise RuntimeError(f"No .txt files found in {POLICIES_DIR}")

    sizes = []
    for _, path in pairs:
        txt = load_text(path)
        sizes.append((path.name, len(txt), est_tokens_from_chars(txt)))
    print("Size report for selected files (name | chars | ~tokens):")
    for name, chars, toks in sizes[:20]:
        print(f"- {name} | {chars} | ~{toks}")
    print(f"... ({len(sizes)} total files)\n")

    results: Dict[str, Any] = {}
    print(f"🔍 Running on {len(pairs)} policies (no chunking)...\n")

    for domain, path in tqdm(pairs):
        txt = load_text(path)
        if not txt.strip():
            results[domain] = {"error": "Empty policy file"}

            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            continue

        approx_tokens = est_tokens_from_chars(txt)
        if approx_tokens > ESTIMATED_TOKEN_SOFT_CAP:
            results[domain] = {
                "error": f"Estimated tokens {approx_tokens} exceed soft cap {ESTIMATED_TOKEN_SOFT_CAP}. "
                         f"Skipped to honor no-chunking."
            }
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            continue

        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_OUTPUT_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "Return only one strict JSON object. No commentary."},
                    {"role": "user", "content": build_prompt(domain, txt)},
                ],
            )
            content = resp.choices[0].message.content or ""
            raw = json.loads(content)
            clean = normalize_result(raw, txt)
            results[domain] = clean
        except Exception as e:
            results[domain] = {"error": str(e)}

        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Done. Results saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import glob
import json
from pathlib import Path

POLICIES_DIR = "policies_text"
OUTPUT_FILE  = "ccpa_policy_audit_all.json"

def iter_policy_domains():
    """Return domain names inferred from policy filenames."""
    files = sorted(glob.glob(os.path.join(POLICIES_DIR, "*.txt")))
    domains = []
    for p in files:
        name = Path(p).stem
        domain = name.split("__combined", 1)[0].replace("_", ".")
        domains.append(domain)
    return set(domains)

def main():

    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            existing = set(json.load(f).keys())
    else:
        existing = set()

    all_domains = iter_policy_domains()
    new_domains = all_domains - existing

    print(f"Total policy files: {len(all_domains)}")
    print(f"Already processed: {len(existing)}")
    print(f"New (unprocessed): {len(new_domains)}\n")

    if new_domains:
        print("New domains:")
        for d in sorted(new_domains):
            print("-", d)
    else:
        print("✅ No new domains. All files are already processed.")

if __name__ == "__main__":
    main()

In [ ]:
import os
import glob
import json
import re
import time
from pathlib import Path
from typing import Dict, Any, List, Tuple

from openai import OpenAI
from tqdm import tqdm

POLICIES_DIR       = "policies_text"
PREV_OUTPUT_FILE   = "ccpa_policy_audit_all.json"
OUTPUT_FILE        = PREV_OUTPUT_FILE
MODEL              = "gpt-4o"
TEMPERATURE        = 0
MAX_OUTPUT_TOKENS  = 1500
ESTIMATED_TOKEN_SOFT_CAP = 120_000

RATE_LIMIT_DELAY_S = 1.5
BACKOFF_BASE_S     = 4
BACKOFF_MAX_S      = 60

API_KEY = "REDACTED-SET-OPENAI_API_KEY-ENV-VAR"
client = OpenAI(api_key=API_KEY)

def iter_policy_files_map() -> Dict[str, Path]:
    """
    Returns a dict {domain: path} from POLICIES_DIR/*.txt.
    Assumes names like 'www_example_com__combined.txt'.
    """
    files = sorted(glob.glob(os.path.join(POLICIES_DIR, "*.txt")))
    mapping = {}
    for p in files:
        name = Path(p).stem
        domain = name.split("__combined", 1)[0].replace("_", ".")
        mapping[domain] = Path(p)
    return mapping

def load_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

def est_tokens_from_chars(s: str) -> int:
    return max(1, len(s) // 4)

def load_prev_results(path: str) -> Dict[str, Any]:
    if not os.path.isfile(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def is_quota_error(entry: Any) -> bool:
    """
    Detect the insufficient_quota / 429 condition from a stored result entry.
    We check both generic 'error' strings and nested dicts just in case.
    """
    if not isinstance(entry, dict):
        return False
    err = entry.get("error")
    if not err:
        return False
    s = json.dumps(err, ensure_ascii=False).lower()
    return ("insufficient_quota" in s) or ("error code: 429" in s) or ("exceeded your current quota" in s)

def build_prompt(domain: str, policy_text: str) -> str:
    return f"""
You are a privacy compliance analyst evaluating a privacy policy strictly under the California Consumer Privacy Act (CCPA) and related U.S. state privacy concepts (CPRA updates included). You must rely ONLY on the policy text provided below—no external knowledge or assumptions.

Analyze the policy for: {domain}

# Output Requirements
Return **one** JSON object matching this exact schema and types:
{{
  "online_data_practices": "string (brief summary or 'Not mentioned.')",
  "offline_data_practices": "string (brief summary or 'Not mentioned.')",
  "rubric_assessment": {{
    "completeness_score": 0-3,
    "completeness_reason": "string",
    "usability_score": 0-3,
    "usability_reason": "string",
    "accuracy_score": 0-3,
    "accuracy_reason": "string",
    "policy_contradiction": true/false,
    "disclosure_map": {{
      "data_collected": true/false,
      "data_shared": true/false,
      "purpose_of_collection": true/false,
      "retention_period": true/false,
      "right_to_access": true/false,
      "right_to_delete": true/false,
      "opt_out": true/false
    }}
  }},
  "behavioral_claims": {{
    "honors_gpc": true/false/"unspecified",
    "respects_dnt": true/false/"unspecified",
    "sets_cookies_before_consent": true/false/"unspecified",
    "sets_cookies_after_rejecting_consent": true/false/"unspecified",
    "deletes_cookies_on_rejection": true/false/"unspecified",
    "uses_tracking_only_after_consent": true/false/"unspecified",
    "sells_data": true/false/"unspecified",
    "shares_with_third_parties": true/false/"unspecified",
    "justifications": {{
      "honors_gpc": "1–2 sentences. NEVER 'unspecified'. If flag is 'unspecified', explicitly say 'No explicit statement found; flag set to unspecified.' and cite the nearest relevant line(s).",
      "respects_dnt": " ... ",
      "sets_cookies_before_consent": " ... ",
      "sets_cookies_after_rejecting_consent": " ... ",
      "deletes_cookies_on_rejection": " ... ",
      "uses_tracking_only_after_consent": " ... ",
      "sells_data": " ... ",
      "shares_with_third_parties": " ... "
    }}
  }}
}}

## Section A — Policy Scope Categorization
- **online_data_practices**: Briefly summarize policy text on web/app tracking and collection (e.g., cookies, pixels, SDKs, analytics, ad tech, web forms, account signup). If not present, return "Not mentioned."
- **offline_data_practices**: Briefly summarize policy text on in-store collection, call centers, paper forms, events, or other in-person channels. If not present, return "Not mentioned."

## Section B — Policy Quality Assessment (Scored 0–3)
Score using only what the policy explicitly states.

### 1) completeness_score (0–3)
Assess whether the policy **mentions** and **explains** consumer rights central to CCPA and how to exercise them.
- **What to look for**: Access/know; deletion; opt-out of sale/share/targeted advertising; (nice-to-have but not required: correction, portability, non-discrimination, appeal).
- **Scoring guide**:
  - **0**: No meaningful mention of rights or how to exercise them.
  - **1**: Vague rights mention, minimal specifics, unclear processes.
  - **2**: Rights identified with at least some concrete instructions (e.g., link/email/form/phone), but gaps remain (e.g., missing one core right or unclear verification).
  - **3**: Rights clearly listed (access/know, delete, opt-out of sale/share/targeted advertising), with actionable mechanisms (webform/email/phone link) and basics on verification, timelines, and scope.

### 2) usability_score (0–3)
Evaluate how **actionable** the mechanisms are.
- **What to look for**: Dedicated links/buttons, DSAR portals, emails, phone numbers, clear steps, eligibility/verification guidance, response timelines, required information.
- **Scoring guide**:
  - **0**: No method to act; purely informational.
  - **1**: A method exists but is hard to find or ambiguous (e.g., generic support email, broken/missing link).
  - **2**: Clear method(s) with some guidance; minor friction or missing details.
  - **3**: Multiple clear methods + step-by-step clarity (verification, timelines, exceptions, appeal) and unambiguous labels (e.g., “Do Not Sell or Share My Personal Information”).

### 3) accuracy_score (0–3)
Evaluate legal/terminology correctness and internal consistency.
- **What to look for**: Correct use of terms (e.g., “sale” includes broad consideration; “share” for targeted advertising; “verified consumer”; “sensitive personal information” limits; “household”/“personal information” definitions), avoidance of outdated/incorrect claims, correct opt-out signal references (e.g., GPC).
- **Scoring guide**:
  - **0**: Clearly incorrect or misleading legal statements.
  - **1**: Several inaccuracies or outdated framing; ambiguous on critical terms.
  - **2**: Mostly accurate with minor issues/ambiguities.
  - **3**: Accurate, current, and aligned with CCPA/CPRA language; exceptions and scope are described correctly.

### policy_contradiction (boolean)
True if the text contains **self-contradictory** or suspicious claims (e.g., says “we do not sell data” but also describes selling, or claims to honor GPC while simultaneously stating universal signals are ignored).

### disclosure_map (booleans)
Mark **true** only if the policy explicitly discloses the item.
- **data_collected**: Lists categories collected (e.g., identifiers, geolocation, internet activity).
- **data_shared**: States data is shared with third parties (including for targeted advertising/cross-context behavioral advertising).
- **purpose_of_collection**: Explains why data is collected (e.g., service provision, security, marketing, analytics).
- **retention_period**: States a duration/criteria for retention.
- **right_to_access**: Explains the right to know/access and how to exercise it.
- **right_to_delete**: Explains the right to delete and how to exercise it; may list exceptions.
- **opt_out**: Provides a mechanism to opt out of **sale** or **share/targeted advertising** (e.g., DNSMPI page, “Your Privacy Choices” link, opt-out signal).

## Section C — Semantic Behavior Claims (True/False/"unspecified")
Use **true/false** ONLY if the policy explicitly supports the claim. Otherwise use **"unspecified"**.
For **every** behavioral flag, provide a **non-empty justification**:
- If explicit: quote/paraphrase the line(s) and cite the section title if available.
- If not explicit: write, “No explicit statement found; flag set to unspecified.” and include the nearest relevant line(s) that discuss cookies/tracking/opt-out signals/third-party sharing, if any.

## Important Rules
- Base everything **only** on the provided policy text.
- Do **not** assume. If unclear or absent, use **false** (for disclosure_map booleans) or **'unspecified'** (for behavioral flags).
- Keep summaries concise, specific, and neutral.
- Return **only** the JSON object—no prose before or after.

================== BEGIN POLICY TEXT ==================
{policy_text}
=================== END POLICY TEXT ===================
""".strip()

RUBRIC_KEYS = [
    "data_collected", "data_shared", "purpose_of_collection", "retention_period",
    "right_to_access", "right_to_delete", "opt_out"
]
BEHAVIOR_KEYS = [
    "honors_gpc", "respects_dnt", "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent", "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent", "sells_data", "shares_with_third_parties"
]

KEYWORDS_MAP = {
    "honors_gpc": ["global privacy control", "gpc", "universal opt-out", "opt-out signal", "ccpa signal"],
    "respects_dnt": ["do not track", "dnt"],
    "sets_cookies_before_consent": ["before consent", "prior to consent", "without consent", "implied consent"],
    "sets_cookies_after_rejecting_consent": ["after rejecting", "after refusal", "even if you reject", "decline"],
    "deletes_cookies_on_rejection": ["delete cookies", "remove cookies", "disable cookies", "withdraw consent"],
    "uses_tracking_only_after_consent": ["only after consent", "after you consent", "upon consent"],
    "sells_data": ["sell", "sold", "sale of personal information", "sale of personal data"],
    "shares_with_third_parties": ["share", "shared", "third parties", "cross-context behavioral advertising", "targeted advertising"]
}

def _tri(v):
    if isinstance(v, bool):
        return True if v else False
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}: return True
    if s in {"false", "no", "0"}: return False
    return "unspecified"

def _bool(v):
    if isinstance(v, bool): return v
    s = str(v).strip().lower()
    return s in {"true", "yes", "1"}

def _text(v, default="Not mentioned."):
    return v.strip() if isinstance(v, str) and v.strip() else default

def _nearest_snippet(policy_text: str, keywords: List[str], window_chars: int = 280) -> str:
    text = policy_text or ""
    low = text.lower()
    for kw in keywords:
        i = low.find(kw)
        if i != -1:
            start = max(0, i - window_chars // 2)
            end = min(len(text), i + len(kw) + window_chars // 2)
            snippet = text[start:end]
            snippet = re.sub(r"\s+", " ", snippet).strip()
            return snippet
    return ""

def normalize_result(raw: dict, policy_text: str) -> dict:
    out = {
        "online_data_practices": _text(raw.get("online_data_practices", "")),
        "offline_data_practices": _text(raw.get("offline_data_practices", "")),
        "rubric_assessment": {
            "completeness_score": int(raw.get("rubric_assessment", {}).get("completeness_score", 0) or 0),
            "completeness_reason": str(raw.get("rubric_assessment", {}).get("completeness_reason", ""))[:2000],
            "usability_score": int(raw.get("rubric_assessment", {}).get("usability_score", 0) or 0),
            "usability_reason": str(raw.get("rubric_assessment", {}).get("usability_reason", ""))[:2000],
            "accuracy_score": int(raw.get("rubric_assessment", {}).get("accuracy_score", 0) or 0),
            "accuracy_reason": str(raw.get("rubric_assessment", {}).get("accuracy_reason", ""))[:2000],
            "policy_contradiction": _bool(raw.get("rubric_assessment", {}).get("policy_contradiction", False)),
            "disclosure_map": {},
        },
        "behavioral_claims": {k: "unspecified" for k in BEHAVIOR_KEYS}
    }

    dm = raw.get("rubric_assessment", {}).get("disclosure_map", {}) or {}
    if "right_to_acces" in dm and "right_to_access" not in dm:
        dm["right_to_access"] = dm.pop("right_to_acces")
    for k in RUBRIC_KEYS:
        out["rubric_assessment"]["disclosure_map"][k] = _bool(dm.get(k, False))

    bc = raw.get("behavioral_claims", {}) or {}
    just = bc.get("justifications", {}) or {}
    for k in BEHAVIOR_KEYS:
        out["behavioral_claims"][k] = _tri(bc.get(k, "unspecified"))

    norm_just = {}
    for k in BEHAVIOR_KEYS:
        j = just.get(k, "")
        j_clean = j.strip() if isinstance(j, str) else ""
        if not j_clean or j_clean.lower() == "unspecified":
            snippet = _nearest_snippet(policy_text, KEYWORDS_MAP.get(k, []))
            if snippet:
                norm_just[k] = f"No explicit statement found; flag set to 'unspecified'. Nearest relevant excerpt: \"{snippet}\""
            else:
                norm_just[k] = "No explicit statement found; flag set to 'unspecified'. Policy does not discuss this item in clear terms."
        else:
            norm_just[k] = j_clean[:1000]
    out["behavioral_claims"]["justifications"] = norm_just
    return out

def call_model(domain: str, txt: str) -> Dict[str, Any]:
    """
    Calls the model with controlled pacing and simple exponential backoff on errors.
    """
    delay = RATE_LIMIT_DELAY_S
    backoff = BACKOFF_BASE_S
    while True:
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_OUTPUT_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "Return only one strict JSON object. No commentary."},
                    {"role": "user", "content": build_prompt(domain, txt)},
                ],
            )
            content = resp.choices[0].message.content or ""
            return json.loads(content)
        except Exception as e:
            msg = str(e).lower()

            if "insufficient_quota" in msg or "exceeded your current quota" in msg:
                raise

            time.sleep(backoff)
            backoff = min(int(backoff * 2), BACKOFF_MAX_S)

def main():
    prev = load_prev_results(PREV_OUTPUT_FILE)
    files_map = iter_policy_files_map()

    pending = [d for d, entry in prev.items() if is_quota_error(entry)]
    print(f"Found {len(pending)} domains with insufficient_quota to retry.\n")

    if not pending:
        print("Nothing to rerun. Exiting.")
        return

    results = dict(prev)

    for domain in tqdm(pending):
        path = files_map.get(domain)
        if not path:
            results[domain] = {"error": "Policy file not found for rerun."}
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            continue

        txt = load_text(path)
        if not txt.strip():
            results[domain] = {"error": "Empty policy file on rerun."}
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            continue

        approx_tokens = est_tokens_from_chars(txt)
        if approx_tokens > ESTIMATED_TOKEN_SOFT_CAP:
            results[domain] = {"error": f"Estimated tokens {approx_tokens} exceed soft cap {ESTIMATED_TOKEN_SOFT_CAP}. Skipped to honor no-chunking."}
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            continue

        try:
            raw = call_model(domain, txt)
            clean = normalize_result(raw, txt)
            results[domain] = clean
            time.sleep(RATE_LIMIT_DELAY_S)
        except Exception as e:
            results[domain] = {"error": f"Rerun failed: {e}"}

        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Rerun complete. Results merged into: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import json
from pathlib import Path
from typing import Dict, Any, List

import pandas as pd
import numpy as np

JSON_PATH = "ccpa_policy_audit_all_clean.json"
POLICY_TEXT_DIR = "policies_text"
OUT_DIR = "analysis_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

RUBRIC_SCORE_COLS = ["completeness_score", "usability_score", "accuracy_score"]
DISCLOSURE_KEYS = [
    "data_collected", "data_shared", "purpose_of_collection",
    "retention_period", "right_to_access", "right_to_delete", "opt_out"
]
BEHAVIOR_KEYS = [
    "honors_gpc", "respects_dnt", "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent", "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent", "sells_data", "shares_with_third_parties"
]

def _tri(v) -> str:
    """Normalize to 'true'/'false'/'unspecified'."""
    if isinstance(v, bool):
        return "true" if v else "false"
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}:
        return "true"
    if s in {"false", "no", "0"}:
        return "false"
    return "unspecified"

def _bool(v) -> bool:
    """Normalize disclosure booleans to True/False."""
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    return s in {"true", "yes", "1"}

def maybe_policy_lengths(policy_dir: str, domains: pd.Index) -> pd.Series:
    """
    If you have policies_text/*.txt, compute character length per domain
    (filenames like: example_com__combined.txt -> domain 'example.com').
    """
    p = Path(policy_dir)
    if not p.is_dir():
        return pd.Series(index=domains, dtype="float64")

    lengths = {}
    for txt in p.glob("*.txt"):
        name = txt.stem
        base = name.split("__combined", 1)[0]
        domain = base.replace("_", ".")
        try:
            chars = txt.read_text(encoding="utf-8", errors="ignore")
            lengths[domain] = len(chars)
        except Exception:
            pass

    return pd.Series(lengths, name="policy_chars").reindex(domains)

def load_results(json_path: str) -> pd.DataFrame:
    """
    Load domain→result mapping and flatten nested structures robustly.
    This avoids NaN inflation that happens when json_normalize is used on Series incorrectly.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    df_raw = pd.DataFrame.from_dict(data, orient="index")

    if "rubric_assessment" in df_raw.columns:
        rub = (
            df_raw["rubric_assessment"]
            .apply(lambda x: x if isinstance(x, dict) else {})
            .apply(pd.Series)
        )

        disc = (
            rub.get("disclosure_map", pd.Series(index=rub.index))
            .apply(lambda x: x if isinstance(x, dict) else {})
            .apply(pd.Series)
        )
        rub = rub.drop(columns=["disclosure_map"], errors="ignore")
    else:
        rub = pd.DataFrame(index=df_raw.index)
        disc = pd.DataFrame(index=df_raw.index)

    if "behavioral_claims" in df_raw.columns:
        beh = (
            df_raw["behavioral_claims"]
            .apply(lambda x: x if isinstance(x, dict) else {})
            .apply(pd.Series)
        )
    else:
        beh = pd.DataFrame(index=df_raw.index)

    summaries = pd.DataFrame(index=df_raw.index)
    for col in ["online_data_practices", "offline_data_practices"]:
        if col in df_raw.columns:
            summaries[col] = df_raw[col]

    for c in RUBRIC_SCORE_COLS:
        if c in rub.columns:
            rub[c] = pd.to_numeric(rub[c], errors="coerce")

    for k in DISCLOSURE_KEYS:
        if k in disc.columns:
            disc[k] = disc[k].apply(_bool)

    just_cols = pd.DataFrame(index=beh.index)
    if "justifications" in beh.columns:
        just_cols = (
            beh["justifications"]
            .apply(lambda x: x if isinstance(x, dict) else {})
            .apply(pd.Series)
        )
        beh = beh.drop(columns=["justifications"], errors="ignore")

    for k in BEHAVIOR_KEYS:
        if k in beh.columns:
            beh[k] = beh[k].apply(_tri)

    disc = disc.add_prefix("disc.")
    beh = beh.add_prefix("beh.")
    just_cols = just_cols.add_prefix("just.")

    out = pd.concat([summaries, rub, disc, beh, just_cols], axis=1)
    out.index.name = "domain"
    return out

def section_dataset_overview(df: pd.DataFrame) -> None:
    print("\n=== 1) Dataset Overview ===")
    print(f"Total websites: {len(df)}")
    present = df.count().sort_values(ascending=False)
    print("\nNon-null counts by field (top 20):")
    print(present.head(20).to_string())
    present.to_csv(os.path.join(OUT_DIR, "field_nonnull_counts.csv"))

def section_rubric_distributions(df: pd.DataFrame) -> None:
    print("\n=== 2) Rubric Scores: Distributions & Summary ===")
    scores = df[RUBRIC_SCORE_COLS].dropna(how="all")
    desc = scores.describe()
    print("\nSummary (describe):")
    print(desc.to_string())
    desc.to_csv(os.path.join(OUT_DIR, "rubric_scores_describe.csv"))

    for col in RUBRIC_SCORE_COLS:
        vc = scores[col].value_counts().sort_index()
        print(f"\nDistribution for {col}:")
        print(vc.to_string())
        vc.to_csv(os.path.join(OUT_DIR, f"dist_{col}.csv"))

def section_disclosure_coverage(df: pd.DataFrame) -> None:
    print("\n=== 3) Disclosure Coverage ===")
    cols = [f"disc.{k}" for k in DISCLOSURE_KEYS if f"disc.{k}" in df.columns]
    sub = df[cols].astype(float)
    coverage = (sub.mean(numeric_only=True) * 100).round(1).sort_values(ascending=False)
    print("\n% of sites explicitly disclosing each item:")
    print(coverage.to_string())
    coverage.to_csv(os.path.join(OUT_DIR, "disclosure_coverage_percent.csv"))

def section_behavioral_prevalence(df: pd.DataFrame) -> None:
    print("\n=== 4) Behavioral Declarations Prevalence (Policy-Level) ===")
    for k in BEHAVIOR_KEYS:
        col = f"beh.{k}"
        if col not in df.columns:
            continue
        vc = df[col].value_counts(dropna=False)
        total = vc.sum()
        pct = (vc / total * 100).round(1)
        table = pd.DataFrame({"count": vc, "percent": pct})
        print(f"\n{k}:")
        print(table.to_string())
        table.to_csv(os.path.join(OUT_DIR, f"behavior_{k}_counts.csv"))

def section_transparency_index(df: pd.DataFrame) -> None:
    print("\n=== 5) Transparency/Opacity Index ===")
    beh_cols = [f"beh.{k}" for k in BEHAVIOR_KEYS if f"beh.{k}" in df.columns]
    if not beh_cols:
        print("No behavioral columns found.")
        return
    unspecified_by_domain = (df[beh_cols] == "unspecified").mean(axis=1)
    print(f"Mean opacity (fraction of 'unspecified' across behaviors): {unspecified_by_domain.mean():.3f}")
    print("Opacity quantiles:")
    print(unspecified_by_domain.quantile([0, .25, .5, .75, 1]).round(3).to_string())
    unspecified_by_domain.rename("opacity_fraction").to_csv(
        os.path.join(OUT_DIR, "domain_opacity_fraction.csv")
    )

    per_field_unspec = (df[beh_cols] == "unspecified").mean().sort_values(ascending=False)
    print("\nUnspecified rates by field:")
    print(per_field_unspec.round(3).to_string())
    per_field_unspec.to_csv(os.path.join(OUT_DIR, "behavior_unspecified_rates.csv"))

def section_score_correlations(df: pd.DataFrame) -> None:
    print("\n=== 6) Score Correlations ===")
    scores = df[RUBRIC_SCORE_COLS].dropna()
    if scores.empty:
        print("No score data.")
        return
    pearson = scores.corr(method="pearson").round(3)
    spearman = scores.corr(method="spearman").round(3)
    print("\nPearson correlation matrix:")
    print(pearson.to_string())
    print("\nSpearman correlation matrix:")
    print(spearman.to_string())
    pearson.to_csv(os.path.join(OUT_DIR, "corr_scores_pearson.csv"))
    spearman.to_csv(os.path.join(OUT_DIR, "corr_scores_spearman.csv"))

def section_exemplars(df: pd.DataFrame) -> None:
    print("\n=== 7) High/Low Exemplars ===")

    lengths = maybe_policy_lengths(POLICY_TEXT_DIR, df.index)
    if lengths.notna().any():
        df2 = df.copy()
        df2["policy_chars"] = lengths
        print("\nPolicy char-length statistics (if available):")
        print(df2["policy_chars"].describe().round(1).to_string())
        corr = df2[RUBRIC_SCORE_COLS + ["policy_chars"]].corr(method="spearman").round(3)
        print("\nSpearman corr with policy length:")
        print(corr.to_string())
        corr.to_csv(os.path.join(OUT_DIR, "corr_scores_with_length_spearman.csv"))

    for col in RUBRIC_SCORE_COLS:
        if col not in df.columns:
            continue
        s = df[col].dropna()
        if s.empty:
            continue
        top = s.sort_values(ascending=False).head(20)
        bot = s.sort_values(ascending=True).head(20)
        print(f"\nTop 20 by {col}:")
        print(top.to_string())
        print(f"\nBottom 20 by {col}:")
        print(bot.to_string())
        top.to_csv(os.path.join(OUT_DIR, f"top20_{col}.csv"))
        bot.to_csv(os.path.join(OUT_DIR, f"bottom20_{col}.csv"))

df = load_results(JSON_PATH)

print("Sample row (first domain):")
print(df.iloc[0].dropna().to_string())

section_dataset_overview(df)
section_rubric_distributions(df)
section_disclosure_coverage(df)
section_behavioral_prevalence(df)
section_transparency_index(df)
section_score_correlations(df)
section_exemplars(df)

print("\n[✓] Analysis complete. CSVs saved to:", os.path.abspath(OUT_DIR))

In [ ]:
import os
import json
import math
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_JSON  = "ccpa_policy_audit_all_clean.json"
OUTPUT_DIR  = "paper_figs"
TABLES_DIR  = "paper_tables"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for domain, res in data.items():
    if not isinstance(res, dict):
        continue

    if not all(k in res for k in ["online_data_practices","offline_data_practices","rubric_assessment","behavioral_claims"]):
        continue

    ra = res.get("rubric_assessment", {}) or {}
    dm = ra.get("disclosure_map", {}) or {}
    bc = res.get("behavioral_claims", {}) or {}
    just = bc.get("justifications", {}) or {}

    row = {
        "domain": domain,
        "online_data_practices": res.get("online_data_practices", ""),
        "offline_data_practices": res.get("offline_data_practices", ""),
        "completeness_score": ra.get("completeness_score", np.nan),
        "completeness_reason": ra.get("completeness_reason", ""),
        "usability_score": ra.get("usability_score", np.nan),
        "usability_reason": ra.get("usability_reason", ""),
        "accuracy_score": ra.get("accuracy_score", np.nan),
        "accuracy_reason": ra.get("accuracy_reason", ""),
        "policy_contradiction": ra.get("policy_contradiction", False),

        "policy_chars": res.get("policy_chars", np.nan),

        "disc.data_collected": bool(dm.get("data_collected", False)),
        "disc.data_shared": bool(dm.get("data_shared", False)),
        "disc.purpose_of_collection": bool(dm.get("purpose_of_collection", False)),
        "disc.retention_period": bool(dm.get("retention_period", False)),
        "disc.right_to_access": bool(dm.get("right_to_access", False)),
        "disc.right_to_delete": bool(dm.get("right_to_delete", False)),
        "disc.opt_out": bool(dm.get("opt_out", False)),

        "beh.honors_gpc": str(bc.get("honors_gpc", "unspecified")),
        "beh.respects_dnt": str(bc.get("respects_dnt", "unspecified")),
        "beh.sets_cookies_before_consent": str(bc.get("sets_cookies_before_consent", "unspecified")),
        "beh.sets_cookies_after_rejecting_consent": str(bc.get("sets_cookies_after_rejecting_consent", "unspecified")),
        "beh.deletes_cookies_on_rejection": str(bc.get("deletes_cookies_on_rejection", "unspecified")),
        "beh.uses_tracking_only_after_consent": str(bc.get("uses_tracking_only_after_consent", "unspecified")),
        "beh.sells_data": str(bc.get("sells_data", "unspecified")),
        "beh.shares_with_third_parties": str(bc.get("shares_with_third_parties", "unspecified")),

        "just.honors_gpc": just.get("honors_gpc", ""),
        "just.respects_dnt": just.get("respects_dnt", ""),
        "just.sets_cookies_before_consent": just.get("sets_cookies_before_consent", ""),
        "just.sets_cookies_after_rejecting_consent": just.get("sets_cookies_after_rejecting_consent", ""),
        "just.deletes_cookies_on_rejection": just.get("deletes_cookies_on_rejection", ""),
        "just.uses_tracking_only_after_consent": just.get("uses_tracking_only_after_consent", ""),
        "just.sells_data": just.get("sells_data", ""),
        "just.shares_with_third_parties": just.get("shares_with_third_parties", ""),
    }
    rows.append(row)

df = pd.DataFrame(rows)
df_scores = df[["completeness_score","usability_score","accuracy_score"]].apply(pd.to_numeric, errors="coerce")

def plot_score_dist(series: pd.Series, title: str, fname: str):
    counts = series.value_counts(dropna=False).reindex([0,1,2,3], fill_value=0)
    plt.figure()
    counts.plot(kind="bar")
    plt.title(title)
    plt.xlabel("Score")
    plt.ylabel("Count")
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, fname)
    plt.savefig(out, dpi=300)
    plt.close()
    return out

fig1 = plot_score_dist(df["completeness_score"], "Completeness Score Distribution", "fig_scores_completeness.png")
fig2 = plot_score_dist(df["usability_score"],    "Usability Score Distribution",    "fig_scores_usability.png")
fig3 = plot_score_dist(df["accuracy_score"],     "Accuracy Score Distribution",     "fig_scores_accuracy.png")

disc_cols = [
    "disc.data_collected", "disc.data_shared", "disc.purpose_of_collection",
    "disc.retention_period", "disc.right_to_access", "disc.right_to_delete", "disc.opt_out"
]
disc_pct = (df[disc_cols].mean() * 100).round(1).sort_values(ascending=False)
disc_pct.to_csv(os.path.join(TABLES_DIR, "table_disclosure_prevalence.csv"))

plt.figure()
disc_pct.plot(kind="barh")
plt.gca().invert_yaxis()
plt.xlabel("Percent of Sites (%)")
plt.title("Disclosure Prevalence")
plt.tight_layout()
fig4 = os.path.join(OUTPUT_DIR, "fig_disclosure_prevalence.png")
plt.savefig(fig4, dpi=300)
plt.close()

beh_cols = [
    "beh.honors_gpc","beh.respects_dnt","beh.sets_cookies_before_consent",
    "beh.sets_cookies_after_rejecting_consent","beh.deletes_cookies_on_rejection",
    "beh.uses_tracking_only_after_consent","beh.sells_data","beh.shares_with_third_parties"
]

def norm_tristate(x):
    s = str(x).strip().lower()
    if s in {"true","false","unspecified"}:
        return s
    return "unspecified"
beh_norm = df[beh_cols].applymap(norm_tristate)

stack = {}
for col in beh_cols:
    counts = beh_norm[col].value_counts()
    stack[col] = {
        "true": counts.get("true", 0),
        "false": counts.get("false", 0),
        "unspecified": counts.get("unspecified", 0),
    }
beh_df = pd.DataFrame(stack).T
beh_df.to_csv(os.path.join(TABLES_DIR, "table_behavior_distributions.csv"))

plt.figure()
bottom = np.zeros(len(beh_df))
for label in ["true","false","unspecified"]:
    plt.bar(beh_df.index, beh_df[label].values, bottom=bottom, label=label)
    bottom += beh_df[label].values
plt.xticks(rotation=60, ha="right")
plt.ylabel("Count")
plt.title("Behavioral Declarations (Stacked)")
plt.legend()
plt.tight_layout()
fig5 = os.path.join(OUTPUT_DIR, "fig_behaviors_stacked.png")
plt.savefig(fig5, dpi=300)
plt.close()

unspec_cols = beh_cols
unspec_frac = (beh_norm.eq("unspecified").sum(axis=1) / len(unspec_cols)).rename("opacity_index")
unspec_frac.to_csv(os.path.join(TABLES_DIR, "table_opacity_index.csv"))

plt.figure()
plt.hist(unspec_frac.values, bins=10)
plt.xlabel("Opacity Index (fraction of 'unspecified' over behaviors)")
plt.ylabel("Number of Sites")
plt.title("Opacity Index Distribution")
plt.tight_layout()
fig6 = os.path.join(OUTPUT_DIR, "fig_opacity_hist.png")
plt.savefig(fig6, dpi=300)
plt.close()

if df["policy_chars"].notna().any():
    for s in ["completeness_score","usability_score","accuracy_score"]:
        plt.figure()
        plt.scatter(df["policy_chars"], df[s], alpha=0.5)
        plt.xlabel("Policy length (characters)")
        plt.ylabel(s.replace("_"," ").title())
        plt.title(f"Policy Length vs {s.replace('_',' ').title()}")
        plt.tight_layout()
        out = os.path.join(OUTPUT_DIR, f"fig_scatter_len_{s}.png")
        plt.savefig(out, dpi=300)
        plt.close()

    corr_df = df_scores.copy()
    corr_df["policy_chars"] = pd.to_numeric(df["policy_chars"], errors="coerce")
    corr = corr_df.corr(method="spearman")
    corr.to_csv(os.path.join(TABLES_DIR, "table_spearman_corr.csv"))

    plt.figure()
    plt.imshow(corr.values, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Spearman Correlations")
    plt.tight_layout()
    fig7 = os.path.join(OUTPUT_DIR, "fig_corr_spearman.png")
    plt.savefig(fig7, dpi=300)
    plt.close()

desc = df_scores.describe().round(3)
desc.to_csv(os.path.join(TABLES_DIR, "table_score_describe.csv"))

for col in ["completeness_score","usability_score","accuracy_score"]:
    vc = df[col].value_counts().reindex([0,1,2,3], fill_value=0)
    vc.to_csv(os.path.join(TABLES_DIR, f"table_dist_{col}.csv"))

print("Saved figures:")
for p in sorted(Path(OUTPUT_DIR).glob("*.png")):
    print("-", p)

print("\nSaved tables:")
for p in sorted(Path(TABLES_DIR).glob("*.csv")):
    print("-", p)

In [ ]:
import json, pandas as pd

path = "ccpa_policy_audit_all.json"
data = json.load(open(path, "r", encoding="utf-8"))

expected_top = {"online_data_practices","offline_data_practices","rubric_assessment","behavioral_claims"}
rubric_keys = {"completeness_score","usability_score","accuracy_score","policy_contradiction","disclosure_map"}
disc_keys = {"data_collected","data_shared","purpose_of_collection","retention_period","right_to_access","right_to_delete","opt_out"}

ok = []
problems = []
for dom, obj in data.items():
    if not isinstance(obj, dict):
        problems.append((dom,"not a dict")); continue
    if not expected_top.issubset(obj.keys()):
        problems.append((dom,"missing top keys")); continue
    rub = obj.get("rubric_assessment", {})
    if not rubric_keys.issubset(rub.keys()):
        problems.append((dom,"missing rubric keys")); continue
    dm = rub.get("disclosure_map", {})

    if "right_to_acces" in dm and "right_to_access" not in dm:
        dm["right_to_access"] = dm.pop("right_to_acces")
    if not disc_keys.issubset(dm.keys()):
        problems.append((dom,"missing disclosure keys")); continue
    ok.append(dom)

print(f"Schema-OK: {len(ok)} / {len(data)}")
print("Examples OK:", ok[:5])
print("Problems (first 10):", problems[:10])

In [ ]:
import json

input_path = "ccpa_policy_audit_all.json"
output_path = "ccpa_policy_audit_all_clean.json"

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

expected_top = {"online_data_practices","offline_data_practices","rubric_assessment","behavioral_claims"}
rubric_keys = {"completeness_score","usability_score","accuracy_score","policy_contradiction","disclosure_map"}
disc_keys = {"data_collected","data_shared","purpose_of_collection","retention_period","right_to_access","right_to_delete","opt_out"}

cleaned_data = {}
problems = {}

for dom, obj in data.items():
    if not isinstance(obj, dict):
        problems[dom] = "not a dict"
        continue

    if not expected_top.issubset(obj.keys()):
        problems[dom] = "missing top keys"
        continue

    rub = obj.get("rubric_assessment", {})
    if not rubric_keys.issubset(rub.keys()):
        problems[dom] = "missing rubric keys"
        continue

    dm = rub.get("disclosure_map", {})

    if "right_to_acces" in dm and "right_to_access" not in dm:
        dm["right_to_access"] = dm.pop("right_to_acces")
    if not disc_keys.issubset(dm.keys()):
        problems[dom] = "missing disclosure keys"
        continue

    cleaned_data[dom] = obj

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

print(f"Original count: {len(data)}")
print(f"Cleaned count: {len(cleaned_data)}")
print(f"Removed count: {len(data) - len(cleaned_data)}")
print("Problem examples:", list(problems.items())[:10])

In [ ]:
import json
import os
from collections import defaultdict
from textwrap import shorten

INPUT_JSON = "ccpa_policy_audit_all.json"
N_EXAMPLES_PER_CASE = 5
MAX_JUST_CHARS = 300

BEHAVIORS = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]

def tri_norm(v):
    """Normalize behavioral flag values to 'true'/'false'/'unspecified' strings."""
    if isinstance(v, bool):
        return "true" if v else "false"
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}:
        return "true"
    if s in {"false", "no", "0"}:
        return "false"
    return "unspecified"

def main():
    if not os.path.isfile(INPUT_JSON):
        raise FileNotFoundError(f"Cannot find file: {INPUT_JSON}")

    with open(INPUT_JSON, "r", encoding="utf-8") as f:
        data = json.load(f)

    examples = {b: defaultdict(list) for b in BEHAVIORS}

    total_domains = 0
    for domain, result in data.items():
        if not isinstance(result, dict):
            continue
        bc = result.get("behavioral_claims", {})
        just = bc.get("justifications", {}) or {}
        if not bc:
            continue
        total_domains += 1
        for b in BEHAVIORS:
            label = tri_norm(bc.get(b, "unspecified"))
            jtxt = just.get(b, "")
            jtxt = jtxt if isinstance(jtxt, str) else ""
            examples[b][label].append((domain, jtxt))

    print(f"\nCollected justification examples from {total_domains} domains\n")
    for b in BEHAVIORS:
        print("=" * 80)
        print(f"Behavior: {b}")
        print("-" * 80)

        counts = {k: len(v) for k, v in examples[b].items()}

        for lab in ("true", "false", "unspecified"):
            counts.setdefault(lab, 0)
        print(f"Counts  ->  true: {counts['true']} | false: {counts['false']} | unspecified: {counts['unspecified']}")

        for lab in ("true", "false", "unspecified"):
            rows = examples[b][lab]
            if not rows:
                continue
            print(f"\n[{lab.upper()}] Examples (up to {N_EXAMPLES_PER_CASE}):")
            for (dom, jtxt) in rows[:N_EXAMPLES_PER_CASE]:
                preview = shorten(jtxt.strip().replace("\n", " "), width=MAX_JUST_CHARS, placeholder=" ...")
                print(f"- {dom}: {preview if preview else '(no justification text provided)'}")
        print("\n")

if __name__ == "__main__":
    main()

In [ ]:
import json
import re
from collections import defaultdict, Counter

INPUT_JSON = "ccpa_policy_audit_all.json"
MAX_EXAMPLES = 5

BEHAVIOR_KEYS = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]

def tri_value(v):
    """Normalize to 'true' | 'false' | 'unspecified' (strings, lowercase)."""
    if isinstance(v, bool):
        return "true" if v else "false"
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}:
        return "true"
    if s in {"false", "no", "0"}:
        return "false"
    return "unspecified"

def clean_text(s):
    s = (s or "").strip()

    s = re.sub(r"\s+", " ", s)
    return s

def main():
    with open(INPUT_JSON, "r", encoding="utf-8") as f:
        data = json.load(f)

    buckets = {b: {"true": [], "false": [], "unspecified": []} for b in BEHAVIOR_KEYS}
    counts = {b: Counter() for b in BEHAVIOR_KEYS}

    domains_counted = 0

    for domain, entry in data.items():
        if not isinstance(entry, dict):
            continue

        bc = entry.get("behavioral_claims", {}) or {}
        just = bc.get("justifications", {}) or {}

        if not bc:
            continue

        for key in BEHAVIOR_KEYS:
            raw_flag = bc.get(key, "unspecified")
            val = tri_value(raw_flag)

            jtxt = clean_text(just.get(key, "")) if isinstance(just, dict) else ""

            counts[key][val] += 1

            if not jtxt:
                jtxt = "No explicit statement found; flag set to unspecified."
            buckets[key][val].append((domain, jtxt))

        domains_counted += 1

    print(f"Collected justification examples from {domains_counted} domains\n")

    for behavior in BEHAVIOR_KEYS:
        print("=" * 80)
        print(f"Behavior: {behavior}")
        print("-" * 80)
        c = counts[behavior]
        print(f"Counts  ->  true: {c['true']} | false: {c['false']} | unspecified: {c['unspecified']}\n")

        print("[TRUE] Examples (up to 5):")
        true_ex = buckets[behavior]["true"][:MAX_EXAMPLES]
        if not true_ex:
            print("- (none)")
        else:
            for dom, j in true_ex:
                print(f"- {dom}: {j}")
        print()

        print("[FALSE] Examples (up to 5):")
        false_ex = buckets[behavior]["false"][:MAX_EXAMPLES]
        if not false_ex:
            print("- (none)")
        else:
            for dom, j in false_ex:
                print(f"- {dom}: {j}")
        print()

        print("[UNSPECIFIED] Examples (up to 5):")
        unspec_ex = buckets[behavior]["unspecified"][:MAX_EXAMPLES]
        if not unspec_ex:
            print("- (none)")
        else:
            for dom, j in unspec_ex:
                print(f"- {dom}: {j}")
        print("\n")

if __name__ == "__main__":
    main()

In [ ]:
import json
import re
import random
from typing import List, Tuple

INPUT_FILE   = "ccpa_policy_audit_all.json"
MAX_EXAMPLES = 5
SNIPPET_CHARS = 500
RANDOM_SEED  = 42

def load_data(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise ValueError("Expected top-level object mapping domains to results.")
    return data

def norm(s):
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s).strip()

def is_not_mentioned(s: str) -> bool:
    s2 = norm(s).lower()
    return s2 in {"not mentioned", "not mentioned.", "n/a", "na", ""}

def summarize_examples_random(pairs: List[Tuple[str, str]], max_examples: int, k: int) -> List[Tuple[str, str]]:
    """
    Randomly sample up to k examples from (domain, text) pairs.
    Trim text to SNIPPET_CHARS for readability.
    """
    if not pairs:
        return []
    sample = random.sample(pairs, k=min(k, len(pairs)))
    out = []
    for domain, text in sample:
        t = norm(text)
        if len(t) > SNIPPET_CHARS:
            t = t[:SNIPPET_CHARS].rstrip() + "…"
        out.append((domain, t))
    return out

def main():
    if RANDOM_SEED is not None:
        random.seed(RANDOM_SEED)

    data = load_data(INPUT_FILE)
    domains = list(data.keys())

    online_present: List[Tuple[str, str]] = []
    online_missing: List[str] = []
    offline_present: List[Tuple[str, str]] = []
    offline_not_mentioned: List[str] = []

    online_lengths: List[int] = []
    offline_lengths_present: List[int] = []

    for d in domains:
        entry = data.get(d, {})
        online  = norm(entry.get("online_data_practices", ""))
        offline = norm(entry.get("offline_data_practices", ""))

        if is_not_mentioned(online):
            online_missing.append(d)
        else:
            online_present.append((d, online))
            online_lengths.append(len(online))

        if is_not_mentioned(offline):
            offline_not_mentioned.append(d)
        else:
            offline_present.append((d, offline))
            offline_lengths_present.append(len(offline))

    total = len(domains)
    online_cnt = len(online_present)
    offline_cnt = len(offline_present)
    online_pct = (online_cnt / total * 100) if total else 0.0
    offline_pct = (offline_cnt / total * 100) if total else 0.0

    print("=== Coverage of Online/Offline Data Practices ===")
    print(f"Total domains: {total}")
    print(f"Online present: {online_cnt} ({online_pct:.1f}%)")
    print(f"Online missing/not mentioned: {len(online_missing)} ({100 - online_pct:.1f}%)")
    print(f"Offline present: {offline_cnt} ({offline_pct:.1f}%)")
    print(f"Offline not mentioned: {len(offline_not_mentioned)} ({100 - offline_pct:.1f}%)")

    if online_lengths:
        mean_online = sum(online_lengths) / len(online_lengths)
        print(f"\nOnline length (chars): mean={mean_online:.1f}, "
              f"min={min(online_lengths)}, max={max(online_lengths)}")
    if offline_lengths_present:
        mean_offline = sum(offline_lengths_present) / len(offline_lengths_present)
        print(f"Offline length (chars): mean={mean_offline:.1f}, "
              f"min={min(offline_lengths_present)}, max={max(offline_lengths_present)}")

    print("\n--- Example online_data_practices (present; random) ---")
    for dom, txt in summarize_examples_random(online_present, MAX_EXAMPLES, MAX_EXAMPLES):
        print(f"- {dom}: {txt}")

    print("\n--- Example offline_data_practices (present; random) ---")
    for dom, txt in summarize_examples_random(offline_present, MAX_EXAMPLES, MAX_EXAMPLES):
        print(f"- {dom}: {txt}")

    print("\n--- Example domains with offline_data_practices = 'Not mentioned' (random) ---")
    if offline_not_mentioned:
        for dom in random.sample(offline_not_mentioned, k=min(MAX_EXAMPLES, len(offline_not_mentioned))):
            print(f"- {dom}")
    else:
        print("(none)")

if __name__ == "__main__":
    main()

In [ ]:
import json
import re
import os
from collections import Counter, defaultdict
from typing import Dict, Any, Tuple

import pandas as pd

AUDIT_JSON = "ccpa_policy_audit_all.json"
CCPA_CSV   = "cookie_data_final - merged_updated.csv"

SAVE_CSV   = True
OUT_DIR    = "by_group_outputs"
MAX_EXAMPLES = 5

BEHAVIOR_KEYS = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]

DISCLOSURE_KEYS = [
    "data_collected",
    "data_shared",
    "purpose_of_collection",
    "retention_period",
    "right_to_access",
    "right_to_delete",
    "opt_out",
]

def norm_domain(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"^https?://", "", s)
    s = s.split("/")[0]
    if s.startswith("www."):
        s = s[4:]
    return s

def is_not_mentioned(s: str) -> bool:
    if not isinstance(s, str):
        return True
    t = re.sub(r"\s+", " ", s).strip().lower()
    return t in {"", "not mentioned", "not mentioned.", "n/a", "na"}

def load_audit(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = {}
    for dom, v in data.items():
        out[norm_domain(dom)] = v
    return out

def load_ccpa_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8")

    if "website" not in df.columns:

        if "domain" in df.columns:
            df["website"] = df["domain"]
        else:
            raise ValueError("CSV must contain 'website' (or 'domain') column.")
    if "ccpa" not in df.columns:
        raise ValueError("CSV must contain 'ccpa' column (e.g., 'subjected'/'not_subjected').")
    if "Industry" not in df.columns:
        raise ValueError("CSV must contain 'Industry' column.")

    df["domain_key"] = df["website"].astype(str).map(norm_domain)

    agg = (df
           .sort_values(["domain_key"])
           .dropna(subset=["domain_key"])
           .drop_duplicates(subset=["domain_key"]))
    return agg[["domain_key", "ccpa", "Industry"]].reset_index(drop=True)

def flatten_one(dom: str, rec: dict) -> dict:
    """Flatten one audit JSON record to a row dict for DataFrame."""
    rub = rec.get("rubric_assessment", {}) or {}
    disc = rub.get("disclosure_map", {}) or {}

    beh = rec.get("behavioral_claims", {}) or {}
    just = beh.get("justifications", {}) or {}

    row = {
        "domain": dom,
        "online_data_practices": rec.get("online_data_practices", ""),
        "offline_data_practices": rec.get("offline_data_practices", ""),
        "completeness_score": rub.get("completeness_score", None),
        "completeness_reason": rub.get("completeness_reason", ""),
        "usability_score": rub.get("usability_score", None),
        "usability_reason": rub.get("usability_reason", ""),
        "accuracy_score": rub.get("accuracy_score", None),
        "accuracy_reason": rub.get("accuracy_reason", ""),
        "policy_contradiction": rub.get("policy_contradiction", None),
    }

    for k in DISCLOSURE_KEYS:
        row[f"disc.{k}"] = bool(disc.get(k, False))

    for k in BEHAVIOR_KEYS:
        val = beh.get(k, "unspecified")
        if isinstance(val, bool):
            row[f"beh.{k}"] = "true" if val else "false"
        else:
            row[f"beh.{k}"] = str(val).strip().lower()
        row[f"just.{k}"] = just.get(k, "")

    row["online_len"] = len(str(row["online_data_practices"]))
    row["offline_len"] = len(str(row["offline_data_practices"]))

    row["online_present"] = not is_not_mentioned(row["online_data_practices"])
    row["offline_present"] = not is_not_mentioned(row["offline_data_practices"])

    for c in ["completeness_score", "usability_score", "accuracy_score"]:
        try:
            row[c] = int(row[c])
        except Exception:
            row[c] = None

    return row

def make_frame(audit: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for dom, rec in audit.items():
        if isinstance(rec, dict) and ("online_data_practices" in rec or "rubric_assessment" in rec):
            rows.append(flatten_one(dom, rec))
    return pd.DataFrame(rows)

def attach_groups(df: pd.DataFrame, ccpa_df: pd.DataFrame) -> pd.DataFrame:
    return df.merge(ccpa_df, how="left", left_on="domain", right_on="domain_key") \
             .drop(columns=["domain_key"])

def behavior_counts(sub: pd.DataFrame) -> pd.DataFrame:
    out = []
    for k in BEHAVIOR_KEYS:
        vc = sub[f"beh.{k}"].fillna("unspecified").value_counts()
        true_c = int(vc.get("true", 0))
        false_c = int(vc.get("false", 0))
        unspec_c = int(vc.get("unspecified", 0))
        out.append({"behavior": k, "true": true_c, "false": false_c, "unspecified": unspec_c})
    return pd.DataFrame(out).set_index("behavior")

def disclosure_prevalence(sub: pd.DataFrame) -> pd.DataFrame:
    out = []
    for k in DISCLOSURE_KEYS:
        pct = float(sub[f"disc.{k}"].mean() * 100.0) if len(sub) else 0.0
        out.append({"disclosure": k, "percent": round(pct, 1)})
    return pd.DataFrame(out).set_index("disclosure")

def score_distributions(sub: pd.DataFrame) -> pd.DataFrame:
    arr = []
    for metric in ["accuracy_score", "completeness_score", "usability_score"]:
        vc = sub[metric].dropna().astype(int).value_counts().reindex([0,1,2,3], fill_value=0)
        for score in [0,1,2,3]:
            arr.append({"metric": metric, "score": score, "count": int(vc.loc[score])})
    return pd.DataFrame(arr).set_index(["metric", "score"])

def score_summary(sub: pd.DataFrame) -> pd.DataFrame:
    return (sub[["accuracy_score","completeness_score","usability_score"]]
            .dropna()
            .describe()
            .T[["count","mean","std","min","25%","50%","75%","max"]])

def behavior_opacity(sub: pd.DataFrame) -> float:

    if len(sub) == 0:
        return float("nan")
    frac = []
    for _, row in sub.iterrows():
        u = 0
        tot = 0
        for k in BEHAVIOR_KEYS:
            v = str(row.get(f"beh.{k}", "unspecified")).lower()
            if v in {"true","false","unspecified"}:
                tot += 1
                if v == "unspecified":
                    u += 1
        if tot > 0:
            frac.append(u / tot)
    return float(pd.Series(frac).mean()) if frac else float("nan")

def score_opacity(sub: pd.DataFrame) -> float:

    if len(sub) == 0:
        return float("nan")
    x = sub[["completeness_score","usability_score"]].dropna()
    if len(x) == 0:
        return float("nan")
    return float((1 - ((x["completeness_score"] + x["usability_score"]) / 6)).mean())

def pretty_print_header(title: str):
    print("\n" + "="*80)
    print(title)
    print("="*80)

def print_group_report(label: str, sub: pd.DataFrame, save_prefix: str = None):
    n = len(sub)
    pretty_print_header(f"Group: {label}  (n={n})")

    online_present = int(sub["online_present"].sum())
    offline_present = int(sub["offline_present"].sum())
    print(f"- Online present: {online_present} ({online_present/n*100:.1f}%)")
    print(f"- Offline present: {offline_present} ({offline_present/n*100:.1f}%)")

    if online_present:
        print(f"- Online length (chars): mean={sub.loc[sub['online_present'],'online_len'].mean():.1f}, "
              f"min={int(sub.loc[sub['online_present'],'online_len'].min())}, "
              f"max={int(sub.loc[sub['online_present'],'online_len'].max())}")
    if offline_present:
        print(f"- Offline length (chars): mean={sub.loc[sub['offline_present'],'offline_len'].mean():.1f}, "
              f"min={int(sub.loc[sub['offline_present'],'offline_len'].min())}, "
              f"max={int(sub.loc[sub['offline_present'],'offline_len'].max())}")

    bdf = behavior_counts(sub)
    print("\nBehavioral compliance counts:")
    print(bdf)

    ddf = disclosure_prevalence(sub)
    print("\nDisclosure prevalence (% true):")
    print(ddf)

    sdf = score_distributions(sub)
    print("\nScore distributions:")
    print(sdf)

    ssum = score_summary(sub)
    print("\nScore summary (describe):")
    print(ssum)

    beh_op = behavior_opacity(sub)
    sc_op = score_opacity(sub)
    print(f"\nBehavioral opacity (mean fraction unspecified): {beh_op:.3f}")
    print(f"Score-based opacity index OI = 1 - (comp+usab)/6 : {sc_op:.3f}")

    def rand_examples(col, present=True):
        df2 = sub.copy()
        if present:
            df2 = df2[df2[col]]
        else:
            df2 = df2[~df2[col]]
        return df2.sample(n=min(MAX_EXAMPLES, len(df2)), random_state=42) if len(df2) else df2

    ex_on = rand_examples("online_present", True)
    ex_off = rand_examples("offline_present", True)
    ex_off_missing = rand_examples("offline_present", False)

    print("\n--- Example online_data_practices (present; random) ---")
    for _, r in ex_on.iterrows():
        txt = str(r["online_data_practices"]).strip()
        if len(txt) > 220:
            txt = txt[:220].rstrip() + "…"
        print(f"- {r['domain']}: {txt}")

    print("\n--- Example offline_data_practices (present; random) ---")
    for _, r in ex_off.iterrows():
        txt = str(r["offline_data_practices"]).strip()
        if len(txt) > 220:
            txt = txt[:220].rstrip() + "…"
        print(f"- {r['domain']}: {txt}")

    print("\n--- Example domains with offline_data_practices = 'Not mentioned' (random) ---")
    for _, r in ex_off_missing.iterrows():
        print(f"- {r['domain']}")

    if SAVE_CSV and save_prefix:
        os.makedirs(OUT_DIR, exist_ok=True)
        bdf.to_csv(os.path.join(OUT_DIR, f"{save_prefix}_behavior_counts.csv"))
        ddf.to_csv(os.path.join(OUT_DIR, f"{save_prefix}_disclosure_prevalence.csv"))
        sdf.to_csv(os.path.join(OUT_DIR, f"{save_prefix}_score_distributions.csv"))
        ssum.to_csv(os.path.join(OUT_DIR, f"{save_prefix}_score_summary.csv"))

def main():

    audit = load_audit(AUDIT_JSON)
    df = make_frame(audit)
    ccpa_df = load_ccpa_csv(CCPA_CSV)
    df = attach_groups(df, ccpa_df)

    df["ccpa"] = df["ccpa"].fillna("unknown").str.strip().str.lower()
    df["Industry"] = df["Industry"].fillna("Unknown").astype(str).str.strip()

    pretty_print_header(f"GLOBAL SUMMARY (all joined rows)  n={len(df)}")
    print(df[["domain","ccpa","Industry"]].head(10).to_string(index=False))

    pretty_print_header("BREAKDOWN BY CCPA SUBJECTIVITY")
    for ccpa_label, sub in df.groupby("ccpa", dropna=False):
        save_prefix = f"by_ccpa_{re.sub(r'[^a-z0-9]+','_', str(ccpa_label))}"
        print_group_report(f"CCPA={ccpa_label}", sub, save_prefix)

    pretty_print_header("BREAKDOWN BY INDUSTRY (top 12 by count)")
    top_inds = (df["Industry"].value_counts().head(12)).index.tolist()
    for ind in top_inds:
        sub = df[df["Industry"] == ind]
        save_prefix = f"by_industry_{re.sub(r'[^a-z0-9]+','_', ind.lower())}"
        print_group_report(f"Industry={ind}", sub, save_prefix)

    for ind, sub in df.groupby("Industry", dropna=False):
        save_prefix = f"by_industry_{re.sub(r'[^a-z0-9]+','_', str(ind).lower())}"
        print_group_report(f"Industry={ind}", sub, save_prefix)

if __name__ == "__main__":
    main()

In [ ]:
import argparse
import collections
import csv
import json
import os
import re
from pathlib import Path
from typing import Dict, Any, Tuple, Optional

try:
    import tldextract
except Exception:
    tldextract = None

def normalize_host(raw: str) -> str:
    """Normalize a URL/host to a bare lowercase host (no scheme, no path)."""
    if not raw:
        return ""
    s = raw.strip().lower()
    s = re.sub(r"^https?://", "", s)
    s = s.split("/")[0].split("?")[0].strip()
    s = s.rstrip(".")
    if s.startswith("www."):
        s = s[4:]
    return s

def etld_plus_one(host: str) -> str:
    """Return eTLD+1 for a host. Uses tldextract when available, else a simple heuristic."""
    h = normalize_host(host)
    if not h:
        return ""
    if tldextract:
        ext = tldextract.extract(h)
        if ext.domain and ext.suffix:
            return f"{ext.domain}.{ext.suffix}"
        return h

    parts = h.split(".")
    if len(parts) >= 2:
        return ".".join(parts[-2:])
    return h

def clean_label(s: Optional[str]) -> str:
    """Trim and standardize simple labels."""
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s).strip()

def normalize_industry(label: str) -> str:
    """Standardize Industry names (light normalization + synonym mapping)."""
    s = clean_label(label)
    if not s:
        return "Unknown"

    s_low = s.lower()
    SYN = {
        "retail & ecommerce": "Retail",
        "retail/ecommerce": "Retail",
        "e-commerce": "Retail",
        "ecommerce": "Retail",
        "media & news": "News/Media",
        "newsmedia": "News/Media",
        "news media": "News/Media",
        "education": "Education",
        "education/nonprofit": "Education",
        "finance": "Financial Services",
        "financial": "Financial Services",
        "financialservices": "Financial Services",
        "healthcare": "Healthcare",
        "health care": "Healthcare",
        "tech": "Technology",
        "technology": "Technology",
        "software": "Technology",
        "nonprofit": "Nonprofit",
        "government": "Government",
        "public sector": "Government",
        "travel": "Travel/Leisure",
        "travel & leisure": "Travel/Leisure",
        "food": "Food/Restaurants",
        "restaurants": "Food/Restaurants",
        "food & beverage": "Food/Restaurants",
        "automotive": "Automotive",
        "auto": "Automotive",
        "gaming": "Gaming",
        "advertising": "Advertising/AdTech",
        "adtech": "Advertising/AdTech",
    }
    if s_low in SYN:
        return SYN[s_low]

    return "/".join([w.capitalize() for w in s.split("/")])

def normalize_ccpa_flag(ccpa_val: str) -> str:
    """Only 'subjected' counts as subject to CCPA. Everything else = non-subject."""
    s = clean_label(ccpa_val).lower()
    return "subjected" if s == "subjected" else "non-subjected"

def majority_vote(values):
    """Return the most common non-empty value; tie-break by first appearance."""
    candidates = [v for v in values if clean_label(v)]
    if not candidates:
        return ""
    counter = collections.Counter(candidates)
    most_common = counter.most_common()
    top_count = most_common[0][1]
    winners = [v for v, c in most_common if c == top_count]

    for v in candidates:
        if v in winners:
            return v
    return candidates[0]

def load_audit_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise ValueError("Policy audit file must be a dict mapping domain -> result.")

    out = {}
    for k, v in data.items():
        out[normalize_host(k)] = v
    return out

def load_cookie_csv(path: str) -> Dict[str, Dict[str, str]]:
    """
    Load cookie CSV and build two maps (host & eTLD+1) to (ccpa, industry) via majority vote.
    The CSV snippet shows columns: ccpa, website, Industry, ...
    We’ll use 'website' if present; else try 'domain'.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(path)
    rows = []
    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        cols = {c.lower(): c for c in reader.fieldnames or []}
        site_col = cols.get("website") or cols.get("domain")
        if not site_col:
            raise ValueError("CSV must contain a 'website' or 'domain' column.")
        for r in reader:
            host_raw = r.get(site_col, "")
            host = normalize_host(host_raw)
            if not host:
                continue
            rows.append({
                "host": host,
                "etld1": etld_plus_one(host),
                "ccpa": clean_label(r.get("ccpa", "")),
                "industry": clean_label(r.get("Industry") or r.get("industry") or ""),
            })

    agg_host = {}
    by_host = collections.defaultdict(lambda: {"ccpa": [], "industry": []})
    for r in rows:
        by_host[r["host"]]["ccpa"].append(r["ccpa"])
        by_host[r["host"]]["industry"].append(r["industry"])
    for h, bags in by_host.items():
        ccpa_mv = normalize_ccpa_flag(majority_vote(bags["ccpa"]))
        ind_mv = normalize_industry(majority_vote(bags["industry"]))
        agg_host[h] = {"ccpa": ccpa_mv, "industry": ind_mv}

    agg_etld1 = {}
    by_etld = collections.defaultdict(lambda: {"ccpa": [], "industry": []})
    for r in rows:
        by_etld[r["etld1"]]["ccpa"].append(r["ccpa"])
        by_etld[r["etld1"]]["industry"].append(r["industry"])
    for h, bags in by_etld.items():
        ccpa_mv = normalize_ccpa_flag(majority_vote(bags["ccpa"]))
        ind_mv = normalize_industry(majority_vote(bags["industry"]))
        agg_etld1[h] = {"ccpa": ccpa_mv, "industry": ind_mv}

    return {"by_host": agg_host, "by_etld1": agg_etld1}

def merge_and_diagnose(audit: Dict[str, Any],
                       cookie_maps: Dict[str, Dict[str, Dict[str, str]]],
                       prefer_exact=True) -> Tuple[Dict[str, Any], Dict[str, int], Dict[str, Any]]:
    by_host = cookie_maps["by_host"]
    by_etld1 = cookie_maps["by_etld1"]

    merged = {}
    stats = collections.Counter()
    unknown_reasons = collections.Counter()

    for dom, payload in audit.items():
        exact = by_host.get(dom)
        if exact:
            stats["matched_exact"] += 1
            ccpa = exact["ccpa"]
            industry = exact["industry"]
            reason = ""
        else:
            et = etld_plus_one(dom)
            wide = by_etld1.get(et)
            if wide:
                stats["matched_etld1"] += 1
                ccpa = wide["ccpa"]
                industry = wide["industry"]
                reason = ""
            else:
                stats["unmatched"] += 1
                ccpa = "non-subjected"
                industry = "Unknown"
                reason = "no_csv_row"

        subject_to_ccpa = (ccpa == "subjected")

        merged[dom] = {
            "domain": dom,
            "etld1": etld_plus_one(dom),
            "subject_to_ccpa": subject_to_ccpa,
            "industry": industry,
            "raw_ccpa_label": ccpa,
            "audit": payload,
        }

        if industry == "Unknown":
            unknown_reasons[reason or "blank_or_unusable"] += 1

    return merged, stats, {"unknown_industry_reasons": dict(unknown_reasons)}

def summarize_merged(merged: Dict[str, Any]) -> None:
    total = len(merged)

    subj_counts = collections.Counter(m["subject_to_ccpa"] for m in merged.values())
    ind_counts = collections.Counter(m["industry"] for m in merged.values())

    print("\n=== Subject-to-CCPA (after rule: only 'subjected' → subject) ===")
    for k in [True, False]:
        print(f"{str(k):>5}: {subj_counts.get(k,0)} ({subj_counts.get(k,0)/total*100:.1f}%)")

    print("\n=== Industry Distribution (normalized) ===")
    for ind, c in ind_counts.most_common():
        print(f"{ind:25s}  {c:5d}  ({c/total*100:.1f}%)")

    unknown_domains = [d for d, m in merged.items() if m["industry"] == "Unknown"]
    if unknown_domains:
        print(f"\nSample Unknown Industry domains (first 25 of {len(unknown_domains)}):")
        for d in unknown_domains[:25]:
            print(" -", d)

def main():
    ap = argparse.ArgumentParser(description="Merge CCPA/Industry into audit results with diagnostics.")
    ap.add_argument("--audit_json", default="ccpa_policy_audit_all.json",
                    help="Path to your policy audit JSON.")
    ap.add_argument("--cookie_csv", default="cookie_data_final - merged_updated.csv",
                    help="Path to your cookie CSV containing columns 'website' (or 'domain'), 'ccpa', and 'Industry'.")
    ap.add_argument("--out_json", default="ccpa_audit_with_labels.json",
                    help="Path to write merged JSON with labels.")
    args = ap.parse_args()

    audit = load_audit_json(args.audit_json)
    cookie_maps = load_cookie_csv(args.cookie_csv)

    print(f"Loaded audit domains: {len(audit)}")
    print(f"Cookie CSV maps — unique hosts: {len(cookie_maps['by_host'])}, eTLD+1: {len(cookie_maps['by_etld1'])}")

    merged, match_stats, extras = merge_and_diagnose(audit, cookie_maps)

    print("\n=== Join Diagnostics ===")
    print(f"Matched by exact host : {match_stats.get('matched_exact',0)}")
    print(f"Matched by eTLD+1     : {match_stats.get('matched_etld1',0)}")
    print(f"Unmatched (defaulted) : {match_stats.get('unmatched',0)}")

    print("\n=== Why Industry is 'Unknown' ===")
    for reason, c in extras["unknown_industry_reasons"].items():
        print(f"{reason:20s}: {c}")

    summarize_merged(merged)

    with open(args.out_json, "w", encoding="utf-8") as f:
        json.dump(merged, f, indent=2, ensure_ascii=False)
    print(f"\n[✓] Wrote merged dataset to: {args.out_json}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import tldextract

file_path = "cookie_data_final - merged_updated.csv"
df = pd.read_csv(file_path)

df['website'] = df['website'].astype(str).str.strip().str.lower()

if 'ccpa' not in df.columns:
    raise ValueError("Missing 'ccpa' column in the dataset.")

df['ccpa_subject'] = df['ccpa'].apply(lambda x: str(x).strip().lower() == "subjected")

print("=== CCPA Subjectivity Counts ===")
print(df['ccpa_subject'].value_counts(dropna=False))
print()

if 'Industry' not in df.columns:
    raise ValueError("Missing 'Industry' column in the dataset.")

df['Industry'] = df['Industry'].astype(str).str.strip()

missing_industry = df['Industry'].isin(['', 'nan', 'None', 'Unknown', 'unknown'])
missing_count = missing_industry.sum()
total_count = len(df)

print(f"=== Industry Field Check ===")
print(f"Total rows: {total_count}")
print(f"Missing/Unknown Industry: {missing_count} ({missing_count/total_count:.2%})")
print()

if missing_count > 0:
    print("Sample rows with missing/unknown Industry:")
    print(df.loc[missing_industry, ['website', 'Industry']].head(10))
    print()

df['etld_plus1'] = df['website'].apply(lambda w: tldextract.extract(w).registered_domain)

industry_by_etld = df.groupby('etld_plus1')['Industry'].apply(lambda x: list(set(i for i in x if i not in ['', 'nan', 'None', 'Unknown', 'unknown'])))
possible_fixes = industry_by_etld[industry_by_etld.apply(len) > 0]

print("=== Possible Industry Fixes via eTLD+1 Matching ===")
print(possible_fixes.head(10))
print()

print("=== Diagnostic Summary ===")
print(f"Total unique websites: {df['website'].nunique()}")
print(f"Total unique eTLD+1: {df['etld_plus1'].nunique()}")
print(f"Websites missing Industry: {missing_count} ({missing_count/total_count:.1%})")
print(f"CCPA-subjected websites: {df['ccpa_subject'].sum()} ({df['ccpa_subject'].mean():.1%})")
print("Run the possible_fixes output to inspect and apply corrections if needed.")

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import tldextract
from pathlib import Path

FILE_COOKIES = "cookie_data_final - merged_updated.csv"
OUT_WEBSITE = "site_level_by_website.csv"
OUT_ETLD1   = "site_level_by_etld1.csv"
UNKNOWN_TOKENS = {"", "nan", "none", "unknown", "n/a", "na", "undef", "undefined"}

def norm_text(x: str) -> str:
    s = str(x or "").strip()

    s = re.sub(r"\s+", " ", s)
    return s

def is_unknown(value: str) -> bool:
    return norm_text(value).lower() in UNKNOWN_TOKENS

def choose_industry(values):
    """
    Pick a canonical Industry from a list:
      1) Drop 'Unknown' and other unknown-like tokens
      2) Take the most frequent remaining label (mode)
      3) If everything is unknown/missing, return 'Unknown'
    """
    clean = [norm_text(v) for v in values if not is_unknown(v)]
    if not clean:
        return "Unknown"
    counts = Counter(clean)

    top_count = max(counts.values())
    candidates = sorted([k for k,v in counts.items() if v == top_count])
    return candidates[0]

def extract_website_host(raw: str) -> str:
    """
    Normalize 'website' column to a stable host string without paths or scheme,
    preserving subdomain if present (this tends to match your ~950 count).
    """
    s = norm_text(raw).lower()

    s = re.sub(r"^[a-z]+://", "", s)

    s = s.split("/", 1)[0]

    s = s[4:] if s.startswith("www.") else s
    return s

def extract_etld1(raw: str) -> str:
    """
    Registered domain (eTLD+1), e.g., store.example.co.uk -> example.co.uk
    """
    parts = tldextract.extract(norm_text(raw).lower())
    return f"{parts.domain}.{parts.suffix}" if parts.suffix else parts.domain

df = pd.read_csv(FILE_COOKIES)
required_cols = {"website", "ccpa", "Industry"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")

df["website_raw"] = df["website"].astype(str)
df["website_norm"] = df["website_raw"].apply(extract_website_host)
df["etld_plus1"]  = df["website_raw"].apply(extract_etld1)

df["Industry"] = df["Industry"].astype(str)
df["ccpa_subject"] = df["ccpa"].astype(str).str.strip().str.lower().eq("subjected")

n_rows = len(df)
u_web = df["website_norm"].nunique()
u_et1 = df["etld_plus1"].nunique()

print("=== Row-Level Diagnostics (raw cookie rows) ===")
print(f"Total rows (cookies): {n_rows}")
print(f"Unique websites (normalized host): {u_web}")
print(f"Unique eTLD+1 roots: {u_et1}")

row_unknown_rate = (df["Industry"].apply(is_unknown)).mean()
print(f"Row-level Industry unknown rate: {row_unknown_rate:.2%}\n")

def agg_site(group):

    industry = choose_industry(group["Industry"].tolist())

    subj = bool(group["ccpa_subject"].any())

    n_cookies = len(group)
    return pd.Series({"Industry": industry, "ccpa_subject": subj, "cookie_rows": n_cookies})

site_web = df.groupby("website_norm", as_index=False).apply(agg_site)

n_sites_web = len(site_web)
n_unknown_web = (site_web["Industry"] == "Unknown").sum()
print("=== Website-Level (grouped by normalized host) ===")
print(f"Total sites: {n_sites_web}")
print(f"Unknown industry sites: {n_unknown_web} ({n_unknown_web/n_sites_web:.2%})")
print(site_web.head(10))
print()

site_et1 = df.groupby("etld_plus1", as_index=False).apply(agg_site)

n_sites_et1 = len(site_et1)
n_unknown_et1 = (site_et1["Industry"] == "Unknown").sum()
print("=== Root-Level (grouped by eTLD+1) ===")
print(f"Total roots: {n_sites_et1}")
print(f"Unknown industry roots: {n_unknown_et1} ({n_unknown_et1/n_sites_et1:.2%})")
print(site_et1.head(10))
print()

host_to_et1 = df[["website_norm", "etld_plus1"]].drop_duplicates()
counts_per_et1 = host_to_et1.groupby("etld_plus1")["website_norm"].nunique().sort_values(ascending=False)
multi_brand = (counts_per_et1 > 1).sum()
print("=== Sub-brand Collisions (host -> eTLD+1) ===")
print(f"eTLD+1 roots with >1 distinct host/brand under them: {multi_brand}")
print("Top examples:")
print(counts_per_et1.head(15))
print()

Path(OUT_WEBSITE).write_text("")
Path(OUT_ETLD1).write_text("")

site_web.to_csv(OUT_WEBSITE, index=False)
site_et1.to_csv(OUT_ETLD1, index=False)

print(f"[✓] Saved site-level tables:\n  - {OUT_WEBSITE}\n  - {OUT_ETLD1}")
print("\nUse `site_level_by_website.csv` if you want to keep sub-brands distinct (≈ your 950 sites).")
print("Use `site_level_by_etld1.csv` if you want to collapse sub-brands under their registrable root (≈ 942 roots).")

In [ ]:
import pandas as pd
import tldextract

file_path = "cookie_data_final - merged_updated.csv"
df = pd.read_csv(file_path)

df["ccpa_subject"] = df["ccpa"].str.lower().eq("subjected")

def get_etld_plus1(url):
    ext = tldextract.extract(url)
    return f"{ext.domain}.{ext.suffix}"

df["etld_plus1"] = df["website"].apply(get_etld_plus1)

site_df = df.groupby("etld_plus1").agg({
    "ccpa_subject": "max",
    "Industry": lambda x: x.dropna().unique()[0] if len(x.dropna().unique()) > 0 else "Unknown"
}).reset_index()

missing_industry = site_df[site_df["Industry"].isin(["Unknown", ""])]
print("=== Industry Field Check (site-level) ===")
print(f"Total unique sites: {len(site_df)}")
print(f"Missing/Unknown Industry: {len(missing_industry)} ({len(missing_industry)/len(site_df)*100:.2f}%)")

if not missing_industry.empty:
    print("\nSample missing industry rows:")
    print(missing_industry.head(10))

possible_fixes = (
    df[~df["Industry"].isin(["Unknown", ""])]
    .groupby("etld_plus1")["Industry"]
    .agg(lambda x: pd.Series(x).mode()[0])
)

possible_fixes = possible_fixes.loc[possible_fixes.index.isin(missing_industry["etld_plus1"])]
if not possible_fixes.empty:
    print("\n=== Possible Industry Fixes ===")
    print(possible_fixes)

ccpa_counts = site_df["ccpa_subject"].value_counts()
print("\n=== CCPA Subjectivity Counts (site-level) ===")
print(ccpa_counts)

grouped = site_df.groupby(["ccpa_subject", "Industry"]).size().reset_index(name="site_count")
print("\n=== Grouped Counts: CCPA × Industry (site-level) ===")
print(grouped.sort_values(["ccpa_subject", "site_count"], ascending=[False, False]))

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
import tldextract

POLICY_JSON = "ccpa_policy_audit_all_clean.json"
COOKIE_CSV  = "cookie_data_final - merged_updated.csv"
OUT_DIR     = "paper_tables"
MIN_INDUSTRY_N = 10

def etld_plus1_from_host(host: str) -> str:
    """eTLD+1 from a host/domain string (no scheme required)."""
    host = (host or "").strip()

    host = re.sub(r"^https?://", "", host, flags=re.I)
    host = host.split("/")[0]
    ext = tldextract.extract(host)
    if ext.suffix:
        return f"{ext.domain}.{ext.suffix}"

    return host

def safe_bool(v):
    """Map value to bool (strict) or np.nan for unknown."""
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}:  return True
    if s in {"false", "no", "0"}:  return False
    return np.nan

def safe_text(s):
    return re.sub(r"\s+", " ", s).strip() if isinstance(s, str) else ""

def norm_offline_online_text(s):
    s2 = safe_text(s).lower()
    if s2 in {"not mentioned", "not mentioned.", "n/a", "na", ""}:
        return ""
    return safe_text(s)

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

cookies = pd.read_csv(COOKIE_CSV)

cookies["ccpa_subject"] = cookies["ccpa"].astype(str).str.lower().eq("subjected")

cookies["etld_plus1"] = cookies["website"].astype(str).apply(etld_plus1_from_host)

def pick_industry(series):
    vals = [v for v in series.astype(str).tolist() if v and v.lower() != "unknown"]
    if len(vals) == 0:

        return "Unknown"

    s = pd.Series(vals)
    modes = s.mode()
    return modes.iloc[0] if not modes.empty else vals[0]

site_meta = (
    cookies.groupby("etld_plus1")
           .agg(
               ccpa_subject=("ccpa_subject", "max"),
               Industry=("Industry", pick_industry)
           )
           .reset_index()
)

print("=== Site-level meta (from cookie CSV) ===")
print(site_meta.head(5))
print(f"Unique sites from CSV: {len(site_meta)}\n")

with open(POLICY_JSON, "r", encoding="utf-8") as f:
    policy_raw = json.load(f)

records = []
for domain, entry in policy_raw.items():
    row = {"domain": domain, "etld_plus1": etld_plus1_from_host(domain)}

    ra = entry.get("rubric_assessment", {}) or {}
    dm = ra.get("disclosure_map", {}) or {}

    row["completeness_score"] = ra.get("completeness_score", np.nan)
    row["usability_score"]    = ra.get("usability_score", np.nan)
    row["accuracy_score"]     = ra.get("accuracy_score", np.nan)

    for k in ["data_collected","data_shared","purpose_of_collection",
              "retention_period","right_to_access","right_to_delete","opt_out"]:
        row[f"disc.{k}"] = bool(dm.get(k, False))

    bc = entry.get("behavioral_claims", {}) or {}
    for k in ["honors_gpc","respects_dnt",
              "sets_cookies_before_consent","sets_cookies_after_rejecting_consent",
              "deletes_cookies_on_rejection","uses_tracking_only_after_consent",
              "sells_data","shares_with_third_parties"]:
        row[f"beh.{k}"] = safe_bool(bc.get(k, "unspecified"))

    row["online_text"]  = norm_offline_online_text(entry.get("online_data_practices", ""))
    row["offline_text"] = norm_offline_online_text(entry.get("offline_data_practices", ""))

    row["online_present"]  = bool(row["online_text"])
    row["offline_present"] = bool(row["offline_text"])

    records.append(row)

policy_df = pd.DataFrame.from_records(records)

print("=== Policy rows (head) ===")
print(policy_df.head(5))
print(f"Total policy domains: {len(policy_df)} | Unique eTLD+1: {policy_df['etld_plus1'].nunique()}\n")

merged = policy_df.merge(site_meta, on="etld_plus1", how="left", validate="m:1")

unmatched = merged[merged["ccpa_subject"].isna()]
missing_industry_sites = merged[merged["Industry"].isna() | (merged["Industry"] == "Unknown")]

print("=== Merge diagnostics ===")
print(f"Policies unmatched to CSV (no ccpa/industry): {len(unmatched)}")
print(f"Policies with missing/Unknown industry after merge: {len(missing_industry_sites)}\n")

ensure_dir(OUT_DIR)
if len(unmatched) > 0:
    unmatched[["domain","etld_plus1"]].drop_duplicates().to_csv(
        os.path.join(OUT_DIR, "diagnostics_unmatched_sites.csv"), index=False
    )
if len(missing_industry_sites) > 0:
    (missing_industry_sites[["domain","etld_plus1","Industry"]]
        .drop_duplicates()
        .to_csv(os.path.join(OUT_DIR, "diagnostics_missing_industry_sites.csv"), index=False)
     )

analytical = merged.dropna(subset=["ccpa_subject"]).copy()

analytical["ccpa_subject"] = analytical["ccpa_subject"].astype(bool)

SCORE_COLS = ["completeness_score","usability_score","accuracy_score"]
DISC_COLS  = [c for c in analytical.columns if c.startswith("disc.")]
BEH_COLS   = [c for c in analytical.columns if c.startswith("beh.")]

def agg_scores(df):
    return df[SCORE_COLS].agg(["count","mean","std","min","median","max"])

def prevalence(df, cols):

    return (df[cols].mean(numeric_only=True) * 100).round(1).to_frame("prevalence_%")

def behavior_distribution(df):

    out = []
    for col in BEH_COLS:
        s = df[col]
        true_pct = (s == True).mean() * 100
        false_pct = (s == False).mean() * 100
        unspec_pct = s.isna().mean() * 100
        out.append({
            "behavior": col.replace("beh.",""),
            "pct_true": round(true_pct, 1),
            "pct_false": round(false_pct, 1),
            "pct_unspecified": round(unspec_pct, 1),
            "n": len(s)
        })
    return pd.DataFrame(out).sort_values("behavior")

def pack_ccpa_group(df):
    """Return a dict of summary tables for a CCPA group."""
    d = {}
    d["n_sites"]          = len(df)
    d["scores"]           = agg_scores(df)
    d["disclosure_prev"]  = prevalence(df, DISC_COLS)
    d["behavior_dist"]    = behavior_distribution(df)
    return d

ccpa_groups = {}
for flag, group in analytical.groupby("ccpa_subject", dropna=False):
    ccpa_groups[flag] = pack_ccpa_group(group)

print("=== Summary by CCPA subjectivity ===")
for flag, bundle in ccpa_groups.items():
    label = "Subject to CCPA" if flag else "Not subject to CCPA"
    print(f"\n--- {label} ---")
    print(f"Sites: {bundle['n_sites']}")
    print("\nScore summary:")
    print(bundle["scores"])
    print("\nDisclosure prevalence (%):")
    print(bundle["disclosure_prev"].T)
    print("\nBehavior distribution (%):")
    print(bundle["behavior_dist"])

ccpa_overview_rows = []
for flag, b in ccpa_groups.items():
    label = "subjected" if flag else "non_subjected"

    scores = b["scores"].copy()
    scores.columns = [f"{c}" for c in scores.columns]
    scores["metric"] = scores.index
    scores["ccpa_subject"] = label

    disc = b["disclosure_prev"].reset_index().rename(columns={"index":"disclosure"})
    disc["ccpa_subject"] = label

    beh = b["behavior_dist"].copy()
    beh["ccpa_subject"] = label

    scores.to_csv(os.path.join(OUT_DIR, f"ccpa_overview_scores__{label}.csv"), index=False)
    disc.to_csv(os.path.join(OUT_DIR, f"ccpa_overview_disclosures__{label}.csv"), index=False)
    beh.to_csv(os.path.join(OUT_DIR, f"ccpa_overview_behaviors__{label}.csv"), index=False)

beh_all = pd.concat(
    [b["behavior_dist"].assign(ccpa_subject=("subjected" if f else "non_subjected"))
     for f, b in ccpa_groups.items()],
    ignore_index=True
)
beh_all.to_csv(os.path.join(OUT_DIR, "ccpa_overview_behaviors_all.csv"), index=False)

industry_sizes = analytical.groupby("Industry").size().rename("n").reset_index()
valid_industries = industry_sizes[industry_sizes["n"] >= MIN_INDUSTRY_N]["Industry"].tolist()

ind_summaries = []
for ind, grp in analytical.groupby("Industry"):
    if ind not in valid_industries:
        continue
    row = {"Industry": ind, "n_sites": len(grp)}

    for c in SCORE_COLS:
        row[f"{c}_mean"] = grp[c].mean()
        row[f"{c}_std"]  = grp[c].std()

    for c in DISC_COLS:
        row[f"{c}_prev%"] = grp[c].mean() * 100

    for c in BEH_COLS:
        row[f"{c}_true%"]        = (grp[c] == True).mean() * 100
        row[f"{c}_false%"]       = (grp[c] == False).mean() * 100
        row[f"{c}_unspecified%"] = grp[c].isna().mean() * 100
    ind_summaries.append(row)

industry_overview = pd.DataFrame(ind_summaries).sort_values("n_sites", ascending=False)
industry_overview.to_csv(os.path.join(OUT_DIR, "industry_overview.csv"), index=False)

print("\n=== Industry overview (top 10 by n_sites) ===")
print(industry_overview[["Industry","n_sites"] + [f"{c}_mean" for c in SCORE_COLS]].head(10))

tw_rows = []
for (flag, ind), grp in analytical.groupby(["ccpa_subject", "Industry"]):
    if ind not in valid_industries:
        continue
    r = {
        "ccpa_subject": "subjected" if flag else "non_subjected",
        "Industry": ind,
        "n_sites": len(grp)
    }
    for c in SCORE_COLS:
        r[f"{c}_mean"] = grp[c].mean()
    for c in DISC_COLS:
        r[f"{c}_prev%"] = grp[c].mean() * 100
    for c in BEH_COLS:
        r[f"{c}_true%"]        = (grp[c] == True).mean() * 100
        r[f"{c}_false%"]       = (grp[c] == False).mean() * 100
        r[f"{c}_unspecified%"] = grp[c].isna().mean() * 100
    tw_rows.append(r)

ccpa_by_industry = pd.DataFrame(tw_rows).sort_values(["ccpa_subject","n_sites"], ascending=[False, False])
ccpa_by_industry.to_csv(os.path.join(OUT_DIR, "ccpa_by_industry.csv"), index=False)

print("\n=== CCPA × Industry overview (head) ===")
print(ccpa_by_industry.head(12))

print(f"\n✅ Done. Tables saved in: {os.path.abspath(OUT_DIR)}")

In [ ]:
import json, re, os
from pathlib import Path
import pandas as pd
import numpy as np
import tldextract
from textwrap import shorten

POLICY_JSON = "ccpa_policy_audit_all.json"
CCPA_CSV    = "cookie_data_final - merged_updated.csv"
OUTDIR      = "split_outputs"
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

BEHAVIOR_KEYS = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]

DISC_KEYS = [
    "data_collected",
    "data_shared",
    "purpose_of_collection",
    "retention_period",
    "right_to_access",
    "right_to_delete",
    "opt_out",
]

SCORE_KEYS = ["accuracy_score", "completeness_score", "usability_score"]

def etld1(host: str) -> str:
    ext = tldextract.extract(host or "")
    if not ext.suffix:
        return (ext.domain or host or "").lower().strip()
    return f"{ext.domain}.{ext.suffix}".lower().strip()

def norm_text(s, width=220):
    if not isinstance(s, str): return ""
    s = " ".join(s.split())
    return shorten(s, width=width, placeholder="…")

def load_policy_df(path_json: str) -> pd.DataFrame:
    with open(path_json, "r", encoding="utf-8") as f:
        data = json.load(f)
    rows = []
    for raw_host, obj in data.items():
        e = etld1(raw_host)
        ra = obj.get("rubric_assessment", {}) or {}
        dm = ra.get("disclosure_map", {}) or {}
        bc = obj.get("behavioral_claims", {}) or {}
        just = bc.get("justifications", {}) or {}
        rows.append({
            "host_raw": raw_host,
            "domain_etld1": e,
            "online_data_practices": obj.get("online_data_practices", ""),
            "offline_data_practices": obj.get("offline_data_practices", ""),

            "accuracy_score": ra.get("accuracy_score", np.nan),
            "completeness_score": ra.get("completeness_score", np.nan),
            "usability_score": ra.get("usability_score", np.nan),

            **{f"disc.{k}": bool(dm.get(k, False)) for k in DISC_KEYS},

            **{f"beh.{k}": bc.get(k, "unspecified") for k in BEHAVIOR_KEYS},

            **{f"just.{k}": just.get(k, "") for k in BEHAVIOR_KEYS},
        })
    df = pd.DataFrame(rows)

    for k in BEHAVIOR_KEYS:
        col = f"beh.{k}"
        df[col] = df[col].map(lambda v: ("true" if v is True else "false" if v is False else str(v).strip().lower()))
        df[col] = df[col].replace({"none":"unspecified","nan":"unspecified","": "unspecified"})

    for k in SCORE_KEYS:
        df[k] = pd.to_numeric(df[k], errors="coerce")
    return df

def load_ccpa_df(path_csv: str) -> pd.DataFrame:
    df = pd.read_csv(path_csv, dtype=str)
    df["website"] = df["website"].astype(str).str.strip()
    df["etld_plus1"] = df["website"].apply(etld1)

    df = df.drop_duplicates(subset=["etld_plus1"], keep="first")

    def ccpa_label(val):
        if pd.isna(val): return "Unspecified"
        s = str(val).strip().lower()
        return "True" if s == "subjected" else "False"
    df["ccpa_subjectivity"] = df["ccpa"].apply(ccpa_label)

    def industry_label(val):
        s = (str(val).strip() if isinstance(val, str) else "").lower()
        if s in {"", "nan", "unknown"}:
            return "Unspecified"
        return str(val).strip()
    df["Industry_norm"] = df["Industry"].apply(industry_label)
    return df[["etld_plus1","ccpa_subjectivity","Industry_norm"]]

def attach_ccpa_industry(policy_df: pd.DataFrame, ccpa_df: pd.DataFrame) -> pd.DataFrame:
    m = policy_df.merge(ccpa_df, left_on="domain_etld1", right_on="etld_plus1", how="left")
    m["ccpa_subjectivity"] = m["ccpa_subjectivity"].fillna("Unspecified")
    m["Industry_norm"] = m["Industry_norm"].fillna("Unspecified")
    return m.drop(columns=["etld_plus1"])

def is_not_mentioned(s: str) -> bool:
    if not isinstance(s, str): return True
    t = " ".join(s.split()).strip().lower()
    return t in {"", "not mentioned", "not mentioned."}

def compute_metrics(df: pd.DataFrame) -> dict:
    total = len(df)

    online_present = (~df["online_data_practices"].map(is_not_mentioned)).sum()
    offline_present = (~df["offline_data_practices"].map(is_not_mentioned)).sum()

    online_lens = df.loc[~df["online_data_practices"].map(is_not_mentioned), "online_data_practices"].map(lambda s: len(" ".join(s.split())))
    offline_lens = df.loc[~df["offline_data_practices"].map(is_not_mentioned), "offline_data_practices"].map(lambda s: len(" ".join(s.split())))

    beh_dist = {}
    for k in BEHAVIOR_KEYS:
        vc = df[f"beh.{k}"].value_counts().reindex(["true","false","unspecified"]).fillna(0).astype(int)
        beh_dist[k] = vc.to_dict()

    disc_prev = {k: float(df[f"disc.{k}"].mean()*100.0) for k in DISC_KEYS}

    score_desc = df[SCORE_KEYS].describe().T

    score_hists = {k: df[k].value_counts().sort_index().astype(int).to_dict() for k in SCORE_KEYS}

    unspecified_frac = (df[[f"beh.{k}" for k in BEHAVIOR_KEYS]] == "unspecified").mean(axis=1)
    opacity_stats = {
        "mean_opacity": float(unspecified_frac.mean()),
        "quantiles": unspecified_frac.quantile([0,0.25,0.5,0.75,1]).to_dict()
    }
    return {
        "n": total,
        "online_present": int(online_present),
        "online_missing": int(total - online_present),
        "offline_present": int(offline_present),
        "offline_missing": int(total - offline_present),
        "online_len": {
            "mean": float(online_lens.mean()) if len(online_lens) else np.nan,
            "min": int(online_lens.min()) if len(online_lens) else np.nan,
            "max": int(online_lens.max()) if len(online_lens) else np.nan,
        },
        "offline_len": {
            "mean": float(offline_lens.mean()) if len(offline_lens) else np.nan,
            "min": int(offline_lens.min()) if len(offline_lens) else np.nan,
            "max": int(offline_lens.max()) if len(offline_lens) else np.nan,
        },
        "behaviors": beh_dist,
        "disclosures_pct": disc_prev,
        "scores_describe": score_desc,
        "scores_hists": score_hists,
        "opacity": opacity_stats,
    }

def print_metrics(label: str, metrics: dict):
    n = metrics["n"]
    print(f"\n=== {label} ===")
    print(f"n = {n}")
    print(f"Online present: {metrics['online_present']} ({metrics['online_present']/n*100:.1f}%), "
          f"missing: {metrics['online_missing']} ({metrics['online_missing']/n*100:.1f}%)")
    print(f"Offline present: {metrics['offline_present']} ({metrics['offline_present']/n*100:.1f}%), "
          f"missing: {metrics['offline_missing']} ({metrics['offline_missing']/n*100:.1f}%)")
    ol = metrics["online_len"]; ofl = metrics["offline_len"]
    if not np.isnan(ol["mean"]): print(f"Online length (chars): mean={ol['mean']:.1f}, min={ol['min']}, max={ol['max']}")
    if not np.isnan(ofl["mean"]): print(f"Offline length (chars): mean={ofl['mean']:.1f}, min={ofl['min']}, max={ofl['max']}")

    print("\nBehavioral distributions (true / false / unspecified):")
    for k, d in metrics["behaviors"].items():
        print(f"- {k:35s} {d['true']:4d} / {d['false']:4d} / {d['unspecified']:4d}")

    print("\nDisclosure prevalence (% true):")
    for k, v in metrics["disclosures_pct"].items():
        print(f"- {k:25s} {v:5.1f}%")

    print("\nScore distributions (counts):")
    for k, hist in metrics["scores_hists"].items():

        line = " ".join([f"{i}:{int(hist.get(i,0))}" for i in [0,1,2,3]])
        print(f"- {k:20s} {line}")
    print("\nScore summary (describe):")
    print(metrics["scores_describe"][["count","mean","std","min","25%","50%","75%","max"]].round(3))

    print("\nOpacity (fraction unspecified across behaviors):")
    print(f"- mean={metrics['opacity']['mean_opacity']:.3f}")
    q = metrics["opacity"]["quantiles"]
    print(f"- quantiles: 0={q[0]:.3f} 0.25={q[0.25]:.3f} 0.5={q[0.5]:.3f} 0.75={q[0.75]:.3f} 1={q[1.0]:.3f}")

def example_rows(df: pd.DataFrame, n=5):

    sel = df.copy()
    sel["has_online"] = ~sel["online_data_practices"].map(is_not_mentioned)
    sel["has_offline"] = ~sel["offline_data_practices"].map(is_not_mentioned)
    ex = sel[(sel["has_online"]) | (sel["has_offline"])].sample(n=min(n, len(sel)), random_state=RANDOM_SEED)
    lines = []
    for _, r in ex.iterrows():
        lines.append(f"- {r['domain_etld1']}: "
                     f"online={norm_text(r['online_data_practices'], 180)} | "
                     f"offline={norm_text(r['offline_data_practices'], 140)}")
    return "\n".join(lines)

def save_group_csvs(group_name: str, metrics: dict, outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)

    beh_rows = []
    for b, d in metrics["behaviors"].items():
        beh_rows.append({"behavior": b, "true": d["true"], "false": d["false"], "unspecified": d["unspecified"]})
    pd.DataFrame(beh_rows).to_csv(outdir / f"{group_name}_behaviors.csv", index=False)

    pd.DataFrame([metrics["disclosures_pct"]]).to_csv(outdir / f"{group_name}_disclosures_pct.csv", index=False)

    sh = []
    for k, hist in metrics["scores_hists"].items():
        for score, cnt in sorted(hist.items()):
            sh.append({"metric": k, "score": score, "count": cnt})
    pd.DataFrame(sh).to_csv(outdir / f"{group_name}_score_hists.csv", index=False)

    metrics["scores_describe"].to_csv(outdir / f"{group_name}_score_describe.csv")

policy_df = load_policy_df(POLICY_JSON)
ccpa_df   = load_ccpa_df(CCPA_CSV)
merged    = attach_ccpa_industry(policy_df, ccpa_df)

print("########## SPLIT BY CCPA SUBJECTIVITY ##########")
for label in ["True", "False", "Unspecified"]:
    sub = merged[merged["ccpa_subjectivity"] == label].copy()
    if sub.empty:
        print(f"\n=== CCPA={label} === (n=0)")
        continue
    m = compute_metrics(sub)
    print_metrics(f"CCPA={label}", m)
    print("\nExamples:")
    print(example_rows(sub, n=5))
    save_group_csvs(f"ccpa_{label.lower()}", m, Path(OUTDIR))

print("\n########## SPLIT BY INDUSTRY ##########")
industries = merged["Industry_norm"].value_counts().index.tolist()
for ind in industries:
    sub = merged[merged["Industry_norm"] == ind].copy()
    m = compute_metrics(sub)
    print_metrics(f"Industry={ind}", m)
    print("\nExamples:")
    print(example_rows(sub, n=5))
    safe_ind = re.sub(r"[^A-Za-z0-9_.-]+", "_", ind)
    save_group_csvs(f"industry_{safe_ind}", m, Path(OUTDIR))

In [ ]:
import json
import pandas as pd
from urllib.parse import urlparse
import tldextract

policy_json_path = "ccpa_policy_audit_all.json"
ccpa_csv_path = "cookie_data_final - merged_updated.csv"

with open(policy_json_path, "r") as f:
    policy_data = json.load(f)

ccpa_df = pd.read_csv(ccpa_csv_path)

def extract_etld_plus1(domain):
    ext = tldextract.extract(domain)
    return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain

policy_domains = set(extract_etld_plus1(k) for k in policy_data.keys())
ccpa_domains = set(extract_etld_plus1(d) for d in ccpa_df['website'].dropna().astype(str))

only_in_policy = policy_domains - ccpa_domains
only_in_ccpa = ccpa_domains - policy_domains
in_both = policy_domains & ccpa_domains

print(f"Total in Policy JSON: {len(policy_domains)}")
print(f"Total in CCPA CSV: {len(ccpa_domains)}")
print(f"Websites only in Policy JSON: {len(only_in_policy)}")
print(f"Websites only in CCPA CSV: {len(only_in_ccpa)}")
print(f"Websites in both: {len(in_both)}\n")

print("=== Sample only in Policy JSON ===")
print(list(only_in_policy)[:20])

print("\n=== Sample only in CCPA CSV ===")
print(list(only_in_ccpa)[:20])

In [ ]:
import json
import pandas as pd
import tldextract

policy_json_path = "ccpa_policy_audit_all.json"
ccpa_csv_path = "cookie_data_final - merged_updated.csv"
output_json_path = "domain_comparison_results.json"

with open(policy_json_path, "r") as f:
    policy_data = json.load(f)

ccpa_df = pd.read_csv(ccpa_csv_path)

def extract_etld_plus1(domain):
    ext = tldextract.extract(domain)
    return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain

policy_domains = set(extract_etld_plus1(k) for k in policy_data.keys())
ccpa_domains = set(extract_etld_plus1(d) for d in ccpa_df['website'].dropna().astype(str))

only_in_policy = sorted(policy_domains - ccpa_domains)
only_in_ccpa = sorted(ccpa_domains - policy_domains)
in_both = sorted(policy_domains & ccpa_domains)

results = {
    "total_in_policy_json": len(policy_domains),
    "total_in_ccpa_csv": len(ccpa_domains),
    "websites_only_in_policy_json": only_in_policy,
    "websites_only_in_ccpa_csv": only_in_ccpa,
    "websites_in_both": in_both
}

with open(output_json_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"✅ Results saved to {output_json_path}")

In [ ]:
import json

INPUT = "ccpa_policy_audit_all.json"

with open(INPUT, "r", encoding="utf-8") as f:
    data = json.load(f)

num_websites = len(data.keys())
print(f"Number of websites: {num_websites}")

print("Sample websites:", list(data.keys())[:10])

In [ ]:
import json
import pandas as pd
from urllib.parse import urlparse

JSON_PATH = "ccpa_policy_audit_all.json"
CSV_PATH  = "cookie_data_final - merged_updated.csv"
CSV_SITE_COL = "website"

try:
    import tldextract
    _USE_TLD = True
except Exception:
    _USE_TLD = False

def canon_host(s: str) -> str:
    """Normalize a hostname-ish string (strip scheme, leading dot, www., lowercase)."""
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    if not s:
        return s
    if s.startswith("."):
        s = s[1:]
    if s.startswith("www."):
        s = s[4:]
    if "://" in s:
        host = urlparse(s).netloc.lower()
        if host.startswith("www."):
            host = host[4:]
        s = host or s
    return s

def reg_domain(host: str) -> str:
    """Return registrable domain (eTLD+1) if possible; else fallback to last two labels."""
    h = canon_host(host)
    if not h:
        return ""
    if _USE_TLD:
        ext = tldextract.extract(h)
        if ext.domain and ext.suffix:
            return f"{ext.domain}.{ext.suffix}"
    parts = h.split(".")
    return ".".join(parts[-2:]) if len(parts) >= 2 else h

with open(JSON_PATH, "r", encoding="utf-8") as f:
    policy_raw = json.load(f)

policy_hosts_raw = list(policy_raw.keys())
policy_regs = {reg_domain(k) for k in policy_hosts_raw if reg_domain(k)}

df = pd.read_csv(CSV_PATH)

if CSV_SITE_COL not in df.columns:
    raise ValueError(f"'{CSV_SITE_COL}' not found in {CSV_PATH} columns: {list(df.columns)}")

csv_regs = {reg_domain(s) for s in df[CSV_SITE_COL].dropna().astype(str) if reg_domain(s)}

only_in_policy = sorted(policy_regs - csv_regs)
only_in_csv    = sorted(csv_regs - policy_regs)
in_both        = sorted(policy_regs & csv_regs)

n_policy = len(policy_regs)
n_csv    = len(csv_regs)
n_both   = len(in_both)
n_only_policy = len(only_in_policy)
n_only_csv    = len(only_in_csv)

def pct(x, d):
    return (100.0 * x / d) if d else 0.0

jaccard = (n_both / (n_policy + n_csv - n_both)) if (n_policy + n_csv - n_both) else 0.0

print("=== Website Set Comparison (registrable domains) ===")
print(f"Policy JSON unique sites (eTLD+1): {n_policy}")
print(f"Merged CSV unique sites (eTLD+1):  {n_csv}")
print(f"In BOTH:                           {n_both}  ({pct(n_both, n_policy):.1f}% of policy; {pct(n_both, n_csv):.1f}% of csv)")
print(f"Only in policy JSON:               {n_only_policy}  ({pct(n_only_policy, n_policy):.1f}% of policy)")
print(f"Only in merged CSV:                {n_only_csv}  ({pct(n_only_csv, n_csv):.1f}% of csv)")
print(f"Jaccard similarity:                {jaccard:.3f}")

pd.Series(in_both, name="domain").to_csv("sites_in_both.csv", index=False)
pd.Series(only_in_policy, name="domain").to_csv("sites_only_in_policy.csv", index=False)
pd.Series(only_in_csv, name="domain").to_csv("sites_only_in_csv.csv", index=False)
print("\n✅ Wrote:")
print(" - sites_in_both.csv")
print(" - sites_only_in_policy.csv")
print(" - sites_only_in_csv.csv")

print("\nSample overlap (up to 10):", in_both[:10])
print("Sample only-in-policy (up to 10):", only_in_policy[:10])
print("Sample only-in-csv (up to 10):", only_in_csv[:10])

In [ ]:
"""
Compare domains between:
  1) ccpa_policy_audit_all.json  (dict keyed by site/URL)
  2) cookie_data_final - merged_updated.csv  (with column 'website')

- Normalize to eTLD+1 via tldextract
- Print overlaps / differences
- WHOIS each base domain (cached) and try to link "only-in-one-file" domains
  to "the other file" by shared registrant/organization/registrar.

INSTALL (in your environment):
    pip install tldextract python-whois

Note: WHOIS lookups can be slow/rate-limited. The script caches results in
      whois_cache.json and sleeps between requests.
"""

import json
import time
import re
import sys
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd

POLICY_JSON = "ccpa_policy_audit_all.json"
COOKIES_CSV = "cookie_data_final - merged_updated.csv"
WEBSITE_COL = "website"

WHOIS_CACHE_PATH = Path("whois_cache.json")
WHOIS_SLEEP_SECS = 2.0
WHOIS_LIMIT_PER_SIDE = 500

try:
    import tldextract
except Exception as e:
    print("ERROR: tldextract not installed. Run: pip install tldextract")
    raise

try:
    import whois
except Exception as e:
    print("ERROR: python-whois not installed. Run: pip install python-whois")
    raise

def canon_host(s: str) -> str:
    """Normalize a hostname-ish string (strip scheme, www., leading dot, lowercase)."""
    if s is None:
        return ""
    s = str(s).strip()
    if not s:
        return ""

    if "://" in s:
        try:
            netloc = urlparse(s).netloc
            if netloc:
                s = netloc
        except Exception:
            pass
    s = s.lower()
    if s.startswith("."):
        s = s[1:]
    if s.startswith("www."):
        s = s[4:]
    return s

def base_domain(s: str) -> str:
    """Return eTLD+1 ('domain.suffix'); falls back gracefully."""
    h = canon_host(s)
    if not h:
        return ""
    ext = tldextract.extract(h)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"

    parts = h.split(".")
    return ".".join(parts[-2:]) if len(parts) >= 2 else h

def load_policy_json(path: str) -> pd.DataFrame:
    """Load the JSON dict and return a DataFrame with original key + base_domain."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    keys = list(data.keys())
    df = pd.DataFrame({"policy_key": keys})
    df["policy_host"] = df["policy_key"].apply(canon_host)
    df["policy_base"] = df["policy_host"].apply(base_domain)

    df = df[df["policy_base"] != ""].drop_duplicates(subset=["policy_base"])
    return df

def load_cookie_csv(path: str, website_col: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if website_col not in df.columns:
        raise ValueError(f"Column '{website_col}' not found in {path}")
    out = df[[website_col]].copy()
    out["site_host"] = out[website_col].apply(canon_host)
    out["site_base"] = out["site_host"].apply(base_domain)
    out = out[out["site_base"] != ""]
    out = out.drop_duplicates(subset=["site_base"])
    return out

def norm_org_string(s: str) -> str:
    """Normalize org/registrant string for matching."""
    if not s:
        return ""
    s = str(s)

    s = s.lower()
    s = re.sub(r"[\u200b\u200c\u200d]", "", s)
    s = re.sub(r"[^a-z0-9&.\-@_\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def whois_lookup(domain: str, cache: dict) -> dict:
    """WHOIS lookup for a base domain with caching. Returns a dict:
       {'org': ..., 'registrar': ..., 'name': ..., 'emails': ...}
    """
    domain = domain.strip().lower()
    if not domain:
        return {}

    if domain in cache:
        return cache[domain]

    info = {}
    try:
        w = whois.whois(domain)

        org = w.get("org") or w.get("orgName")
        registrar = w.get("registrar")
        name = w.get("name")
        emails = w.get("emails")

        if isinstance(emails, list):
            emails = ",".join(sorted(set([str(x) for x in emails if x])))

        info = {
            "org": norm_org_string(org),
            "registrar": norm_org_string(registrar),
            "name": norm_org_string(name),
            "emails": norm_org_string(emails),
        }
    except Exception as e:
        info = {"error": str(e)}

    cache[domain] = info

    try:
        with open(WHOIS_CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)
    except Exception:
        pass

    time.sleep(WHOIS_SLEEP_SECS)
    return info

def load_whois_cache() -> dict:
    if WHOIS_CACHE_PATH.exists():
        try:
            return json.loads(WHOIS_CACHE_PATH.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}

def print_section(title: str):
    print("\n" + "="*len(title))
    print(title)
    print("="*len(title))

def main():

    print_section("Loading & normalizing")
    df_policy = load_policy_json(POLICY_JSON)
    df_ccpa   = load_cookie_csv(COOKIES_CSV, WEBSITE_COL)

    policy_set = set(df_policy["policy_base"])
    ccpa_set   = set(df_ccpa["site_base"])

    inter = sorted(policy_set & ccpa_set)
    only_policy = sorted(policy_set - ccpa_set)
    only_ccpa   = sorted(ccpa_set - policy_set)

    print(f"Policy JSON unique base domains: {len(policy_set)}")
    print(f"CCPA CSV   unique base domains: {len(ccpa_set)}")
    print(f"In both: {len(inter)}")
    print(f"Only in JSON: {len(only_policy)}")
    print(f"Only in CSV : {len(only_ccpa)}")

    pd.Series(inter, name="base_domain").to_csv("domains_in_both.csv", index=False)
    pd.Series(only_policy, name="base_domain").to_csv("domains_only_in_json.csv", index=False)
    pd.Series(only_ccpa, name="base_domain").to_csv("domains_only_in_csv.csv", index=False)
    print("Wrote: domains_in_both.csv, domains_only_in_json.csv, domains_only_in_csv.csv")

    print_section("WHOIS enrichment (cached)")
    cache = load_whois_cache()

    policy_targets = only_policy if WHOIS_LIMIT_PER_SIDE is None else only_policy[:WHOIS_LIMIT_PER_SIDE]
    ccpa_targets   = only_ccpa if WHOIS_LIMIT_PER_SIDE is None else only_ccpa[:WHOIS_LIMIT_PER_SIDE]

    domains_to_lookup = set(policy_targets) | set(ccpa_targets) | set(inter)

    whois_rows = []
    for d in domains_to_lookup:
        info = whois_lookup(d, cache)
        whois_rows.append({
            "base_domain": d,
            "org": info.get("org", ""),
            "registrar": info.get("registrar", ""),
            "name": info.get("name", ""),
            "emails": info.get("emails", ""),
            "error": info.get("error", "")
        })
    df_whois = pd.DataFrame(whois_rows)

    df_whois["side"] = ""
    df_whois.loc[df_whois["base_domain"].isin(only_policy), "side"] = "JSON_only"
    df_whois.loc[df_whois["base_domain"].isin(only_ccpa),   "side"] = "CSV_only"
    df_whois.loc[df_whois["base_domain"].isin(inter),       "side"] = "BOTH"

    df_whois.to_csv("whois_enriched_domains.csv", index=False)
    print("Wrote: whois_enriched_domains.csv")

    print_section("Cross-link by registrant/org/registrar/email")

    def match_key(row):

        for k in ("org", "registrar", "name", "emails"):
            v = row.get(k, "") if isinstance(row, dict) else row[k]
            if v:
                return (k, v)
        return ("", "")

    df_whois["best_key_kind"] = ""
    df_whois["best_key_val"]  = ""
    for i, r in df_whois.iterrows():
        k, v = match_key(r)
        df_whois.at[i, "best_key_kind"] = k
        df_whois.at[i, "best_key_val"]  = v

    json_only = df_whois[(df_whois["side"] == "JSON_only") & (df_whois["best_key_val"] != "")]
    csv_only  = df_whois[(df_whois["side"] == "CSV_only")  & (df_whois["best_key_val"] != "")]

    from collections import defaultdict
    idx = defaultdict(list)
    for _, r in csv_only.iterrows():
        idx[(r["best_key_kind"], r["best_key_val"])].append(r["base_domain"])

    suggestions = []
    for _, r in json_only.iterrows():
        key = (r["best_key_kind"], r["best_key_val"])
        candidates = idx.get(key, [])
        if candidates:
            suggestions.append({
                "json_domain": r["base_domain"],
                "csv_candidates": ", ".join(sorted(set(candidates))),
                "link_kind": key[0],
                "link_value": key[1]
            })

    df_suggest = pd.DataFrame(suggestions)
    if not df_suggest.empty:
        df_suggest.to_csv("suggested_crossfile_links_by_whois.csv", index=False)
        print("Wrote: suggested_crossfile_links_by_whois.csv")
        print(df_suggest.head(20).to_string(index=False))
    else:
        print("No cross-links found by WHOIS keys (org/registrar/name/emails).")

    print_section("Done")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nInterrupted.", file=sys.stderr)

In [ ]:
import os
import glob
import json
import re
from pathlib import Path
from typing import Dict, Any, List, Tuple

from openai import OpenAI
from tqdm import tqdm

POLICIES_DIR = "policies_text"
OUTPUT_FILE  = "ccpa_policy_audit_all.json"
MODEL        = "gpt-4o"
TEMPERATURE  = 0
MAX_OUTPUT_TOKENS = 1500

API_KEY = "REDACTED-SET-OPENAI_API_KEY-ENV-VAR"
client = OpenAI(api_key=API_KEY)

def iter_policy_files() -> List[Tuple[str, Path]]:
    """
    Returns (domain, path) pairs from POLICIES_DIR/*.txt.
    Assumes names like 'www_example_com__combined.txt'.
    """
    files = sorted(glob.glob(os.path.join(POLICIES_DIR, "*.txt")))
    pairs = []
    for p in files:
        name = Path(p).stem
        domain = name.split("__combined", 1)[0].replace("_", ".")
        pairs.append((domain, Path(p)))
    return pairs

def load_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

def load_existing_results(path: str) -> Dict[str, Any]:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            try:
                return json.load(f) or {}
            except json.JSONDecodeError:

                backup = path + ".bak"
                try:
                    os.replace(path, backup)
                    print(f"⚠️ Existing JSON was invalid. Backed up to {backup}. Starting a new file.")
                except Exception:
                    print("⚠️ Existing JSON was invalid and could not be backed up. Starting a new file.")
                return {}
    return {}

def build_prompt(domain: str, policy_text: str) -> str:
    """
    Returns a detailed instruction prompt for the LLM with clear scoring rubrics,
    definition of each metric/flag, and a strict JSON schema to follow.
    Also instructs to always provide non-empty justifications.
    """
    return f"""
You are a privacy compliance analyst evaluating a privacy policy strictly under the California Consumer Privacy Act (CCPA) and related U.S. state privacy concepts (CPRA updates included). You must rely ONLY on the policy text provided below—no external knowledge or assumptions.

Analyze the policy for: {domain}

# Output Requirements
Return **one** JSON object matching this exact schema and types:
{{
  "online_data_practices": "string (brief summary or 'Not mentioned.')",
  "offline_data_practices": "string (brief summary or 'Not mentioned.')",
  "rubric_assessment": {{
    "completeness_score": 0-3,
    "completeness_reason": "string",
    "usability_score": 0-3,
    "usability_reason": "string",
    "accuracy_score": 0-3,
    "accuracy_reason": "string",
    "policy_contradiction": true/false,
    "disclosure_map": {{
      "data_collected": true/false,
      "data_shared": true/false,
      "purpose_of_collection": true/false,
      "retention_period": true/false,
      "right_to_access": true/false,
      "right_to_delete": true/false,
      "opt_out": true/false
    }}
  }},
  "behavioral_claims": {{
    "honors_gpc": true/false/"unspecified",
    "respects_dnt": true/false/"unspecified",
    "sets_cookies_before_consent": true/false/"unspecified",
    "sets_cookies_after_rejecting_consent": true/false/"unspecified",
    "deletes_cookies_on_rejection": true/false/"unspecified",
    "uses_tracking_only_after_consent": true/false/"unspecified",
    "sells_data": true/false/"unspecified",
    "shares_with_third_parties": true/false/"unspecified",
    "justifications": {{
      "honors_gpc": "1–2 sentences. NEVER 'unspecified'. If flag is 'unspecified', explicitly say 'No explicit statement found; flag set to unspecified.' and cite the nearest relevant line(s).",
      "respects_dnt": " ... ",
      "sets_cookies_before_consent": " ... ",
      "sets_cookies_after_rejecting_consent": " ... ",
      "deletes_cookies_on_rejection": " ... ",
      "uses_tracking_only_after_consent": " ... ",
      "sells_data": " ... ",
      "shares_with_third_parties": " ... "
    }}
  }}
}}

## Section A — Policy Scope Categorization
- **online_data_practices**: Briefly summarize policy text on web/app tracking and collection (e.g., cookies, pixels, SDKs, analytics, ad tech, web forms, account signup). If not present, return "Not mentioned."
- **offline_data_practices**: Briefly summarize policy text on in-store collection, call centers, paper forms, events, or other in-person channels. If not present, return "Not mentioned."

## Section B — Policy Quality Assessment (Scored 0–3)
Score using only what the policy explicitly states.

### 1) completeness_score (0–3)
Assess whether the policy **mentions** and **explains** consumer rights central to CCPA and how to exercise them.
- **What to look for**: Access/know; deletion; opt-out of sale/share/targeted advertising; (nice-to-have but not required: correction, portability, non-discrimination, appeal).
- **Scoring guide**:
  - **0**: No meaningful mention of rights or how to exercise them.
  - **1**: Vague rights mention, minimal specifics, unclear processes.
  - **2**: Rights identified with at least some concrete instructions (e.g., link/email/form/phone), but gaps remain (e.g., missing one core right or unclear verification).
  - **3**: Rights clearly listed (access/know, delete, opt-out of sale/share/targeted advertising), with actionable mechanisms (webform/email/phone link) and basics on verification, timelines, and scope.

### 2) usability_score (0–3)
Evaluate how **actionable** the mechanisms are.
- **What to look for**: Dedicated links/buttons, DSAR portals, emails, phone numbers, clear steps, eligibility/verification guidance, response timelines, required information.
- **Scoring guide**:
  - **0**: No method to act; purely informational.
  - **1**: A method exists but is hard to find or ambiguous (e.g., generic support email, broken/missing link).
  - **2**: Clear method(s) with some guidance; minor friction or missing details.
  - **3**: Multiple clear methods + step-by-step clarity (verification, timelines, exceptions, appeal) and unambiguous labels (e.g., “Do Not Sell or Share My Personal Information”).

### 3) accuracy_score (0–3)
Evaluate legal/terminology correctness and internal consistency.
- **What to look for**: Correct use of terms (e.g., “sale” includes broad consideration; “share” for targeted advertising; “verified consumer”; “sensitive personal information” limits; “household”/“personal information” definitions), avoidance of outdated/incorrect claims, correct opt-out signal references (e.g., GPC).
- **Scoring guide**:
  - **0**: Clearly incorrect or misleading legal statements.
  - **1**: Several inaccuracies or outdated framing; ambiguous on critical terms.
  - **2**: Mostly accurate with minor issues/ambiguities.
  - **3**: Accurate, current, and aligned with CCPA/CPRA language; exceptions and scope are described correctly.

### policy_contradiction (boolean)
True if the text contains **self-contradictory** or suspicious claims (e.g., says “we do not sell data” but also describes selling, or claims to honor GPC while simultaneously stating universal signals are ignored).

### disclosure_map (booleans)
Mark **true** only if the policy explicitly discloses the item.
- **data_collected**: Lists categories collected (e.g., identifiers, geolocation, internet activity).
- **data_shared**: States data is shared with third parties (including for targeted advertising/cross-context behavioral advertising).
- **purpose_of_collection**: Explains why data is collected (e.g., service provision, security, marketing, analytics).
- **retention_period**: States a duration/criteria for retention.
- **right_to_access**: Explains the right to know/access and how to exercise it.
- **right_to_delete**: Explains the right to delete and how to exercise it; may list exceptions.
- **opt_out**: Provides a mechanism to opt out of **sale** or **share/targeted advertising** (e.g., DNSMPI page, “Your Privacy Choices” link, opt-out signal).

## Section C — Semantic Behavior Claims (True/False/"unspecified")
Use **true/false** ONLY if the policy explicitly supports the claim. Otherwise use **"unspecified"**.
For **every** behavioral flag, provide a **non-empty justification**:
- If explicit: quote/paraphrase the line(s) and cite the section title if available.
- If not explicit: write, “No explicit statement found; flag set to unspecified.” and include the nearest relevant line(s) that discuss cookies/tracking/opt-out signals/third-party sharing, if any.

## Important Rules
- Base everything **only** on the provided policy text.
- Do **not** assume. If unclear or absent, use **false** (for disclosure_map booleans) or **'unspecified'** (for behavioral flags).
- Keep summaries concise, specific, and neutral.
- Return **only** the JSON object—no prose before or after.

================== BEGIN POLICY TEXT ==================
{policy_text}
=================== END POLICY TEXT ===================
""".strip()

RUBRIC_KEYS = [
    "data_collected", "data_shared", "purpose_of_collection", "retention_period",
    "right_to_access", "right_to_delete", "opt_out"
]
BEHAVIOR_KEYS = [
    "honors_gpc", "respects_dnt", "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent", "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent", "sells_data", "shares_with_third_parties"
]

KEYWORDS_MAP = {
    "honors_gpc": ["global privacy control", "gpc", "universal opt-out", "opt-out signal", "ccpa signal"],
    "respects_dnt": ["do not track", "dnt"],
    "sets_cookies_before_consent": ["before consent", "prior to consent", "without consent", "implied consent"],
    "sets_cookies_after_rejecting_consent": ["after rejecting", "after refusal", "even if you reject", "decline"],
    "deletes_cookies_on_rejection": ["delete cookies", "remove cookies", "disable cookies", "withdraw consent"],
    "uses_tracking_only_after_consent": ["only after consent", "after you consent", "upon consent"],
    "sells_data": ["sell", "sold", "sale of personal information", "sale of personal data"],
    "shares_with_third_parties": ["share", "shared", "third parties", "cross-context behavioral advertising", "targeted advertising"]
}

def _tri(v):

    if isinstance(v, bool):
        return True if v else False
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}: return True
    if s in {"false", "no", "0"}: return False
    return "unspecified"

def _bool(v):
    if isinstance(v, bool): return v
    s = str(v).strip().lower()
    return s in {"true", "yes", "1"}

def _text(v, default="Not mentioned."):
    return v.strip() if isinstance(v, str) and v.strip() else default

def _nearest_snippet(policy_text: str, keywords: List[str], window_chars: int = 280) -> str:
    """Return a short context window around the first keyword hit, if any."""
    text = policy_text or ""
    low = text.lower()
    for kw in keywords:
        i = low.find(kw)
        if i != -1:
            start = max(0, i - window_chars // 2)
            end = min(len(text), i + len(kw) + window_chars // 2)
            snippet = text[start:end]
            snippet = re.sub(r"\s+", " ", snippet).strip()
            return snippet
    return ""

def normalize_result(raw: dict, policy_text: str) -> dict:
    out = {
        "online_data_practices": _text(raw.get("online_data_practices", "")),
        "offline_data_practices": _text(raw.get("offline_data_practices", "")),
        "rubric_assessment": {
            "completeness_score": int(raw.get("rubric_assessment", {}).get("completeness_score", 0) or 0),
            "completeness_reason": str(raw.get("rubric_assessment", {}).get("completeness_reason", ""))[:2000],
            "usability_score": int(raw.get("rubric_assessment", {}).get("usability_score", 0) or 0),
            "usability_reason": str(raw.get("rubric_assessment", {}).get("usability_reason", ""))[:2000],
            "accuracy_score": int(raw.get("rubric_assessment", {}).get("accuracy_score", 0) or 0),
            "accuracy_reason": str(raw.get("rubric_assessment", {}).get("accuracy_reason", ""))[:2000],
            "policy_contradiction": _bool(raw.get("rubric_assessment", {}).get("policy_contradiction", False)),
            "disclosure_map": {},
        },
        "behavioral_claims": {k: "unspecified" for k in BEHAVIOR_KEYS}
    }

    dm = raw.get("rubric_assessment", {}).get("disclosure_map", {}) or {}
    if "right_to_acces" in dm and "right_to_access" not in dm:
        dm["right_to_access"] = dm.pop("right_to_acces")
    for k in RUBRIC_KEYS:
        out["rubric_assessment"]["disclosure_map"][k] = _bool(dm.get(k, False))

    bc = raw.get("behavioral_claims", {}) or {}
    just = bc.get("justifications", {}) or {}
    for k in BEHAVIOR_KEYS:
        out["behavioral_claims"][k] = _tri(bc.get(k, "unspecified"))

    norm_just = {}
    for k in BEHAVIOR_KEYS:
        j = just.get(k, "")
        j_clean = j.strip() if isinstance(j, str) else ""
        if not j_clean or j_clean.lower() == "unspecified":
            snippet = _nearest_snippet(policy_text, KEYWORDS_MAP.get(k, []))
            if snippet:
                norm_just[k] = (
                    "No explicit statement found; flag set to 'unspecified'. "
                    f'Nearest relevant excerpt: "{snippet}"'
                )
            else:
                norm_just[k] = (
                    "No explicit statement found; flag set to 'unspecified'. "
                    "Policy does not discuss this item in clear terms."
                )
        else:
            norm_just[k] = j_clean[:1000]
    out["behavioral_claims"]["justifications"] = norm_just
    return out

def main():
    pairs = iter_policy_files()
    if not pairs:
        raise RuntimeError(f"No .txt files found in {POLICIES_DIR}")

    results: Dict[str, Any] = load_existing_results(OUTPUT_FILE)
    existing_domains = set(results.keys())

    new_pairs = [(d, p) for (d, p) in pairs if d not in existing_domains]

    print(f"Total policy files in folder: {len(pairs)}")
    print(f"Already in JSON: {len(existing_domains)}")
    print(f"New to process: {len(new_pairs)}\n")

    if not new_pairs:
        print("✅ No new domains. Nothing to do.")
        return

    print("🔍 Running on NEW policies (no chunking; no soft token cap)...\n")

    for domain, path in tqdm(new_pairs):
        txt = load_text(path)
        if not txt.strip():
            results[domain] = {"error": "Empty policy file"}

            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            continue

        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_OUTPUT_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "Return only one strict JSON object. No commentary."},
                    {"role": "user", "content": build_prompt(domain, txt)},
                ],
            )
            content = resp.choices[0].message.content or ""
            raw = json.loads(content)
            clean = normalize_result(raw, txt)
            results[domain] = clean
        except Exception as e:

            results[domain] = {"error": str(e)}

        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Done. Appended new results to {OUTPUT_FILE}")

if __name__ == "__main__":

    if not os.getenv("OPENAI_API_KEY"):
        print("⚠️ Please set OPENAI_API_KEY in your environment before running.")
    main()

In [ ]:
import os
import json
import pandas as pd
import tldextract
from collections import defaultdict
from typing import Dict, Any

POLICY_JSON_PATH = "ccpa_policy_audit_all.json"
CCPA_DATA_PATH   = "cookie_data_final - merged_updated.csv"

OUT_JSON = "policy_domains_ccpa_subjectivity.json"
OUT_CSV  = "policy_domains_ccpa_subjectivity.csv"

def norm_domain(s: str) -> str:
    """
    Normalize to registrable (eTLD+1), e.g.:
    'www.microsoft.com' -> 'microsoft.com'
    '.1800petmeds.com'  -> '1800petmeds.com'
    'news.bbc.co.uk'    -> 'bbc.co.uk'
    """
    if not isinstance(s, str) or not s.strip():
        return ""
    s = s.strip().lstrip(".")
    ext = tldextract.extract(s)
    if not ext.domain or not ext.suffix:
        return s.lower()
    return f"{ext.domain}.{ext.suffix}".lower()

def load_policy_domains(policy_json_path: str) -> Dict[str, Any]:
    with open(policy_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    normalized = {}
    for k, v in data.items():
        nd = norm_domain(k)
        if nd:
            normalized[nd] = v
    return normalized

def read_ccpa_table(path: str) -> pd.DataFrame:
    """
    Reads cookie/ccpa dataset from JSON (list of records) or CSV/TSV.
    Auto-detects delimiter for CSV/TSV. Keeps only needed columns if present.
    """
    _, ext = os.path.splitext(path.lower())
    if ext == ".json":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        if isinstance(obj, list):
            df = pd.DataFrame(obj)
        elif isinstance(obj, dict):
            try:
                df = pd.json_normalize(obj)
            except Exception:
                df = pd.DataFrame([obj])
        else:
            df = pd.DataFrame(obj)
    else:
        df = pd.read_csv(path, sep=None, engine="python", dtype=str)
    for col in ["ccpa", "website", "domain"]:
        if col in df.columns:
            df[col] = df[col].astype(str)
    return df

def build_ccpa_subject_map(df: pd.DataFrame) -> Dict[str, bool]:
    """
    Build a map: registrable_domain -> is_subjected (True/False),
    where True iff any row for that domain has ccpa == 'subjected' (case-insensitive).
    """
    domain_col = "website" if "website" in df.columns else ("domain" if "domain" in df.columns else None)
    if domain_col is None or "ccpa" not in df.columns:
        raise ValueError("Input CCPA dataset must have at least one of ['website','domain'] and a 'ccpa' column.")

    agg: Dict[str, bool] = defaultdict(bool)
    for _, row in df.iterrows():
        dom_raw = row.get(domain_col, "")
        ccpa_val = row.get("ccpa", "")
        nd = norm_domain(dom_raw)
        if not nd:
            continue
        is_subj = isinstance(ccpa_val, str) and ccpa_val.strip().lower() == "subjected"
        agg[nd] = agg[nd] or is_subj
    return dict(agg)

def main():

    policy_results = load_policy_domains(POLICY_JSON_PATH)
    policy_domains = set(policy_results.keys())
    if not policy_domains:
        raise RuntimeError(f"No policy domains found in {POLICY_JSON_PATH}")

    df = read_ccpa_table(CCPA_DATA_PATH)
    ccpa_map = build_ccpa_subject_map(df)

    classification = {}
    counts = {"subjected": 0, "not_subjected": 0, "unknown": 0}

    for dom in sorted(policy_domains):
        if dom in ccpa_map:
            label = "subjected" if ccpa_map[dom] else "not_subjected"
        else:
            label = "unknown"

        classification[dom] = label
        counts[label] += 1

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(classification, f, indent=2, ensure_ascii=False)

    pd.DataFrame(
        [{"domain": d, "ccpa_subjectivity": classification[d]} for d in sorted(classification.keys())]
    ).to_csv(OUT_CSV, index=False)

    total = len(policy_domains)
    print("=== CCPA Subjectivity Classification (from cookie dataset) ===")
    print(f"Total policy domains:          {total}")
    print(f"Subjected:                     {counts['subjected']}")
    print(f"Not subjected:                 {counts['not_subjected']}")
    print(f"Unknown (absent from dataset): {counts['unknown']}")
    print(f"\nSaved JSON: {OUT_JSON}")
    print(f"Saved CSV:  {OUT_CSV}")

if __name__ == "__main__":
    main()

In [ ]:
import json
from collections import Counter

INPUT_FILE = "policy_domains_ccpa_subjectivity.json"

def main():

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    counts = Counter(data.values())

    total = len(data)
    print("=== CCPA Subjectivity Counts ===")
    print(f"Total domains: {total}")
    for label, count in counts.items():
        print(f"{label}: {count}")

if __name__ == "__main__":
    main()

In [ ]:
import json
import pandas as pd
import tldextract
from pathlib import Path

POLICY_JSON = "policy_domains_ccpa_subjectivity.json"
COOKIE_FILE = "cookie_data_final - merged_updated.csv"

UNIQUE_COOKIE_TXT = "unique_cookie_websites.txt"
ONLY_IN_POLICY_TXT = "only_in_policy.txt"
ONLY_IN_COOKIE_TXT = "only_in_cookie.txt"
COMMON_TXT = "common.txt"

def norm_domain(s: str) -> str | None:
    if not isinstance(s, str):
        return None
    s = s.strip().lstrip(".").lower()
    if not s:
        return None
    ext = tldextract.extract(s)
    if not ext.domain or not ext.suffix:

        return s if "." in s else None
    return f"{ext.domain}.{ext.suffix}"

def load_policy_domains(path: str) -> set[str]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = set()
    for k in data.keys():
        nd = norm_domain(k)
        if nd:
            out.add(nd)
    return out

def load_cookie_unique_domains(path: str) -> set[str]:

    df = pd.read_csv(path, sep=None, engine="python", dtype=str, encoding="utf-8-sig")

    df.columns = [c.strip() for c in df.columns]

    candidates = []
    for col in ("website", "domain"):
        if col in df.columns:
            candidates.append(df[col].dropna().astype(str))
    if not candidates:
        raise ValueError("Cookie file must include a 'website' and/or 'domain' column.")

    series = pd.concat(candidates, ignore_index=True)

    series = series[series.str.strip().ne("")]

    normalized = series.apply(norm_domain).dropna().astype(str)
    return set(normalized.unique())

def main():
    policy_domains = load_policy_domains(POLICY_JSON)
    cookie_domains = load_cookie_unique_domains(COOKIE_FILE)

    Path(UNIQUE_COOKIE_TXT).write_text("\n".join(sorted(cookie_domains)), encoding="utf-8")

    common = policy_domains & cookie_domains
    only_in_policy = policy_domains - cookie_domains
    only_in_cookie = cookie_domains - policy_domains

    print("=== Domain Set Comparison (normalized to registrable domain) ===")
    print(f"Total policy domains: {len(policy_domains)}")
    print(f"Total cookie domains (unique): {len(cookie_domains)}")
    print(f"Domains in common: {len(common)}")
    print(f"Domains only in policy JSON: {len(only_in_policy)}")
    print(f"Domains only in cookie file: {len(only_in_cookie)}")

    def sample(s, n=20):
        return ", ".join(list(sorted(s))[:n]) if s else "(none)"

    print("\nSample only-in-policy (up to 20):")
    print(sample(only_in_policy))
    print("\nSample only-in-cookie (up to 20):")
    print(sample(only_in_cookie))

    Path(ONLY_IN_POLICY_TXT).write_text("\n".join(sorted(only_in_policy)), encoding="utf-8")
    Path(ONLY_IN_COOKIE_TXT).write_text("\n".join(sorted(only_in_cookie)), encoding="utf-8")
    Path(COMMON_TXT).write_text("\n".join(sorted(common)), encoding="utf-8")

    print(f"\nSaved unique cookie domains to: {UNIQUE_COOKIE_TXT}")
    print(f"Saved only-in-policy list to:   {ONLY_IN_POLICY_TXT}")
    print(f"Saved only-in-cookie list to:   {ONLY_IN_COOKIE_TXT}")
    print(f"Saved common list to:           {COMMON_TXT}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import tldextract
import json

csv_path = "cookie_data_final - merged_updated.csv"

df = pd.read_csv(csv_path)

def normalize_domain(raw_domain: str) -> str:
    """Extract eTLD+1 (registrable domain)."""
    if pd.isna(raw_domain):
        return None
    raw_domain = raw_domain.strip().lower()
    ext = tldextract.extract(raw_domain)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return raw_domain

raw_domains = set()
normalized_domains = set()

for col in ["website", "domain"]:
    if col in df.columns:
        for val in df[col].dropna():
            raw = val.strip().lower()
            raw_domains.add(raw)
            normalized = normalize_domain(raw)
            if normalized:
                normalized_domains.add(normalized)

print(f"Unique raw domains/websites: {len(raw_domains)}")
print(f"Unique normalized eTLD+1 domains: {len(normalized_domains)}")

with open("unique_normalized_domains.txt", "w") as f:
    for d in sorted(normalized_domains):
        f.write(d + "\n")

with open("unique_normalized_domains.json", "w") as f:
    json.dump(sorted(normalized_domains), f, indent=2)

print("Saved unique domains to unique_normalized_domains.txt and unique_normalized_domains.json")

In [ ]:
import pandas as pd
import tldextract
import json
import os

file_path = "cookie_data_final - Banner_present.csv"

ext = os.path.splitext(file_path)[1].lower()
if ext == ".csv":
    df = pd.read_csv(file_path)
elif ext in [".xlsx", ".xls"]:
    df = pd.read_excel(file_path)
else:
    raise ValueError("Unsupported file type. Please use CSV or Excel.")

def normalize_domain(raw_domain: str) -> str:
    """Extract eTLD+1 (registrable domain)."""
    if pd.isna(raw_domain):
        return None
    raw_domain = raw_domain.strip().lower()
    ext = tldextract.extract(raw_domain)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return raw_domain

raw_domains = set()
normalized_domains = set()

for col in ["website", "domain"]:
    if col in df.columns:
        for val in df[col].dropna():
            raw = val.strip().lower()
            raw_domains.add(raw)
            normalized = normalize_domain(raw)
            if normalized:
                normalized_domains.add(normalized)

print(f"Unique raw domains/websites: {len(raw_domains)}")
print(f"Unique normalized eTLD+1 domains: {len(normalized_domains)}")

with open("banner_raw_domains.json", "w") as f:
    json.dump(sorted(raw_domains), f, indent=2)

with open("banner_normalized_domains.json", "w") as f:
    json.dump(sorted(normalized_domains), f, indent=2)

print("Saved banner_raw_domains.json and banner_normalized_domains.json")

In [ ]:
import pandas as pd
import tldextract
import json
import os

file_path = "cookie_data_final - Banner_not_present.csv"

ext = os.path.splitext(file_path)[1].lower()
if ext == ".csv":
    df = pd.read_csv(file_path)
elif ext in [".xlsx", ".xls"]:
    df = pd.read_excel(file_path)
else:
    raise ValueError("Unsupported file type. Please use CSV or Excel.")

def normalize_domain(raw_domain: str) -> str:
    """Extract eTLD+1 (registrable domain)."""
    if pd.isna(raw_domain):
        return None
    raw_domain = raw_domain.strip().lower()
    ext = tldextract.extract(raw_domain)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return raw_domain

raw_domains = set()
normalized_domains = set()

for col in ["website", "domain"]:
    if col in df.columns:
        for val in df[col].dropna():
            raw = val.strip().lower()
            raw_domains.add(raw)
            normalized = normalize_domain(raw)
            if normalized:
                normalized_domains.add(normalized)

print(f"Unique raw domains/websites: {len(raw_domains)}")
print(f"Unique normalized eTLD+1 domains: {len(normalized_domains)}")

with open("banner_raw_domains.json", "w") as f:
    json.dump(sorted(raw_domains), f, indent=2)

with open("banner_normalized_domains.json", "w") as f:
    json.dump(sorted(normalized_domains), f, indent=2)

print("Saved banner_raw_domains.json and banner_normalized_domains.json")

In [ ]:
import pandas as pd
import tldextract
import json
import os

file_path = "cookie_data_final - Banner_present.csv"

ext = os.path.splitext(file_path)[1].lower()
if ext == ".csv":
    df = pd.read_csv(file_path)
elif ext in [".xlsx", ".xls"]:
    df = pd.read_excel(file_path)
else:
    raise ValueError("Unsupported file type. Please use CSV or Excel.")

def normalize_domain(raw_domain: str) -> str:
    """Extract eTLD+1 (registrable domain)."""
    if pd.isna(raw_domain):
        return None
    raw_domain = str(raw_domain).strip().lower()
    ext = tldextract.extract(raw_domain)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return raw_domain

raw_domains = set()
normalized_domains = set()

for val in df["domain"].dropna():
    raw = str(val).strip().lower()
    raw_domains.add(raw)
    normalized = normalize_domain(raw)
    if normalized:
        normalized_domains.add(normalized)

print(f"Unique raw 'domain' entries: {len(raw_domains)}")
print(f"Unique normalized eTLD+1 domains: {len(normalized_domains)}")

with open("banner_raw_domains.json", "w") as f:
    json.dump(sorted(raw_domains), f, indent=2)

with open("banner_normalized_domains.json", "w") as f:
    json.dump(sorted(normalized_domains), f, indent=2)

print("Saved banner_raw_domains.json and banner_normalized_domains.json")

In [ ]:
import pandas as pd
import tldextract
import json
from urllib.parse import urlsplit

csv_path = "cookie_data_final - merged_updated.csv"

df = pd.read_csv(csv_path, sep=None, engine="python", dtype=str, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]

def to_host(raw: str) -> str | None:
    """
    Convert a raw input (possibly a URL or bare domain) into a cleaned host:
    - strips scheme, credentials, port, path
    - removes leading dots and lowercases
    """
    if not isinstance(raw, str):
        return None
    s = raw.strip()
    if not s:
        return None

    parts = urlsplit(s if "://" in s else "http://" + s)
    host = parts.netloc or parts.path

    host = host.split("@")[-1].split(":")[0]
    host = host.strip().lstrip(".").lower()
    return host or None

def normalize_etld1(raw: str) -> str | None:
    """
    Extract eTLD+1 (registrable domain) from a raw input.
    Falls back to cleaned host if tldextract cannot parse.
    """
    host = to_host(raw)
    if not host:
        return None
    ext = tldextract.extract(host)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}".lower()
    return host

raw_domains = set()
normalized_domains = set()

for col in ["website", "domain"]:
    if col in df.columns:
        for val in df[col].dropna():
            host = to_host(val)
            if host:
                raw_domains.add(host)
                etld1 = normalize_etld1(host)
                if etld1:
                    normalized_domains.add(etld1)

print(f"Unique raw domains/websites: {len(raw_domains)}")
print(f"Unique normalized eTLD+1 domains: {len(normalized_domains)}")

with open("unique_raw_domains.txt", "w", encoding="utf-8") as f:
    for d in sorted(raw_domains):
        f.write(d + "\n")
with open("unique_raw_domains.json", "w", encoding="utf-8") as f:
    json.dump(sorted(raw_domains), f, indent=2)

with open("unique_normalized_domains.txt", "w", encoding="utf-8") as f:
    for d in sorted(normalized_domains):
        f.write(d + "\n")
with open("unique_normalized_domains.json", "w", encoding="utf-8") as f:
    json.dump(sorted(normalized_domains), f, indent=2)

print("Saved:")
print(" - unique_raw_domains.txt / .json")
print(" - unique_normalized_domains.txt / .json")

In [ ]:
import pandas as pd, json, re, os
from urllib.parse import urlsplit
from pathlib import Path

CSV_PATH = "data_source.csv"
POLICY_JSON_PATH = "policy_domains_ccpa_subjectivity.json"

try:
    import tldextract
    HAVE_TLDEXTRACT = True
except Exception:
    HAVE_TLDEXTRACT = False

def to_host(raw: str) -> str | None:
    """
    Convert arbitrary input (URL or domain) to a cleaned host:
    - adds dummy scheme if needed to parse
    - strips credentials, port, path
    - trims leading dots; lowercases
    """
    if not isinstance(raw, str):
        return None
    s = raw.strip()
    if not s:
        return None
    parts = urlsplit(s if "://" in s else "http://" + s)
    host = (parts.netloc or parts.path).split("@")[-1].split(":")[0].strip().lstrip(".").lower()
    return host or None

COMMON_PUBLIC_SUFFIX_2 = {
    "co.uk","org.uk","gov.uk","ac.uk","sch.uk",
    "com.au","net.au","org.au","edu.au","gov.au",
    "com.br","com.mx","com.tr","com.cn","com.hk","com.sg","com.tw",
    "co.in","co.jp","co.kr","co.id","co.za","co.nz","co.ke","co.il",
    "gov.cn","gov.br","gov.mx","gov.tr",
}

def etld1_heuristic(host: str) -> str | None:
    labels = host.split(".")
    if len(labels) < 2:
        return None
    last_two = ".".join(labels[-2:])
    last_three = ".".join(labels[-3:]) if len(labels) >= 3 else last_two
    if last_two in COMMON_PUBLIC_SUFFIX_2 and len(labels) >= 3:
        return last_three
    return last_two

def normalize_etld1(raw: str) -> str | None:
    """Normalize any input to eTLD+1; prefer tldextract if available."""
    host = to_host(raw)
    if not host:
        return None
    if HAVE_TLDEXTRACT:
        ext = tldextract.extract(host)
        if ext.domain and ext.suffix:
            return f"{ext.domain}.{ext.suffix}".lower()
        return host
    return etld1_heuristic(host) or host

df = pd.read_csv(CSV_PATH, sep=None, engine="python", dtype=str, encoding="utf-8-sig")
with open(POLICY_JSON_PATH, "r", encoding="utf-8") as f:
    policy_map = json.load(f)

candidate_cols = []
for c in df.columns:
    if re.search(r'\b(domain|website|host(name)?|url)\b', c, re.I):
        candidate_cols.append(c)
candidate_cols = list(dict.fromkeys(candidate_cols))

raw_hosts = set()
csv_etld1 = set()
for col in candidate_cols:
    for val in df[col].dropna():
        host = to_host(val)
        if host:
            raw_hosts.add(host)
            nd = normalize_etld1(host)
            if nd:
                csv_etld1.add(nd)

policy_etld1 = set()
for k in policy_map.keys():
    nd = normalize_etld1(k)
    if nd:
        policy_etld1.add(nd)

common = csv_etld1 & policy_etld1
only_in_csv = csv_etld1 - policy_etld1
only_in_policy = policy_etld1 - csv_etld1

print("=== Website Comparison (normalized to eTLD+1) ===")
print(f"CSV file: {CSV_PATH}")
print(f"Policy JSON: {POLICY_JSON_PATH}")
print(f"CSV shape: {df.shape}")
print(f"Detected domain-like columns: {candidate_cols}")
print(f"Unique CSV hosts (raw): {len(raw_hosts)}")
print(f"Unique CSV eTLD+1 domains: {len(csv_etld1)}")
print(f"Unique Policy eTLD+1 domains: {len(policy_etld1)}")
print(f"Common domains: {len(common)}")
print(f"Only in CSV: {len(only_in_csv)}")
print(f"Only in Policy JSON: {len(only_in_policy)}")
print(f"tldextract available: {HAVE_TLDEXTRACT}")

out_dir = Path(".")
(out_dir / "domains_common.txt").write_text("\n".join(sorted(common)), encoding="utf-8")
(out_dir / "domains_only_in_csv.txt").write_text("\n".join(sorted(only_in_csv)), encoding="utf-8")
(out_dir / "domains_only_in_policy.txt").write_text("\n".join(sorted(only_in_policy)), encoding="utf-8")

print("\nSaved:")
print(" - domains_common.txt")
print(" - domains_only_in_csv.txt")
print(" - domains_only_in_policy.txt")

In [ ]:
import json
from pathlib import Path
import pandas as pd
import tldextract

def norm_domain(s):
    if s is None: return None
    s = str(s).strip().lower()
    if not s: return None
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return None

def as_bool(x):
    if isinstance(x, bool): return x
    if x is None: return None
    s = str(x).strip().lower()
    if s in {"true","yes","1"}: return True
    if s in {"false","no","0"}: return False
    return None

def as_tri(x):
    if isinstance(x, str) and x.strip().lower()=="unspecified":
        return "unspecified"
    b = as_bool(x)
    if b is True: return "true"
    if b is False: return "false"
    return "unspecified"

def tri_dist(series):
    counts = series.value_counts(dropna=False)
    return pd.Series({
        "true": int(counts.get("true",0)),
        "false": int(counts.get("false",0)),
        "unspecified": int(counts.get("unspecified",0)),
    })

policies_path = Path("ccpa_policy_audit_all_clean.json")
subjectivity_path = Path("policy_domains_ccpa_subjectivity.json")

with open(subjectivity_path,"r",encoding="utf-8") as f:
    subj_map = json.load(f)
with open(policies_path,"r",encoding="utf-8") as f:
    policies_data = json.load(f)

subj_df = pd.DataFrame([
    {"domain": norm_domain(k), "subjectivity": str(v)}
    for k,v in subj_map.items() if norm_domain(k)
]).drop_duplicates("domain")

rows=[]
for key, blob in policies_data.items():
    d = norm_domain(key)
    if not d: continue
    rubric = (blob or {}).get("rubric_assessment",{}) or {}
    disclosure = rubric.get("disclosure_map",{}) or {}
    beh = (blob or {}).get("behavioral_claims",{}) or {}
    rows.append({
        "domain": d,
        "completeness_score": rubric.get("completeness_score"),
        "usability_score": rubric.get("usability_score"),
        "accuracy_score": rubric.get("accuracy_score"),
        "policy_contradiction": as_bool(rubric.get("policy_contradiction")),
        "disc_data_collected": as_bool(disclosure.get("data_collected")),
        "disc_data_shared": as_bool(disclosure.get("data_shared")),
        "disc_purpose_of_collection": as_bool(disclosure.get("purpose_of_collection")),
        "disc_retention_period": as_bool(disclosure.get("retention_period")),
        "disc_right_to_access": as_bool(disclosure.get("right_to_access")),
        "disc_right_to_delete": as_bool(disclosure.get("right_to_delete")),
        "disc_opt_out": as_bool(disclosure.get("opt_out")),
        "honors_gpc": as_tri(beh.get("honors_gpc")),
        "respects_dnt": as_tri(beh.get("respects_dnt")),
        "sets_cookies_before_consent": as_tri(beh.get("sets_cookies_before_consent")),
        "sets_cookies_after_rejecting_consent": as_tri(beh.get("sets_cookies_after_rejecting_consent")),
        "deletes_cookies_on_rejection": as_tri(beh.get("deletes_cookies_on_rejection")),
        "uses_tracking_only_after_consent": as_tri(beh.get("uses_tracking_only_after_consent")),
        "sells_data": as_tri(beh.get("sells_data")),
        "shares_with_third_parties": as_tri(beh.get("shares_with_third_parties")),
    })
pol_df=pd.DataFrame(rows)

merged=pol_df.merge(subj_df,on="domain",how="left")
merged["subjectivity"]=merged["subjectivity"].fillna("unknown")

counts=merged.groupby("subjectivity")["domain"].nunique().rename("n_sites").reset_index()

rubric_cols=["completeness_score","usability_score","accuracy_score"]
rubric_stats=(merged.groupby("subjectivity")[rubric_cols]
              .agg(["mean","median","std","min","max","count"]))
rubric_stats.columns=["_".join(c).strip() for c in rubric_stats.columns.to_flat_index()]
rubric_stats=rubric_stats.reset_index()

disc_cols=[c for c in merged.columns if c.startswith("disc_")]
disc_df=merged.copy()
for c in disc_cols: disc_df[c]=disc_df[c].astype("float")
disclosure_rates=disc_df.groupby("subjectivity")[disc_cols].mean().reset_index()

beh_cols=["honors_gpc","respects_dnt","sets_cookies_before_consent",
          "sets_cookies_after_rejecting_consent","deletes_cookies_on_rejection",
          "uses_tracking_only_after_consent","sells_data","shares_with_third_parties"]
beh_frames=[]
for col in beh_cols:
    d=(merged.groupby("subjectivity")[col].apply(tri_dist).unstack().fillna(0).astype(int).reset_index())
    d.insert(1,"claim",col)
    beh_frames.append(d)
behavioral_distribution=pd.concat(beh_frames,ignore_index=True)

pc=merged.copy()
pc["policy_contradiction_num"]=pc["policy_contradiction"].astype("float")
contradiction_rate=(pc.groupby("subjectivity")["policy_contradiction_num"]
                    .mean().rename("contradiction_rate").reset_index())

print("=== Counts by subjectivity ===")
print(counts.to_string(index=False))

print("\n=== Rubric mean scores (per subjectivity) ===")
keep=[c for c in rubric_stats.columns if c.endswith("_mean") or c=="subjectivity"]
print(rubric_stats[keep].to_string(index=False))

print("\n=== Disclosure coverage ===")
print(disclosure_rates.to_string(index=False))

print("\n=== Policy contradiction rate ===")
print(contradiction_rate.to_string(index=False))

print("\n=== Behavioral distribution (sample) ===")
print(behavioral_distribution.head(12).to_string(index=False))

In [ ]:
import json
import pandas as pd
import tldextract

def norm_domain(s):
    if s is None: return None
    s = str(s).strip().lower()
    if not s: return None
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return None

policies_path = "ccpa_policy_audit_all_clean.json"
subjectivity_path = "policy_domains_ccpa_subjectivity.json"

with open(policies_path, "r", encoding="utf-8") as f:
    policies = json.load(f)

policy_domains = {norm_domain(k) for k in policies.keys() if norm_domain(k)}
print(f"Number of unique domains in policies file: {len(policy_domains)}")

with open(subjectivity_path, "r", encoding="utf-8") as f:
    subj_map = json.load(f)

subj_domains = {norm_domain(k) for k in subj_map.keys() if norm_domain(k)}
print(f"Number of unique domains in subjectivity file: {len(subj_domains)}")

In [ ]:
import json, sys
from pathlib import Path
from collections import Counter
import tldextract

P_POL = Path("/mnt/data/ccpa_policy_audit_all_clean.json")
P_SUB = Path("/mnt/data/policy_domains_ccpa_subjectivity.json")
if not P_POL.exists(): P_POL = Path("ccpa_policy_audit_all_clean.json")
if not P_SUB.exists(): P_SUB = Path("policy_domains_ccpa_subjectivity.json")

def norm_domain(s: str):
    if s is None: return None
    s = str(s).strip().lower().strip(".")
    if not s: return None

    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return None

try:
    policies = json.loads(Path(P_POL).read_text(encoding="utf-8"))
except Exception as e:
    print(f"ERROR reading policies file {P_POL}: {e}"); raise

try:
    subj_map = json.loads(Path(P_SUB).read_text(encoding="utf-8"))
except Exception as e:
    print(f"ERROR reading subjectivity file {P_SUB}: {e}"); raise

pol_raw_keys = list(policies.keys())
sub_raw_keys = list(subj_map.keys())

pol_norm_list = [norm_domain(k) for k in pol_raw_keys]
sub_norm_list = [norm_domain(k) for k in sub_raw_keys]

pol_norm = {d for d in pol_norm_list if d}
sub_norm = {d for d in sub_norm_list if d}

pol_invalid = [k for k, d in zip(pol_raw_keys, pol_norm_list) if d is None]
sub_invalid = [k for k, d in zip(sub_raw_keys, sub_norm_list) if d is None]

def dup_report(raw_keys, norm_list, label, topn=10):
    c = Counter([d for d in norm_list if d])
    dups = [(dom, cnt) for dom, cnt in c.items() if cnt > 1]
    dups.sort(key=lambda x: x[1], reverse=True)
    print(f"\nTop {min(topn,len(dups))} domains with duplicate raw entries in {label}:")
    for dom, cnt in dups[:topn]:
        print(f"  {dom:35s} x{cnt}")

in_both = pol_norm & sub_norm
only_in_policies = pol_norm - sub_norm
only_in_subjectivity = sub_norm - pol_norm

print("=== FILES ===")
print(f"Policies file:     {P_POL}")
print(f"Subjectivity file: {P_SUB}")

print("\n=== RAW KEY COUNTS (before normalization) ===")
print(f"Policies:     {len(pol_raw_keys)}")
print(f"Subjectivity: {len(sub_raw_keys)}")

print("\n=== NORMALIZED UNIQUE DOMAIN COUNTS (eTLD+1) ===")
print(f"Policies:     {len(pol_norm)}")
print(f"Subjectivity: {len(sub_norm)}")

print("\n=== INVALID / UNPARSEABLE KEYS (ignored after normalization) ===")
print(f"Policies invalid:     {len(pol_invalid)}")
print(f"Subjectivity invalid: {len(sub_invalid)}")

print("\n=== OVERLAP ===")
print(f"In both:              {len(in_both)}")
print(f"Only in policies:     {len(only_in_policies)}")
print(f"Only in subjectivity: {len(only_in_subjectivity)}")

def sample(lst, n=20):
    return sorted(list(lst))[:min(n, len(lst))]

print("\nSample ONLY IN POLICIES (first 20):")
for d in sample(only_in_policies): print("  ", d)

print("\nSample ONLY IN SUBJECTIVITY (first 20):")
for d in sample(only_in_subjectivity): print("  ", d)

dup_report(pol_raw_keys, pol_norm_list, "policies")
dup_report(sub_raw_keys, sub_norm_list, "subjectivity")

out_dir = Path("domain_diff_outputs"); out_dir.mkdir(exist_ok=True)
(Path(out_dir/"only_in_policies.txt")).write_text("\n".join(sorted(only_in_policies)), encoding="utf-8")
(Path(out_dir/"only_in_subjectivity.txt")).write_text("\n".join(sorted(only_in_subjectivity)), encoding="utf-8")
(Path(out_dir/"in_both.txt")).write_text("\n".join(sorted(in_both)), encoding="utf-8")
(Path(out_dir/"policies_invalid_keys.txt")).write_text("\n".join(map(str, pol_invalid)), encoding="utf-8")
(Path(out_dir/"subjectivity_invalid_keys.txt")).write_text("\n".join(map(str, sub_invalid)), encoding="utf-8")

print(f"\nWrote detailed lists to: {out_dir.resolve()}")

In [ ]:
import json, re, unicodedata
from pathlib import Path
from collections import Counter
import tldextract

P_POL = Path("/mnt/data/ccpa_policy_audit_all_clean.json")
P_SUB = Path("/mnt/data/policy_domains_ccpa_subjectivity.json")
if not P_POL.exists(): P_POL = Path("ccpa_policy_audit_all_clean.json")
if not P_SUB.exists(): P_SUB = Path("policy_domains_ccpa_subjectivity.json")

_ZW = ''.join(chr(c) for c in range(0x110000) if unicodedata.category(chr(c)).startswith(('C','Z')))

SAFE = re.compile(r"[^A-Za-z0-9\.\-/:?&=_]")

def strip_invisibles(s: str) -> str:

    return ''.join(ch for ch in s if not unicodedata.category(ch).startswith('C')).strip()

def soft_clean(s: str) -> str:

    s = strip_invisibles(s)
    s = s.strip().strip('"\''"“”‘’`")
    s = re.sub(r"\s+", " ", s)
    return s

def norm_domain(s):
    if s is None: return None
    s = soft_clean(str(s).lower())
    if not s: return None

    if " " in s: s = s.split(" ")[0]

    s = SAFE.sub("", s)

    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        host = f"{ext.domain}.{ext.suffix}"

        try:
            host = host.encode("idna").decode("ascii")
        except Exception:
            pass
        return host
    return None

def collect_domains_from_json(obj):
    """Recursively walk any JSON-like object and extract eTLD+1 from any string-looking values."""
    out = set()
    def walk(x):
        if isinstance(x, dict):
            for k,v in x.items():
                d = norm_domain(k)
                if d: out.add(d)
                walk(v)
        elif isinstance(x, (list, tuple, set)):
            for it in x: walk(it)
        elif isinstance(x, str):
            d = norm_domain(x)
            if d: out.add(d)

    walk(obj)
    return out

policies = json.loads(P_POL.read_text(encoding="utf-8"))
subj_map = json.loads(P_SUB.read_text(encoding="utf-8"))

subj_domains = {norm_domain(k) for k in subj_map.keys() if norm_domain(k)}

pol_top = {norm_domain(k) for k in policies.keys() if norm_domain(k)}

pol_deep = collect_domains_from_json(policies)

pol_all = pol_top | pol_deep

in_both = pol_all & subj_domains
only_in_policies = pol_all - subj_domains
only_in_subjectivity = subj_domains - pol_all

print("=== FILES ===")
print(f"Policies file:     {P_POL}")
print(f"Subjectivity file: {P_SUB}")

print("\n=== POLICY DOMAINS ===")
print(f"Top-level unique (normalized): {len(pol_top)}")
print(f"Deep-scan unique (normalized): {len(pol_deep)}")
print(f"Union (policy_all):            {len(pol_all)}")

print("\n=== SUBJECTIVITY DOMAINS ===")
print(f"Unique (normalized): {len(subj_domains)}")

print("\n=== OVERLAP ===")
print(f"In both:                {len(in_both)}")
print(f"Only in policies:       {len(only_in_policies)}")
print(f"Only in subjectivity:   {len(only_in_subjectivity)}")

check_examples = [
    "aessuccess.org","allrecipes.com","arkive.org","avast.com","avira.com",
    "berries.com","bhg.com","bkstr.com","bookoo.com","calibre-ebook.com",
    "carwale.com","cnbc.com","cnn.com","coastalliving.com","consumerreports.org",
    "coupons.com","dailymail.co.uk","deezer.com","dx.com","elclasificado.com"
]
print("\n=== CHECK EXAMPLES ===")
for d in check_examples:
    nd = norm_domain(d)
    print(f"{d:22s} -> {nd:22s} | in policies: {nd in pol_all} | in subjectivity: {nd in subj_domains}")

out_dir = Path("domain_diff_outputs"); out_dir.mkdir(exist_ok=True)
(out_dir/"policies_top.txt").write_text("\n".join(sorted(pol_top)), encoding="utf-8")
(out_dir/"policies_deep.txt").write_text("\n".join(sorted(pol_deep)), encoding="utf-8")
(out_dir/"policies_all.txt").write_text("\n".join(sorted(pol_all)), encoding="utf-8")
(out_dir/"subjectivity_all.txt").write_text("\n".join(sorted(subj_domains)), encoding="utf-8")
(out_dir/"in_both.txt").write_text("\n".join(sorted(in_both)), encoding="utf-8")
(out_dir/"only_in_policies.txt").write_text("\n".join(sorted(only_in_policies)), encoding="utf-8")
(out_dir/"only_in_subjectivity.txt").write_text("\n".join(sorted(only_in_subjectivity)), encoding="utf-8")
print(f"\nWrote detailed lists to: {out_dir.resolve()}")

In [ ]:
import json
import tldextract

def norm_domain(s):
    if s is None: return None
    s = str(s).strip().lower()
    if not s: return None
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return None

json_path = "ccpa_policy_audit_all.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

domains = {norm_domain(k) for k in data.keys() if norm_domain(k)}

print(f"Number of unique domains in {json_path}: {len(domains)}\n")

print("=== List of domains ===")
for d in sorted(domains):
    print(d)

In [ ]:
import json, re, unicodedata
from pathlib import Path
import tldextract

P_POL = Path("/mnt/data/ccpa_policy_audit_all.json")
P_SUB = Path("/mnt/data/policy_domains_ccpa_subjectivity.json")
if not P_POL.exists(): P_POL = Path("ccpa_policy_audit_all.json")
if not P_SUB.exists(): P_SUB = Path("policy_domains_ccpa_subjectivity.json")

SAFE = re.compile(r"[^A-Za-z0-9\.\-/:?&=_]")

def strip_invisibles(s: str) -> str:

    return ''.join(ch for ch in s if not unicodedata.category(ch).startswith('C')).strip()

def norm_domain(s):
    """Normalize a domain/URL to eTLD+1 (registrable domain)."""
    if s is None: return None
    s = strip_invisibles(str(s).lower()).strip('"\''"“”‘’`")
    if not s: return None
    if " " in s: s = s.split(" ")[0]
    s = SAFE.sub("", s)
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        host = f"{ext.domain}.{ext.suffix}"
        try:
            host = host.encode("idna").decode("ascii")
        except Exception:
            pass
        return host
    return None

policies = json.loads(P_POL.read_text(encoding="utf-8"))
subj_map = json.loads(P_SUB.read_text(encoding="utf-8"))

pol_domains = {norm_domain(k) for k in policies.keys() if norm_domain(k)}

subj_domains = {norm_domain(k) for k in subj_map.keys() if norm_domain(k)}

in_both = pol_domains & subj_domains
only_in_policies = pol_domains - subj_domains
only_in_subjectivity = subj_domains - pol_domains

print("=== FILES ===")
print(f"Policies:     {P_POL}")
print(f"Subjectivity: {P_SUB}")

print("\n=== UNIQUE DOMAIN COUNTS (normalized eTLD+1) ===")
print(f"Policies:                    {len(pol_domains)}")
print(f"Subjectivity:                {len(subj_domains)}")
print(f"Overlap (in both):           {len(in_both)}")
print(f"Only in policies:            {len(only_in_policies)}")
print(f"Only in subjectivity:        {len(only_in_subjectivity)}")

def show_sample(title, items, n=20):
    print(f"\n{title} (showing up to {n}):")
    for d in sorted(list(items))[:min(n, len(items))]:
        print("  ", d)

show_sample("Only in policies", only_in_policies)
show_sample("Only in subjectivity", only_in_subjectivity)

out_dir = Path("domain_compare_outputs"); out_dir.mkdir(exist_ok=True)
(out_dir/"policies_all.txt").write_text("\n".join(sorted(pol_domains)), encoding="utf-8")
(out_dir/"subjectivity_all.txt").write_text("\n".join(sorted(subj_domains)), encoding="utf-8")
(out_dir/"in_both.txt").write_text("\n".join(sorted(in_both)), encoding="utf-8")
(out_dir/"only_in_policies.txt").write_text("\n".join(sorted(only_in_policies)), encoding="utf-8")
(out_dir/"only_in_subjectivity.txt").write_text("\n".join(sorted(only_in_subjectivity)), encoding="utf-8")
print(f"\nWrote detailed lists to: {out_dir.resolve()}")

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import tldextract

POLICIES_PATH = "ccpa_policy_audit_all.json"
SUBJECTIVITY_PATH = "policy_domains_ccpa_subjectivity.json"

def norm_domain(s):
    if s is None: return None
    s = str(s).strip().lower()
    if not s: return None
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return None

def as_bool(x):
    if isinstance(x, bool): return x
    if x is None: return None
    s = str(x).strip().lower()
    if s in {"true","yes","1"}: return True
    if s in {"false","no","0"}: return False
    return None

def as_tri(x):

    if isinstance(x, str) and x.strip().lower()=="unspecified":
        return "unspecified"
    b = as_bool(x)
    if b is True: return "true"
    if b is False: return "false"
    return "unspecified"

def tri_dist(series: pd.Series) -> pd.Series:
    counts = series.value_counts(dropna=False)
    return pd.Series({
        "true": int(counts.get("true", 0)),
        "false": int(counts.get("false", 0)),
        "unspecified": int(counts.get("unspecified", 0)),
    })

def fmt_pct(x, digits=1):
    if pd.isna(x): return ""
    return f"{100*x:.{digits}f}\\%"

def latex_table(df: pd.DataFrame, caption: str, label: str, index=False):

    print(df.to_latex(index=index, escape=False, caption=caption, label=label, longtable=False, bold_rows=False, float_format="%.3f", na_rep=""))
    print("\n")

pol_data = json.loads(Path(POLICIES_PATH).read_text(encoding="utf-8"))
subj_map = json.loads(Path(SUBJECTIVITY_PATH).read_text(encoding="utf-8"))

subj_df = pd.DataFrame(
    [{"domain": norm_domain(k), "subjectivity": str(v)} for k, v in subj_map.items() if norm_domain(k)]
).drop_duplicates("domain")

rows = []
for key, blob in pol_data.items():
    d = norm_domain(key)
    if not d:
        continue
    blob = blob or {}
    rubric = blob.get("rubric_assessment", {}) or {}
    disc = rubric.get("disclosure_map", {}) or {}
    beh = blob.get("behavioral_claims", {}) or {}
    rows.append({
        "domain": d,

        "completeness_score": rubric.get("completeness_score"),
        "usability_score": rubric.get("usability_score"),
        "accuracy_score": rubric.get("accuracy_score"),
        "policy_contradiction": as_bool(rubric.get("policy_contradiction")),

        "disc_data_collected": as_bool(disc.get("data_collected")),
        "disc_data_shared": as_bool(disc.get("data_shared")),
        "disc_purpose_of_collection": as_bool(disc.get("purpose_of_collection")),
        "disc_retention_period": as_bool(disc.get("retention_period")),
        "disc_right_to_access": as_bool(disc.get("right_to_access")),
        "disc_right_to_delete": as_bool(disc.get("right_to_delete")),
        "disc_opt_out": as_bool(disc.get("opt_out")),

        "honors_gpc": as_tri(beh.get("honors_gpc")),
        "respects_dnt": as_tri(beh.get("respects_dnt")),
        "sets_cookies_before_consent": as_tri(beh.get("sets_cookies_before_consent")),
        "sets_cookies_after_rejecting_consent": as_tri(beh.get("sets_cookies_after_rejecting_consent")),
        "deletes_cookies_on_rejection": as_tri(beh.get("deletes_cookies_on_rejection")),
        "uses_tracking_only_after_consent": as_tri(beh.get("uses_tracking_only_after_consent")),
        "sells_data": as_tri(beh.get("sells_data")),
        "shares_with_third_parties": as_tri(beh.get("shares_with_third_parties")),
    })
pol_df = pd.DataFrame(rows)

merged = pol_df.merge(subj_df, on="domain", how="left")
merged["subjectivity"] = merged["subjectivity"].fillna("unknown")

counts = merged.groupby("subjectivity")["domain"].nunique().rename("n_sites").reset_index()
print("=== Counts by subjectivity ===")
print(counts.to_string(index=False), "\n")
latex_table(
    counts,
    caption="Counts of unique websites by CCPA subjectivity.",
    label="tab:counts_subjectivity",
    index=False
)

rubric_cols = ["completeness_score","usability_score","accuracy_score"]
rubric_stats = (
    merged.groupby("subjectivity")[rubric_cols]
    .agg(["mean","std","median","min","max","count"])
)
rubric_stats.columns = ["_".join(c).strip() for c in rubric_stats.columns.to_flat_index()]
rubric_stats = rubric_stats.reset_index()

compact = merged.groupby("subjectivity")[rubric_cols].agg(['mean','std'])
compact = compact.applymap(lambda x: np.nan if pd.isna(x) else x)
def mean_std_fmt(m,s):
    if pd.isna(m) or pd.isna(s): return ""
    return f"{m:.2f} ($\\pm$ {s:.2f})"
cm = pd.DataFrame(index=compact.index)
for c in rubric_cols:
    cm[c] = [mean_std_fmt(compact.loc[idx,(c,'mean')], compact.loc[idx,(c,'std')]) for idx in compact.index]
cm = cm.reset_index().rename(columns={"subjectivity":"Subjectivity","completeness_score":"Completeness","usability_score":"Usability","accuracy_score":"Accuracy"})

print("=== Rubric stats (mean, std, median, min, max, count) ===")
print(rubric_stats.to_string(index=False), "\n")
latex_table(
    cm,
    caption="Rubric scores by subjectivity: mean ($\\pm$ std).",
    label="tab:rubric_subjectivity",
    index=False
)

disc_cols = [c for c in merged.columns if c.startswith("disc_")]
disc_bool = merged.copy()
for c in disc_cols:
    disc_bool[c] = disc_bool[c].astype("float")
disc_rates = disc_bool.groupby("subjectivity")[disc_cols].mean().reset_index()

disc_pretty = disc_rates.copy()
for c in disc_cols:
    disc_pretty[c] = disc_pretty[c].map(fmt_pct)
disc_pretty = disc_pretty.rename(columns={
    "subjectivity":"Subjectivity",
    "disc_data_collected":"Data collected",
    "disc_data_shared":"Data shared",
    "disc_purpose_of_collection":"Purpose of collection",
    "disc_retention_period":"Retention period",
    "disc_right_to_access":"Right to access",
    "disc_right_to_delete":"Right to delete",
    "disc_opt_out":"Opt-out",
})

print("=== Disclosure coverage (fraction True) ===")
print(disc_rates.to_string(index=False), "\n")
latex_table(
    disc_pretty,
    caption="Disclosure coverage in policies by subjectivity (percentage of policies mentioning each item).",
    label="tab:disclosures_subjectivity",
    index=False
)

beh_cols = [
    "honors_gpc","respects_dnt","sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent","deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent","sells_data","shares_with_third_parties"
]

beh_frames = []
for col in beh_cols:
    d = (
        merged.groupby("subjectivity")[col]
        .apply(tri_dist)
        .unstack()
        .fillna(0)
        .astype(int)
        .reset_index()
        .rename(columns={"subjectivity":"Subjectivity"})
    )
    d.insert(1, "Claim", col)

    tot = (d["true"] + d["false"] + d["unspecified"]).replace(0, np.nan)
    d["true_%"] = (d["true"]/tot).map(lambda x: "" if pd.isna(x) else fmt_pct(x))
    d["false_%"] = (d["false"]/tot).map(lambda x: "" if pd.isna(x) else fmt_pct(x))
    d["unspecified_%"] = (d["unspecified"]/tot).map(lambda x: "" if pd.isna(x) else fmt_pct(x))
    beh_frames.append(d)

behavioral = pd.concat(beh_frames, ignore_index=True)

beh_pct = behavioral.pivot(index=["Subjectivity"], columns="Claim", values=["true_%","false_%","unspecified_%"])

beh_pct.columns = [f"{c[1]} {c[0].replace('_',' ')}" for c in beh_pct.columns.to_flat_index()]
beh_pct = beh_pct.reset_index()

print("=== Behavioral distributions (counts & %) — head ===")
print(behavioral.head(12).to_string(index=False), "\n")
latex_table(
    beh_pct,
    caption="Behavioral claim distributions by subjectivity (percent of policies per claim labeled true/false/unspecified).",
    label="tab:behavior_subjectivity",
    index=False
)

pc = merged.copy()
pc["policy_contradiction_num"] = pc["policy_contradiction"].astype("float")
contr_rate = (
    pc.groupby("subjectivity")["policy_contradiction_num"]
    .mean()
    .rename("contradiction_rate")
    .reset_index()
)
contr_pretty = contr_rate.copy()
contr_pretty["contradiction_rate"] = contr_pretty["contradiction_rate"].map(fmt_pct)
contr_pretty = contr_pretty.rename(columns={"subjectivity":"Subjectivity","contradiction_rate":"Contradiction rate"})

print("=== Policy contradiction rate (share True) ===")
print(contr_rate.to_string(index=False), "\n")
latex_table(
    contr_pretty,
    caption="Share of policies flagged with a policy–behavior contradiction by subjectivity.",
    label="tab:contradiction_subjectivity",
    index=False
)

outdir = Path("out_subjectivity_analysis"); outdir.mkdir(exist_ok=True)
counts.to_csv(outdir/"counts_by_subjectivity.csv", index=False)
rubric_stats.to_csv(outdir/"rubric_stats_full_by_subjectivity.csv", index=False)
disc_rates.to_csv(outdir/"disclosure_rates_by_subjectivity.csv", index=False)
behavioral.to_csv(outdir/"behavioral_distribution_by_subjectivity.csv", index=False)
contr_rate.to_csv(outdir/"contradiction_rate_by_subjectivity.csv", index=False)
print(f"Wrote CSVs to: {outdir.resolve()}")

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import tldextract
import textwrap

PATHS = {
    "policies": "ccpa_policy_audit_all.json",
    "subjectivity": "policy_domains_ccpa_subjectivity.json",
}
MAX_SNIPPET_CHARS = 280
CLAIMS = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]

def norm_domain(s):
    if s is None: return None
    s = str(s).strip().lower()
    if not s: return None
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return None

def as_bool(x):
    if isinstance(x, bool): return x
    if x is None: return None
    s = str(x).strip().lower()
    if s in {"true","yes","1"}: return True
    if s in {"false","no","0"}: return False
    return None

def as_tri(x):

    if isinstance(x, str) and x.strip().lower()=="unspecified":
        return "unspecified"
    b = as_bool(x)
    if b is True: return "true"
    if b is False: return "false"
    return "unspecified"

def tri_dist(series: pd.Series) -> pd.Series:
    counts = series.value_counts(dropna=False)
    return pd.Series({
        "true": int(counts.get("true", 0)),
        "false": int(counts.get("false", 0)),
        "unspecified": int(counts.get("unspecified", 0)),
    })

def fmt_pct(x, digits=1):
    if pd.isna(x): return ""
    return f"{100*x:.{digits}f}\\%"

def latex_table(df: pd.DataFrame, caption: str, label: str, index=False, longtable=False):
    print(df.to_latex(index=index, escape=False, caption=caption, label=label,
                      longtable=longtable, bold_rows=False, na_rep=""))
    print("\n")

def clean_snippet(s: str, limit: int = MAX_SNIPPET_CHARS) -> str:
    s = (s or "").strip()
    if not s: return ""
    s = " ".join(s.split())
    return (s[:limit] + "…") if len(s) > limit else s

pol_data = json.loads(Path(PATHS["policies"]).read_text(encoding="utf-8"))
subj_map = json.loads(Path(PATHS["subjectivity"]).read_text(encoding="utf-8"))

subj_df = pd.DataFrame(
    [{"domain": norm_domain(k), "subjectivity": str(v)} for k, v in subj_map.items() if norm_domain(k)]
).drop_duplicates("domain")

rows = []
for key, blob in pol_data.items():
    d = norm_domain(key)
    if not d:
        continue
    blob = blob or {}
    rubric = blob.get("rubric_assessment", {}) or {}
    disc = rubric.get("disclosure_map", {}) or {}
    beh = blob.get("behavioral_claims", {}) or {}
    just = beh.get("justifications", {}) or {}
    row = {
        "domain": d,

        "completeness_score": rubric.get("completeness_score"),
        "usability_score": rubric.get("usability_score"),
        "accuracy_score": rubric.get("accuracy_score"),
        "policy_contradiction": as_bool(rubric.get("policy_contradiction")),

        "disc_data_collected": as_bool(disc.get("data_collected")),
        "disc_data_shared": as_bool(disc.get("data_shared")),
        "disc_purpose_of_collection": as_bool(disc.get("purpose_of_collection")),
        "disc_retention_period": as_bool(disc.get("retention_period")),
        "disc_right_to_access": as_bool(disc.get("right_to_access")),
        "disc_right_to_delete": as_bool(disc.get("right_to_delete")),
        "disc_opt_out": as_bool(disc.get("opt_out")),
    }

    for c in CLAIMS:
        row[c] = as_tri(beh.get(c))
        row[f"just_{c}"] = just.get(c) or ""

    row["online_data_practices"] = (blob.get("online_data_practices") or "").strip()
    row["offline_data_practices"] = (blob.get("offline_data_practices") or "").strip()
    rows.append(row)

pol_df = pd.DataFrame(rows)

merged = pol_df.merge(subj_df, on="domain", how="left")
merged["subjectivity"] = merged["subjectivity"].fillna("unknown")

counts = merged.groupby("subjectivity")["domain"].nunique().rename("n_sites").reset_index()
print("=== Counts by subjectivity ===")
print(counts.to_string(index=False), "\n")

rubric_cols = ["completeness_score","usability_score","accuracy_score"]
rubric_stats = (
    merged.groupby("subjectivity")[rubric_cols]
    .agg(["mean","std","median","min","max","count"])
)
rubric_stats.columns = ["_".join(c).strip() for c in rubric_stats.columns.to_flat_index()]
rubric_stats = rubric_stats.reset_index()
print("=== Rubric stats (mean, std, median, min, max, count) ===")
print(rubric_stats.to_string(index=False), "\n")

compact = merged.groupby("subjectivity")[rubric_cols].agg(['mean','std'])
def mean_std_fmt(m,s):
    if pd.isna(m) or pd.isna(s): return ""
    return f"{m:.2f} ($\\pm$ {s:.2f})"
cm = pd.DataFrame(index=compact.index)
for c in rubric_cols:
    cm[c] = [mean_std_fmt(compact.loc[idx,(c,'mean')], compact.loc[idx,(c,'std')]) for idx in compact.index]
cm = cm.reset_index().rename(columns={
    "subjectivity":"Subjectivity",
    "completeness_score":"Completeness",
    "usability_score":"Usability",
    "accuracy_score":"Accuracy"
})

disc_cols = [c for c in merged.columns if c.startswith("disc_")]
disc_bool = merged.copy()
for c in disc_cols:
    disc_bool[c] = disc_bool[c].astype("float")
disc_rates = disc_bool.groupby("subjectivity")[disc_cols].mean().reset_index()

disc_pretty = disc_rates.copy()
for c in disc_cols:
    disc_pretty[c] = disc_pretty[c].map(fmt_pct)
disc_pretty = disc_pretty.rename(columns={
    "subjectivity":"Subjectivity",
    "disc_data_collected":"Data collected",
    "disc_data_shared":"Data shared",
    "disc_purpose_of_collection":"Purpose of collection",
    "disc_retention_period":"Retention period",
    "disc_right_to_access":"Right to access",
    "disc_right_to_delete":"Right to delete",
    "disc_opt_out":"Opt-out",
})

print("=== Disclosure coverage (fraction True) ===")
print(disc_rates.to_string(index=False), "\n")

dist_frames = []
for claim in CLAIMS:
    d = merged.groupby("subjectivity")[claim].apply(tri_dist).unstack().fillna(0).astype(int).reset_index()
    d.insert(1, "claim", claim)

    total = (d["true"] + d["false"] + d["unspecified"]).replace(0, np.nan)
    d["true_pct"] = (d["true"]/total)
    d["false_pct"] = (d["false"]/total)
    d["unspecified_pct"] = (d["unspecified"]/total)
    dist_frames.append(d)
behavior_by_subj = pd.concat(dist_frames, ignore_index=True)

print("=== Behavioral distributions BY SUBJECTIVITY (exact counts) — head ===")
print(behavior_by_subj.head(12).to_string(index=False), "\n")

overall_frames = []
for claim in CLAIMS:
    s = tri_dist(merged[claim])
    overall_frames.append({
        "claim": claim,
        "true": int(s["true"]),
        "false": int(s["false"]),
        "unspecified": int(s["unspecified"]),
        "n": int(s["true"] + s["false"] + s["unspecified"])
    })
behavior_overall = pd.DataFrame(overall_frames)
behavior_overall["true_pct"] = behavior_overall["true"] / behavior_overall["n"]
behavior_overall["false_pct"] = behavior_overall["false"] / behavior_overall["n"]
behavior_overall["unspecified_pct"] = behavior_overall["unspecified"] / behavior_overall["n"]

print("=== Behavioral distributions OVERALL (exact counts) ===")
print(behavior_overall.to_string(index=False), "\n")

example_rows = []
for claim in CLAIMS:
    for value in ["true","false","unspecified"]:
        subset = merged[merged[claim] == value]

        just_col = f"just_{claim}"
        with_just = subset[subset[just_col].astype(str).str.strip() != ""]
        pick_df = with_just if not with_just.empty else subset
        if not pick_df.empty:
            row = pick_df.iloc[0]
            example_rows.append({
                "claim": claim,
                "value": value,
                "domain": row["domain"],
                "subjectivity": row["subjectivity"],
                "justification": clean_snippet(row.get(just_col,"")),
            })
        else:
            example_rows.append({
                "claim": claim,
                "value": value,
                "domain": "",
                "subjectivity": "",
                "justification": ""
            })

examples_df = pd.DataFrame(example_rows)
print("=== Qualitative examples (one per claim/value) ===")
print(examples_df.to_string(index=False), "\n")

ex_tex = examples_df.copy()
ex_tex["Example"] = ex_tex.apply(lambda r: f"\\texttt{{{r['domain']}}} ({r['subjectivity']}) — {r['justification']}".replace('%','\\%'), axis=1)
ex_tex = ex_tex.pivot(index="claim", columns="value", values="Example").reset_index()
ex_tex = ex_tex[["claim","true","false","unspecified"]].rename(columns={
    "claim":"Claim","true":"True example","false":"False example","unspecified":"Unspecified example"
})

pc = merged.copy()
pc["policy_contradiction_num"] = pc["policy_contradiction"].astype("float")
contr = pc.groupby("subjectivity")["policy_contradiction"].value_counts(dropna=False).unstack(fill_value=0)
contr["n"] = contr.sum(axis=1)
contr_rate = (pc.groupby("subjectivity")["policy_contradiction_num"].mean()
              .rename("contradiction_rate").to_frame())
contr_exact = contr.join(contr_rate)
print("=== Policy contradiction (exact counts per subjectivity) ===")
print(contr_exact.to_string(), "\n")

latex_table(counts.rename(columns={"subjectivity":"Subjectivity","n_sites":"$n_{\\text{sites}}$"}),
            caption="Counts of unique websites by CCPA subjectivity.",
            label="tab:counts_subjectivity", index=False)

latex_table(cm, caption="Rubric scores by subjectivity: mean ($\\pm$ std).",
            label="tab:rubric_subjectivity", index=False)

latex_table(disc_pretty, caption="Disclosure coverage in policies by subjectivity (percentage of policies mentioning each item).",
            label="tab:disclosures_subjectivity", index=False)

beh_counts_pivot = behavior_by_subj.pivot(index="subjectivity", columns="claim", values=["true","false","unspecified"])
beh_counts_pivot.columns = [f"{v} {c}" for v,c in beh_counts_pivot.columns.to_flat_index()]
beh_counts_pivot = beh_counts_pivot.reset_index().rename(columns={"subjectivity":"Subjectivity"})
latex_table(beh_counts_pivot, caption="Behavioral claim distributions by subjectivity (exact counts of true/false/unspecified).",
            label="tab:behavior_counts_subjectivity", index=False, longtable=True)

beh_overall_tex = behavior_overall.copy()
for col in ["true_pct","false_pct","unspecified_pct"]:
    beh_overall_tex[col] = beh_overall_tex[col].map(fmt_pct)
latex_table(beh_overall_tex[["claim","n","true","false","unspecified","true_pct","false_pct","unspecified_pct"]]
            .rename(columns={"claim":"Claim","n":"$n$","true":"True","false":"False","unspecified":"Unspecified",
                             "true_pct":"True \\%","false_pct":"False \\%","unspecified_pct":"Unspec. \\%"}),
            caption="Behavioral claim distributions overall (counts and percentages).",
            label="tab:behavior_overall", index=False)

latex_table(ex_tex, caption="Qualitative examples (one domain + justification snippet per claim/value).",
            label="tab:behavior_examples", index=False, longtable=True)

contr_tex = contr_exact.copy()
if "True" in contr_tex.columns: contr_tex = contr_tex.rename(columns={True:"True", False:"False", np.nan:"NA"})
contr_tex = contr_tex.reset_index().rename(columns={"subjectivity":"Subjectivity"})
contr_tex["contradiction_rate"] = contr_tex["contradiction_rate"].map(fmt_pct)
latex_table(contr_tex, caption="Policy--behavior contradiction counts and rates by subjectivity.",
            label="tab:contradiction_subjectivity", index=False)

outdir = Path("out_subjectivity_analysis"); outdir.mkdir(exist_ok=True)
counts.to_csv(outdir/"counts_by_subjectivity.csv", index=False)
rubric_stats.to_csv(outdir/"rubric_stats_full_by_subjectivity.csv", index=False)
disc_rates.to_csv(outdir/"disclosure_rates_by_subjectivity.csv", index=False)
behavior_by_subj.to_csv(outdir/"behavior_by_subjectivity_counts.csv", index=False)
behavior_overall.to_csv(outdir/"behavior_overall_counts.csv", index=False)
examples_df.to_csv(outdir/"behavior_examples_one_per_value.csv", index=False)
contr_exact.to_csv(outdir/"contradiction_by_subjectivity_counts.csv", index=True)

print(f"Wrote CSVs to: {outdir.resolve()}")

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import tldextract

PATHS = {
    "policies": "ccpa_policy_audit_all.json",
    "subjectivity": "policy_domains_ccpa_subjectivity.json",
}
MAX_SNIPPET_CHARS = 280
CLAIMS = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]

def norm_domain(s: str | None) -> str | None:
    if s is None:
        return None
    s = str(s).strip().lower()
    if not s:
        return None
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return None

def as_bool(x):
    if isinstance(x, bool): return x
    if x is None: return None
    s = str(x).strip().lower()
    if s in {"true","yes","1"}: return True
    if s in {"false","no","0"}: return False
    return None

def as_tri(x):

    if isinstance(x, str) and x.strip().lower() == "unspecified":
        return "unspecified"
    b = as_bool(x)
    if b is True: return "true"
    if b is False: return "false"
    return "unspecified"

def fmt_pct(x, digits=1):
    if pd.isna(x): return ""
    return f"{100*float(x):.{digits}f}\\%"

def clean_snippet(s: str, limit: int = MAX_SNIPPET_CHARS) -> str:
    s = (s or "").strip()
    if not s: return ""
    s = " ".join(s.split())
    return (s[:limit] + "…") if len(s) > limit else s

def latex_table(df: pd.DataFrame, caption: str, label: str, index=False, longtable=False):
    print(df.to_latex(index=index, escape=False, caption=caption, label=label,
                      longtable=longtable, bold_rows=False, na_rep=""))
    print("\n")

pol_data = json.loads(Path(PATHS["policies"]).read_text(encoding="utf-8"))
subj_map = json.loads(Path(PATHS["subjectivity"]).read_text(encoding="utf-8"))

pol_domains = {norm_domain(k) for k in pol_data.keys()}
pol_domains = {d for d in pol_domains if d}
subj_domains = {norm_domain(k) for k in subj_map.keys()}
subj_domains = {d for d in subj_domains if d}

intersect = sorted(pol_domains & subj_domains)

print("=== DOMAIN UNIVERSE (normalized eTLD+1) ===")
print(f"Policies domains:         {len(pol_domains)}")
print(f"Subjectivity domains:     {len(subj_domains)}")
print(f"Intersection (used):      {len(intersect)}")
print()

subj_df = pd.DataFrame(
    [{"domain": norm_domain(k), "subjectivity": str(v)}
     for k, v in subj_map.items() if norm_domain(k) in intersect]
).drop_duplicates("domain")

rows = []
for key, blob in pol_data.items():
    d = norm_domain(key)
    if d not in intersect:
        continue
    blob = blob or {}
    rubric = blob.get("rubric_assessment", {}) or {}
    disc = rubric.get("disclosure_map", {}) or {}
    beh = blob.get("behavioral_claims", {}) or {}
    just = beh.get("justifications", {}) or {}
    row = {
        "domain": d,

        "completeness_score": rubric.get("completeness_score"),
        "usability_score": rubric.get("usability_score"),
        "accuracy_score": rubric.get("accuracy_score"),
        "policy_contradiction": as_bool(rubric.get("policy_contradiction")),

        "disc_data_collected": as_bool(disc.get("data_collected")),
        "disc_data_shared": as_bool(disc.get("data_shared")),
        "disc_purpose_of_collection": as_bool(disc.get("purpose_of_collection")),
        "disc_retention_period": as_bool(disc.get("retention_period")),
        "disc_right_to_access": as_bool(disc.get("right_to_access")),
        "disc_right_to_delete": as_bool(disc.get("right_to_delete")),
        "disc_opt_out": as_bool(disc.get("opt_out")),
    }

    for c in CLAIMS:
        row[c] = as_tri(beh.get(c))
        row[f"just_{c}"] = (just.get(c) or "").strip()
    rows.append(row)

pol_df = pd.DataFrame(rows).drop_duplicates("domain")

merged = pol_df.merge(subj_df, on="domain", how="inner")

merged = merged.drop_duplicates("domain")
assert merged["domain"].nunique() == len(intersect), "Post-merge domain count mismatch!"

counts = merged.groupby("subjectivity")["domain"].nunique().rename("n_sites").reset_index()
total_sites = counts["n_sites"].sum()
counts["pct"] = counts["n_sites"] / total_sites

print("=== Counts by subjectivity (intersection only) ===")
print(counts.assign(pct=counts["pct"].map(lambda x: float(f"{x:.4f}"))).to_string(index=False), "\n")

denom_by_subj = dict(zip(counts["subjectivity"], counts["n_sites"]))
denom_overall = int(total_sites)

rubric_cols = ["completeness_score","usability_score","accuracy_score"]
rubric_stats = (
    merged.groupby("subjectivity")[rubric_cols]
    .agg(["mean","std","median","min","max","count"])
)
rubric_stats.columns = ["_".join(c).strip() for c in rubric_stats.columns.to_flat_index()]
rubric_stats = rubric_stats.reset_index()

print("=== Rubric stats (mean, std, median, min, max, count) ===")
print(rubric_stats.to_string(index=False), "\n")

compact = merged.groupby("subjectivity")[rubric_cols].agg(['mean','std'])
def mean_std_fmt(m,s):
    if pd.isna(m) or pd.isna(s): return ""
    return f"{m:.2f} ($\\pm$ {s:.2f})"
cm = pd.DataFrame(index=compact.index)
for c in rubric_cols:
    cm[c] = [mean_std_fmt(compact.loc[idx,(c,'mean')], compact.loc[idx,(c,'std')]) for idx in compact.index]
cm = cm.reset_index().rename(columns={
    "subjectivity":"Subjectivity",
    "completeness_score":"Completeness",
    "usability_score":"Usability",
    "accuracy_score":"Accuracy"
})

disc_cols = [c for c in merged.columns if c.startswith("disc_")]
disc_norm = merged.copy()

for c in disc_cols:
    disc_norm[c] = disc_norm[c].fillna(False).astype(int)

disc_counts = disc_norm.groupby("subjectivity")[disc_cols].sum().reset_index()
disc_rates = disc_counts.copy()
for col in disc_cols:
    disc_rates[col] = disc_rates[col] / disc_rates["subjectivity"].map(denom_by_subj)

print("=== Disclosure coverage (counts and share True) — normalized to group size ===")
disp = disc_counts.copy()
for col in disc_cols:
    disp[f"{col}_pct"] = disc_rates[col]
print(disp.to_string(index=False), "\n")

disc_pretty = disc_counts.copy()
for col in disc_cols:
    disc_pretty[col] = disc_pretty[col].astype(int).astype(str) + " (" + disc_rates[col].map(fmt_pct) + ")"
disc_pretty = disc_pretty.rename(columns={
    "subjectivity":"Subjectivity",
    "disc_data_collected":"Data collected",
    "disc_data_shared":"Data shared",
    "disc_purpose_of_collection":"Purpose of collection",
    "disc_retention_period":"Retention period",
    "disc_right_to_access":"Right to access",
    "disc_right_to_delete":"Right to delete",
    "disc_opt_out":"Opt-out",
})

def tri_counts_by_group(df: pd.DataFrame, claim: str) -> pd.DataFrame:
    g = df.groupby("subjectivity")[claim].value_counts(dropna=False).unstack(fill_value=0)
    for v in ["true","false","unspecified"]:
        if v not in g.columns:
            g[v] = 0
    g = g[["true","false","unspecified"]].reset_index()

    g["n_group"] = g["subjectivity"].map(denom_by_subj).astype(int)
    for v in ["true","false","unspecified"]:
        g[f"{v}_pct"] = g[v] / g["n_group"]
    g.insert(1, "claim", claim)
    return g

by_subj_frames = [tri_counts_by_group(merged, c) for c in CLAIMS]
behavior_by_subj = pd.concat(by_subj_frames, ignore_index=True)

print("=== Behavioral distributions BY SUBJECTIVITY (counts normalized by group n) — head ===")
print(behavior_by_subj.head(12).to_string(index=False), "\n")

overall_records = []
for claim in CLAIMS:
    vc = merged[claim].value_counts(dropna=False)
    t = int(vc.get("true", 0))
    f = int(vc.get("false", 0))
    u = int(vc.get("unspecified", 0))
    overall_records.append({
        "claim": claim,
        "true": t, "false": f, "unspecified": u,
        "n": denom_overall,
        "true_pct": t/denom_overall,
        "false_pct": f/denom_overall,
        "unspecified_pct": u/denom_overall
    })
behavior_overall = pd.DataFrame(overall_records)

print("=== Behavioral distributions OVERALL (normalized by total intersection n) ===")
print(behavior_overall.to_string(index=False), "\n")

example_rows = []
for claim in CLAIMS:
    for value in ["true","false","unspecified"]:
        subset = merged[merged[claim] == value]
        just_col = f"just_{claim}"
        with_just = subset[subset[just_col].astype(str).str.strip() != ""]
        pick_df = with_just if not with_just.empty else subset
        if not pick_df.empty:
            row = pick_df.iloc[0]
            example_rows.append({
                "claim": claim,
                "value": value,
                "domain": row["domain"],
                "subjectivity": row["subjectivity"],
                "justification": clean_snippet(row.get(just_col,"")),
            })
        else:
            example_rows.append({
                "claim": claim,
                "value": value,
                "domain": "",
                "subjectivity": "",
                "justification": ""
            })
examples_df = pd.DataFrame(example_rows)

print("=== Qualitative examples (one per claim/value) ===")
print(examples_df.to_string(index=False), "\n")

pc = merged.copy()
pc["policy_contradiction_num"] = pc["policy_contradiction"].fillna(False).astype(int)
contr_counts = pc.groupby("subjectivity")["policy_contradiction_num"].sum().rename("contr_true").to_frame()
contr_counts["n_group"] = contr_counts.index.map(denom_by_subj).astype(int)
contr_counts["rate"] = contr_counts["contr_true"] / contr_counts["n_group"]
print("=== Policy contradiction (counts and rate, normalized) ===")
print(contr_counts.to_string(), "\n")

counts_tex = counts.copy()
counts_tex["$n_{\\text{sites}}$"] = counts_tex.apply(
    lambda r: f"{int(r['n_sites'])} ({fmt_pct(r['pct'])})", axis=1
)
latex_table(counts_tex[["subjectivity","$n_{\\text{sites}}$"]].rename(columns={"subjectivity":"Subjectivity"}),
            caption="Counts of unique websites by CCPA subjectivity (intersection only).",
            label="tab:counts_subjectivity_norm", index=False)

latex_table(cm, caption="Rubric scores by subjectivity (intersection): mean ($\\pm$ std).",
            label="tab:rubric_subjectivity_norm", index=False)

latex_table(disc_pretty, caption="Disclosure coverage by subjectivity (intersection). Entries show count (percent of group).",
            label="tab:disclosures_subjectivity_norm", index=False)

beh_tex = behavior_by_subj.copy()
for v in ["true","false","unspecified"]:
    beh_tex[v] = beh_tex.apply(lambda r, vv=v: f"{int(r[vv])} ({fmt_pct(r[f'{vv}_pct'])})", axis=1)
beh_tex = beh_tex[["subjectivity","claim","true","false","unspecified"]].rename(columns={
    "subjectivity":"Subjectivity","claim":"Claim",
    "true":"True","false":"False","unspecified":"Unspecified"
})
latex_table(beh_tex, caption="Behavioral claim distributions by subjectivity (counts with percent of group).",
            label="tab:behavior_counts_subjectivity_norm", index=False)

beh_overall_tex = behavior_overall.copy()
for col in ["true","false","unspecified"]:
    beh_overall_tex[col] = beh_overall_tex.apply(lambda r, c=col: f"{int(r[c])} ({fmt_pct(r[c + '_pct'])})", axis=1)
latex_table(beh_overall_tex[["claim","true","false","unspecified","n"]]
            .rename(columns={"claim":"Claim","n":"$n$ (intersection)"}),
            caption="Behavioral claim distributions overall (normalized by total intersection).",
            label="tab:behavior_overall_norm", index=False)

ex_tex = examples_df.copy()
ex_tex["Example"] = ex_tex.apply(
    lambda r: f"\\texttt{{{r['domain']}}} ({r['subjectivity']}) — {r['justification']}".replace('%','\\%'),
    axis=1
)
ex_tex = ex_tex.pivot(index="claim", columns="value", values="Example").reset_index()
ex_tex = ex_tex[["claim","true","false","unspecified"]].rename(columns={
    "claim":"Claim","true":"True example","false":"False example","unspecified":"Unspecified example"
})
latex_table(ex_tex, caption="Qualitative examples: one domain + justification snippet per claim/value (intersection).",
            label="tab:behavior_examples_norm", index=False)

contr_tex = contr_counts.reset_index().rename(columns={"index":"Subjectivity"})
contr_tex["rate"] = contr_tex["rate"].map(fmt_pct)
latex_table(contr_tex.rename(columns={"policy_contradiction_num":"Contradiction (count)",
                                      "n_group":"$n$ (group)","rate":"Rate"}),
            caption="Policy contradiction counts and rates by subjectivity (intersection).",
            label="tab:contradiction_subjectivity_norm", index=False)

outdir = Path("out_subjectivity_analysis_norm"); outdir.mkdir(exist_ok=True)
counts.to_csv(outdir/"counts_by_subjectivity.csv", index=False)
rubric_stats.to_csv(outdir/"rubric_stats_full_by_subjectivity.csv", index=False)
disc_counts.to_csv(outdir/"disclosure_counts_by_subjectivity.csv", index=False)
disc_rates.to_csv(outdir/"disclosure_rates_by_subjectivity.csv", index=False)
behavior_by_subj.to_csv(outdir/"behavior_by_subjectivity_counts_normalized.csv", index=False)
behavior_overall.to_csv(outdir/"behavior_overall_counts_normalized.csv", index=False)
examples_df.to_csv(outdir/"behavior_examples_one_per_value.csv", index=False)
contr_counts.to_csv(outdir/"contradiction_by_subjectivity_counts_normalized.csv", index=True)

print(f"Wrote CSVs to: {outdir.resolve()}")

In [ ]:
import json
from pathlib import Path
import re
import numpy as np
import pandas as pd
import tldextract

PATH_POLICIES = "ccpa_policy_audit_all.json"
PATH_SUBJECTIVITY = "policy_domains_ccpa_subjectivity.json"
PATH_INDUSTRY_CSV = "data_source.csv"

MIN_INDUSTRY_SIZE    = 15
MIN_GROUP_SIZE       = 8
MAX_SHOW             = 25
ROUND_PCT_DIGITS     = 3
CSV_OUTDIR           = Path("out_industry_analysis")
CSV_OUTDIR.mkdir(exist_ok=True)

CLAIMS = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]
DISC_COLS = [
    "disc_data_collected",
    "disc_data_shared",
    "disc_purpose_of_collection",
    "disc_retention_period",
    "disc_right_to_access",
    "disc_right_to_delete",
    "disc_opt_out",
]
RUBRIC_COLS = ["completeness_score","usability_score","accuracy_score"]

def norm_domain(s):
    """Return normalized eTLD+1 or None."""
    if s is None:
        return None
    s = str(s).strip().lower()
    if not s:
        return None

    if s.startswith("http://") or s.startswith("https://"):

        s = re.sub(r"^https?://", "", s)
        s = s.split("/")[0]

    if "@" in s:
        s = s.split("@")[-1]
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return None

def as_bool(x):
    if isinstance(x, bool): return x
    if x is None: return None
    s = str(x).strip().lower()
    if s in {"true","yes","1"}: return True
    if s in {"false","no","0"}: return False
    return None

def as_tri(x):
    """tri-state string: 'true'/'false'/'unspecified'"""
    if isinstance(x, str) and x.strip().lower() == "unspecified":
        return "unspecified"
    b = as_bool(x)
    if b is True: return "true"
    if b is False: return "false"
    return "unspecified"

def pct_fmt(x, digits=1):
    if x is None or pd.isna(x): return ""
    return f"{100*float(x):.{digits}f}%"

def latex_table(df: pd.DataFrame, caption: str, label: str, index=False):
    print(df.to_latex(index=index, escape=False, caption=caption, label=label))
    print()

def autodetect_sep(path):
    """Try robust read for CSV/TSV (comma or tab)."""
    try:
        return pd.read_csv(path, sep=None, engine="python")
    except Exception:

        try:
            return pd.read_csv(path, sep="\t")
        except Exception:

            return pd.read_csv(path)

def find_col(df, candidates):
    """Find first matching column (case-insensitive); return exact name or None."""
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]

    norm_map = {"".join(c.lower().split()).replace("_",""): c for c in df.columns}
    for cand in candidates:
        key = "".join(cand.lower().split()).replace("_","")
        if key in norm_map:
            return norm_map[key]
    return None

pol_data = json.loads(Path(PATH_POLICIES).read_text(encoding="utf-8"))
subj_map = json.loads(Path(PATH_SUBJECTIVITY).read_text(encoding="utf-8"))

rows = []
for key, blob in pol_data.items():
    d = norm_domain(key)
    if not d:
        continue
    blob = blob or {}
    rubric = blob.get("rubric_assessment", {}) or {}
    disc   = rubric.get("disclosure_map", {}) or {}
    beh    = blob.get("behavioral_claims", {}) or {}

    row = {
        "domain": d,
        "completeness_score": rubric.get("completeness_score"),
        "usability_score":    rubric.get("usability_score"),
        "accuracy_score":     rubric.get("accuracy_score"),

        "disc_data_collected":        as_bool(disc.get("data_collected")),
        "disc_data_shared":           as_bool(disc.get("data_shared")),
        "disc_purpose_of_collection": as_bool(disc.get("purpose_of_collection")),
        "disc_retention_period":      as_bool(disc.get("retention_period")),
        "disc_right_to_access":       as_bool(disc.get("right_to_access")),
        "disc_right_to_delete":       as_bool(disc.get("right_to_delete")),
        "disc_opt_out":               as_bool(disc.get("opt_out")),
    }
    for c in CLAIMS:
        row[c] = as_tri(beh.get(c))
    rows.append(row)

pol_df = pd.DataFrame(rows).drop_duplicates("domain")

subj_rows = []
for k, v in subj_map.items():
    d = norm_domain(k)
    if not d:
        continue
    if isinstance(v, dict):
        subj_rows.append({"domain": d, "subjectivity": str(v.get("subjectivity","unknown"))})
    else:
        subj_rows.append({"domain": d, "subjectivity": str(v)})
subj_df = pd.DataFrame(subj_rows).drop_duplicates("domain")

universe = sorted(set(pol_df["domain"]) & set(subj_df["domain"]))

pol_df = pol_df[pol_df["domain"].isin(universe)].copy()
subj_df = subj_df[subj_df["domain"].isin(universe)].copy()

ind_raw = autodetect_sep(PATH_INDUSTRY_CSV)

col_industry   = find_col(ind_raw, ["Industry"])
col_subcat     = find_col(ind_raw, ["sub category","subcategory","sub_category","sub-cat","subcat"])
col_profit     = find_col(ind_raw, ["Profit-nonprofit","profit_nonprofit","profit"])
col_website    = find_col(ind_raw, ["website","domain"])
col_url        = find_col(ind_raw, ["url","link","homepage"])
col_ccpa_group = find_col(ind_raw, ["ccpa","ccpa_group","ccpa segment"])

if not (col_industry and (col_website or col_url)):
    raise ValueError("Could not find required columns in data_source.csv. "
                     "Need at least 'Industry' and 'website' (or 'url').")

ind_rows = []
for _, r in ind_raw.iterrows():
    dom = None
    if col_website and pd.notna(r.get(col_website)):
        dom = norm_domain(r.get(col_website))
    if not dom and col_url and pd.notna(r.get(col_url)):
        dom = norm_domain(r.get(col_url))
    if not dom:
        continue
    ind_rows.append({
        "domain": dom,
        "industry": str(r.get(col_industry,"")).strip() or "unknown",
        "industry_subcategory": str(r.get(col_subcat,"")).strip() if col_subcat else "",
        "profit_nonprofit": str(r.get(col_profit,"")).strip() if col_profit else "",
        "industry_ccpa_group": str(r.get(col_ccpa_group,"")).strip() if col_ccpa_group else "",
    })
ind_df = pd.DataFrame(ind_rows).drop_duplicates("domain")

ind_df = ind_df[ind_df["domain"].isin(universe)].copy()

merged = (
    pol_df.merge(subj_df, on="domain", how="left")
          .merge(ind_df, on="domain", how="left")
)
merged["subjectivity"] = merged["subjectivity"].fillna("unknown")
merged["industry"] = merged["industry"].fillna("unknown").replace("", "unknown")

print("=== COVERAGE ===")
print(f"Domains in intersection: {len(universe)}")
print(f"Domains with industry info: {merged['industry'].notna().sum()} "
      f"({(merged['industry'].notna().mean()*100):.1f}%)")
print("Top industries by domain count:")
print(merged["industry"].value_counts().head(10).to_string(), "\n")

ind_counts = merged.groupby("industry")["domain"].nunique().rename("n_domains").reset_index()
ind_counts["share"] = ind_counts["n_domains"] / ind_counts["n_domains"].sum()

ind_subj_counts = (
    merged.groupby(["industry","subjectivity"])["domain"]
    .nunique().rename("n_domains").reset_index()
)

print("=== Industry counts (overall) — top ===")
print(ind_counts.sort_values("n_domains", ascending=False).head(MAX_SHOW).to_string(index=False), "\n")

rubric_by_ind = (
    merged.groupby("industry")[RUBRIC_COLS]
          .agg(["mean","std","count"])
)
rubric_by_ind.columns = ["_".join(c) for c in rubric_by_ind.columns.to_flat_index()]
rubric_by_ind = rubric_by_ind.reset_index()
rb = rubric_by_ind[rubric_by_ind["completeness_score_count"] >= MIN_INDUSTRY_SIZE].copy()

print("=== Top industries by Accuracy (mean) — n >= MIN_INDUSTRY_SIZE ===")
print(rb.sort_values("accuracy_score_mean", ascending=False)[["industry","accuracy_score_mean","accuracy_score_count"]]
      .head(MAX_SHOW).to_string(index=False), "\n")

disc = merged[["industry"] + DISC_COLS].copy()
for c in DISC_COLS:
    disc[c] = disc[c].fillna(False).astype(int)

disc_counts = disc.groupby("industry")[DISC_COLS].agg(["sum","count"])

disc_rates = pd.DataFrame({
    col: disc_counts[(col,"sum")] / disc_counts[(col,"count")] for col in DISC_COLS
})
disc_rates["n"] = disc_counts[(DISC_COLS[0],"count")]
disc_rates = disc_rates.reset_index()

print("=== Lowest retention disclosure industries — n >= MIN_INDUSTRY_SIZE ===")
ret_low = disc_rates[disc_rates["n"] >= MIN_INDUSTRY_SIZE].copy() \
                   .sort_values("disc_retention_period", ascending=True)
ret_low["disc_retention_period"] = ret_low["disc_retention_period"].round(ROUND_PCT_DIGITS)
print(ret_low[["industry","n","disc_retention_period"]].head(MAX_SHOW).to_string(index=False), "\n")

def tri_share_by_group(df, group_cols, claim):
    vc = (df.groupby(group_cols)[claim].value_counts(dropna=False).unstack(fill_value=0))
    for v in ["true","false","unspecified"]:
        if v not in vc.columns: vc[v] = 0
    vc["n_group"] = vc.sum(axis=1)
    for v in ["true","false","unspecified"]:
        vc[f"{v}_share"] = vc[v] / vc["n_group"].replace(0, np.nan)
    vc = vc.reset_index()
    vc.insert(len(group_cols), "claim", claim)
    return vc

by_ind_subj_frames = [tri_share_by_group(merged, ["industry","subjectivity"], c) for c in CLAIMS]
by_ind_subj = pd.concat(by_ind_subj_frames, ignore_index=True)

def gap_table(claim):
    sub = by_ind_subj[by_ind_subj["claim"] == claim].copy()
    piv = sub.pivot_table(index="industry", columns="subjectivity", values="true_share")
    n_piv = sub.pivot_table(index="industry", columns="subjectivity", values="n_group", aggfunc="sum")
    ok = (n_piv.get("subjected", pd.Series(0,index=n_piv.index)) >= MIN_GROUP_SIZE) & \
         (n_piv.get("not_subjected", pd.Series(0,index=n_piv.index)) >= MIN_GROUP_SIZE)
    piv = piv[ok]; n_piv = n_piv[ok]
    piv["gap_true_share"] = piv.get("subjected", np.nan) - piv.get("not_subjected", np.nan)
    piv["n_subj"] = n_piv.get("subjected", 0); piv["n_not"] = n_piv.get("not_subjected", 0)
    return piv.reset_index()

gap_honors_gpc = gap_table("honors_gpc").sort_values("gap_true_share", ascending=False)
gap_sells      = gap_table("sells_data").sort_values("gap_true_share", ascending=False)
gap_share3p    = gap_table("shares_with_third_parties").sort_values("gap_true_share", ascending=False)

disc_long = merged[["industry","subjectivity"] + DISC_COLS].copy()
for c in DISC_COLS:
    disc_long[c] = disc_long[c].fillna(False).astype(int)
disc_by_ind_subj = disc_long.groupby(["industry","subjectivity"])[DISC_COLS].agg(["sum","count"])

rows = []
for (ind, subj), grp in disc_by_ind_subj.groupby(level=[0,1]):
    n = grp[(DISC_COLS[0],"count")].iloc[0]
    rec = {"industry": ind, "subjectivity": subj, "n_group": n}
    for c in DISC_COLS:
        rec[c] = grp[(c,"sum")].iloc[0] / n if n else np.nan
    rows.append(rec)
disc_share = pd.DataFrame(rows)

def disc_gap(col):
    piv = disc_share.pivot_table(index="industry", columns="subjectivity", values=col)
    n_piv = disc_share.pivot_table(index="industry", columns="subjectivity", values="n_group", aggfunc="sum")
    ok = (n_piv.get("subjected", pd.Series(0,index=n_piv.index)) >= MIN_GROUP_SIZE) & \
         (n_piv.get("not_subjected", pd.Series(0,index=n_piv.index)) >= MIN_GROUP_SIZE)
    piv = piv[ok]; n_piv = n_piv[ok]
    piv["gap_true_share"] = piv.get("subjected", np.nan) - piv.get("not_subjected", np.nan)
    piv["n_subj"] = n_piv.get("subjected", 0); piv["n_not"] = n_piv.get("not_subjected", 0)
    return piv.reset_index()

gap_optout = disc_gap("disc_opt_out").sort_values("gap_true_share", ascending=False)

print("=== HIGHLIGHTS (paper-ready) ===")

print("* Top industries by Accuracy (mean) — n≥{}:".format(MIN_INDUSTRY_SIZE))
acc_top = rb.sort_values("accuracy_score_mean", ascending=False)[["industry","accuracy_score_mean","accuracy_score_count"]]
print(acc_top.head(MAX_SHOW).to_string(index=False), "\n")

print("* Lowest Retention disclosure (share True) — n≥{}:".format(MIN_INDUSTRY_SIZE))
print(ret_low[["industry","n","disc_retention_period"]].head(MAX_SHOW).to_string(index=False), "\n")

def show_gap(title, df):
    tmp = df.copy()
    tmp["gap_true_share"] = (100*tmp["gap_true_share"]).round(1)
    print(title)
    print(tmp[["industry","n_subj","n_not","gap_true_share"]].head(MAX_SHOW).to_string(index=False))
    print()

show_gap("* Largest SUBJECTED–NOT_SUBJECTED gaps in honoring GPC (pp):", gap_honors_gpc)
show_gap("* Largest SUBJECTED–NOT_SUBJECTED gaps in OPT-OUT disclosure (pp):", gap_optout)
show_gap("* Largest SUBJECTED–NOT_SUBJECTED gaps in SELLS DATA (pp):", gap_sells)
show_gap("* Largest SUBJECTED–NOT_SUBJECTED gaps in SHARES WITH THIRD PARTIES (pp):", gap_share3p)

ind_counts_tex = ind_counts.sort_values("n_domains", ascending=False).copy()
ind_counts_tex["share"] = ind_counts_tex["share"].map(lambda x: f"{100*x:.1f}\\%")
latex_table(ind_counts_tex.head(40),
            caption="Industry distribution of domains (policies ∩ subjectivity).",
            label="tab:ind_counts")

rb_tex = merged.groupby("industry")[RUBRIC_COLS].agg(["mean","std","count"]).reset_index()
for c in RUBRIC_COLS:
    rb_tex[(c,"mean")] = rb_tex[(c,"mean")].round(2)
    rb_tex[(c,"std")]  = rb_tex[(c,"std")].round(2)
def fmt_row(r, base):
    m, s, n = r[(base,"mean")], r[(base,"std")], int(r[(base,"count")])
    return f"{m} ($\\pm$ {s}), $n$={n}"
rb_pretty = pd.DataFrame({
    "Industry": rb_tex["industry"],
    "Completeness": rb_tex.apply(lambda r: fmt_row(r,"completeness_score"), axis=1),
    "Usability":    rb_tex.apply(lambda r: fmt_row(r,"usability_score"), axis=1),
    "Accuracy":     rb_tex.apply(lambda r: fmt_row(r,"accuracy_score"), axis=1),
})
latex_table(rb_pretty, caption="Rubric scores by industry: mean ($\\pm$ std) with sample size.", label="tab:ind_rubric")

disc_counts_ind = disc.groupby("industry")[DISC_COLS].agg(["sum","count"]).reset_index()
def mk_pretty(col):
    return disc_counts_ind.apply(
        lambda r: f"{int(r[(col,'sum')])} ({100*float(r[(col,'sum')])/max(int(r[(col,'count')]),1):.1f}\\%)", axis=1)
disc_pretty = pd.DataFrame({
    "Industry": disc_counts_ind["industry"],
    "Retention": mk_pretty("disc_retention_period"),
    "Opt-out":   mk_pretty("disc_opt_out"),
    "Data collected": mk_pretty("disc_data_collected"),
    "Data shared":    mk_pretty("disc_data_shared"),
})
sizes = disc_counts_ind[("disc_data_collected","count")]
disc_pretty["n"] = sizes.values
latex_table(disc_pretty.sort_values("n", ascending=False).drop(columns=["n"]).head(30),
            caption="Disclosure coverage by industry (counts with percent of industry).",
            label="tab:ind_disclosures")

def gap_tex(df, title, label):
    t = df[["industry","n_subj","n_not","gap_true_share"]].copy()
    t.rename(columns={"industry":"Industry","n_subj":"$n_{subj}$","n_not":"$n_{not}$","gap_true_share":"Gap (pp)"},
             inplace=True)
    t["Gap (pp)"] = (100*t["Gap (pp)"]).round(1)
    latex_table(t.head(20), caption=title, label=label)

gap_tex(gap_honors_gpc, "Largest subjected$-$not\\_subjected gaps by industry in honoring GPC (percentage points).", "tab:gap_ind_gpc")
gap_tex(gap_optout,     "Largest subjected$-$not\\_subjected gaps by industry in opt-out disclosure (percentage points).", "tab:gap_ind_optout")
gap_tex(gap_sells,      "Largest subjected$-$not\\_subjected gaps by industry in selling data (percentage points).", "tab:gap_ind_sells")
gap_tex(gap_share3p,    "Largest subjected$-$not\\_subjected gaps by industry in sharing with third parties (percentage points).", "tab:gap_ind_share3p")

ind_counts.to_csv(CSV_OUTDIR/"industry_counts.csv", index=False)
ind_subj_counts.to_csv(CSV_OUTDIR/"industry_by_subjectivity_counts.csv", index=False)
rubric_by_ind.to_csv(CSV_OUTDIR/"industry_rubric_stats.csv", index=False)
disc_rates.to_csv(CSV_OUTDIR/"industry_disclosure_rates.csv", index=False)
by_ind_subj.to_csv(CSV_OUTDIR/"behavior_by_industry_subjectivity.csv", index=False)
gap_honors_gpc.to_csv(CSV_OUTDIR/"gap_industry_honors_gpc.csv", index=False)
gap_optout.to_csv(CSV_OUTDIR/"gap_industry_optout.csv", index=False)
gap_sells.to_csv(CSV_OUTDIR/"gap_industry_sells.csv", index=False)
gap_share3p.to_csv(CSV_OUTDIR/"gap_industry_shares3p.csv", index=False)

print(f"Saved CSVs to: {CSV_OUTDIR.resolve()}")

In [ ]:
import pandas as pd
import tldextract

industry_df = pd.read_csv("data_source.csv")

def norm_domain(s):
    if pd.isna(s): return None
    s = str(s).strip().lower()
    if not s: return None
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return None

if "website" in industry_df.columns:
    industry_df["domain"] = industry_df["website"].apply(norm_domain)
elif "url" in industry_df.columns:
    industry_df["domain"] = industry_df["url"].apply(norm_domain)

unknown_domains = industry_df[industry_df["Industry"].isna() | (industry_df["Industry"].astype(str).str.strip()=="")]

print("=== Domains with unknown industries ===")
print(unknown_domains["domain"].dropna().unique())

In [ ]:
import json, re
from pathlib import Path
import pandas as pd
import numpy as np
import tldextract
from difflib import get_close_matches

PATH_POLICIES = "ccpa_policy_audit_all.json"
PATH_SUBJECTIVITY = "policy_domains_ccpa_subjectivity.json"
PATH_INDUSTRY_CSV = "data_source.csv"

def norm_domain(s):
    """Return normalized eTLD+1 or None."""
    if s is None:
        return None
    s = str(s).strip().lower()
    if not s:
        return None

    s = re.sub(r"^https?://", "", s)

    s = re.sub(r"^[^/]*@", "", s)

    s = s.split("/")[0]

    s = s[:-1] if s.endswith(".") else s

    if "@" in s:
        s = s.split("@")[-1]
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return None

def autodetect_sep(path):
    try:
        return pd.read_csv(path, sep=None, engine="python")
    except Exception:
        try:
            return pd.read_csv(path, sep="\t")
        except Exception:
            return pd.read_csv(path)

def find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]

    norm_map = {"".join(c.lower().split()).replace("_",""): c for c in df.columns}
    for cand in candidates:
        key = "".join(cand.lower().split()).replace("_","")
        if key in norm_map:
            return norm_map[key]
    return None

pol_data = json.loads(Path(PATH_POLICIES).read_text(encoding="utf-8"))
subj_map = json.loads(Path(PATH_SUBJECTIVITY).read_text(encoding="utf-8"))
ind_raw  = autodetect_sep(PATH_INDUSTRY_CSV)

pol_set  = {norm_domain(k) for k in pol_data.keys() if norm_domain(k)}
subj_set = {norm_domain(k) for k in subj_map.keys() if norm_domain(k)}
universe = sorted(pol_set & subj_set)

print("=== DOMAIN UNIVERSE ===")
print(f"Policies (normalized unique):     {len(pol_set)}")
print(f"Subjectivity (normalized unique): {len(subj_set)}")
print(f"Intersection (used):              {len(universe)}\n")

rows = []
for d in universe:
    blob = pol_data.get(d)
    if blob is None:

        for k, v in pol_data.items():
            if norm_domain(k) == d:
                blob = v; break
    blob = blob or {}
    rubric = blob.get("rubric_assessment", {}) or {}
    disc   = rubric.get("disclosure_map", {}) or {}
    beh    = blob.get("behavioral_claims", {}) or {}
    row = {
        "domain": d,
        "completeness_score": rubric.get("completeness_score"),
        "usability_score":    rubric.get("usability_score"),
        "accuracy_score":     rubric.get("accuracy_score"),
    }
    rows.append(row)
pol_df = pd.DataFrame(rows)

subj_rows = []
for k, v in subj_map.items():
    d = norm_domain(k)
    if d not in universe:
        continue
    subj = v["subjectivity"] if isinstance(v, dict) and "subjectivity" in v else str(v)
    subj_rows.append({"domain": d, "subjectivity": subj})
subj_df = pd.DataFrame(subj_rows).drop_duplicates("domain")

col_industry = find_col(ind_raw, ["Industry"])
col_website  = find_col(ind_raw, ["website","domain"])
col_url      = find_col(ind_raw, ["url","link","homepage"])

if not col_industry:
    raise ValueError("Couldn't find an 'Industry' column in data_source.csv")
if not (col_website or col_url):
    raise ValueError("Couldn't find a 'website' or 'url' column in data_source.csv")

if "Industry_clean" in ind_tmp.columns:
    blanks_in_csv = ind_tmp[ind_tmp["Industry_clean"].astype(str).str.strip() == ""]
else:

    if col_industry is None:
        blanks_in_csv = pd.DataFrame(columns=ind_tmp.columns)
    else:
        blanks_in_csv = ind_tmp[ind_tmp[col_industry].astype(str).str.strip() == ""]

blank_domains = blanks_in_csv["domain"].dropna().unique().tolist()
print("=== CSV rows that have a domain but BLANK Industry (sanity) ===")
print(f"Count: {len(blank_domains)}")
print("Sample:", blank_domains[:20], "\n")

print("=== Why 'unknown'? Breakdown ===")
print(f"- Missing from CSV entirely: {len(unmatched_in_intersection)}")

present_but_blank = sorted(set(unmatched_in_intersection) & set(blank_domains))
print(f"- Present in CSV but Industry blank: {len(present_but_blank)}")
print("Sample:", present_but_blank[:20], "\n")

print("=== NORMALIZATION SPOT-CHECK (first 10 non-empty CSV rows) ===")
show = ind_raw.dropna(how="all").head(10).copy()
wcol = col_website if col_website else ""
ucol = col_url if col_url else ""
for _, r in show.iterrows():
    ws = r.get(wcol, "")
    url = r.get(ucol, "")
    print(f"raw website={ws!r} -> {norm_domain(ws)!r} | raw url={url!r} -> {norm_domain(url)!r}")

unmatched_in_intersection = sorted(set(universe) - csv_domains_set)
extra_in_csv = sorted(csv_domains_set - set(universe))

print("=== COVERAGE COMPARISON ===")
print(f"In intersection but NOT in CSV (become 'unknown' after merge): {len(unmatched_in_intersection)}")
print(f"In CSV but NOT in intersection:                               {len(extra_in_csv)}\n")

print("Sample (up to 20) unmatched_in_intersection:")
for d in unmatched_in_intersection[:20]:
    print("  ", d)
print()

print("Sample (up to 20) extra_in_csv:")
for d in extra_in_csv[:20]:
    print("  ", d)
print()

merged = (
    pol_df.merge(subj_df, on="domain", how="left")
          .merge(ind_map_df, on="domain", how="left")
)
merged["industry"] = merged["industry"].fillna("unknown").replace("", "unknown")

mapped_cnt  = (merged["industry"] != "unknown").sum()
unknown_cnt = (merged["industry"] == "unknown").sum()

print("=== MERGE STATUS ===")
print(f"Domains in intersection:          {len(merged)}")
print(f"Domains with mapped industry:     {mapped_cnt} ({mapped_cnt/len(merged)*100:.1f}%)")
print(f"Domains with 'unknown' industry:  {unknown_cnt} ({unknown_cnt/len(merged)*100:.1f}%)\n")

unknown_domains_after_merge = merged.loc[merged["industry"]=="unknown","domain"].tolist()
check_equal = set(unknown_domains_after_merge) == set(unmatched_in_intersection)
print(f"Consistency check (unknown == unmatched_in_intersection)? {check_equal}\n")

def near_matches_for(d, pool, n=5):
    return get_close_matches(d, pool, n=n, cutoff=0.6)

if unmatched_in_intersection:
    print("=== NEAR MATCHES for a few 'unknown' domains (to spot formatting issues) ===")
    pool = list(csv_domains_set)
    for d in unmatched_in_intersection[:10]:
        nm = near_matches_for(d, pool, n=5)
        print(f"{d:>30}  ->  near matches: {nm}")
    print()

true_unknown_in_csv = ind_tmp[ind_tmp["industry"].astype(str).str.strip()==""]["domain"].unique()
print("=== CSV rows that have a domain but BLANK Industry (sanity) ===")
print(f"Count: {len(true_unknown_in_csv)}")
if len(true_unknown_in_csv) > 0:
    print("Sample:", true_unknown_in_csv[:20])
print()

print("=== NORMALIZATION SPOT-CHECK (first 10 from CSV) ===")
for _, r in ind_raw.head(10).iterrows():
    ws = r[col_website] if col_website else ""
    url = r[col_url] if col_url else ""
    print(f"raw website={ws!r} -> {norm_domain(ws)!r} | raw url={url!r} -> {norm_domain(url)!r}")

In [ ]:
import pandas as pd
import json
import os
import tldextract

csv_path = "data_source_with_ppurl.csv"
json_path = "policy_domains_ccpa_subjectivity.json"
out_dir = "ccpa_compare_out"
os.makedirs(out_dir, exist_ok=True)

def normalize_domain(s: str) -> str:
    """Return registrable domain (eTLD+1) from a URL or bare domain."""
    if not isinstance(s, str) or not s.strip():
        return None
    s = s.strip().lower()
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return s

def csv_value_to_subjectivity(v: str) -> str:
    """
    Treat any value that starts with 'subjected' (e.g., 'subjected', 'subjected, location')
    as 'subjected'; everything else as 'not_subjected'.
    """
    v = (str(v) if v is not None else "").strip().lower()
    return "subjected" if v.startswith("subjected") else "not_subjected"

df = pd.read_csv(csv_path)

with open(json_path, "r") as f:
    json_data = json.load(f)

domain_candidates = df["companyWebsite"].fillna("") if "companyWebsite" in df.columns else pd.Series([""] * len(df))
if "url" in df.columns:
    domain_candidates = domain_candidates.where(domain_candidates.str.strip() != "", df["url"].fillna(""))

df["_domain_norm"] = domain_candidates.map(normalize_domain)

ccpa_col = "CCPA required"
if ccpa_col not in df.columns:
    raise KeyError(f"Expected column '{ccpa_col}' in {csv_path}")

df["_csv_subjectivity"] = df[ccpa_col].map(csv_value_to_subjectivity)

keep_cols = [c for c in [
    "_domain_norm", "_csv_subjectivity",
    "CCPA required", "Index", "category", "sub category",
    "Profit-nonprofit", "companyWebsite", "url", "Privacy Policy URL"
] if c in df.columns or c in ["_domain_norm", "_csv_subjectivity"]]

csv_by_domain = (
    df[keep_cols]
    .dropna(subset=["_domain_norm"])
    .drop_duplicates(subset=["_domain_norm"], keep="first")
    .set_index("_domain_norm", drop=True)
)

json_mapping = {}
for k, v in json_data.items():
    dom = normalize_domain(k)
    if dom:
        json_mapping[dom] = v.strip().lower()

json_df = pd.DataFrame.from_dict(json_mapping, orient="index", columns=["json_subjectivity"])
json_df.index.name = "_domain_norm"

common_df = csv_by_domain.join(json_df, how="inner")
common_df["match"] = (common_df["_csv_subjectivity"] == common_df["json_subjectivity"])

conflicts_df = common_df[~common_df["match"]].copy()

only_in_csv_df = csv_by_domain.join(json_df, how="left", rsuffix="_json")
only_in_csv_df = only_in_csv_df[only_in_csv_df["json_subjectivity"].isna()].copy()
only_in_csv_df = only_in_csv_df.drop(columns=["json_subjectivity"], errors="ignore")

only_in_json_df = json_df.join(csv_by_domain, how="left", rsuffix="_csv")
only_in_json_df = only_in_json_df[only_in_json_df["_csv_subjectivity"].isna()].copy()
only_in_json_df = only_in_json_df[["json_subjectivity"]]

common_out = os.path.join(out_dir, "common.csv")
conflicts_out = os.path.join(out_dir, "conflicts.csv")
only_csv_out = os.path.join(out_dir, "only_in_csv.csv")
only_json_out = os.path.join(out_dir, "only_in_json.csv")

common_df.reset_index().to_csv(common_out, index=False)
conflicts_df.reset_index().to_csv(conflicts_out, index=False)
only_in_csv_df.reset_index().to_csv(only_csv_out, index=False)
only_in_json_df.reset_index().to_csv(only_json_out, index=False)

print("=== CCPA Subjectivity Comparison ===")
print(f"CSV domains (unique, normalized): {csv_by_domain.shape[0]}")
print(f"JSON domains (unique, normalized): {json_df.shape[0]}")
print(f"Common domains: {common_df.shape[0]}")
print(f"Conflicts: {conflicts_df.shape[0]}")
print(f"Only in CSV: {only_in_csv_df.shape[0]}")
print(f"Only in JSON: {only_in_json_df.shape[0]}")
print("\nOutputs written to:", os.path.abspath(out_dir))

In [ ]:
import pandas as pd
import json
import os
import tldextract

csv_path = "data_source_with_ppurl.csv"
json_path = "policy_domains_ccpa_subjectivity.json"
out_dir = "ccpa_compare_out"
os.makedirs(out_dir, exist_ok=True)

def normalize_domain(s: str) -> str:
    """Return registrable domain (eTLD+1) from a URL or bare domain."""
    if not isinstance(s, str) or not s.strip():
        return None
    s = s.strip().lower()
    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return s

def csv_value_to_subjectivity(v: str) -> str:
    """
    Treat any value that starts with 'subjected' (e.g., 'subjected', 'subjected, location')
    as 'subjected'; everything else as 'not_subjected'.
    """
    v = (str(v) if v is not None else "").strip().lower()
    return "subjected" if v.startswith("subjected") else "not_subjected"

df = pd.read_csv(csv_path)

with open(json_path, "r") as f:
    json_data = json.load(f)

json_mapping = {}
for k, v in json_data.items():
    dom = normalize_domain(k)
    if dom:
        json_mapping[dom] = str(v).strip().lower()
json_domains = set(json_mapping.keys())

if "website" in df.columns:
    website_col = "website"
elif "companyWebsite" in df.columns:
    website_col = "companyWebsite"
elif "url" in df.columns:
    website_col = "url"
else:
    raise KeyError("No suitable website column found. Expected one of: 'website', 'companyWebsite', or 'url'.")

pp_col = "Privacy Policy URL"
if pp_col not in df.columns:
    raise KeyError(f"Expected column '{pp_col}' in {csv_path}")

ccpa_col = "CCPA required"
if ccpa_col not in df.columns:
    raise KeyError(f"Expected column '{ccpa_col}' in {csv_path}")

rows = []
for i, row in df.iterrows():
    website_dom = normalize_domain(row.get(website_col))
    pp_dom = normalize_domain(row.get(pp_col))
    csv_subjectivity = csv_value_to_subjectivity(row.get(ccpa_col, ""))

    matched_on = "no_json_match"
    match_domain = None

    if website_dom and website_dom in json_domains:
        match_domain = website_dom
        matched_on = "website"
    elif pp_dom and pp_dom in json_domains:
        match_domain = pp_dom
        matched_on = "privacy_policy"

    rows.append({
        "_row_id": i,
        "website_raw": row.get(website_col),
        "privacy_policy_url_raw": row.get(pp_col),
        "website_domain": website_dom,
        "pp_domain": pp_dom,
        "_csv_subjectivity": csv_subjectivity,
        "_match_domain": match_domain,
        "matched_on": matched_on,

        "CCPA required": row.get(ccpa_col, None),
        "Index": row.get("Index", None),
        "category": row.get("category", None),
        "sub category": row.get("sub category", None),
        "Profit-nonprofit": row.get("Profit-nonprofit", None),
        "url": row.get("url", None),
    })

cmp_df = pd.DataFrame(rows)

matched_mask = cmp_df["matched_on"].isin(["website", "privacy_policy"])
matched_df = cmp_df[matched_mask].copy()
matched_df = matched_df.drop_duplicates(subset=["_match_domain"], keep="first")

matched_df["json_subjectivity"] = matched_df["_match_domain"].map(json_mapping)
matched_df["match"] = (matched_df["_csv_subjectivity"] == matched_df["json_subjectivity"])

conflicts_df = matched_df[~matched_df["match"]].copy()

only_in_csv_df = cmp_df[cmp_df["matched_on"] == "no_json_match"].copy()

matched_json_domains = set(matched_df["_match_domain"].dropna().tolist())
only_in_json_domains = sorted(list(json_domains - matched_json_domains))
only_in_json_df = pd.DataFrame({
    "_domain_norm": only_in_json_domains,
    "json_subjectivity": [json_mapping[d] for d in only_in_json_domains]
})

common_out = os.path.join(out_dir, "common.csv")
conflicts_out = os.path.join(out_dir, "conflicts.csv")
only_csv_out = os.path.join(out_dir, "only_in_csv.csv")
only_json_out = os.path.join(out_dir, "only_in_json.csv")

common_cols = [
    "_match_domain", "matched_on",
    "_csv_subjectivity", "json_subjectivity", "match",
    "website_raw", "website_domain", "privacy_policy_url_raw", "pp_domain",
    "CCPA required", "Index", "category", "sub category", "Profit-nonprofit", "url"
]
matched_df[common_cols].to_csv(common_out, index=False)

conflicts_df[common_cols].to_csv(conflicts_out, index=False)

only_csv_cols = [
    "website_raw", "website_domain", "privacy_policy_url_raw", "pp_domain",
    "_csv_subjectivity", "CCPA required", "Index", "category", "sub category", "Profit-nonprofit", "url"
]
only_in_csv_df[only_csv_cols].to_csv(only_csv_out, index=False)

only_in_json_df.to_csv(only_json_out, index=False)

print("=== CCPA Subjectivity Comparison (website-first, fallback to PP URL) ===")
print(f"CSV rows: {len(df)}")
print(f"JSON domains (unique): {len(json_domains)}")
print(f"Common (matched via website/PP): {matched_df.shape[0]}")
print(f"Conflicts: {conflicts_df.shape[0]}")
print(f"Only in CSV (no match on website or PP): {only_in_csv_df.shape[0]}")
print(f"Only in JSON (unmatched by CSV): {only_in_json_df.shape[0]}")
print("\nOutputs written to:", os.path.abspath(out_dir))

In [ ]:
import pandas as pd
from urllib.parse import urlsplit

csv_path = "data_source_with_ppurl.csv"

IGNORE_SCHEME = True
DROP_QUERY_AND_FRAGMENT = True
STRIP_TRAILING_SLASH = True

def find_col(df, target_name: str) -> str:
    """
    Return the actual column name matching target_name, case/space/underscore-insensitive.
    Raises KeyError if not found.
    """
    target = target_name.lower().replace(" ", "").replace("_", "")
    for c in df.columns:
        canon = c.lower().replace(" ", "").replace("_", "")
        if canon == target:
            return c
    raise KeyError(f"Column '{target_name}' not found in CSV headers: {list(df.columns)}")

def canonicalize_url(u: str) -> str | None:
    """Canonicalize a URL for duplicate detection."""
    if not isinstance(u, str) or not u.strip():
        return None
    s = u.strip()
    parts = urlsplit(s)

    netloc = parts.netloc.lower()
    path = parts.path or ""
    if STRIP_TRAILING_SLASH and path != "/":
        path = path.rstrip("/")

    if DROP_QUERY_AND_FRAGMENT:

        if IGNORE_SCHEME:

            return f"{netloc}{path}"
        else:

            scheme = (parts.scheme or "").lower()
            return f"{scheme}://{netloc}{path}"
    else:

        scheme = (parts.scheme or "").lower()
        query = parts.query
        fragment = parts.fragment
        if IGNORE_SCHEME:
            return f"{netloc}{path}?{query}#{fragment}".rstrip("?#")
        else:
            base = f"{scheme}://{netloc}{path}"
            if query:
                base += f"?{query}"
            if fragment:
                base += f"#{fragment}"
            return base

df = pd.read_csv(csv_path)

pp_col = find_col(df, "Privacy Policy URL")

df["_ppurl_norm"] = df[pp_col].map(canonicalize_url)

counts = (
    df.dropna(subset=["_ppurl_norm"])
      .groupby("_ppurl_norm", as_index=False)
      .size()
      .rename(columns={"size": "count"})
      .sort_values("count", ascending=False)
)

dups = counts[counts["count"] > 1]

if dups.empty:
    print("No duplicate Privacy Policy URLs found (after normalization).")
else:
    print("Duplicate Privacy Policy URLs (normalized):")

    print(dups.to_string(index=False))

    print("\nDetails per duplicate (original URLs and any helpful context):")

    context_cols = [c for c in [
        "website", "companyWebsite", "url", "Index", "category", "sub category", "Profit-nonprofit", pp_col
    ] if c in df.columns]

    for norm in dups["_ppurl_norm"]:
        block = df[df["_ppurl_norm"] == norm][context_cols].copy()
        print(f"\n=== {norm} — {block.shape[0]} rows ===")

        with pd.option_context("display.max_rows", 200):
            print(block.to_string(index=False))

if not dups.empty:
    out = "duplicate_privacy_policy_urls.csv"
    dups.to_csv(out, index=False)
    print(f"\nSaved summary to: {out}")

In [ ]:
import os
import re
import hashlib
import pandas as pd
from urllib.parse import urlsplit
import tldextract

csv_path = "data_source_with_ppurl.csv"
out_dir = "ppurl_duplicates"
os.makedirs(out_dir, exist_ok=True)

IGNORE_SCHEME = True
DROP_QUERY_AND_FRAGMENT = True
STRIP_TRAILING_SLASH = True

def find_col(df, target_name: str) -> str:
    """
    Return the actual column name matching target_name,
    case/space/underscore-insensitive. Raises KeyError if not found.
    """
    target = target_name.lower().replace(" ", "").replace("_", "")
    for c in df.columns:
        canon = c.lower().replace(" ", "").replace("_", "")
        if canon == target:
            return c
    raise KeyError(f"Column '{target_name}' not found. CSV columns: {list(df.columns)}")

def canonicalize_url(u: str) -> str | None:
    """Canonicalize a URL for duplicate detection (host+path, lowercased, no query/fragment)."""
    if not isinstance(u, str) or not u.strip():
        return None
    s = u.strip()
    parts = urlsplit(s)

    netloc = parts.netloc.lower()
    path = parts.path or ""
    if STRIP_TRAILING_SLASH and path != "/":
        path = path.rstrip("/")

    if DROP_QUERY_AND_FRAGMENT:
        if IGNORE_SCHEME:
            return f"{netloc}{path}"
        else:
            scheme = (parts.scheme or "").lower()
            return f"{scheme}://{netloc}{path}"
    else:
        scheme = (parts.scheme or "").lower()
        query = parts.query
        fragment = parts.fragment
        if IGNORE_SCHEME:
            base = f"{netloc}{path}"
        else:
            base = f"{scheme}://{netloc}{path}"
        if query:
            base += f"?{query}"
        if fragment:
            base += f"#{fragment}"
        return base

def normalize_domain(s: str) -> str | None:
    """Return registrable domain (eTLD+1) from a URL or bare domain."""
    if not isinstance(s, str) or not s.strip():
        return None
    s = s.strip()

    ext = tldextract.extract(s)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}".lower()
    return None

def safe_filename(text: str, suffix: str, max_len: int = 120) -> str:
    """
    Make a filesystem-safe filename from arbitrary text.
    Appends an 8-char SHA1 to avoid collisions.
    """
    h = hashlib.sha1(text.encode("utf-8")).hexdigest()[:8]

    base = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")

    base = base[:max_len]
    return f"{base}__{h}.{suffix}"

df = pd.read_csv(csv_path)

pp_col = find_col(df, "Privacy Policy URL")

try:
    cw_col = find_col(df, "companyWebsite")
except KeyError:

    cw_col = find_col(df, "website") if "website" in [c.lower() for c in df.columns] else find_col(df, "url")

df["_ppurl_norm"] = df[pp_col].map(canonicalize_url)

counts = (
    df.dropna(subset=["_ppurl_norm"])
      .groupby("_ppurl_norm", as_index=False)
      .size()
      .rename(columns={"size": "count"})
      .sort_values("count", ascending=False)
)

dups = counts[counts["count"] > 1].copy()

index_rows = []

for _, row in dups.iterrows():
    norm = row["_ppurl_norm"]
    n = int(row["count"])

    block = df[df["_ppurl_norm"] == norm].copy()

    rep_raw_pp = next((v for v in block[pp_col].dropna().astype(str).tolist() if v.strip()), norm)

    parts = urlsplit(rep_raw_pp)
    host_path = (parts.netloc + parts.path) if parts.netloc else norm
    fname = safe_filename(host_path.lower(), "csv")

    out_cols = [c for c in [
        pp_col, cw_col,
        "website", "url", "Index", "category", "sub category", "Profit-nonprofit"
    ] if c in block.columns]

    block["_company_domain_norm"] = block[cw_col].map(normalize_domain)
    out_cols = out_cols + ["_company_domain_norm"]

    out_path = os.path.join(out_dir, fname)
    block[out_cols].drop_duplicates().to_csv(out_path, index=False)

    distinct_companies = (
        block[cw_col].dropna().astype(str).str.strip().replace("", pd.NA).dropna().unique().tolist()
    )
    index_rows.append({
        "ppurl_normalized": norm,
        "duplicate_count": n,
        "sample_ppurl": rep_raw_pp,
        "output_file": fname,
        "distinct_companyWebsite_count": len(distinct_companies),
        "distinct_companyWebsite_preview": "; ".join(distinct_companies[:10]),
    })

index_df = pd.DataFrame(index_rows).sort_values("duplicate_count", ascending=False)
index_df.to_csv(os.path.join(out_dir, "INDEX_ppurl_duplicates.csv"), index=False)

print("Wrote per-Privacy-Policy URL CSVs for duplicates to:", os.path.abspath(out_dir))
print(f"Total duplicate PP URLs: {len(index_df)}")
if not index_df.empty:
    print(index_df.head(10).to_string(index=False))

In [ ]:
import pandas as pd
from urllib.parse import urlsplit
import tldextract

csv_path = "data_source_with_ppurl.csv"
out_csv  = "combined_ppurl_duplicates.csv"

IGNORE_SCHEME = True
DROP_QUERY_AND_FRAGMENT = True
STRIP_TRAILING_SLASH = True

def find_col(df, target_name: str) -> str:
    """Return actual column name matching target_name (case/space/underscore-insensitive)."""
    target = target_name.lower().replace(" ", "").replace("_", "")
    for c in df.columns:
        canon = c.lower().replace(" ", "").replace("_", "")
        if canon == target:
            return c
    raise KeyError(f"Column '{target_name}' not found. CSV columns: {list(df.columns)}")

def canonicalize_url(u: str) -> str | None:
    """Canonicalize URL for duplicate detection (host+path, lowercased; drops query/fragment)."""
    if not isinstance(u, str) or not u.strip():
        return None
    parts = urlsplit(u.strip())
    netloc = parts.netloc.lower()
    path = parts.path or ""
    if STRIP_TRAILING_SLASH and path != "/":
        path = path.rstrip("/")
    if DROP_QUERY_AND_FRAGMENT:
        return f"{netloc}{path}" if IGNORE_SCHEME else f"{(parts.scheme or '').lower()}://{netloc}{path}"

    base = f"{netloc}{path}" if IGNORE_SCHEME else f"{(parts.scheme or '').lower()}://{netloc}{path}"
    if parts.query:    base += f"?{parts.query}"
    if parts.fragment: base += f"#{parts.fragment}"
    return base

def normalize_domain(s: str) -> str | None:
    """Return registrable domain (eTLD+1) from URL or bare domain."""
    if not isinstance(s, str) or not s.strip():
        return None
    ext = tldextract.extract(s.strip())
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}".lower()
    return None

df = pd.read_csv(csv_path)

pp_col = find_col(df, "Privacy Policy URL")
cw_col = find_col(df, "companyWebsite")

df["_ppurl_norm"] = df[pp_col].map(canonicalize_url)

counts = (
    df.dropna(subset=["_ppurl_norm"])
      .groupby("_ppurl_norm", as_index=False)
      .size()
      .rename(columns={"size": "duplicate_count"})
)

dup_keys = counts[counts["duplicate_count"] > 1]["_ppurl_norm"]

combined = df[df["_ppurl_norm"].isin(dup_keys)].copy()

combined = combined.merge(counts, on="_ppurl_norm", how="left")
combined["_company_domain_norm"] = combined[cw_col].map(normalize_domain)

rep_pp = (
    combined.groupby("_ppurl_norm")[pp_col]
    .apply(lambda s: next((v for v in s.dropna().astype(str) if v.strip()), ""))
    .rename("sample_ppurl")
    .reset_index()
)
combined = combined.merge(rep_pp, on="_ppurl_norm", how="left")

keep_cols = [c for c in [
    "_ppurl_norm", "sample_ppurl", "duplicate_count",
    pp_col, cw_col, "_company_domain_norm",
    "website", "url", "Index", "category", "sub category", "Profit-nonprofit"
] if c in combined.columns]

combined = combined[keep_cols].drop_duplicates()

combined = combined.sort_values(["duplicate_count", "_ppurl_norm", cw_col], ascending=[False, True, True])

combined.to_csv(out_csv, index=False)
print(f"Wrote: {out_csv}")
print(f"Duplicate PP URLs: {combined['_ppurl_norm'].nunique()}  |  Rows: {len(combined)}")

In [ ]:
import json
import collections
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

import tldextract

POLICY_JSON = "ccpa_policy_audit_all.json"
SUBJECTIVITY_JSON = "policy_domains_ccpa_subjectivity.json"

WRITE_JOINED_CSV = True
JOINED_CSV_PATH = "policy_subjectivity_joined.csv"

ALIAS_FIXES = {

    "angielist.com": "angieslist.com",

}

def etld1(domain: str) -> str:
    """
    Normalize any hostname or domain to its registrable domain (eTLD+1).
    Returns empty string if input is empty/invalid.
    """
    if not domain:
        return ""
    d = domain.strip().lower()
    ext = tldextract.extract(d)
    if not ext.domain or not ext.suffix:

        return d
    return f"{ext.domain}.{ext.suffix}"

def apply_alias(domain: str) -> str:
    """Apply alias/typo corrections after eTLD+1 normalization."""
    return ALIAS_FIXES.get(domain, domain)

p_policy = Path(POLICY_JSON)
p_subj = Path(SUBJECTIVITY_JSON)

if not p_policy.exists():
    raise FileNotFoundError(f"Missing: {p_policy.resolve()}")
if not p_subj.exists():
    raise FileNotFoundError(f"Missing: {p_subj.resolve()}")

with open(p_policy, "r") as f:
    policy_data = json.load(f)

with open(p_subj, "r") as f:
    subjectivity_map = json.load(f)

policy_raw_domains = list(policy_data.keys())

policy_parent_by_raw = {}
policy_children_by_parent = collections.defaultdict(list)
for raw in policy_raw_domains:
    parent = apply_alias(etld1(raw))
    policy_parent_by_raw[raw] = parent
    policy_children_by_parent[parent].append(raw)

policy_parents = set(policy_parent_by_raw.values())

subj_parent_by_raw = {}
for raw_dom, label in subjectivity_map.items():
    parent = apply_alias(etld1(raw_dom))
    subj_parent_by_raw[raw_dom.strip().lower()] = (parent, str(label).strip().lower())

subj_by_parent = {}
collisions = collections.defaultdict(set)
for raw_dom, (parent, label) in subj_parent_by_raw.items():
    if parent in subj_by_parent and subj_by_parent[parent] != label:
        collisions[parent].update([subj_by_parent[parent], label])
    subj_by_parent[parent] = label

subjectivity_parents = set(subj_by_parent.keys())

intersection = policy_parents & subjectivity_parents
policy_only = policy_parents - subjectivity_parents
subjectivity_only = subjectivity_parents - policy_parents

n_policy_raw = len(policy_raw_domains)
n_policy_parents = len(policy_parents)
n_subj_raw = len(subjectivity_map)
n_subj_parents = len(subjectivity_parents)
n_intersection = len(intersection)
n_policy_only = len(policy_only)
n_subjectivity_only = len(subjectivity_only)

label_counts = collections.Counter(subj_by_parent[p] for p in intersection)

print("=== Mapping Summary ===")
print(f"Policy file: raw entries           = {n_policy_raw}")
print(f"Policy file: unique eTLD+1 parents = {n_policy_parents}")
print(f"Subjectivity file: raw entries     = {n_subj_raw}")
print(f"Subjectivity file: unique parents  = {n_subj_parents}")
print()
print(f"Intersection (by eTLD+1 parents)   = {n_intersection}")
print(f"Policy-only parents                = {n_policy_only}")
print(f"Subjectivity-only parents          = {n_subjectivity_only}")
print()

if collisions:
    print("! Warning: Label collisions for the same parent domain (in subjectivity map):")
    for parent, labels in list(collisions.items())[:20]:
        print(f"  - {parent}: {sorted(labels)}")
    if len(collisions) > 20:
        print(f"  ... and {len(collisions) - 20} more")
    print()

print("Intersection by subjectivity label (within intersection):")
for label, cnt in label_counts.most_common():
    print(f"  {label:>15}: {cnt}")

SHOW_N = 15
if policy_only:
    print("\nExamples — Policy-only parents (not found in subjectivity):")
    for d in list(sorted(policy_only))[:SHOW_N]:
        kids = policy_children_by_parent.get(d, [])
        preview = f" (e.g., {kids[0]})" if kids else ""
        print(f"  - {d}{preview}")
    if len(policy_only) > SHOW_N:
        print(f"  ... and {len(policy_only) - SHOW_N} more")

if subjectivity_only:
    print("\nExamples — Subjectivity-only parents (no policy audit entry):")
    for d in list(sorted(subjectivity_only))[:SHOW_N]:
        print(f"  - {d}")
    if len(subjectivity_only) > SHOW_N:
        print(f"  ... and {len(subjectivity_only) - SHOW_N} more")

if WRITE_JOINED_CSV:
    if pd is None:
        print("\n[INFO] Skipping CSV export because pandas is not installed.")
    else:
        rows = []
        for raw_policy_domain, parent in policy_parent_by_raw.items():
            label = subj_by_parent.get(parent, None)
            rows.append({
                "raw_policy_domain": raw_policy_domain,
                "parent_domain": parent,
                "subjectivity_label": label,
            })
        df = pd.DataFrame(rows).sort_values(["subjectivity_label", "parent_domain", "raw_policy_domain"], na_position="last")
        df.to_csv(JOINED_CSV_PATH, index=False)
        print(f"\nJoined CSV written to: {JOINED_CSV_PATH}")

In [ ]:
import json
import collections
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import tldextract

POLICY_JSON = "ccpa_policy_audit_all.json"
SUBJECTIVITY_JSON = "policy_domains_ccpa_subjectivity.json"
CSV_SOURCE = "data_source_with_ppurl.csv"

JOINED_POLICY_SUBJ_CSV = "policy_subjectivity_joined.csv"
CSV_MATCH_REPORT = "csv_match_report.csv"
PPURL_DUPES_CSV = "ppurl_duplicates_mapping.csv"

ALIAS_FIXES = {

    "angielist.com": "angieslist.com",

}

CSV_WEBSITE_COL_CANDIDATES = ["companyWebsite", "website", "Website", "domain"]
CSV_PPURL_COL_CANDIDATES = ["Privacy Policy URL", "privacy_policy_url", "privacy_policy", "policy_url"]

def etld1(domain_or_host: str) -> str:
    """Normalize any hostname or domain to its registrable domain (eTLD+1)."""
    if not domain_or_host or not str(domain_or_host).strip():
        return ""
    d = str(domain_or_host).strip().lower()
    ext = tldextract.extract(d)
    if not ext.domain or not ext.suffix:
        return d
    return f"{ext.domain}.{ext.suffix}"

def url_to_host(url: str) -> str:
    """Extract netloc/host from a URL (no scheme assumptions)."""
    if not url or not str(url).strip():
        return ""
    s = str(url).strip()

    if "://" not in s:
        s = "http://" + s
    try:
        return urlparse(s).netloc.lower()
    except Exception:
        return ""

def apply_alias(domain: str) -> str:
    """Apply alias/typo corrections after eTLD+1 normalization."""
    return ALIAS_FIXES.get(domain, domain)

def choose_first_present_column(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of the expected columns found. Tried: {candidates}")

p_policy = Path(POLICY_JSON)
p_subj = Path(SUBJECTIVITY_JSON)
p_csv = Path(CSV_SOURCE)

if not p_policy.exists():
    raise FileNotFoundError(f"Missing: {p_policy.resolve()}")
if not p_subj.exists():
    raise FileNotFoundError(f"Missing: {p_subj.resolve()}")
if not p_csv.exists():
    raise FileNotFoundError(f"Missing: {p_csv.resolve()}")

with open(p_policy, "r") as f:
    policy_data = json.load(f)

with open(p_subj, "r") as f:
    subjectivity_map = json.load(f)

policy_raw_domains = list(policy_data.keys())

policy_parent_by_raw = {}
policy_children_by_parent = collections.defaultdict(list)
for raw in policy_raw_domains:
    parent = apply_alias(etld1(raw))
    policy_parent_by_raw[raw] = parent
    policy_children_by_parent[parent].append(raw)

policy_parents = set(policy_parent_by_raw.values())

subj_parent_by_raw = {}
for raw_dom, label in subjectivity_map.items():
    parent = apply_alias(etld1(raw_dom))
    subj_parent_by_raw[str(raw_dom).strip().lower()] = (parent, str(label).strip().lower())

subj_by_parent = {}
collisions = collections.defaultdict(set)
for raw_dom, (parent, label) in subj_parent_by_raw.items():
    if parent in subj_by_parent and subj_by_parent[parent] != label:
        collisions[parent].update([subj_by_parent[parent], label])
    subj_by_parent[parent] = label

subjectivity_parents = set(subj_by_parent.keys())

intersection = policy_parents & subjectivity_parents
policy_only = policy_parents - subjectivity_parents
subjectivity_only = subjectivity_parents - policy_parents

csv_df = pd.read_csv(p_csv)
website_col = choose_first_present_column(csv_df, CSV_WEBSITE_COL_CANDIDATES)
ppurl_col = choose_first_present_column(csv_df, CSV_PPURL_COL_CANDIDATES)

csv_df["_website_raw"] = csv_df[website_col].astype(str).str.strip()
csv_df["_website_parent"] = csv_df["_website_raw"].apply(lambda x: apply_alias(etld1(url_to_host(x) or x)))

csv_df["_ppurl_raw"] = csv_df[ppurl_col].astype(str).str.strip()
csv_df["_ppurl_host"] = csv_df["_ppurl_raw"].apply(url_to_host)
csv_df["_ppurl_parent"] = csv_df["_ppurl_host"].apply(lambda h: apply_alias(etld1(h)))

ppurl_counts = csv_df["_ppurl_raw"].value_counts(dropna=False)
dupe_ppurls = set(ppurl_counts[ppurl_counts > 1].index)

dupe_rows = (csv_df[csv_df["_ppurl_raw"].isin(dupe_ppurls)]
             .groupby("_ppurl_raw")[website_col]
             .apply(lambda s: sorted({str(x).strip() for x in s if str(x).strip()}))
             .reset_index(name="companyWebsites"))

dupe_rows.to_csv(PPURL_DUPES_CSV, index=False)

def match_parent_return_label(parent: str):
    """Return (in_policy, in_subjectivity, label_or_None)."""
    in_pol = parent in policy_parents
    label = subj_by_parent.get(parent)
    in_subj = parent in subjectivity_parents
    return in_pol, in_subj, label

match_route = []
in_policy_flags = []
in_subjectivity_flags = []
labels = []
matched_parent_used = []
policy_children_example = []

for _, row in csv_df.iterrows():
    w_parent = row["_website_parent"]
    p_parent = row["_ppurl_parent"]

    in_pol, in_subj, lab = match_parent_return_label(w_parent)
    if w_parent and (in_pol or in_subj):
        match_route.append("website")
        in_policy_flags.append(in_pol)
        in_subjectivity_flags.append(in_subj)
        labels.append(lab)
        matched_parent_used.append(w_parent)
        kids = policy_children_by_parent.get(w_parent, [])
        policy_children_example.append(kids[0] if kids else "")
        continue

    in_pol, in_subj, lab = match_parent_return_label(p_parent)
    if p_parent and (in_pol or in_subj):
        match_route.append("ppurl")
        in_policy_flags.append(in_pol)
        in_subjectivity_flags.append(in_subj)
        labels.append(lab)
        matched_parent_used.append(p_parent)
        kids = policy_children_by_parent.get(p_parent, [])
        policy_children_example.append(kids[0] if kids else "")
        continue

    match_route.append("unmatched")
    in_policy_flags.append(False)
    in_subjectivity_flags.append(False)
    labels.append(None)
    matched_parent_used.append("")
    policy_children_example.append("")

csv_df["_match_route"] = match_route
csv_df["_matched_parent"] = matched_parent_used
csv_df["_in_policy"] = in_policy_flags
csv_df["_in_subjectivity"] = in_subjectivity_flags
csv_df["_subjectivity_label"] = labels
csv_df["_policy_child_example"] = policy_children_example

csv_df_out_cols = [
    website_col, "_website_parent",
    ppurl_col, "_ppurl_parent",
    "_match_route", "_matched_parent",
    "_in_policy", "_in_subjectivity", "_subjectivity_label",
    "_policy_child_example"
]
csv_df[csv_df_out_cols].to_csv(CSV_MATCH_REPORT, index=False)

rows = []
for raw_policy_domain, parent in policy_parent_by_raw.items():
    label = subj_by_parent.get(parent, None)
    rows.append({
        "raw_policy_domain": raw_policy_domain,
        "parent_domain": parent,
        "subjectivity_label": label,
    })
pd.DataFrame(rows).sort_values(
    ["subjectivity_label", "parent_domain", "raw_policy_domain"],
    na_position="last"
).to_csv(JOINED_POLICY_SUBJ_CSV, index=False)

n_policy_raw = len(policy_raw_domains)
n_policy_parents = len(policy_parents)
n_subj_raw = len(subjectivity_map)
n_subj_parents = len(subjectivity_parents)
n_intersection = len(intersection)
n_policy_only = len(policy_only)
n_subjectivity_only = len(subjectivity_only)

label_counts = collections.Counter(subj_by_parent[p] for p in intersection)

print("=== Policy vs Subjectivity (by eTLD+1) ===")
print(f"Policy file: raw entries           = {n_policy_raw}")
print(f"Policy file: unique eTLD+1 parents = {n_policy_parents}")
print(f"Subjectivity file: raw entries     = {n_subj_raw}")
print(f"Subjectivity file: unique parents  = {n_subj_parents}")
print(f"Intersection                       = {n_intersection}")
print(f"Policy-only parents                = {n_policy_only}")
print(f"Subjectivity-only parents          = {n_subjectivity_only}")
if collisions:
    print("\n! Warning: Label collisions (same parent, differing labels):")
    for parent, labels_set in list(collisions.items())[:20]:
        print(f"  - {parent}: {sorted(labels_set)}")
    more = len(collisions) - min(20, len(collisions))
    if more > 0:
        print(f"  ... and {more} more")

print("\nIntersection by subjectivity label:")
for label, cnt in label_counts.most_common():
    print(f"  {label:>15}: {cnt}")

total_rows = len(csv_df)
via_website = (csv_df["_match_route"] == "website").sum()
via_ppurl = (csv_df["_match_route"] == "ppurl").sum()
unmatched = (csv_df["_match_route"] == "unmatched").sum()

print("\n=== CSV Matching (data_source_with_ppurl.csv) ===")
print(f"Total CSV rows                 = {total_rows}")
print(f"Matched via WEBSITE parent     = {via_website}")
print(f"Matched via PP-URL parent      = {via_ppurl}")
print(f"Unmatched rows                 = {unmatched}")

matched_mask = csv_df["_match_route"].isin(["website", "ppurl"])
in_pol_count = int(csv_df.loc[matched_mask, "_in_policy"].sum())
in_subj_count = int(csv_df.loc[matched_mask, "_in_subjectivity"].sum())

print("\nAmong matched rows:")
print(f"  In policy set                = {in_pol_count}")
print(f"  In subjectivity set          = {in_subj_count}")

print("\n=== Duplicate Privacy Policy URLs in CSV ===")
print(f"Unique PP URLs total           = {csv_df['_ppurl_raw'].nunique(dropna=False)}")
print(f"PP URLs appearing > 1 time     = {len(dupe_ppurls)}")
if len(dupe_ppurls) > 0:
    print("Examples:")
    for pp in list(dupe_ppurls)[:10]:
        print(f"  - {pp} (count={ppurl_counts[pp]})")

print(f"\nWrote:")
print(f"  - {JOINED_POLICY_SUBJ_CSV}")
print(f"  - {CSV_MATCH_REPORT}")
print(f"  - {PPURL_DUPES_CSV}")

unmatched_df = csv_df[csv_df["_match_route"] == "unmatched"]

print("\n=== Unmatched CSV Rows (no match in policy/subjectivity) ===")
print(f"Total unmatched rows = {len(unmatched_df)}")

if len(unmatched_df) > 0:

    cols_to_show = [website_col, "_website_parent",
                    ppurl_col, "_ppurl_parent"]
    print(unmatched_df[cols_to_show].to_string(index=False))

    unmatched_df.to_csv("csv_unmatched_rows.csv", index=False)
    print("\nSaved unmatched rows to: csv_unmatched_rows.csv")

In [ ]:
"""
Full analysis of policy audits by CCPA subjectivity.

Inputs:
  - ccpa_policy_audit_all.json
      {
        "<domain or subdomain>": {
          "online_data_practices": str,
          "offline_data_practices": str,
          "rubric_assessment": {
              "completeness_score": int,
              "usability_score": int,
              "accuracy_score": int,
              "policy_contradiction": bool,
              "disclosure_map": {
                  "data_collected": bool,
                  "data_shared": bool,
                  "purpose_of_collection": bool,
                  "retention_period": bool,
                  "right_to_access": bool,
                  "right_to_delete": bool,
                  "opt_out": bool
              },
              "..._reason": str
          },
          "behavioral_claims": {
              "honors_gpc": True|False|"unspecified",
              "respects_dnt": True|False|"unspecified",
              "sets_cookies_before_consent": True|False|"unspecified",
              "sets_cookies_after_rejecting_consent": True|False|"unspecified",
              "deletes_cookies_on_rejection": True|False|"unspecified",
              "uses_tracking_only_after_consent": True|False|"unspecified",
              "sells_data": True|False|"unspecified",
              "shares_with_third_parties": True|False|"unspecified",
              "justifications": {...}   # ignored here
          }
        },
        ...
      }

  - policy_domains_ccpa_subjectivity.json
      { "<etld+1>": "subjected" | "not_subjected" }

Outputs (CSV):
  - joined_policy_subjectivity.csv
  - rubric_stats_by_subjectivity.csv
  - disclosure_coverage_by_subjectivity.csv
  - behavioral_claims_by_subjectivity.csv
  - policy_contradictions_by_subjectivity.csv
  - universe_counts.csv

Also prints a structured summary for paper use.
"""

import json
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import tldextract
import numpy as np

POLICY_JSON = "ccpa_policy_audit_all.json"
SUBJECTIVITY_JSON = "policy_domains_ccpa_subjectivity.json"

ALIAS_FIXES = {

}

JOINED_CSV = "joined_policy_subjectivity.csv"
RUBRIC_CSV = "rubric_stats_by_subjectivity.csv"
DISCLOSURE_CSV = "disclosure_coverage_by_subjectivity.csv"
BEHAVIORAL_CSV = "behavioral_claims_by_subjectivity.csv"
CONTRADICT_CSV = "policy_contradictions_by_subjectivity.csv"
UNIVERSE_COUNTS_CSV = "universe_counts.csv"

def etld1(host: str) -> str:
    """Normalize a hostname/domain to registrable domain (eTLD+1)."""
    if not host:
        return ""
    h = str(host).strip().lower()
    ext = tldextract.extract(h)
    if not ext.domain or not ext.suffix:
        return h
    return f"{ext.domain}.{ext.suffix}"

def apply_alias(d: str) -> str:
    return ALIAS_FIXES.get(d, d)

def safe_get(d: dict, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def to_tfu(value):
    """
    Normalize behavioral-claims value into one of:
      'true', 'false', 'unspecified'
    """
    if isinstance(value, bool):
        return "true" if value else "false"
    return "unspecified"

p_policy = Path(POLICY_JSON)
p_subj = Path(SUBJECTIVITY_JSON)
if not p_policy.exists():
    raise FileNotFoundError(f"Missing {p_policy.resolve()}")
if not p_subj.exists():
    raise FileNotFoundError(f"Missing {p_subj.resolve()}")

with open(p_policy, "r") as f:
    policy_data = json.load(f)

with open(p_subj, "r") as f:
    subj_map_raw = json.load(f)

policy_rows = []
for raw_dom, rec in policy_data.items():
    parent = apply_alias(etld1(raw_dom))
    policy_rows.append({
        "raw_policy_domain": raw_dom,
        "parent_domain": parent,
        "record": rec
    })
policy_df = pd.DataFrame(policy_rows)

subj_rows = []
for dom, label in subj_map_raw.items():
    parent = apply_alias(etld1(dom))
    subj_rows.append({
        "parent_domain": parent,
        "subjectivity": str(label).strip().lower()
    })
subj_df = pd.DataFrame(subj_rows).drop_duplicates("parent_domain")

joined = policy_df.merge(subj_df, on="parent_domain", how="left")

def extract_rubric(rec):
    r = rec.get("rubric_assessment", {}) if isinstance(rec, dict) else {}
    return pd.Series({
        "completeness_score": r.get("completeness_score", np.nan),
        "usability_score": r.get("usability_score", np.nan),
        "accuracy_score": r.get("accuracy_score", np.nan),
        "policy_contradiction": r.get("policy_contradiction", False),

        "disclosure_data_collected": safe_get(r, "disclosure_map", "data_collected", default=False),
        "disclosure_data_shared": safe_get(r, "disclosure_map", "data_shared", default=False),
        "disclosure_purpose_of_collection": safe_get(r, "disclosure_map", "purpose_of_collection", default=False),
        "disclosure_retention_period": safe_get(r, "disclosure_map", "retention_period", default=False),
        "disclosure_right_to_access": safe_get(r, "disclosure_map", "right_to_access", default=False),
        "disclosure_right_to_delete": safe_get(r, "disclosure_map", "right_to_delete", default=False),
        "disclosure_opt_out": safe_get(r, "disclosure_map", "opt_out", default=False),
    })

def extract_behavioral(rec):
    b = rec.get("behavioral_claims", {}) if isinstance(rec, dict) else {}
    return pd.Series({
        "honors_gpc": to_tfu(b.get("honors_gpc", "unspecified")),
        "respects_dnt": to_tfu(b.get("respects_dnt", "unspecified")),
        "sets_cookies_before_consent": to_tfu(b.get("sets_cookies_before_consent", "unspecified")),
        "sets_cookies_after_rejecting_consent": to_tfu(b.get("sets_cookies_after_rejecting_consent", "unspecified")),
        "deletes_cookies_on_rejection": to_tfu(b.get("deletes_cookies_on_rejection", "unspecified")),
        "uses_tracking_only_after_consent": to_tfu(b.get("uses_tracking_only_after_consent", "unspecified")),
        "sells_data": to_tfu(b.get("sells_data", "unspecified")),
        "shares_with_third_parties": to_tfu(b.get("shares_with_third_parties", "unspecified")),
    })

rubric_df = joined["record"].apply(extract_rubric)
behav_df = joined["record"].apply(extract_behavioral)

data = pd.concat([joined.drop(columns=["record"]), rubric_df, behav_df], axis=1)

policy_raw = len(policy_df)
policy_parents = policy_df["parent_domain"].nunique()
subj_raw = len(subj_df)
subj_parents = subj_df["parent_domain"].nunique()
intersection_parents = pd.Series(sorted(set(policy_df["parent_domain"]) & set(subj_df["parent_domain"]))).nunique()
policy_only_parents = policy_parents - intersection_parents
subj_only_parents = subj_parents - intersection_parents

universe_counts = pd.DataFrame([{
    "policy_raw_entries": policy_raw,
    "policy_unique_parents": policy_parents,
    "subjectivity_raw_entries": subj_raw,
    "subjectivity_unique_parents": subj_parents,
    "intersection_parents": intersection_parents,
    "policy_only_parents": policy_only_parents,
    "subjectivity_only_parents": subj_only_parents
}])
universe_counts.to_csv(UNIVERSE_COUNTS_CSV, index=False)

data["subjectivity"] = data["subjectivity"].fillna("unlabeled")
inters_mask = data["subjectivity"].isin(["subjected", "not_subjected"])
data_inter = data.loc[inters_mask].copy()

group_sizes = data_inter["subjectivity"].value_counts().to_dict()

def summarize_scores(df, cols):
    rows = []
    for label, g in df.groupby("subjectivity"):
        row = {"subjectivity": label, "n": len(g)}
        for c in cols:
            vals = g[c].dropna().astype(float)
            row[f"{c}_mean"] = vals.mean() if len(vals) else np.nan
            row[f"{c}_median"] = vals.median() if len(vals) else np.nan
            row[f"{c}_std"] = vals.std(ddof=0) if len(vals) else np.nan
        rows.append(row)
    return pd.DataFrame(rows).sort_values("subjectivity")

rubric_cols = ["completeness_score", "usability_score", "accuracy_score"]
rubric_stats = summarize_scores(data_inter, rubric_cols)
rubric_stats.to_csv(RUBRIC_CSV, index=False)

disclosure_cols = [
    "disclosure_data_collected",
    "disclosure_data_shared",
    "disclosure_purpose_of_collection",
    "disclosure_retention_period",
    "disclosure_right_to_access",
    "disclosure_right_to_delete",
    "disclosure_opt_out",
]

disc_rows = []
for label, g in data_inter.groupby("subjectivity"):
    n = len(g)
    for col in disclosure_cols:
        true_count = int(g[col].fillna(False).astype(bool).sum())
        disc_rows.append({
            "subjectivity": label,
            "metric": col,
            "true_count": true_count,
            "group_n": n,
            "true_pct": (true_count / n) if n else np.nan
        })
disclosure_coverage = pd.DataFrame(disc_rows).sort_values(["subjectivity", "metric"])
disclosure_coverage.to_csv(DISCLOSURE_CSV, index=False)

behavioral_cols = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]

behav_rows = []
for label, g in data_inter.groupby("subjectivity"):
    n = len(g)
    for col in behavioral_cols:
        counts = g[col].fillna("unspecified").value_counts()
        t = int(counts.get("true", 0))
        f = int(counts.get("false", 0))
        u = int(counts.get("unspecified", 0))
        behav_rows.append({
            "subjectivity": label,
            "claim": col,
            "true_count": t,
            "false_count": f,
            "unspecified_count": u,
            "group_n": n,
            "true_pct": t / n if n else np.nan,
            "false_pct": f / n if n else np.nan,
            "unspecified_pct": u / n if n else np.nan,
        })
behavioral_claims = pd.DataFrame(behav_rows).sort_values(["subjectivity", "claim"])
behavioral_claims.to_csv(BEHAVIORAL_CSV, index=False)

contr_rows = []
for label, g in data_inter.groupby("subjectivity"):
    n = len(g)
    c_true = int(g["policy_contradiction"].fillna(False).astype(bool).sum())
    contr_rows.append({
        "subjectivity": label,
        "contradictions": c_true,
        "group_n": n,
        "rate": c_true / n if n else np.nan
    })
contr_stats = pd.DataFrame(contr_rows).sort_values("subjectivity")
contr_stats.to_csv(CONTRADICT_CSV, index=False)

data_out_cols = [
    "raw_policy_domain", "parent_domain", "subjectivity",
    "completeness_score", "usability_score", "accuracy_score",
    "disclosure_data_collected", "disclosure_data_shared",
    "disclosure_purpose_of_collection", "disclosure_retention_period",
    "disclosure_right_to_access", "disclosure_right_to_delete",
    "disclosure_opt_out",
    "honors_gpc", "respects_dnt",
    "sets_cookies_before_consent", "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection", "uses_tracking_only_after_consent",
    "sells_data", "shares_with_third_parties",
    "policy_contradiction",
]
data[data_out_cols].to_csv(JOINED_CSV, index=False)

print("\n=== Domain Universe & Intersection (by eTLD+1) ===")
print(universe_counts.to_string(index=False))

print("\n=== Group Sizes (intersection only) ===")
for k in ["subjected", "not_subjected"]:
    print(f"{k:>15}: {group_sizes.get(k, 0)}")

print("\n=== Rubric Stats by Subjectivity ===")
print(rubric_stats.to_string(index=False, float_format=lambda x: f"{x:.3f}" if pd.notna(x) else "NA"))

print("\n=== Disclosure Coverage by Subjectivity (counts & pct of group) ===")
for label in ["subjected", "not_subjected"]:
    sub = disclosure_coverage[disclosure_coverage["subjectivity"] == label]
    if sub.empty:
        continue
    print(f"\n[{label}] n={int(sub['group_n'].iloc[0])}")
    for _, row in sub.iterrows():
        metric = row["metric"].replace("disclosure_", "")
        print(f"  {metric:>26}: {int(row['true_count']):4d}  ({row['true_pct']*100:5.1f}%)")

print("\n=== Behavioral Claims by Subjectivity (counts & pct of group) ===")
for label in ["subjected", "not_subjected"]:
    sub = behavioral_claims[behavioral_claims["subjectivity"] == label]
    if sub.empty:
        continue
    n = int(sub["group_n"].iloc[0])
    print(f"\n[{label}] n={n}")
    for _, row in sub.iterrows():
        name = row["claim"]
        t, f, u = int(row["true_count"]), int(row["false_count"]), int(row["unspecified_count"])
        tp, fp, up = row["true_pct"]*100, row["false_pct"]*100, row["unspecified_pct"]*100
        print(f"  {name:>34}: T={t:3d} ({tp:5.1f}%) | F={f:3d} ({fp:5.1f}%) | U={u:3d} ({up:5.1f}%)")

print("\n=== Policy Contradictions by Subjectivity ===")
print(contr_stats.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nWrote:")
print(f"  - {JOINED_CSV}")
print(f"  - {RUBRIC_CSV}")
print(f"  - {DISCLOSURE_CSV}")
print(f"  - {BEHAVIORAL_CSV}")
print(f"  - {CONTRADICT_CSV}")
print(f"  - {UNIVERSE_COUNTS_CSV}")

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

OUTDIR = "paper/figures/new_figures"
os.makedirs(OUTDIR, exist_ok=True)

CSV_PATH = "rubric_stats_by_subjectivity.csv"
FNAME = "barplot_rubric_scores_ccpa_split"

USE_95CI = False

ANNOTATE_MEDIANS = True

METRIC_MAP = {
    "completeness_score": "Completeness",
    "usability_score":    "Usability",
    "accuracy_score":     "Accuracy",
}
METRIC_ORDER = ["completeness_score", "usability_score", "accuracy_score"]

GROUP_MAP = {
    "subjected":      "Subjected",
    "not_subjected":  "Non-Subj.",
}
GROUP_ORDER = ["Subjected", "Non-Subj."]

PALETTE = {"Subjected": "tab:blue", "Non-Subj.": "tab:orange"}

def _prep_from_csv(csv_path=CSV_PATH):
    """
    Load rubric_stats_by_subjectivity.csv and reshape into long-form:
    columns: Metric, Group, mean, std, median, n
    """
    df = pd.read_csv(csv_path)

    if "subjectivity" not in df.columns:
        raise ValueError("CSV missing 'subjectivity' column.")
    df["Group"] = df["subjectivity"].map(GROUP_MAP)
    if df["Group"].isna().any():
        missing = sorted(set(df.loc[df["Group"].isna(), "subjectivity"]))
        raise ValueError(f"Unknown subjectivity labels in CSV: {missing}")

    rows = []
    for _, r in df.iterrows():
        for base in METRIC_ORDER:
            mean_col   = f"{base}_mean"
            std_col    = f"{base}_std"
            median_col = f"{base}_median"
            for col in [mean_col, std_col, median_col]:
                if col not in df.columns:
                    raise ValueError(f"CSV missing '{col}' column.")
            rows.append({
                "MetricKey": base,
                "Metric": METRIC_MAP[base],
                "Group": r["Group"],
                "mean": float(r[mean_col]),
                "std": float(r[std_col]),
                "median": float(r[median_col]),
                "n": int(r["n"]) if "n" in df.columns else np.nan,
            })

    out = pd.DataFrame(rows)
    out["Metric"] = pd.Categorical(
        out["Metric"], [METRIC_MAP[k] for k in METRIC_ORDER], ordered=True
    )
    out["Group"] = pd.Categorical(out["Group"], GROUP_ORDER, ordered=True)
    return out

def _compute_yerr(row):
    """Return error-bar value for a row (std or 95% CI)."""
    if not USE_95CI:
        return row["std"]

    n = row.get("n", np.nan)
    if n and n > 1:
        return 1.96 * row["std"] / np.sqrt(n)
    return np.nan

def fig_rubric_bars_ccpa_pretty(
    stats_df=None,
    csv_path=CSV_PATH,
    fname=FNAME,
    annotate_medians=ANNOTATE_MEDIANS
):
    """
    Grouped barplot (mean ± std or 95% CI) for Completeness/Usability/Accuracy,
    comparing Subjected vs. Non-Subj. in your seaborn "pretty" style.
    """
    if stats_df is None:
        stats_df = _prep_from_csv(csv_path)
    else:

        stats_df["Metric"] = pd.Categorical(
            stats_df["Metric"], [METRIC_MAP[k] for k in METRIC_ORDER], ordered=True
        )
        stats_df["Group"] = pd.Categorical(stats_df["Group"], GROUP_ORDER, ordered=True)

    stats_df = stats_df.copy()
    stats_df["yerr"] = stats_df.apply(_compute_yerr, axis=1)

    sns.set(style="ticks")
    fig, ax = plt.subplots(figsize=(8.6, 4.6))

    bp = sns.barplot(
        x="Metric", y="mean", hue="Group",
        data=stats_df,
        order=[METRIC_MAP[k] for k in METRIC_ORDER],
        hue_order=GROUP_ORDER,
        palette=PALETTE,
        width=0.65, edgecolor="black", linewidth=1.0,
        ax=ax, errorbar=None
    )

    yerr_lookup = {(r["Metric"], r["Group"]): r["yerr"] for _, r in stats_df.iterrows()}
    med_lookup  = {(r["Metric"], r["Group"]): r["median"] for _, r in stats_df.iterrows()}
    n_lookup    = {(r["Metric"], r["Group"]): r["n"] for _, r in stats_df.iterrows()}

    patches = iter(bp.patches)
    for metric_name in [METRIC_MAP[k] for k in METRIC_ORDER]:
        for group_name in GROUP_ORDER:
            bar = next(patches)
            x_center = bar.get_x() + bar.get_width() / 2.0
            mean_val = bar.get_height()
            yerr = yerr_lookup[(metric_name, group_name)]
            med  = med_lookup[(metric_name, group_name)]
            nval = n_lookup[(metric_name, group_name)]

            if np.isfinite(yerr) and yerr > 0:
                ax.errorbar(
                    x=x_center, y=mean_val, yerr=yerr, fmt="none",
                    elinewidth=1.2, capsize=4, capthick=1.2, color="black"
                )

            if annotate_medians:
                bump = (yerr if (np.isfinite(yerr) and yerr > 0) else 0) + 0.06
                ax.text(
                    x_center, mean_val + bump,
                    f"median={med:.0f}\n(n={int(nval) if pd.notna(nval) else '—'})",
                    ha="center", va="bottom", fontsize=9
                )

    ax.set_xlabel("")
    ylab = "Rubric score (mean ± std)" if not USE_95CI else "Rubric score (mean ± 95% CI)"
    ax.set_ylabel(ylab, fontsize=13)

    ymax = stats_df["mean"].max()
    eb   = stats_df["yerr"].replace([np.inf, -np.inf], np.nan).dropna()
    ymax += (eb.max() if not eb.empty else 0) + 0.6
    ax.set_ylim(0, max(3.2, ymax))

    ax.grid(True, axis="y", linewidth=0.5, alpha=0.35)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        title="",
        ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.14),
        frameon=False, fontsize=12, handlelength=1.8,
        columnspacing=1.2, handletextpad=0.6
    )

    ax.tick_params(axis="y", labelsize=11, direction="in", length=4, width=1)
    ax.margins(x=0.04)

    try:
        format_axes(ax)
    except Exception:
        pass

    fig.subplots_adjust(bottom=0.16, top=0.84)

    pdf_path = os.path.join(OUTDIR, f"{fname}.pdf")
    png_path = os.path.join(OUTDIR, f"{fname}.png")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {pdf_path}  |  {png_path}")

if __name__ == "__main__":

    if Path(CSV_PATH).exists():
        fig_rubric_bars_ccpa_pretty(csv_path=CSV_PATH)
    else:

        inline = pd.DataFrame([
            {"subjectivity":"not_subjected","n":386,
             "completeness_score_mean":1.889,"completeness_score_median":2.0,"completeness_score_std":0.982,
             "usability_score_mean":1.733,"usability_score_median":2.0,"usability_score_std":1.038,
             "accuracy_score_mean":2.168,"accuracy_score_median":2.0,"accuracy_score_std":0.738},
            {"subjectivity":"subjected","n":523,
             "completeness_score_mean":2.331,"completeness_score_median":3.0,"completeness_score_std":0.909,
             "usability_score_mean":2.287,"usability_score_median":3.0,"usability_score_std":0.947,
             "accuracy_score_mean":2.447,"accuracy_score_median":3.0,"accuracy_score_std":0.813},
        ])

        rows=[]
        for _, r in inline.iterrows():
            group = GROUP_MAP[r["subjectivity"]]
            for base in METRIC_ORDER:
                rows.append({
                    "MetricKey": base,
                    "Metric": METRIC_MAP[base],
                    "Group": group,
                    "mean":   float(r[f"{base}_mean"]),
                    "std":    float(r[f"{base}_std"]),
                    "median": float(r[f"{base}_median"]),
                    "n": int(r["n"]),
                })
        stats_df = pd.DataFrame(rows)
        stats_df["Metric"] = pd.Categorical(
            stats_df["Metric"], [METRIC_MAP[k] for k in METRIC_ORDER], ordered=True
        )
        stats_df["Group"] = pd.Categorical(stats_df["Group"], GROUP_ORDER, ordered=True)
        fig_rubric_bars_ccpa_pretty(stats_df=stats_df)

In [ ]:
"""
Full policy analysis split by industry instead of CCPA subjectivity.
Inputs:
  - ccpa_policy_audit_all.json (policy audits)
  - data_source_with_ppurl.csv (with columns: companyWebsite, Privacy Policy URL, industry/category)
"""

import json
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import tldextract

POLICY_JSON = "ccpa_policy_audit_all.json"
CSV_SOURCE = "data_source_with_ppurl.csv"

JOINED_CSV = "joined_policy_industry.csv"
RUBRIC_CSV = "rubric_stats_by_industry.csv"
DISCLOSURE_CSV = "disclosure_coverage_by_industry.csv"
BEHAVIORAL_CSV = "behavioral_claims_by_industry.csv"
CONTRADICT_CSV = "policy_contradictions_by_industry.csv"
UNIVERSE_COUNTS_CSV = "universe_counts_industry.csv"

ALIAS_FIXES = {

}

def etld1(host: str) -> str:
    if not host:
        return ""
    ext = tldextract.extract(str(host).strip().lower())
    return f"{ext.domain}.{ext.suffix}" if ext.domain and ext.suffix else str(host).strip().lower()

def apply_alias(d: str) -> str:
    return ALIAS_FIXES.get(d, d)

def to_tfu(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    return "unspecified"

with open(POLICY_JSON, "r") as f:
    policy_data = json.load(f)

csv_df = pd.read_csv(CSV_SOURCE)

csv_df["parent_domain"] = csv_df["companyWebsite"].apply(lambda x: apply_alias(etld1(x)))
if "industry" not in csv_df.columns and "category" in csv_df.columns:
    csv_df.rename(columns={"category": "industry"}, inplace=True)

industry_map = csv_df.set_index("parent_domain")["industry"].to_dict()

rows = []
for raw_dom, rec in policy_data.items():
    parent = apply_alias(etld1(raw_dom))
    industry = industry_map.get(parent, "Unknown")
    row = {
        "raw_policy_domain": raw_dom,
        "parent_domain": parent,
        "industry": industry,
        "completeness_score": rec.get("rubric_assessment", {}).get("completeness_score"),
        "usability_score": rec.get("rubric_assessment", {}).get("usability_score"),
        "accuracy_score": rec.get("rubric_assessment", {}).get("accuracy_score"),
        "policy_contradiction": rec.get("rubric_assessment", {}).get("policy_contradiction", False),
    }

    dmap = rec.get("rubric_assessment", {}).get("disclosure_map", {})
    for key in ["data_collected","data_shared","purpose_of_collection",
                "retention_period","right_to_access","right_to_delete","opt_out"]:
        row[f"disclosure_{key}"] = dmap.get(key, False)

    bmap = rec.get("behavioral_claims", {})
    for key in ["honors_gpc","respects_dnt","sets_cookies_before_consent",
                "sets_cookies_after_rejecting_consent","deletes_cookies_on_rejection",
                "uses_tracking_only_after_consent","sells_data","shares_with_third_parties"]:
        row[key] = to_tfu(bmap.get(key, "unspecified"))
    rows.append(row)

df = pd.DataFrame(rows)

universe_counts = df["industry"].value_counts().reset_index()
universe_counts.columns = ["industry", "n_policies"]
universe_counts.to_csv(UNIVERSE_COUNTS_CSV, index=False)

rubric_cols = ["completeness_score","usability_score","accuracy_score"]
rubric_stats = df.groupby("industry")[rubric_cols].agg(["mean","median","std"])
rubric_stats.to_csv(RUBRIC_CSV)

disclosure_cols = [c for c in df.columns if c.startswith("disclosure_")]
disc_rows = []
for ind, g in df.groupby("industry"):
    n = len(g)
    for col in disclosure_cols:
        true_count = int(g[col].fillna(False).astype(bool).sum())
        disc_rows.append({
            "industry": ind,
            "metric": col,
            "true_count": true_count,
            "group_n": n,
            "true_pct": true_count / n if n else np.nan
        })
disclosure_coverage = pd.DataFrame(disc_rows)
disclosure_coverage.to_csv(DISCLOSURE_CSV, index=False)

behavioral_cols = ["honors_gpc","respects_dnt","sets_cookies_before_consent",
                   "sets_cookies_after_rejecting_consent","deletes_cookies_on_rejection",
                   "uses_tracking_only_after_consent","sells_data","shares_with_third_parties"]

behav_rows = []
for ind, g in df.groupby("industry"):
    n = len(g)
    for col in behavioral_cols:
        counts = g[col].fillna("unspecified").value_counts()
        t,f,u = counts.get("true",0), counts.get("false",0), counts.get("unspecified",0)
        behav_rows.append({
            "industry": ind, "claim": col,
            "true_count": t, "false_count": f, "unspecified_count": u,
            "group_n": n,
            "true_pct": t/n if n else np.nan,
            "false_pct": f/n if n else np.nan,
            "unspecified_pct": u/n if n else np.nan,
        })
behavioral_claims = pd.DataFrame(behav_rows)
behavioral_claims.to_csv(BEHAVIORAL_CSV, index=False)

contr = df.groupby("industry")["policy_contradiction"].agg(["sum","count"])
contr["rate"] = contr["sum"]/contr["count"]
contr.to_csv(CONTRADICT_CSV)

df.to_csv(JOINED_CSV, index=False)

print("Wrote outputs:")
print(f" - {JOINED_CSV}")
print(f" - {RUBRIC_CSV}")
print(f" - {DISCLOSURE_CSV}")
print(f" - {BEHAVIORAL_CSV}")
print(f" - {CONTRADICT_CSV}")
print(f" - {UNIVERSE_COUNTS_CSV}")

In [ ]:
"""
Full policy analysis split by INDUSTRY (from CSV `category` column).

Inputs (put these in the same folder or update the paths below):
  - ccpa_policy_audit_all.json        # rich policy audit per (sub)domain
  - data_source_with_ppurl.csv        # has `category` (industry) + website columns

What this script does:
  1) Normalizes domains to eTLD+1 and maps each parent domain -> industry (from CSV `category`)
     - Uses website column first; falls back to Privacy Policy URL host if needed.
     - If multiple categories exist for the same parent domain, the majority label is used.
  2) Joins the policy JSON to industry labels and extracts:
     - Rubric scores (completeness/usability/accuracy)
     - Disclosure flags (data_collected, data_shared, purpose_of_collection, retention_period,
                        right_to_access, right_to_delete, opt_out)
     - Behavioral claims (honors_gpc, respects_dnt, sets_cookies_before_consent, etc.)
     - policy_contradiction (bool)
  3) Computes paper-friendly outputs by industry:
     - Universe counts
     - Rubric stats (mean/median/std)
     - Disclosure coverage (true count + %)
     - Behavioral claims (T/F/U counts + %)
     - Contradiction counts and rates
  4) Writes CSVs and prints a concise summary.

Deps: pandas, numpy, tldextract
Install: pip install pandas numpy tldextract
"""

import json
import math
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import tldextract
from urllib.parse import urlparse

POLICY_JSON = "ccpa_policy_audit_all.json"
CSV_SOURCE  = "data_source_with_ppurl.csv"

JOINED_CSV             = "joined_policy_industry.csv"
UNIVERSE_COUNTS_CSV    = "universe_counts_industry.csv"
RUBRIC_CSV             = "rubric_stats_by_industry.csv"
DISCLOSURE_CSV         = "disclosure_coverage_by_industry.csv"
BEHAVIORAL_CSV         = "behavioral_claims_by_industry.csv"
CONTRADICT_CSV         = "policy_contradictions_by_industry.csv"

ALIAS_FIXES = {

}

WEBSITE_COL_CANDIDATES = ["companyWebsite", "website", "Website", "domain", "site", "url"]
PPURL_COL_CANDIDATES   = ["Privacy Policy URL", "privacy_policy_url", "policy_url", "PrivacyPolicyURL"]
CATEGORY_COL_CANDIDATES = ["category", "Category", "industry", "Industry"]

def etld1(host: str) -> str:
    """Normalize a hostname/domain to its registrable domain (eTLD+1)."""
    if not host or not str(host).strip():
        return ""
    h = str(host).strip().lower()
    ext = tldextract.extract(h)
    if not ext.domain or not ext.suffix:
        return h
    return f"{ext.domain}.{ext.suffix}"

def apply_alias(d: str) -> str:
    return ALIAS_FIXES.get(d, d)

def url_to_host(url: str) -> str:
    """Extract netloc from a URL. Accepts bare hosts without scheme."""
    if not url or not str(url).strip():
        return ""
    s = str(url).strip()
    if "://" not in s:
        s = "http://" + s
    try:
        return urlparse(s).netloc.lower()
    except Exception:
        return ""

def choose_first_present_column(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of the expected columns found. Tried: {candidates}")

def to_tfu(value):
    """Normalize behavioral value to 'true'/'false'/'unspecified'."""
    if isinstance(value, bool):
        return "true" if value else "false"
    return "unspecified"

p_policy = Path(POLICY_JSON)
p_csv    = Path(CSV_SOURCE)
if not p_policy.exists():
    raise FileNotFoundError(f"Missing {p_policy.resolve()}")
if not p_csv.exists():
    raise FileNotFoundError(f"Missing {p_csv.resolve()}")

with open(p_policy, "r") as f:
    policy_data = json.load(f)

csv_df = pd.read_csv(p_csv)

website_col  = choose_first_present_column(csv_df, WEBSITE_COL_CANDIDATES)
category_col = choose_first_present_column(csv_df, CATEGORY_COL_CANDIDATES)
ppurl_col    = None
try:
    ppurl_col = choose_first_present_column(csv_df, PPURL_COL_CANDIDATES)
except KeyError:
    pass

csv_df["_website_raw"]   = csv_df[website_col].astype(str).str.strip()
csv_df["_website_host"]  = csv_df["_website_raw"].apply(lambda x: url_to_host(x) or x)
csv_df["_website_parent"] = csv_df["_website_host"].apply(lambda x: apply_alias(etld1(x)))

if ppurl_col:
    csv_df["_ppurl_raw"]   = csv_df[ppurl_col].astype(str).str.strip()
    csv_df["_ppurl_host"]  = csv_df["_ppurl_raw"].apply(url_to_host)
    csv_df["_ppurl_parent"] = csv_df["_ppurl_host"].apply(lambda x: apply_alias(etld1(x)))
else:
    csv_df["_ppurl_parent"] = ""

csv_df["_parent_for_map"] = np.where(csv_df["_website_parent"] != "", csv_df["_website_parent"], csv_df["_ppurl_parent"])

csv_df["_category_norm"] = csv_df[category_col].astype(str).str.strip()
csv_df.loc[csv_df["_category_norm"] == "", "_category_norm"] = "Unknown"

cat_groups = csv_df.groupby(["_parent_for_map", "_category_norm"]).size().reset_index(name="count")
majority_idx = cat_groups.sort_values(["_parent_for_map", "count"], ascending=[True, False]).drop_duplicates("_parent_for_map")
industry_map = dict(zip(majority_idx["_parent_for_map"], majority_idx["_category_norm"]))

rows = []
for raw_dom, rec in policy_data.items():
    parent = apply_alias(etld1(raw_dom))
    industry = industry_map.get(parent, "Unknown")

    r_assess = rec.get("rubric_assessment", {}) if isinstance(rec, dict) else {}
    dmap = r_assess.get("disclosure_map", {}) if isinstance(r_assess, dict) else {}
    bmap = rec.get("behavioral_claims", {}) if isinstance(rec, dict) else {}

    row = {
        "raw_policy_domain": raw_dom,
        "parent_domain": parent,
        "industry": industry,

        "completeness_score": r_assess.get("completeness_score", np.nan),
        "usability_score":    r_assess.get("usability_score",    np.nan),
        "accuracy_score":     r_assess.get("accuracy_score",     np.nan),
        "policy_contradiction": bool(r_assess.get("policy_contradiction", False)),

        "disclosure_data_collected":        bool(dmap.get("data_collected", False)),
        "disclosure_data_shared":           bool(dmap.get("data_shared", False)),
        "disclosure_purpose_of_collection": bool(dmap.get("purpose_of_collection", False)),
        "disclosure_retention_period":      bool(dmap.get("retention_period", False)),
        "disclosure_right_to_access":       bool(dmap.get("right_to_access", False)),
        "disclosure_right_to_delete":       bool(dmap.get("right_to_delete", False)),
        "disclosure_opt_out":               bool(dmap.get("opt_out", False)),

        "honors_gpc":                         to_tfu(bmap.get("honors_gpc", "unspecified")),
        "respects_dnt":                       to_tfu(bmap.get("respects_dnt", "unspecified")),
        "sets_cookies_before_consent":        to_tfu(bmap.get("sets_cookies_before_consent", "unspecified")),
        "sets_cookies_after_rejecting_consent": to_tfu(bmap.get("sets_cookies_after_rejecting_consent", "unspecified")),
        "deletes_cookies_on_rejection":       to_tfu(bmap.get("deletes_cookies_on_rejection", "unspecified")),
        "uses_tracking_only_after_consent":   to_tfu(bmap.get("uses_tracking_only_after_consent", "unspecified")),
        "sells_data":                         to_tfu(bmap.get("sells_data", "unspecified")),
        "shares_with_third_parties":          to_tfu(bmap.get("shares_with_third_parties", "unspecified")),
    }
    rows.append(row)

df = pd.DataFrame(rows)

universe_counts = df["industry"].value_counts(dropna=False).reset_index()
universe_counts.columns = ["industry", "n_policies"]
universe_counts.sort_values("n_policies", ascending=False, inplace=True)
universe_counts.to_csv(UNIVERSE_COUNTS_CSV, index=False)

rubric_cols = ["completeness_score", "usability_score", "accuracy_score"]
rubric_stats = (
    df.groupby("industry")[rubric_cols]
      .agg(["mean", "median", "std"])
      .sort_index()
)
rubric_stats.to_csv(RUBRIC_CSV)

disclosure_cols = [
    "disclosure_data_collected",
    "disclosure_data_shared",
    "disclosure_purpose_of_collection",
    "disclosure_retention_period",
    "disclosure_right_to_access",
    "disclosure_right_to_delete",
    "disclosure_opt_out",
]
disc_rows = []
for ind, g in df.groupby("industry"):
    n = len(g)
    for col in disclosure_cols:
        true_count = int(g[col].fillna(False).astype(bool).sum())
        disc_rows.append({
            "industry": ind,
            "metric": col.replace("disclosure_", ""),
            "true_count": true_count,
            "group_n": n,
            "true_pct": (true_count / n) if n else np.nan,
        })
disclosure_coverage = pd.DataFrame(disc_rows).sort_values(["industry", "metric"])
disclosure_coverage.to_csv(DISCLOSURE_CSV, index=False)

behavioral_cols = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]
behav_rows = []
for ind, g in df.groupby("industry"):
    n = len(g)
    for col in behavioral_cols:
        counts = g[col].fillna("unspecified").value_counts()
        t = int(counts.get("true", 0))
        f = int(counts.get("false", 0))
        u = int(counts.get("unspecified", 0))
        behav_rows.append({
            "industry": ind,
            "claim": col,
            "true_count": t,
            "false_count": f,
            "unspecified_count": u,
            "group_n": n,
            "true_pct": (t / n) if n else np.nan,
            "false_pct": (f / n) if n else np.nan,
            "unspecified_pct": (u / n) if n else np.nan,
        })
behavioral_claims = pd.DataFrame(behav_rows).sort_values(["industry", "claim"])
behavioral_claims.to_csv(BEHAVIORAL_CSV, index=False)

contr = df.groupby("industry")["policy_contradiction"].agg(["sum", "count"])
contr["rate"] = contr["sum"] / contr["count"]
contr.sort_index(inplace=True)
contr.to_csv(CONTRADICT_CSV)

df.to_csv(JOINED_CSV, index=False)

print("\n=== Universe (policies per industry) ===")
print(universe_counts.to_string(index=False))

print("\n=== Rubric stats by industry (mean/median/std) ===")
print(rubric_stats.round(3).to_string())

print("\n=== Disclosure coverage by industry (true count and % of group) — first few rows ===")
print(disclosure_coverage.head(14).to_string(index=False))

print("\n=== Behavioral claims by industry (T/F/U counts and %) — first few rows ===")
print(behavioral_claims.head(14).to_string(index=False))

print("\n=== Policy contradictions by industry ===")
print(contr.round(3).to_string())

print("\nWrote:")
print(f"  - {JOINED_CSV}")
print(f"  - {UNIVERSE_COUNTS_CSV}")
print(f"  - {RUBRIC_CSV}")
print(f"  - {DISCLOSURE_CSV}")
print(f"  - {BEHAVIORAL_CSV}")
print(f"  - {CONTRADICT_CSV}")

In [ ]:
import json
import pandas as pd
import numpy as np
import tldextract
from urllib.parse import urlparse
from pathlib import Path
from collections import defaultdict

SUBJECTIVITY_JSON = "policy_domains_ccpa_subjectivity.json"
CSV_SOURCE = "data_source_with_ppurl.csv"
OUT_CSV = "subjectivity_multi_hit_rows.csv"

ALIAS_FIXES = {

}

LIKELY_COLS = [
    "companyWebsite",
    "website",
    "Website",
    "Privacy Policy URL",
    "privacy_policy_url",
    "policy_url",
    "Homepage",
    "home_url",
    "url",
    "companyWebsite"
]

def etld1(host: str) -> str:
    """Normalize a hostname (or bare domain) to registrable domain (eTLD+1)."""
    if not host or not str(host).strip():
        return ""
    h = str(host).strip().lower()
    ext = tldextract.extract(h)
    if not ext.domain or not ext.suffix:
        return h
    dom = f"{ext.domain}.{ext.suffix}"
    return ALIAS_FIXES.get(dom, dom)

def url_to_host_maybe(value: str) -> str:
    """
    Extract host from a URL-ish string.
    - If it's a bare domain like 'example.com', treat it as host.
    - If it's a full URL, parse netloc.
    - If it's something else, return ''.
    """
    if value is None:
        return ""
    s = str(value).strip()
    if not s:
        return ""

    if " " in s:
        return ""

    s_for_parse = s if "://" in s else "http://" + s
    try:
        netloc = urlparse(s_for_parse).netloc.lower()

        return netloc or ""
    except Exception:
        return ""

def pick_candidate_columns(df: pd.DataFrame):
    """Pick a set of columns to scan: known likely + any with 'url'/'website' in their name."""
    cols = set()
    for c in df.columns:
        lc = c.lower()
        if c in LIKELY_COLS or "url" in lc or "website" in lc or "domain" in lc or "host" in lc:
            cols.add(c)
    return list(cols)

p_subj = Path(SUBJECTIVITY_JSON)
p_csv = Path(CSV_SOURCE)
if not p_subj.exists():
    raise FileNotFoundError(f"Missing: {p_subj.resolve()}")
if not p_csv.exists():
    raise FileNotFoundError(f"Missing: {p_csv.resolve()}")

with open(p_subj, "r") as f:
    subj_map = json.load(f)

subjectivity_domains = set()
for raw_dom in subj_map.keys():
    parent = etld1(raw_dom)
    if parent:
        subjectivity_domains.add(parent)

df = pd.read_csv(p_csv)
scan_cols = pick_candidate_columns(df)
if not scan_cols:
    raise ValueError("No candidate URL/website columns found to scan.")

results = []
multi_hit_count = 0
any_hit_count = 0

for idx, row in df.iterrows():
    found = {}
    for col in scan_cols:
        val = row.get(col, None)
        if pd.isna(val):
            continue
        host = url_to_host_maybe(val)
        if not host:
            continue
        parent = etld1(host)
        if not parent:
            continue
        if parent in subjectivity_domains:
            found.setdefault(parent, []).append(col)

    if found:
        any_hit_count += 1

    if len(found) >= 2:
        multi_hit_count += 1
        results.append({
            "row_index": idx,
            "matches_count": len(found),
            "matched_domains": ";".join(sorted(found.keys())),
            "columns_per_domain": "; ".join([f"{dom}:{'|'.join(sorted(cols))}" for dom, cols in sorted(found.items())]),
        })

out_df = pd.DataFrame(results).sort_values(["matches_count", "row_index"], ascending=[False, True])
out_df.to_csv(OUT_CSV, index=False)

print("=== Subjectivity Multi-Hit Scan ===")
print(f"CSV rows scanned:                 {len(df)}")
print(f"Candidate columns scanned:        {len(scan_cols)} -> {scan_cols}")
print(f"Rows with >=1 subjectivity domain: {any_hit_count}")
print(f"Rows with >=2 subjectivity domains: {multi_hit_count}")
print(f"Output written: {OUT_CSV}")

print("\nExamples (up to 10):")
print(out_df.head(10).to_string(index=False) if not out_df.empty else "(none)")

In [ ]:
"""
Full policy analysis split by INDUSTRY (from CSV `category` column) — with improved
parent-domain mapping and diagnostics to reduce/understand "Unknown" labels.

Inputs (same folder or update paths below):
  - ccpa_policy_audit_all.json        # rich policy audit per (sub)domain
  - data_source_with_ppurl.csv        # has `category` (industry) + website/URL columns

What this script does (v2):
  1) Normalizes domains to eTLD+1 and maps each parent domain -> industry (from CSV)
     • Uses BOTH website domain and privacy-policy URL domain per row (deduped).
     • If a parent shows multiple categories, it picks the majority NON-"Unknown" if available;
       otherwise the overall majority (which may be "Unknown").
  2) Joins the policy JSON to industry labels and extracts:
     • Rubric scores (completeness/usability/accuracy)
     • Disclosure flags (data_collected, data_shared, purpose_of_collection, retention_period,
                        right_to_access, right_to_delete, opt_out)
     • Behavioral claims (honors_gpc, respects_dnt, sets_cookies_before_consent, etc.)
     • policy_contradiction (bool)
  3) Computes paper-friendly outputs by industry:
     • Universe counts
     • Rubric stats (mean/median/std)
     • Disclosure coverage (true count + %)
     • Behavioral claims (T/F/U counts + %)
     • Contradiction counts and rates
  4) Writes CSVs and prints a concise summary + mapping diagnostics.

Deps: pandas, numpy, tldextract
Install: pip install pandas numpy tldextract
"""

import json
from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import urlparse

import pandas as pd
import numpy as np
import tldextract

POLICY_JSON = "ccpa_policy_audit_all.json"
CSV_SOURCE  = "data_source_with_ppurl.csv"

JOINED_CSV              = "joined_policy_industry.csv"
UNIVERSE_COUNTS_CSV     = "universe_counts_industry.csv"
RUBRIC_CSV              = "rubric_stats_by_industry.csv"
DISCLOSURE_CSV          = "disclosure_coverage_by_industry.csv"
BEHAVIORAL_CSV          = "behavioral_claims_by_industry.csv"
CONTRADICT_CSV          = "policy_contradictions_by_industry.csv"
UNKNOWN_DIAG_CSV        = "unknown_industry_parents_diagnostics.csv"
PARENT_CATEGORY_COUNTS  = "parent_to_category_counts.csv"

ALIAS_FIXES = {

}

WEBSITE_COL_CANDIDATES  = ["companyWebsite", "website", "Website", "domain", "site", "url"]
PPURL_COL_CANDIDATES    = ["Privacy Policy URL", "privacy_policy_url", "policy_url", "PrivacyPolicyURL"]
CATEGORY_COL_CANDIDATES = ["category", "Category", "industry", "Industry"]

def etld1(host: str) -> str:
    """Normalize a hostname/domain to its registrable domain (eTLD+1)."""
    if not host or not str(host).strip():
        return ""
    h = str(host).strip().lower()
    ext = tldextract.extract(h)
    if not ext.domain or not ext.suffix:
        return h
    return f"{ext.domain}.{ext.suffix}"

def apply_alias(d: str) -> str:
    return ALIAS_FIXES.get(d, d)

def url_to_host(value: str) -> str:
    """Extract netloc from a URL-ish string; accepts bare hosts without scheme."""
    if value is None:
        return ""
    s = str(value).strip()
    if not s:
        return ""
    if "://" not in s:
        s = "http://" + s
    try:
        return urlparse(s).netloc.lower()
    except Exception:
        return ""

def choose_first_present_column(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of the expected columns found. Tried: {candidates}")

def to_tfu(value):
    """Normalize behavioral value to 'true'/'false'/'unspecified'."""
    if isinstance(value, bool):
        return "true" if value else "false"
    return "unspecified"

p_policy = Path(POLICY_JSON)
p_csv    = Path(CSV_SOURCE)
if not p_policy.exists():
    raise FileNotFoundError(f"Missing {p_policy.resolve()}")
if not p_csv.exists():
    raise FileNotFoundError(f"Missing {p_csv.resolve()}")

with open(p_policy, "r") as f:
    policy_data = json.load(f)

csv_df = pd.read_csv(p_csv)

website_col  = choose_first_present_column(csv_df, WEBSITE_COL_CANDIDATES)
category_col = choose_first_present_column(csv_df, CATEGORY_COL_CANDIDATES)
try:
    ppurl_col = choose_first_present_column(csv_df, PPURL_COL_CANDIDATES)
except KeyError:
    ppurl_col = None

csv_df["_website_raw"]    = csv_df[website_col].astype(str).str.strip()
csv_df["_website_host"]   = csv_df["_website_raw"].apply(lambda x: url_to_host(x) or x)
csv_df["_website_parent"] = csv_df["_website_host"].apply(lambda x: apply_alias(etld1(x)))

if ppurl_col:
    csv_df["_ppurl_raw"]    = csv_df[ppurl_col].astype(str).str.strip()
    csv_df["_ppurl_host"]   = csv_df["_ppurl_raw"].apply(url_to_host)
    csv_df["_ppurl_parent"] = csv_df["_ppurl_host"].apply(lambda x: apply_alias(etld1(x)))
else:
    csv_df["_ppurl_parent"] = ""

csv_df["_category_norm"] = csv_df[category_col].astype(str).str.strip()
csv_df.loc[csv_df["_category_norm"] == "", "_category_norm"] = "Unknown"

parent_to_cat_counts = defaultdict(Counter)
for _, row in csv_df.iterrows():
    cat = row["_category_norm"]
    parents = set()
    if row["_website_parent"]:
        parents.add(row["_website_parent"])
    if row["_ppurl_parent"]:
        parents.add(row["_ppurl_parent"])
    for p in parents:
        parent_to_cat_counts[p][cat] += 1

def pick_label(counter: Counter) -> str:
    """
    Choose industry label for a parent domain:
      1) Prefer the most frequent NON-'Unknown' label if it exists.
      2) Otherwise, return the overall most common (which may be 'Unknown').
    """
    if not counter:
        return "Unknown"
    most_common = counter.most_common()
    non_unknown = [(lab, n) for lab, n in most_common if lab.lower() != "unknown"]
    return non_unknown[0][0] if non_unknown else most_common[0][0]

industry_map = {p: pick_label(cnts) for p, cnts in parent_to_cat_counts.items()}

pd.DataFrame(
    [{"parent_domain": p, "label": pick_label(cnts), "counts": dict(cnts)} for p, cnts in parent_to_cat_counts.items()]
).to_csv(PARENT_CATEGORY_COUNTS, index=False)

rows = []
for raw_dom, rec in policy_data.items():
    parent = apply_alias(etld1(raw_dom))
    industry = industry_map.get(parent, "Unknown")

    r_assess = rec.get("rubric_assessment", {}) if isinstance(rec, dict) else {}
    dmap = r_assess.get("disclosure_map", {}) if isinstance(r_assess, dict) else {}
    bmap = rec.get("behavioral_claims", {}) if isinstance(rec, dict) else {}

    row = {
        "raw_policy_domain": raw_dom,
        "parent_domain": parent,
        "industry": industry,

        "completeness_score": r_assess.get("completeness_score", np.nan),
        "usability_score":    r_assess.get("usability_score",    np.nan),
        "accuracy_score":     r_assess.get("accuracy_score",     np.nan),
        "policy_contradiction": bool(r_assess.get("policy_contradiction", False)),

        "disclosure_data_collected":        bool(dmap.get("data_collected", False)),
        "disclosure_data_shared":           bool(dmap.get("data_shared", False)),
        "disclosure_purpose_of_collection": bool(dmap.get("purpose_of_collection", False)),
        "disclosure_retention_period":      bool(dmap.get("retention_period", False)),
        "disclosure_right_to_access":       bool(dmap.get("right_to_access", False)),
        "disclosure_right_to_delete":       bool(dmap.get("right_to_delete", False)),
        "disclosure_opt_out":               bool(dmap.get("opt_out", False)),

        "honors_gpc":                         to_tfu(bmap.get("honors_gpc", "unspecified")),
        "respects_dnt":                       to_tfu(bmap.get("respects_dnt", "unspecified")),
        "sets_cookies_before_consent":        to_tfu(bmap.get("sets_cookies_before_consent", "unspecified")),
        "sets_cookies_after_rejecting_consent": to_tfu(bmap.get("sets_cookies_after_rejecting_consent", "unspecified")),
        "deletes_cookies_on_rejection":       to_tfu(bmap.get("deletes_cookies_on_rejection", "unspecified")),
        "uses_tracking_only_after_consent":   to_tfu(bmap.get("uses_tracking_only_after_consent", "unspecified")),
        "sells_data":                         to_tfu(bmap.get("sells_data", "unspecified")),
        "shares_with_third_parties":          to_tfu(bmap.get("shares_with_third_parties", "unspecified")),
    }
    rows.append(row)

df = pd.DataFrame(rows)

policy_parents = set(df["parent_domain"])
mapped_parents = set(industry_map.keys())

parents_missing_in_csv = sorted(policy_parents - mapped_parents)
parents_labeled_unknown = sorted(
    p for p in (policy_parents & mapped_parents) if industry_map.get(p, "Unknown") == "Unknown"
)

diag_rows = []
for p in parents_missing_in_csv[:200]:
    sample = df.loc[df["parent_domain"] == p, "raw_policy_domain"].iloc[0]
    diag_rows.append({"parent_domain": p, "reason": "not_in_csv_map", "example_policy_domain": sample})
for p in parents_labeled_unknown[:200]:
    sample = df.loc[df["parent_domain"] == p, "raw_policy_domain"].iloc[0]
    cnts = parent_to_cat_counts.get(p, Counter())
    diag_rows.append({
        "parent_domain": p,
        "reason": "csv_category_unknown",
        "example_policy_domain": sample,
        "csv_category_counts": dict(cnts)
    })

pd.DataFrame(diag_rows).to_csv(UNKNOWN_DIAG_CSV, index=False)

print("\n=== Industry mapping coverage diagnostics ===")
print(f"Policy parents total:               {len(policy_parents)}")
print(f"Policy parents found in CSV map:    {len(policy_parents & mapped_parents)}")
print(f"Policy parents missing in CSV map:  {len(parents_missing_in_csv)}")
print(f"Policy parents labeled 'Unknown':   {len(parents_labeled_unknown)}")
print(f"Wrote diagnostics: {UNKNOWN_DIAG_CSV}")
print(f"Wrote parent->category counts: {PARENT_CATEGORY_COUNTS}")

universe_counts = df["industry"].value_counts(dropna=False).reset_index()
universe_counts.columns = ["industry", "n_policies"]
universe_counts.sort_values("n_policies", ascending=False, inplace=True)
universe_counts.to_csv(UNIVERSE_COUNTS_CSV, index=False)

rubric_cols = ["completeness_score", "usability_score", "accuracy_score"]
rubric_stats = (
    df.groupby("industry")[rubric_cols]
      .agg(["mean", "median", "std"])
      .sort_index()
)
rubric_stats.to_csv(RUBRIC_CSV)

disclosure_cols = [
    "disclosure_data_collected",
    "disclosure_data_shared",
    "disclosure_purpose_of_collection",
    "disclosure_retention_period",
    "disclosure_right_to_access",
    "disclosure_right_to_delete",
    "disclosure_opt_out",
]
disc_rows = []
for ind, g in df.groupby("industry"):
    n = len(g)
    for col in disclosure_cols:
        true_count = int(g[col].fillna(False).astype(bool).sum())
        disc_rows.append({
            "industry": ind,
            "metric": col.replace("disclosure_", ""),
            "true_count": true_count,
            "group_n": n,
            "true_pct": (true_count / n) if n else np.nan,
        })
disclosure_coverage = pd.DataFrame(disc_rows).sort_values(["industry", "metric"])
disclosure_coverage.to_csv(DISCLOSURE_CSV, index=False)

behavioral_cols = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]
behav_rows = []
for ind, g in df.groupby("industry"):
    n = len(g)
    for col in behavioral_cols:
        counts = g[col].fillna("unspecified").value_counts()
        t = int(counts.get("true", 0))
        f = int(counts.get("false", 0))
        u = int(counts.get("unspecified", 0))
        behav_rows.append({
            "industry": ind,
            "claim": col,
            "true_count": t,
            "false_count": f,
            "unspecified_count": u,
            "group_n": n,
            "true_pct": (t / n) if n else np.nan,
            "false_pct": (f / n) if n else np.nan,
            "unspecified_pct": (u / n) if n else np.nan,
        })
behavioral_claims = pd.DataFrame(behav_rows).sort_values(["industry", "claim"])
behavioral_claims.to_csv(BEHAVIORAL_CSV, index=False)

contr = df.groupby("industry")["policy_contradiction"].agg(["sum", "count"])
contr["rate"] = contr["sum"] / contr["count"]
contr.sort_index(inplace=True)
contr.to_csv(CONTRADICT_CSV)

df.to_csv(JOINED_CSV, index=False)

unknown_mask = df["industry"] == "Unknown"
unknown_parents_df = (
    df.loc[unknown_mask, ["parent_domain", "raw_policy_domain"]]
      .drop_duplicates("parent_domain")
      .sort_values("parent_domain")
      .reset_index(drop=True)
)

print("\n=== Policies with industry 'Unknown' ===")
print(f"Unique parent domains with 'Unknown': {len(unknown_parents_df)}")
if unknown_parents_df.empty:
    print("(none)")
else:

    print(unknown_parents_df.to_string(index=False))

policy_parents_unknown = set(unknown_parents_df["parent_domain"])
mapped_parents = set(industry_map.keys())

parents_not_in_csv_map = sorted(policy_parents_unknown - mapped_parents)
parents_csv_category_unknown = sorted(policy_parents_unknown & mapped_parents)

print("\n--- Reason breakdown for 'Unknown' parents ---")
print(f"Not in CSV map (never seen in data_source_with_ppurl.csv): {len(parents_not_in_csv_map)}")
print(f"In CSV but labeled 'Unknown' (per CSV categories):        {len(parents_csv_category_unknown)}")

MAX_SHOW = 200

if parents_not_in_csv_map:
    print("\n[not_in_csv_map] parent_domain → example_raw_policy_domain")
    for p in parents_not_in_csv_map[:MAX_SHOW]:
        example = unknown_parents_df.loc[unknown_parents_df["parent_domain"] == p, "raw_policy_domain"].iloc[0]
        print(f"  {p:35s}  ->  {example}")
    if len(parents_not_in_csv_map) > MAX_SHOW:
        print(f"  ... and {len(parents_not_in_csv_map) - MAX_SHOW} more")

if parents_csv_category_unknown:
    print("\n[csv_category_unknown] parent_domain → CSV category counts seen")
    for p in parents_csv_category_unknown[:MAX_SHOW]:
        cnts = parent_to_cat_counts.get(p, Counter())

        cnts_sorted = dict(sorted(cnts.items(), key=lambda kv: kv[1], reverse=True))
        print(f"  {p:35s}  ->  {cnts_sorted}")
    if len(parents_csv_category_unknown) > MAX_SHOW:
        print(f"  ... and {len(parents_csv_category_unknown) - MAX_SHOW} more")

print("\n=== Universe (policies per industry) ===")
print(universe_counts.to_string(index=False))

print("\n=== Rubric stats by industry (mean/median/std) ===")
print(rubric_stats.round(3).to_string())

print("\n=== Disclosure coverage by industry (true count and % of group) — first few rows ===")
print(disclosure_coverage.head(14).to_string(index=False))

print("\n=== Behavioral claims by industry (T/F/U counts and %) — first few rows ===")
print(behavioral_claims.head(14).to_string(index=False))

print("\n=== Policy contradictions by industry ===")
print(contr.round(3).to_string())

print("\nWrote:")
print(f"  - {JOINED_CSV}")
print(f"  - {UNIVERSE_COUNTS_CSV}")
print(f"  - {RUBRIC_CSV}")
print(f"  - {DISCLOSURE_CSV}")
print(f"  - {BEHAVIORAL_CSV}")
print(f"  - {CONTRADICT_CSV}")
print(f"  - {UNKNOWN_DIAG_CSV}")
print(f"  - {PARENT_CATEGORY_COUNTS}")

In [ ]:
"""
Run full policy analysis BY INDUSTRY using your already-joined file
(e.g., 'joined_policy_industry_updated.csv') where 'industry' is corrected.

Inputs:
  - joined_policy_industry_updated.csv   # or 'filejoined_policy_industry_updated.csv'
      Expected columns (robust to slight naming):
        - industry (or category)
        - raw_policy_domain, parent_domain
        - rubric scores: completeness_score, usability_score, accuracy_score
        - policy_contradiction (bool/str)
        - disclosures:    data_collected, data_shared, purpose_of_collection,
                          retention_period, right_to_access, right_to_delete, opt_out
          (either as the plain names above OR prefixed with 'disclosure_')
        - behavioral:     honors_gpc, respects_dnt, sets_cookies_before_consent,
                          sets_cookies_after_rejecting_consent, deletes_cookies_on_rejection,
                          uses_tracking_only_after_consent, sells_data, shares_with_third_parties
          (values may be bools or 'true'/'false'/'unspecified')

Outputs (CSV):
  - universe_counts_industry_from_joined.csv
  - rubric_stats_by_industry_from_joined.csv
  - disclosure_coverage_by_industry_from_joined.csv
  - behavioral_claims_by_industry_from_joined.csv
  - policy_contradictions_by_industry_from_joined.csv
  - joined_policy_industry_cleaned.csv (normalized booleans/strings; same rows as input)
"""

from pathlib import Path
import pandas as pd
import numpy as np

INPUT_CANDIDATES = [
    "joined_policy_industry_newww.csv",
]

UNIVERSE_COUNTS_CSV = "universe_counts_industry_from_joined.csv"
RUBRIC_CSV          = "rubric_stats_by_industry_from_joined.csv"
DISCLOSURE_CSV      = "disclosure_coverage_by_industry_from_joined.csv"
BEHAVIORAL_CSV      = "behavioral_claims_by_industry_from_joined.csv"
CONTRADICT_CSV      = "policy_contradictions_by_industry_from_joined.csv"
CLEANED_JOINED_OUT  = "joined_policy_industry_cleaned.csv"

RUBRIC_KEYS = ["completeness_score", "usability_score", "accuracy_score"]
DISCLOSURE_BASES = [
    "data_collected", "data_shared", "purpose_of_collection",
    "retention_period", "right_to_access", "right_to_delete", "opt_out",
]
BEHAVIORAL_KEYS = [
    "honors_gpc", "respects_dnt",
    "sets_cookies_before_consent", "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection", "uses_tracking_only_after_consent",
    "sells_data", "shares_with_third_parties",
]

def find_input_path(candidates):
    for name in candidates:
        p = Path(name)
        if p.exists():
            return p
    raise FileNotFoundError(
        "Could not find any of: " + ", ".join(candidates) +
        " (set INPUT_CANDIDATES or rename your file accordingly)."
    )

def coerce_bool_series(s):
    """
    Convert heterogeneous truthy/falsey to booleans.
    Treat anything not clearly 'true' as False (including NaN).
    """
    if s is None:
        return pd.Series([False] * 0)
    if s.dtype == bool:
        return s.fillna(False)

    m = s.astype(str).str.strip().str.lower()
    truthy = {"true", "t", "1", "yes", "y"}
    return m.apply(lambda x: True if x in truthy else False)

def coerce_tfu_series(s):
    """
    Normalize behavioral values to 'true'/'false'/'unspecified'.
    Booleans -> 'true'/'false', missing/other -> 'unspecified'.
    """
    if s is None:
        return pd.Series(["unspecified"] * 0)
    if s.dtype == bool:
        return s.fillna(False).map({True: "true", False: "false"})
    m = s.astype(str).str.strip().str.lower()
    def norm(x):
        if x in {"true", "t", "1", "yes", "y"}:
            return "true"
        if x in {"false", "f", "0", "no", "n"}:
            return "false"
        return "unspecified"
    return m.map(norm)

def get_first_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def find_disclosure_columns(df):
    """
    Return dict { canonical_name -> actual_column_in_df }
    Accept either 'disclosure_X' or 'X'.
    """
    mapping = {}
    for base in DISCLOSURE_BASES:
        pref = f"disclosure_{base}"
        if pref in df.columns:
            mapping[base] = pref
        elif base in df.columns:
            mapping[base] = base
        else:

            mapping[base] = None
    return mapping

in_path = find_input_path(INPUT_CANDIDATES)
df = pd.read_csv(in_path)

industry_col = get_first_col(df, ["industry", "category", "Industry", "Category"])
if industry_col is None:
    raise ValueError("No industry/category column found in the joined file.")
df["industry"] = df[industry_col].astype(str).str.strip()

for k in RUBRIC_KEYS:
    if k not in df.columns:
        raise ValueError(f"Missing rubric column '{k}' in the joined file.")
    df[k] = pd.to_numeric(df[k], errors="coerce")

if "policy_contradiction" not in df.columns:
    raise ValueError("Missing 'policy_contradiction' in the joined file.")
df["policy_contradiction"] = coerce_bool_series(df["policy_contradiction"])

disc_map = find_disclosure_columns(df)
for base, col in disc_map.items():
    if col is None:
        df[f"disclosure_{base}"] = False
    else:
        df[f"disclosure_{base}"] = coerce_bool_series(df[col])

for k in BEHAVIORAL_KEYS:
    if k in df.columns:
        df[k] = coerce_tfu_series(df[k])
    else:

        df[k] = "unspecified"

df.to_csv(CLEANED_JOINED_OUT, index=False)

universe_counts = (
    df["industry"].value_counts(dropna=False)
      .rename_axis("industry").reset_index(name="n_policies")
      .sort_values("n_policies", ascending=False)
)
universe_counts.to_csv(UNIVERSE_COUNTS_CSV, index=False)

rubric_stats = (
    df.groupby("industry")[RUBRIC_KEYS]
      .agg(["mean", "median", "std"])
      .sort_index()
)
rubric_stats.to_csv(RUBRIC_CSV)

disclosure_cols_final = [f"disclosure_{b}" for b in DISCLOSURE_BASES]
disc_rows = []
for ind, g in df.groupby("industry"):
    n = len(g)
    for base in DISCLOSURE_BASES:
        col = f"disclosure_{base}"
        true_count = int(g[col].fillna(False).astype(bool).sum())
        disc_rows.append({
            "industry": ind,
            "metric": base,
            "true_count": true_count,
            "group_n": n,
            "true_pct": (true_count / n) if n else np.nan,
        })
disclosure_coverage = pd.DataFrame(disc_rows).sort_values(["industry", "metric"])
disclosure_coverage.to_csv(DISCLOSURE_CSV, index=False)

behav_rows = []
for ind, g in df.groupby("industry"):
    n = len(g)
    for col in BEHAVIORAL_KEYS:
        counts = g[col].fillna("unspecified").value_counts()
        t = int(counts.get("true", 0))
        f = int(counts.get("false", 0))
        u = int(counts.get("unspecified", 0))
        behav_rows.append({
            "industry": ind,
            "claim": col,
            "true_count": t,
            "false_count": f,
            "unspecified_count": u,
            "group_n": n,
            "true_pct": (t / n) if n else np.nan,
            "false_pct": (f / n) if n else np.nan,
            "unspecified_pct": (u / n) if n else np.nan,
        })
behavioral_claims = pd.DataFrame(behav_rows).sort_values(["industry", "claim"])
behavioral_claims.to_csv(BEHAVIORAL_CSV, index=False)

contr = df.groupby("industry")["policy_contradiction"].agg(["sum", "count"])
contr["rate"] = contr["sum"] / contr["count"]
contr.sort_index(inplace=True)
contr.to_csv(CONTRADICT_CSV)

print(f"\nRead: {in_path}")
print(f"Saved cleaned copy: {CLEANED_JOINED_OUT}")

print("\n=== Universe (policies per industry) ===")
print(universe_counts.to_string(index=False))

print("\n=== Rubric stats by industry (mean/median/std) ===")
print(rubric_stats.round(3).to_string())

print("\n=== Disclosure coverage by industry (true count and % of group) — first few rows ===")
print(disclosure_coverage.head(14).to_string(index=False))

print("\n=== Behavioral claims by industry (T/F/U counts and %) — first few rows ===")
print(behavioral_claims.head(14).to_string(index=False))

print("\n=== Policy contradictions by industry ===")
print(contr.round(3).to_string())

print("\nWrote:")
print(f"  - {UNIVERSE_COUNTS_CSV}")
print(f"  - {RUBRIC_CSV}")
print(f"  - {DISCLOSURE_CSV}")
print(f"  - {BEHAVIORAL_CSV}")
print(f"  - {CONTRADICT_CSV}")

In [ ]:
import os
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

CSV_PATH      = "rubric_stats_by_subjectivity.csv"
OUTDIR        = "paper/figures/new_figures"
FNAME_SINGLE  = "barplot_rubric_scores_ccpa_split"
FNAME_FACETS  = "barplot_rubric_scores_ccpa_split_facets"

USE_95CI      = False

FIGSIZE_SINGLE = (11.6, 6.2)
FIGSIZE_FACETS = (12.6, 4.6)
DPI            = 300

SCORE_MIN, SCORE_MAX = 0.0, 3.0

PALETTE      = {"Subjected": "#3b6fb6", "Non-Subj.": "#e19342"}
GROUP_ORDER  = ["Subjected", "Non-Subj."]

METRIC_ORDER = ["completeness_score", "usability_score", "accuracy_score"]
METRIC_MAP   = {
    "completeness_score": "Completeness",
    "usability_score":    "Usability",
    "accuracy_score":     "Accuracy",
}

mpl.rcParams.update({
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "savefig.transparent": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})

GROUP_CANON = {
    "subjected": "Subjected",
    "subject": "Subjected",
    "ccpa_subject": "Subjected",
    "ccpa_subj": "Subjected",
    "not_subjected": "Non-Subj.",
    "non-subj.": "Non-Subj.",
    "non-subj": "Non-Subj.",
    "non_subj": "Non-Subj.",
    "nonsubj": "Non-Subj.",
    "non subjected": "Non-Subj.",
    "not subjected": "Non-Subj.",
    "ccpa non-subject": "Non-Subj.",
}
def canon_group(x: str) -> str:
    if pd.isna(x): return None
    key = str(x).strip().lower()
    return GROUP_CANON.get(
        key,
        "Subjected" if ("subject" in key and "non" not in key and "not" not in key) else
        ("Non-Subj." if ("non" in key or "not" in key) else None)
    )

def compute_yerr(row, use_ci: bool):
    if not use_ci:
        return float(row["std"])
    n = row.get("n", np.nan)
    return 1.96 * float(row["std"]) / np.sqrt(n) if (pd.notna(n) and n > 1) else np.nan

def canonical_group_n(long_df: pd.DataFrame) -> dict:
    out = {}
    for g, gdf in long_df.groupby("Group", observed=True):
        ns = [int(x) for x in gdf["n"].dropna().tolist()]
        if not ns:
            out[g] = np.nan
            continue
        c = Counter(ns)
        out[g] = max(c.items(), key=lambda kv: (kv[1], kv[0]))[0]
    return out

def load_long_from_csv(csv_path: str):
    p = Path(csv_path)
    if not p.exists():
        raise FileNotFoundError(f"CSV not found: {p.resolve()}")
    wide = pd.read_csv(p)

    grp_col = next((c for c in ["subjectivity", "Group", "group"] if c in wide.columns), None)
    if grp_col is None:
        raise ValueError("CSV must include 'subjectivity' (preferred) or 'Group'/'group'.")

    wide["Group"] = wide[grp_col].apply(canon_group)
    if wide["Group"].isna().any():
        raise ValueError(f"Unrecognized group labels: {sorted(wide.loc[wide['Group'].isna(), grp_col].unique())}")

    rows, n_by_metric = [], {m: (f"{m}_n" if f"{m}_n" in wide.columns else None) for m in METRIC_ORDER}
    has_global_n = "n" in wide.columns

    for _, r in wide.iterrows():
        g = r["Group"]
        for m in METRIC_ORDER:
            mc, sc, xc = f"{m}_mean", f"{m}_std", f"{m}_median"
            for col in (mc, sc, xc):
                if col not in wide.columns:
                    raise ValueError(f"CSV missing '{col}'.")

            nval = np.nan
            if n_by_metric[m] is not None:
                nval = pd.to_numeric(r[n_by_metric[m]], errors="coerce")
            elif has_global_n:
                nval = pd.to_numeric(r["n"], errors="coerce")

            rows.append({
                "MetricKey": m,
                "Metric": METRIC_MAP[m],
                "Group": g,
                "mean":   float(r[mc]),
                "std":    float(r[sc]),
                "median": float(r[xc]),
                "n":      np.nan if pd.isna(nval) else int(nval),
            })

    long = pd.DataFrame(rows)
    long["Metric"] = pd.Categorical(long["Metric"], [METRIC_MAP[k] for k in METRIC_ORDER], ordered=True)
    long["Group"]  = pd.Categorical(long["Group"], GROUP_ORDER, ordered=True)

    long["yerr_sym"] = long.apply(lambda row: compute_yerr(row, USE_95CI), axis=1)
    long["yerr_lo"]  = np.maximum(0.0, np.minimum(long["yerr_sym"], long["mean"] - SCORE_MIN))
    long["yerr_hi"]  = np.maximum(0.0, np.minimum(long["yerr_sym"], SCORE_MAX - long["mean"]))

    return long, canonical_group_n(long)

def validate_inputs(df: pd.DataFrame):
    print("\n=== Parsed Data (first 6 rows) ===")
    print(df.head(6).to_string(index=False))
    print("\nGroups:", df['Group'].unique().tolist())
    print("Metrics:", df['Metric'].unique().tolist())
    for (m, g), sub in df.groupby(['Metric', 'Group'], observed=True):
        means = sub['mean'].tolist()
        if len(means) != 1:
            print(f"[WARN] Duplicate rows for {m}, {g}: {len(means)} rows -> means={means}")

def plot_grouped(csv_path=CSV_PATH, outdir=OUTDIR, fname=FNAME_SINGLE):
    df, group_n = load_long_from_csv(csv_path)
    validate_inputs(df)
    Path(outdir).mkdir(parents=True, exist_ok=True)

    metrics = [METRIC_MAP[k] for k in METRIC_ORDER]
    groups  = GROUP_ORDER
    means   = np.array([[df[(df['Metric']==m) & (df['Group']==g)]['mean'].iloc[0] for g in groups] for m in metrics])
    lo_err  = np.array([[df[(df['Metric']==m) & (df['Group']==g)]['yerr_lo'].iloc[0] for g in groups] for m in metrics])
    hi_err  = np.array([[df[(df['Metric']==m) & (df['Group']==g)]['yerr_hi'].iloc[0] for g in groups] for m in metrics])

    fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE, constrained_layout=True)

    x = np.arange(len(metrics))
    width = 0.34
    offsets = [-width/2, width/2]

    bars = []
    for j, g in enumerate(groups):
        centers = x + offsets[j]
        b = ax.bar(
            centers, means[:, j],
            width=width, label=g,
            color=PALETTE[g], edgecolor="black", linewidth=0.9, zorder=3
        )

        yerr = np.vstack((lo_err[:, j], hi_err[:, j]))
        ax.errorbar(
            centers, means[:, j], yerr=yerr,
            fmt="none", ecolor="black", elinewidth=1.1, capsize=4, capthick=1.1, zorder=4
        )
        bars.append(b)

    ax.set_xticks(x, metrics)
    ax.set_xlabel("")
    ax.set_ylabel("Rubric score (mean ± {})".format("95% CI" if USE_95CI else "SD"))
    ax.set_ylim(SCORE_MIN, SCORE_MAX + 0.15)
    ax.grid(True, axis="y", linewidth=0.6, alpha=0.35, zorder=0)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis="y", direction="in", length=4, width=0.9)
    ax.margins(x=0.06)

    labels = [
        f"{g} (n={int(group_n[g])})" if (g in group_n and pd.notna(group_n[g])) else g
        for g in groups
    ]
    handles = [Patch(facecolor=PALETTE[g], edgecolor="black") for g in groups]
    ax.legend(handles, labels, title="", ncol=2, frameon=False,
              loc="upper center", bbox_to_anchor=(0.5, -0.12))
    fig.subplots_adjust(bottom=0.20)

    pdf = Path(outdir) / f"{fname}.pdf"
    png = Path(outdir) / f"{fname}.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {pdf} | {png}")

def plot_facets(csv_path=CSV_PATH, outdir=OUTDIR, fname=FNAME_FACETS):
    df, group_n = load_long_from_csv(csv_path)
    validate_inputs(df)
    Path(outdir).mkdir(parents=True, exist_ok=True)

    metrics = [METRIC_MAP[k] for k in METRIC_ORDER]
    groups  = GROUP_ORDER

    fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_FACETS, sharey=True, constrained_layout=True)
    width = 0.42
    offsets = [-width/2, width/2]
    x = np.arange(len(groups))

    for ax, m in zip(axes, metrics):
        sub = df[df["Metric"] == m].copy().sort_values("Group")

        means = np.array([sub[sub["Group"] == g]["mean"].iloc[0] for g in groups])
        lo    = np.array([sub[sub["Group"] == g]["yerr_lo"].iloc[0] for g in groups])
        hi    = np.array([sub[sub["Group"] == g]["yerr_hi"].iloc[0] for g in groups])

        for j, g in enumerate(groups):
            center = j
            b = ax.bar(
                center, means[j], width=0.6,
                color=PALETTE[g], edgecolor="black", linewidth=0.9, zorder=3
            )
            ax.errorbar(
                center, means[j], yerr=np.array([[lo[j]], [hi[j]]]),
                fmt="none", ecolor="black", elinewidth=1.1, capsize=4, capthick=1.1, zorder=4
            )

        ax.set_title(m)
        ax.set_xlabel("")
        ax.set_xticks([0, 1], groups)
        ax.set_ylim(SCORE_MIN, SCORE_MAX + 0.15)
        ax.grid(True, axis="y", linewidth=0.6, alpha=0.35, zorder=0)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)
        ax.tick_params(axis="y", direction="in", length=4, width=0.9)

    axes[0].set_ylabel("Rubric score (mean ± {})".format("95% CI" if USE_95CI else "SD"))

    handles = [Patch(facecolor=PALETTE[g], edgecolor="black") for g in groups]
    labels  = [
        f"{g} (n={int(group_n[g])})" if (g in group_n and pd.notna(group_n[g])) else g
        for g in groups
    ]
    fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.04))

    pdf = Path(outdir) / f"{fname}.pdf"
    png = Path(outdir) / f"{fname}.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {pdf} | {png}")

if __name__ == "__main__":
    Path(OUTDIR).mkdir(parents=True, exist_ok=True)
    plot_grouped(CSV_PATH, OUTDIR, FNAME_SINGLE)
    plot_facets(CSV_PATH, OUTDIR, FNAME_FACETS)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from math import sqrt

SPINE_COLOR = 'gray'
font = {'family': 'Times', 'weight': 'normal', 'size': 12}
matplotlib.rc('font', **font)

def latexify(fig_width=None, fig_height=None, columns=1):
    assert columns in [1, 2]
    if fig_width is None:
        fig_width = 3.39 if columns == 1 else 6.9
    if fig_height is None:
        golden_mean = (sqrt(5) - 1.0) / 2.0
        fig_height = fig_width * golden_mean
    MAX_HEIGHT_INCHES = 8.0
    if fig_height > MAX_HEIGHT_INCHES:
        fig_height = MAX_HEIGHT_INCHES
    params = {
        'backend': 'ps',
        'text.latex.preamble': r'\usepackage{gensymb}',
        'axes.labelsize': 10,
        'axes.titlesize': 12,
        'font.size': 10,
        'legend.fontsize': 10,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'text.usetex': False,
        'figure.figsize': [fig_width, fig_height],
        'font.family': 'Times'
    }
    matplotlib.rcParams.update(params)
    return params

def format_axes(ax):
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color(SPINE_COLOR)
        ax.spines[spine].set_linewidth(0.8)
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')
    for axis in [ax.xaxis, ax.yaxis]:
        axis.set_tick_params(direction='in', color=SPINE_COLOR, length=4, width=0.8)
    ax.grid(True, axis='y', linestyle='--', alpha=0.35)
    return ax

latexify(columns=2)

DISCLOSURE_CSV = "disclosure_coverage_by_industry_from_joined.csv"
CLAIMS_CSV     = "behavioral_claims_by_industry_from_joined.csv"

OUT_DISC_PDF = "boxplot_disclosure_truepct_by_metric.pdf"
OUT_DISC_PNG = "boxplot_disclosure_truepct_by_metric.png"

OUT_CLAIMS_PDF = "boxplot_claims_true_false_unspec_facets.pdf"
OUT_CLAIMS_PNG = "boxplot_claims_true_false_unspec_facets.png"

SHOW_JITTER = True
JITTER_ALPHA = 0.45
JITTER_SIZE = 10

def _sorted_categories_by_median(df, cat_col, val_col):
    meds = df.groupby(cat_col, as_index=False)[val_col].median().sort_values(val_col, ascending=False)
    return meds[cat_col].tolist()

def _jitter_positions(x, n, spread=0.06, seed=0):
    rng = np.random.default_rng(seed)
    return x + rng.uniform(-spread, +spread, size=n)

def plot_disclosure_boxplots(csv_path=DISCLOSURE_CSV,
                             out_pdf=OUT_DISC_PDF,
                             out_png=OUT_DISC_PNG):
    df = pd.read_csv(csv_path)

    req = {"industry", "metric", "true_pct"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"Disclosure CSV missing columns: {missing}")

    metric_order = _sorted_categories_by_median(df, "metric", "true_pct")

    data = [df.loc[df["metric"] == m, "true_pct"].dropna().values for m in metric_order]

    n_cats = len(metric_order)
    fig_w = min(18.0, 9.0 + 0.25 * max(0, n_cats - 10))
    fig, ax = plt.subplots(figsize=(fig_w, 5.4), dpi=300)

    positions = np.arange(n_cats, dtype=float)
    bp = ax.boxplot(
        data,
        positions=positions,
        widths=0.6,
        patch_artist=True,
        showfliers=False,
        manage_ticks=False,
        notch=True,
    )

    for box in bp['boxes']:
        box.set(facecolor="#4E79A7", edgecolor="black", alpha=0.80, linewidth=0.9)
    for median in bp['medians']:
        median.set(color="black", linewidth=1.2)
    for whisk in bp['whiskers'] + bp['caps']:
        whisk.set(color="black", linewidth=0.9)

    if SHOW_JITTER:
        for i, m in enumerate(metric_order):
            y = df.loc[df["metric"] == m, "true_pct"].dropna().values
            x = _jitter_positions(i, len(y), spread=0.06, seed=42 + i)
            ax.scatter(x, y, s=JITTER_SIZE, c="black", alpha=JITTER_ALPHA, linewidths=0)

    ax.set_xticks(positions)
    ax.set_xticklabels(metric_order, rotation=45, ha="right")
    ax.set_ylabel("Disclosure coverage (share of policies with disclosure)")
    ax.set_ylim(0.0, 1.0)
    ax.set_title("Disclosure Coverage by Metric (Distribution Across Industries)", pad=10)

    format_axes(ax)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.26)

    plt.savefig(out_pdf, bbox_inches="tight")
    plt.savefig(out_png, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_pdf} | {out_png}")

def plot_claims_faceted_boxplots(csv_path=CLAIMS_CSV,
                                 out_pdf=OUT_CLAIMS_PDF,
                                 out_png=OUT_CLAIMS_PNG):
    df = pd.read_csv(csv_path)

    needed = {"industry", "claim", "true_pct", "false_pct", "unspecified_pct"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Claims CSV missing columns: {missing}")

    long = df.melt(
        id_vars=["industry", "claim"],
        value_vars=["true_pct", "false_pct", "unspecified_pct"],
        var_name="status",
        value_name="pct"
    )

    status_map = {
        "true_pct": "True",
        "false_pct": "False",
        "unspecified_pct": "Unspecified",
    }
    long["status"] = long["status"].map(status_map)

    claim_order = _sorted_categories_by_median(long[long["status"] == "True"], "claim", "pct")

    statuses = ["True", "False", "Unspecified"]
    fig_w = min(18.0, 6.9)
    fig, axes = plt.subplots(1, 3, figsize=(fig_w, 3.6), dpi=300, sharey=True)

    colors = {"True": "#59A14F", "False": "#E15759", "Unspecified": "#9C9C9C"}

    for ax, st in zip(axes, statuses):
        sub = long[long["status"] == st].copy()

        data = [sub.loc[sub["claim"] == c, "pct"].dropna().values for c in claim_order]
        pos = np.arange(len(claim_order), dtype=float)

        bp = ax.boxplot(
            data,
            positions=pos,
            widths=0.6,
            patch_artist=True,
            showfliers=False,
            manage_ticks=False,
            notch=True,
        )
        for box in bp['boxes']:
            box.set(facecolor=colors[st], edgecolor="black", alpha=0.80, linewidth=0.9)
        for median in bp['medians']:
            median.set(color="black", linewidth=1.2)
        for whisk in bp['whiskers'] + bp['caps']:
            whisk.set(color="black", linewidth=0.9)

        if SHOW_JITTER:
            for i, c in enumerate(claim_order):
                y = sub.loc[sub["claim"] == c, "pct"].dropna().values
                x = _jitter_positions(i, len(y), spread=0.06, seed=99 + i)
                ax.scatter(x, y, s=JITTER_SIZE, c="black", alpha=JITTER_ALPHA, linewidths=0)

        ax.set_title(st)
        ax.set_xticks(pos)
        ax.set_xticklabels(claim_order, rotation=60, ha="right")
        format_axes(ax)

    axes[0].set_ylabel("Share of policies making the claim")
    axes[0].set_ylim(0.0, 1.0)
    fig.suptitle("Behavioral Claims Across Industries (Box Plots by Claim and Status)", y=1.02, fontsize=12)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.42, top=0.90)

    plt.savefig(out_pdf, bbox_inches="tight")
    plt.savefig(out_png, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_pdf} | {out_png}")

if __name__ == "__main__":
    plot_disclosure_boxplots()
    plot_claims_faceted_boxplots()

In [ ]:
import os
import re
import glob
import json
from pathlib import Path
from typing import Dict, Any, List, Tuple
from collections import defaultdict

import pandas as pd
from tqdm import tqdm
from openai import OpenAI

POLICIES_DIR = "policies_text"
MAIN_JSON    = "ccpa_policy_audit_all.json"
SUMMARY_CSV  = "ccpa_policy_extras_summary.csv"
MODEL        = "gpt-4o"
TEMPERATURE  = 0
MAX_OUTPUT_TOKENS = 900

API_KEY = "REDACTED-SET-OPENAI_API_KEY-ENV-VAR"
client = OpenAI(api_key=API_KEY)

def iter_policy_files() -> List[Tuple[str, Path]]:
    """Return (domain, path) from POLICIES_DIR/*.txt with names like 'www_example_com__combined.txt'."""
    files = sorted(glob.glob(os.path.join(POLICIES_DIR, "*.txt")))
    out = []
    for p in files:
        name = Path(p).stem
        domain = name.split("__combined", 1)[0].replace("_", ".")
        out.append((domain, Path(p)))
    return out

def load_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

def load_json(path: str) -> Dict[str, Any]:
    if not os.path.exists(path):
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f) or {}
    except Exception:
        return {}

def save_json(obj: Dict[str, Any], path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def build_prompt(domain: str, policy_text: str) -> str:
    return f"""
You are auditing a privacy policy for {domain}. Work ONLY from the text below. Return exactly one JSON object with this schema:

{{
  "cookie_practices_mentioned": true/false/"unspecified",
  "cookie_practices_excerpt": "string (≤ 600 chars). If unspecified, say: 'No explicit mention found.'",
  "mentions_ccpa": true/false/"unspecified",
  "mentions_ccpa_excerpt": "string (≤ 600 chars). If unspecified, say: 'No explicit mention found.'",
  "contacts": {{
    "opt_out": {{
      "form_urls": ["..."],
      "emails": ["..."],
      "phone_numbers": ["..."],
      "other_urls": ["..."]
    }},
    "access_or_delete": {{
      "form_urls": ["..."],
      "emails": ["..."],
      "phone_numbers": ["..."],
      "other_urls": ["..."]
    }}
  }},
  "contacts_flat": [
    {{
      "type": "form_url|email|phone|other_url",
      "for": "opt_out|access|delete|access_delete|general_privacy",
      "value": "string",
      "label": "string or empty"
    }}
  ]
}}

Strict rules:
- Use true/false ONLY if explicit. Otherwise use "unspecified".
- cookie_practices_mentioned = true if the policy discusses cookies (e.g., cookies, pixels, SDKs, analytics cookies) at all, regardless of consent steps.
- mentions_ccpa = true if the policy explicitly mentions “CCPA”, “CPRA”, “California Consumer Privacy Act”, “California privacy rights” (in a legal-rights sense).
- Extract concrete contact methods for (a) opt-out of sale/share/targeted advertising, and (b) access/delete requests (right to know/delete). Put each into the appropriate bucket(s).
- Provide a flattened list in contacts_flat with the intended purpose (for) and a guessed label if present (e.g., “Your Privacy Choices”, “Do Not Sell or Share”).
- If no contacts are found for a category, return empty arrays.

================== BEGIN POLICY TEXT ==================
{policy_text}
=================== END POLICY TEXT ===================
""".strip()

def _tri(v):
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}:
        return True
    if s in {"false", "no", "0"}:
        return False
    return "unspecified"

def _s(v, default=""):
    return (v or "").strip() if isinstance(v, str) else default

def _list(v):
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    return []

def normalize_extras(raw: dict) -> dict:
    out = {
        "cookie_practices_mentioned": _tri(raw.get("cookie_practices_mentioned", "unspecified")),
        "cookie_practices_excerpt": _s(raw.get("cookie_practices_excerpt"), "No explicit mention found.")[:600],
        "mentions_ccpa": _tri(raw.get("mentions_ccpa", "unspecified")),
        "mentions_ccpa_excerpt": _s(raw.get("mentions_ccpa_excerpt"), "No explicit mention found.")[:600],
        "contacts": {
            "opt_out": {
                "form_urls": _list(raw.get("contacts", {}).get("opt_out", {}).get("form_urls", [])),
                "emails": _list(raw.get("contacts", {}).get("opt_out", {}).get("emails", [])),
                "phone_numbers": _list(raw.get("contacts", {}).get("opt_out", {}).get("phone_numbers", [])),
                "other_urls": _list(raw.get("contacts", {}).get("opt_out", {}).get("other_urls", [])),
            },
            "access_or_delete": {
                "form_urls": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("form_urls", [])),
                "emails": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("emails", [])),
                "phone_numbers": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("phone_numbers", [])),
                "other_urls": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("other_urls", [])),
            },
        },
        "contacts_flat": []
    }
    flat = raw.get("contacts_flat", [])
    if isinstance(flat, list):
        for item in flat:
            if not isinstance(item, dict):
                continue
            t = (_s(item.get("type")) or "").lower()
            fr = (_s(item.get("for")) or "").lower() or "general_privacy"
            val = _s(item.get("value"))
            lab = _s(item.get("label"))
            if t in {"form_url", "email", "phone", "other_url"} and val:
                out["contacts_flat"].append({"type": t, "for": fr, "value": val, "label": lab})
    return out

def main():

    pairs = iter_policy_files()
    if not pairs:
        raise RuntimeError(f"No .txt files found in {POLICIES_DIR}")

    master = load_json(MAIN_JSON)

    for domain, path in tqdm(pairs, desc="Analyzing policy extras"):
        text = load_text(path)
        if not text.strip():

            master.setdefault(domain, {})
            master[domain]["policy_extras"] = {"error": "Empty policy file"}
            save_json(master, MAIN_JSON)
            continue

        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_OUTPUT_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "Return exactly one strict JSON object. No commentary."},
                    {"role": "user", "content": build_prompt(domain, text)},
                ],
            )
            content = resp.choices[0].message.content or "{}"
            raw = json.loads(content)
            extras = normalize_extras(raw)

            master.setdefault(domain, {})
            master[domain]["policy_extras"] = extras

        except Exception as e:
            master.setdefault(domain, {})
            master[domain]["policy_extras"] = {"error": str(e)}

        save_json(master, MAIN_JSON)

    rows = []
    for domain, obj in master.items():
        pe = (obj or {}).get("policy_extras", {})
        if not pe or "error" in pe:
            rows.append({
                "domain": domain,
                "cookie_practices_mentioned": None,
                "mentions_ccpa": None,
                "has_opt_out_contacts": None,
                "has_access_delete_contacts": None,
                "opt_out_forms": 0, "opt_out_emails": 0, "opt_out_phones": 0, "opt_out_urls": 0,
                "ad_forms": 0, "ad_emails": 0, "ad_phones": 0, "ad_urls": 0,
            })
            continue

        opt_out = pe["contacts"]["opt_out"]
        ad = pe["contacts"]["access_or_delete"]

        rows.append({
            "domain": domain,
            "cookie_practices_mentioned": pe.get("cookie_practices_mentioned"),
            "mentions_ccpa": pe.get("mentions_ccpa"),
            "has_opt_out_contacts": any([
                opt_out["form_urls"], opt_out["emails"], opt_out["phone_numbers"], opt_out["other_urls"]
            ]),
            "has_access_delete_contacts": any([
                ad["form_urls"], ad["emails"], ad["phone_numbers"], ad["other_urls"]
            ]),
            "opt_out_forms": len(opt_out["form_urls"]),
            "opt_out_emails": len(opt_out["emails"]),
            "opt_out_phones": len(opt_out["phone_numbers"]),
            "opt_out_urls": len(opt_out["other_urls"]),
            "ad_forms": len(ad["form_urls"]),
            "ad_emails": len(ad["emails"]),
            "ad_phones": len(ad["phone_numbers"]),
            "ad_urls": len(ad["other_urls"]),
        })

    df = pd.DataFrame(rows)

    def _count_true(series):
        return int((series == True).sum())

    summary = {
        "total_domains": len(df),
        "mentions_cookie_practices": _count_true(df["cookie_practices_mentioned"]),
        "mentions_ccpa": _count_true(df["mentions_ccpa"]),
        "has_opt_out_contacts": _count_true(df["has_opt_out_contacts"]),
        "has_access_delete_contacts": _count_true(df["has_access_delete_contacts"]),
        "opt_out_forms_total": df["opt_out_forms"].sum(),
        "opt_out_emails_total": df["opt_out_emails"].sum(),
        "opt_out_phones_total": df["opt_out_phones"].sum(),
        "opt_out_urls_total": df["opt_out_urls"].sum(),
        "access_delete_forms_total": df["ad_forms"].sum(),
        "access_delete_emails_total": df["ad_emails"].sum(),
        "access_delete_phones_total": df["ad_phones"].sum(),
        "access_delete_urls_total": df["ad_urls"].sum(),
    }
    df.loc[len(df)] = {**{"domain": "__DATASET_SUMMARY__"}, **summary}

    df.to_csv(SUMMARY_CSV, index=False)
    print(f"✅ Done. Updated {MAIN_JSON} and wrote {SUMMARY_CSV}")

if __name__ == "__main__":
    main()

In [ ]:
import json, os, shutil
from copy import deepcopy

SRC_PATH = "ccpa_policy_audit_all.json"
DST_PATH = "ccpa_policy_audit_all_pastversion.json"
ONLY_INTERSECTION = True

def load_json(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON in {path}: {e}")
    if not isinstance(data, dict):
        raise TypeError(f"Top-level JSON must be an object/dict in {path}")
    return data

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

src = load_json(SRC_PATH)
dst = load_json(DST_PATH)

src_domains = set(src.keys())
dst_domains = set(dst.keys())
common = src_domains & dst_domains

if ONLY_INTERSECTION:
    domains_to_update = sorted(common)
else:

    domains_to_update = sorted(dst_domains)

bak_path = DST_PATH + ".bak"
shutil.copyfile(DST_PATH, bak_path)

updated = 0
missing_in_src_extras = []
skipped_not_in_src = []
created_dst_objs = 0

for domain in domains_to_update:
    if domain not in src:
        skipped_not_in_src.append(domain)
        continue

    src_obj = src.get(domain, {})
    pe = src_obj.get("policy_extras", None)

    if pe is None:
        missing_in_src_extras.append(domain)
        continue

    if not isinstance(dst.get(domain), dict):
        dst[domain] = {}
        created_dst_objs += 1

    dst[domain]["policy_extras"] = deepcopy(pe)
    updated += 1

save_json(dst, DST_PATH)

extra_dst_not_in_src = sorted(dst_domains - src_domains)

print("=== Merge Summary ===")
print(f"Source file              : {SRC_PATH}")
print(f"Destination file         : {DST_PATH}")
print(f"Backup created           : {bak_path}")
print(f"Source domains           : {len(src_domains)}")
print(f"Destination domains      : {len(dst_domains)}")
print(f"Common domains           : {len(common)}")
print(f"ONLY_INTERSECTION        : {ONLY_INTERSECTION}")
print(f"Updated 'policy_extras'  : {updated}")
print(f"Created empty dst objs   : {created_dst_objs}")
print(f"Missing extras in source : {len(missing_in_src_extras)}")
if missing_in_src_extras:
    print("  examples:", ", ".join(missing_in_src_extras[:10]), "..." if len(missing_in_src_extras) > 10 else "")
if ONLY_INTERSECTION and extra_dst_not_in_src:
    print(f"Domains only in dst (not in src): {len(extra_dst_not_in_src)}")
    print("  examples:", ", ".join(extra_dst_not_in_src[:10]), "..." if len(extra_dst_not_in_src) > 10 else "")
print("\n✅ Done.")

In [ ]:
import os
import re
import glob
import json
from pathlib import Path
from typing import Dict, Any, List, Tuple

import pandas as pd
from tqdm import tqdm
from openai import OpenAI

POLICIES_DIR = "policies_text"
MAIN_JSON    = "ccpa_policy_audit_all_pastversion.json"
SUMMARY_CSV  = "ccpa_policy_extras_summary.csv"
MODEL        = "gpt-4o"
TEMPERATURE  = 0
MAX_OUTPUT_TOKENS = 900

SCHEMA_VERSION = 1

API_KEY = "REDACTED-SET-OPENAI_API_KEY-ENV-VAR"
client = OpenAI(api_key=API_KEY)

def iter_policy_files() -> List[Tuple[str, Path]]:
    """Return (domain, path) from POLICIES_DIR/*.txt with names like 'www_example_com__combined.txt'."""
    files = sorted(glob.glob(os.path.join(POLICIES_DIR, "*.txt")))
    out = []
    for p in files:
        name = Path(p).stem
        domain = name.split("__combined", 1)[0].replace("_", ".")
        out.append((domain, Path(p)))
    return out

def load_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

def load_json(path: str) -> Dict[str, Any]:
    if not os.path.exists(path):
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f) or {}
    except Exception:
        return {}

def save_json(obj: Dict[str, Any], path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def extras_is_complete(pe: dict) -> bool:
    if not isinstance(pe, dict):
        return False
    if "error" in pe:
        return False
    required = {
        "cookie_practices_mentioned",
        "cookie_practices_excerpt",
        "mentions_ccpa",
        "mentions_ccpa_excerpt",
        "contacts",
        "contacts_flat",
    }
    if not required.issubset(pe.keys()):
        return False

    contacts = pe.get("contacts", {})
    for bucket in ("opt_out", "access_or_delete"):
        if bucket not in contacts or not isinstance(contacts[bucket], dict):
            return False
        for k in ("form_urls", "emails", "phone_numbers", "other_urls"):
            if k not in contacts[bucket]:
                return False

    if pe.get("_schema_v", SCHEMA_VERSION) != SCHEMA_VERSION:
        return False
    return True

def build_prompt(domain: str, policy_text: str) -> str:
    return f"""
You are auditing a privacy policy for {domain}. Work ONLY from the text below. Return exactly one JSON object with this schema:

{{
  "cookie_practices_mentioned": true/false/"unspecified",
  "cookie_practices_excerpt": "string (≤ 600 chars). If unspecified, say: 'No explicit mention found.'",
  "mentions_ccpa": true/false/"unspecified",
  "mentions_ccpa_excerpt": "string (≤ 600 chars). If unspecified, say: 'No explicit mention found.'",
  "contacts": {{
    "opt_out": {{
      "form_urls": ["..."],
      "emails": ["..."],
      "phone_numbers": ["..."],
      "other_urls": ["..."]
    }},
    "access_or_delete": {{
      "form_urls": ["..."],
      "emails": ["..."],
      "phone_numbers": ["..."],
      "other_urls": ["..."]
    }}
  }},
  "contacts_flat": [
    {{
      "type": "form_url|email|phone|other_url",
      "for": "opt_out|access|delete|access_delete|general_privacy",
      "value": "string",
      "label": "string or empty"
    }}
  ]
}}

Strict rules:
- Use true/false ONLY if explicit. Otherwise use "unspecified".
- cookie_practices_mentioned = true if the policy discusses cookies (e.g., cookies, pixels, SDKs, analytics cookies) at all, regardless of consent steps.
- mentions_ccpa = true if the policy explicitly mentions “CCPA”, “CPRA”, “California Consumer Privacy Act”, “California privacy rights” (in a legal-rights sense).
- Extract concrete contact methods for (a) opt-out of sale/share/targeted advertising, and (b) access/delete requests (right to know/delete). Put each into the appropriate bucket(s).
- Provide a flattened list in contacts_flat with the intended purpose (for) and a guessed label if present (e.g., “Your Privacy Choices”, “Do Not Sell or Share”).
- If no contacts are found for a category, return empty arrays.

================== BEGIN POLICY TEXT ==================
{policy_text}
=================== END POLICY TEXT ===================
""".strip()

def _tri(v):
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}:
        return True
    if s in {"false", "no", "0"}:
        return False
    return "unspecified"

def _s(v, default=""):
    return (v or "").strip() if isinstance(v, str) else default

def _list(v):
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    return []

def normalize_extras(raw: dict) -> dict:
    out = {
        "cookie_practices_mentioned": _tri(raw.get("cookie_practices_mentioned", "unspecified")),
        "cookie_practices_excerpt": _s(raw.get("cookie_practices_excerpt"), "No explicit mention found.")[:600],
        "mentions_ccpa": _tri(raw.get("mentions_ccpa", "unspecified")),
        "mentions_ccpa_excerpt": _s(raw.get("mentions_ccpa_excerpt"), "No explicit mention found.")[:600],
        "contacts": {
            "opt_out": {
                "form_urls": _list(raw.get("contacts", {}).get("opt_out", {}).get("form_urls", [])),
                "emails": _list(raw.get("contacts", {}).get("opt_out", {}).get("emails", [])),
                "phone_numbers": _list(raw.get("contacts", {}).get("opt_out", {}).get("phone_numbers", [])),
                "other_urls": _list(raw.get("contacts", {}).get("opt_out", {}).get("other_urls", [])),
            },
            "access_or_delete": {
                "form_urls": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("form_urls", [])),
                "emails": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("emails", [])),
                "phone_numbers": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("phone_numbers", [])),
                "other_urls": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("other_urls", [])),
            },
        },
        "contacts_flat": []
    }
    flat = raw.get("contacts_flat", [])
    if isinstance(flat, list):
        for item in flat:
            if not isinstance(item, dict):
                continue
            t = (_s(item.get("type")) or "").lower()
            fr = (_s(item.get("for")) or "").lower() or "general_privacy"
            val = _s(item.get("value"))
            lab = _s(item.get("label"))
            if t in {"form_url", "email", "phone", "other_url"} and val:
                out["contacts_flat"].append({"type": t, "for": fr, "value": val, "label": lab})
    out["_schema_v"] = SCHEMA_VERSION
    return out

def main():
    pairs = iter_policy_files()
    if not pairs:
        raise RuntimeError(f"No .txt files found in {POLICIES_DIR}")

    master = load_json(MAIN_JSON)

    todo_pairs = []
    for domain, path in pairs:
        pe = (master.get(domain, {}) or {}).get("policy_extras")
        if extras_is_complete(pe):
            continue
        todo_pairs.append((domain, path))

    if not todo_pairs:
        print("✅ Nothing to do. All domains have complete 'policy_extras'.")
    else:
        print(f"➡️ Domains needing 'policy_extras': {len(todo_pairs)}")

    for domain, path in tqdm(todo_pairs, desc="Analyzing policy extras"):
        text = load_text(path)
        master.setdefault(domain, {})

        if not text.strip():
            master[domain]["policy_extras"] = {"error": "Empty policy file", "_schema_v": SCHEMA_VERSION}
            save_json(master, MAIN_JSON)
            continue

        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_OUTPUT_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "Return exactly one strict JSON object. No commentary."},
                    {"role": "user", "content": build_prompt(domain, text)},
                ],
            )
            content = resp.choices[0].message.content or "{}"
            raw = json.loads(content)
            extras = normalize_extras(raw)
            master[domain]["policy_extras"] = extras

        except Exception as e:
            master[domain]["policy_extras"] = {"error": str(e), "_schema_v": SCHEMA_VERSION}

        save_json(master, MAIN_JSON)

    rows = []
    for domain, obj in master.items():
        pe = (obj or {}).get("policy_extras", {})
        if not pe or "error" in pe or not extras_is_complete(pe):
            rows.append({
                "domain": domain,
                "cookie_practices_mentioned": None,
                "mentions_ccpa": None,
                "has_opt_out_contacts": None,
                "has_access_delete_contacts": None,
                "opt_out_forms": 0, "opt_out_emails": 0, "opt_out_phones": 0, "opt_out_urls": 0,
                "ad_forms": 0, "ad_emails": 0, "ad_phones": 0, "ad_urls": 0,
            })
            continue

        opt_out = pe["contacts"]["opt_out"]
        ad = pe["contacts"]["access_or_delete"]

        rows.append({
            "domain": domain,
            "cookie_practices_mentioned": pe.get("cookie_practices_mentioned"),
            "mentions_ccpa": pe.get("mentions_ccpa"),
            "has_opt_out_contacts": any([
                opt_out["form_urls"], opt_out["emails"], opt_out["phone_numbers"], opt_out["other_urls"]
            ]),
            "has_access_delete_contacts": any([
                ad["form_urls"], ad["emails"], ad["phone_numbers"], ad["other_urls"]
            ]),
            "opt_out_forms": len(opt_out["form_urls"]),
            "opt_out_emails": len(opt_out["emails"]),
            "opt_out_phones": len(opt_out["phone_numbers"]),
            "opt_out_urls": len(opt_out["other_urls"]),
            "ad_forms": len(ad["form_urls"]),
            "ad_emails": len(ad["emails"]),
            "ad_phones": len(ad["phone_numbers"]),
            "ad_urls": len(ad["other_urls"]),
        })

    df = pd.DataFrame(rows)

    def _count_true(series):
        return int((series == True).sum())

    summary = {
        "total_domains": len(df),
        "mentions_cookie_practices": _count_true(df["cookie_practices_mentioned"]),
        "mentions_ccpa": _count_true(df["mentions_ccpa"]),
        "has_opt_out_contacts": _count_true(df["has_opt_out_contacts"]),
        "has_access_delete_contacts": _count_true(df["has_access_delete_contacts"]),
        "opt_out_forms_total": df["opt_out_forms"].sum(),
        "opt_out_emails_total": df["opt_out_emails"].sum(),
        "opt_out_phones_total": df["opt_out_phones"].sum(),
        "opt_out_urls_total": df["opt_out_urls"].sum(),
        "access_delete_forms_total": df["ad_forms"].sum(),
        "access_delete_emails_total": df["ad_emails"].sum(),
        "access_delete_phones_total": df["ad_phones"].sum(),
        "access_delete_urls_total": df["ad_urls"].sum(),
    }
    df.loc[len(df)] = {**{"domain": "__DATASET_SUMMARY__"}, **summary}
    df.to_csv(SUMMARY_CSV, index=False)

    print(f"\n✅ Done. Updated {MAIN_JSON} and wrote {SUMMARY_CSV}")
    print(f"   Skipped already-complete domains automatically.")

if __name__ == "__main__":
    main()

In [ ]:
import os
import re
import glob
import json
import time
import random
from pathlib import Path
from typing import Dict, Any, List, Tuple

import pandas as pd
from tqdm import tqdm
from openai import OpenAI

POLICIES_DIR = "policies_text"
MAIN_JSON    = "ccpa_policy_audit_all.json"
SUMMARY_CSV  = "ccpa_policy_extras_summary.csv"
MODEL        = "gpt-4o"
TEMPERATURE  = 0
MAX_OUTPUT_TOKENS = 900

SCHEMA_VERSION = 1
RERUN_ALL_ERRORS = False
RETRY_MAX_ATTEMPTS = 3
RETRY_BASE_SECONDS = 5
RETRY_MAX_SECONDS = 60
RETRIABLE_ERROR_SUBSTRINGS = [
    "429", "rate limit", "insufficient_quota", "quota", "temporarily unavailable", "overloaded"
]

API_KEY = "REDACTED-SET-OPENAI_API_KEY-ENV-VAR"
client = OpenAI(api_key=API_KEY)

def iter_policy_files() -> List[Tuple[str, Path]]:
    """Return (domain, path) from POLICIES_DIR/*.txt with names like 'www_example_com__combined.txt'."""
    files = sorted(glob.glob(os.path.join(POLICIES_DIR, "*.txt")))
    out = []
    for p in files:
        name = Path(p).stem
        domain = name.split("__combined", 1)[0].replace("_", ".")
        out.append((domain, Path(p)))
    return out

def load_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

def load_json(path: str) -> Dict[str, Any]:
    if not os.path.exists(path):
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f) or {}
    except Exception:
        return {}

def save_json(obj: Dict[str, Any], path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def extras_is_complete(pe: dict) -> bool:
    """True if policy_extras exists, is structured, and matches current schema."""
    if not isinstance(pe, dict):
        return False
    if "error" in pe:
        return False
    required = {
        "cookie_practices_mentioned",
        "cookie_practices_excerpt",
        "mentions_ccpa",
        "mentions_ccpa_excerpt",
        "contacts",
        "contacts_flat",
    }
    if not required.issubset(pe.keys()):
        return False
    contacts = pe.get("contacts", {})
    for bucket in ("opt_out", "access_or_delete"):
        b = contacts.get(bucket, {})
        if not isinstance(b, dict):
            return False
        for k in ("form_urls", "emails", "phone_numbers", "other_urls"):
            if k not in b:
                return False
    if pe.get("_schema_v", SCHEMA_VERSION) != SCHEMA_VERSION:
        return False
    return True

def extras_needs_rerun(pe: dict) -> bool:
    """Domains to process: missing/incomplete OR 429/quota errors (and optionally all errors)."""
    if not isinstance(pe, dict):
        return True

    if extras_is_complete(pe):
        return False

    if "error" in pe:
        err = str(pe.get("error", "")).lower()
        if "empty policy file" in err:
            return False
        if any(s in err for s in RETRIABLE_ERROR_SUBSTRINGS):
            return True
        return bool(RERUN_ALL_ERRORS)

    return True

def build_prompt(domain: str, policy_text: str) -> str:
    return f"""
You are auditing a privacy policy for {domain}. Work ONLY from the text below. Return exactly one JSON object with this schema:

{{
  "cookie_practices_mentioned": true/false/"unspecified",
  "cookie_practices_excerpt": "string (≤ 600 chars). If unspecified, say: 'No explicit mention found.'",
  "mentions_ccpa": true/false/"unspecified",
  "mentions_ccpa_excerpt": "string (≤ 600 chars). If unspecified, say: 'No explicit mention found.'",
  "contacts": {{
    "opt_out": {{
      "form_urls": ["..."],
      "emails": ["..."],
      "phone_numbers": ["..."],
      "other_urls": ["..."]
    }},
    "access_or_delete": {{
      "form_urls": ["..."],
      "emails": ["..."],
      "phone_numbers": ["..."],
      "other_urls": ["..."]
    }}
  }},
  "contacts_flat": [
    {{
      "type": "form_url|email|phone|other_url",
      "for": "opt_out|access|delete|access_delete|general_privacy",
      "value": "string",
      "label": "string or empty"
    }}
  ]
}}

Strict rules:
- Use true/false ONLY if explicit. Otherwise use "unspecified".
- cookie_practices_mentioned = true if the policy discusses cookies (e.g., cookies, pixels, SDKs, analytics cookies) at all, regardless of consent steps.
- mentions_ccpa = true if the policy explicitly mentions “CCPA”, “CPRA”, “California Consumer Privacy Act”, “California privacy rights” (in a legal-rights sense).
- Extract concrete contact methods for (a) opt-out of sale/share/targeted advertising, and (b) access/delete requests (right to know/delete). Put each into the appropriate bucket(s).
- Provide a flattened list in contacts_flat with the intended purpose (for) and a guessed label if present (e.g., “Your Privacy Choices”, “Do Not Sell or Share”).
- If no contacts are found for a category, return empty arrays.

================== BEGIN POLICY TEXT ==================
{policy_text}
=================== END POLICY TEXT ===================
""".strip()

def _tri(v):
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}:
        return True
    if s in {"false", "no", "0"}:
        return False
    return "unspecified"

def _s(v, default=""):
    return (v or "").strip() if isinstance(v, str) else default

def _list(v):
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    return []

def normalize_extras(raw: dict) -> dict:
    out = {
        "cookie_practices_mentioned": _tri(raw.get("cookie_practices_mentioned", "unspecified")),
        "cookie_practices_excerpt": _s(raw.get("cookie_practices_excerpt"), "No explicit mention found.")[:600],
        "mentions_ccpa": _tri(raw.get("mentions_ccpa", "unspecified")),
        "mentions_ccpa_excerpt": _s(raw.get("mentions_ccpa_excerpt"), "No explicit mention found.")[:600],
        "contacts": {
            "opt_out": {
                "form_urls": _list(raw.get("contacts", {}).get("opt_out", {}).get("form_urls", [])),
                "emails": _list(raw.get("contacts", {}).get("opt_out", {}).get("emails", [])),
                "phone_numbers": _list(raw.get("contacts", {}).get("opt_out", {}).get("phone_numbers", [])),
                "other_urls": _list(raw.get("contacts", {}).get("opt_out", {}).get("other_urls", [])),
            },
            "access_or_delete": {
                "form_urls": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("form_urls", [])),
                "emails": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("emails", [])),
                "phone_numbers": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("phone_numbers", [])),
                "other_urls": _list(raw.get("contacts", {}).get("access_or_delete", {}).get("other_urls", [])),
            },
        },
        "contacts_flat": []
    }
    flat = raw.get("contacts_flat", [])
    if isinstance(flat, list):
        for item in flat:
            if not isinstance(item, dict):
                continue
            t = (_s(item.get("type")) or "").lower()
            fr = (_s(item.get("for")) or "").lower() or "general_privacy"
            val = _s(item.get("value"))
            lab = _s(item.get("label"))
            if t in {"form_url", "email", "phone", "other_url"} and val:
                out["contacts_flat"].append({"type": t, "for": fr, "value": val, "label": lab})
    out["_schema_v"] = SCHEMA_VERSION
    return out

def call_model_with_retry(domain: str, text: str):
    """Call the model with basic exponential backoff on retriable errors."""
    for attempt in range(1, RETRY_MAX_ATTEMPTS + 1):
        try:
            return client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_OUTPUT_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "Return exactly one strict JSON object. No commentary."},
                    {"role": "user", "content": build_prompt(domain, text)},
                ],
            )
        except Exception as e:
            msg = str(e).lower()
            retriable = any(s in msg for s in RETRIABLE_ERROR_SUBSTRINGS)
            if not retriable or attempt == RETRY_MAX_ATTEMPTS:
                raise

            sleep_s = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** (attempt - 1))) + random.uniform(0, 1.5)
            print(f"⚠️  {domain}: retriable error '{msg[:120]}...' -> retry {attempt}/{RETRY_MAX_ATTEMPTS} in {sleep_s:.1f}s")
            time.sleep(sleep_s)

def main():
    pairs = iter_policy_files()
    if not pairs:
        raise RuntimeError(f"No .txt files found in {POLICIES_DIR}")

    master = load_json(MAIN_JSON)

    todo_pairs = []
    skipped_complete = 0
    skipped_permanent = 0

    for domain, path in pairs:
        pe = (master.get(domain, {}) or {}).get("policy_extras")
        if extras_needs_rerun(pe):
            todo_pairs.append((domain, path))
        else:
            skipped_complete += 1

    if not todo_pairs:
        print("✅ Nothing to do. All domains complete or only permanent errors remain.")
        return

    print(f"➡️ Domains queued for (re)run: {len(todo_pairs)}  |  Skipped complete: {skipped_complete}")

    for domain, path in tqdm(todo_pairs, desc="Analyzing policy extras"):
        text = load_text(path)
        master.setdefault(domain, {})

        if not text.strip():
            master[domain]["policy_extras"] = {"error": "Empty policy file", "_schema_v": SCHEMA_VERSION}
            save_json(master, MAIN_JSON)
            skipped_permanent += 1
            continue

        try:
            resp = call_model_with_retry(domain, text)
            content = resp.choices[0].message.content or "{}"
            raw = json.loads(content)
            extras = normalize_extras(raw)
            master[domain]["policy_extras"] = extras

        except Exception as e:
            master[domain]["policy_extras"] = {"error": str(e), "_schema_v": SCHEMA_VERSION}

        save_json(master, MAIN_JSON)

    rows = []
    for domain, obj in master.items():
        pe = (obj or {}).get("policy_extras", {})
        if not pe or "error" in pe or not extras_is_complete(pe):
            rows.append({
                "domain": domain,
                "cookie_practices_mentioned": None,
                "mentions_ccpa": None,
                "has_opt_out_contacts": None,
                "has_access_delete_contacts": None,
                "opt_out_forms": 0, "opt_out_emails": 0, "opt_out_phones": 0, "opt_out_urls": 0,
                "ad_forms": 0, "ad_emails": 0, "ad_phones": 0, "ad_urls": 0,
            })
            continue

        opt_out = pe["contacts"]["opt_out"]
        ad = pe["contacts"]["access_or_delete"]

        rows.append({
            "domain": domain,
            "cookie_practices_mentioned": pe.get("cookie_practices_mentioned"),
            "mentions_ccpa": pe.get("mentions_ccpa"),
            "has_opt_out_contacts": any([
                opt_out["form_urls"], opt_out["emails"], opt_out["phone_numbers"], opt_out["other_urls"]
            ]),
            "has_access_delete_contacts": any([
                ad["form_urls"], ad["emails"], ad["phone_numbers"], ad["other_urls"]
            ]),
            "opt_out_forms": len(opt_out["form_urls"]),
            "opt_out_emails": len(opt_out["emails"]),
            "opt_out_phones": len(opt_out["phone_numbers"]),
            "opt_out_urls": len(opt_out["other_urls"]),
            "ad_forms": len(ad["form_urls"]),
            "ad_emails": len(ad["emails"]),
            "ad_phones": len(ad["phone_numbers"]),
            "ad_urls": len(ad["other_urls"]),
        })

    df = pd.DataFrame(rows)

    def _count_true(series):
        return int((series == True).sum())

    summary = {
        "total_domains": len(df),
        "mentions_cookie_practices": _count_true(df["cookie_practices_mentioned"]),
        "mentions_ccpa": _count_true(df["mentions_ccpa"]),
        "has_opt_out_contacts": _count_true(df["has_opt_out_contacts"]),
        "has_access_delete_contacts": _count_true(df["has_access_delete_contacts"]),
        "opt_out_forms_total": df["opt_out_forms"].sum(),
        "opt_out_emails_total": df["opt_out_emails"].sum(),
        "opt_out_phones_total": df["opt_out_phones"].sum(),
        "opt_out_urls_total": df["opt_out_urls"].sum(),
        "access_delete_forms_total": df["ad_forms"].sum(),
        "access_delete_emails_total": df["ad_emails"].sum(),
        "access_delete_phones_total": df["ad_phones"].sum(),
        "access_delete_urls_total": df["ad_urls"].sum(),
    }
    df.loc[len(df)] = {**{"domain": "__DATASET_SUMMARY__"}, **summary}
    df.to_csv(SUMMARY_CSV, index=False)

    print(f"\n✅ Done. Updated {MAIN_JSON} and wrote {SUMMARY_CSV}")
    print(f"   Skipped complete: {skipped_complete} | Permanent-skip (empty files): {skipped_permanent}")

if __name__ == "__main__":
    main()

In [ ]:
import json
import os
from typing import Dict, Any, List, Tuple
from collections import defaultdict

import pandas as pd
import tldextract

MAIN_JSON = "ccpa_policy_audit_all.json"
SUBJ_JSON = "policy_domains_ccpa_subjectivity.json"

OUT_DOMAIN_CSV = "policy_extras_domain_level__subjectivity_superset.csv"
OUT_SUMMARY_SUBJ_CSV = "policy_extras_summary_by_subjectivity__subjectivity_superset.csv"
OUT_TRI_SUBJ_CSV = "policy_extras_tri_distribution_by_subjectivity__subjectivity_superset.csv"
OUT_MISSING_IN_MASTER = "subjectivity_domains_missing_in_master.csv"

def normalize_domain(raw: str) -> str:
    if not isinstance(raw, str) or not raw.strip():
        return ""
    raw = raw.strip().lower()
    ext = tldextract.extract(raw)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return raw

def tri_normalize(v) -> str:
    """
    Normalize tri-state to one of {'true','false','unspecified','missing'}.
    """
    if v is None:
        return "missing"
    if isinstance(v, bool):
        return "true" if v else "false"
    s = str(v).strip().lower()
    if s in {"true", "yes", "1"}:
        return "true"
    if s in {"false", "no", "0"}:
        return "false"
    if s in {"unspecified", ""}:
        return "unspecified"
    return "unspecified"

def safe_len(x):
    return len(x) if isinstance(x, (list, tuple, set)) else 0

def load_json(path: str) -> Dict[str, Any]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f) or {}

def extract_subjectivity_map(obj: Any) -> Dict[str, str]:
    """
    Produce: {domain_raw: subjectivity_str}, tolerant to different schemas:
      - { "example.com": "subjected" }
      - { "example.com": {"subjectivity": "subjected"} }
      - [ {"domain": "example.com", "subjectivity": "subjected"}, ... ]
      - booleans/flags: {"example.com": {"ccpa": true}} -> 'subjected'
    """
    def coerce_subjectivity(v) -> str:
        if isinstance(v, str):
            s = v.strip().lower()
            if s in {"subjected", "subject", "ccpa", "yes", "true", "1"}:
                return "subjected"
            if s in {"not_subjected", "non-subjected", "no", "false", "0", "not subject"}:
                return "not_subjected"
            return s
        if isinstance(v, bool):
            return "subjected" if v else "not_subjected"
        if isinstance(v, (int, float)):
            return "subjected" if v else "not_subjected"
        return "unknown"

    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            subj = None
            if isinstance(v, dict):
                for key in ["subjectivity", "ccpa_subjectivity", "ccpa", "subjected"]:
                    if key in v:
                        subj = coerce_subjectivity(v[key])
                        break
                if subj is None:

                    truthy = None
                    for key in v:
                        if isinstance(v[key], (bool, int, float)):
                            truthy = bool(v[key]); break
                    subj = "subjected" if truthy else ("not_subjected" if truthy is not None else "unknown")
            else:
                subj = coerce_subjectivity(v)
            out[k.strip().lower()] = subj
    elif isinstance(obj, list):
        for item in obj:
            if not isinstance(item, dict):
                continue
            dom = (item.get("domain") or item.get("website") or item.get("host") or "").strip().lower()
            if not dom:
                continue
            if "subjectivity" in item:
                subj = coerce_subjectivity(item["subjectivity"])
            elif "ccpa" in item:
                subj = coerce_subjectivity(item["ccpa"])
            elif "subjected" in item:
                subj = coerce_subjectivity(item["subjected"])
            else:
                subj = "unknown"
            out[dom] = subj
    return out

master = load_json(MAIN_JSON)
subj_obj = load_json(SUBJ_JSON)
subj_map_raw = extract_subjectivity_map(subj_obj)

master_by_raw: Dict[str, Dict[str, Any]] = {}
master_by_norm: Dict[str, List[str]] = defaultdict(list)
for d_raw, obj in master.items():
    key_raw = (d_raw or "").strip().lower()
    if not key_raw:
        continue
    master_by_raw[key_raw] = obj
    nd = normalize_domain(key_raw)
    if nd:
        master_by_norm[nd].append(key_raw)

rows = []
missing_in_master = []

for subj_dom_raw, subj_val in subj_map_raw.items():
    subj_dom_raw_l = subj_dom_raw.strip().lower()
    nd = normalize_domain(subj_dom_raw_l)

    join_status = "raw_match"
    obj = master_by_raw.get(subj_dom_raw_l)

    if obj is None and nd:

        candidates = master_by_norm.get(nd, [])
        if candidates:
            obj = master_by_raw.get(candidates[0])
            join_status = "norm_match"
        else:
            join_status = "missing_in_master"

    if obj is None:

        rows.append({
            "domain_subjectivity_raw": subj_dom_raw_l,
            "domain_subjectivity_norm": nd,
            "domain_master_raw": None,
            "join_status": join_status,
            "subjectivity": subj_val or "unknown",
            "cookie_practices_mentioned": "missing",
            "mentions_ccpa": "missing",
            "has_opt_out_contacts": None,
            "has_access_delete_contacts": None,
            "opt_out_forms": 0, "opt_out_emails": 0, "opt_out_phones": 0, "opt_out_urls": 0,
            "ad_forms": 0, "ad_emails": 0, "ad_phones": 0, "ad_urls": 0,
            "extras_status": "missing",
        })
        missing_in_master.append({"subjectivity_domain": subj_dom_raw_l, "normalized": nd})
        continue

    pe = (obj or {}).get("policy_extras")

    if isinstance(pe, dict) and "error" not in pe:
        cookie_tri = tri_normalize(pe.get("cookie_practices_mentioned"))
        ccpa_tri   = tri_normalize(pe.get("mentions_ccpa"))

        contacts = pe.get("contacts", {}) if isinstance(pe.get("contacts"), dict) else {}
        opt_out = contacts.get("opt_out", {}) if isinstance(contacts.get("opt_out"), dict) else {}
        ad      = contacts.get("access_or_delete", {}) if isinstance(contacts.get("access_or_delete"), dict) else {}

        has_opt_out = any([
            safe_len(opt_out.get("form_urls")),
            safe_len(opt_out.get("emails")),
            safe_len(opt_out.get("phone_numbers")),
            safe_len(opt_out.get("other_urls")),
        ])
        has_access_delete = any([
            safe_len(ad.get("form_urls")),
            safe_len(ad.get("emails")),
            safe_len(ad.get("phone_numbers")),
            safe_len(ad.get("other_urls")),
        ])

        rows.append({
            "domain_subjectivity_raw": subj_dom_raw_l,
            "domain_subjectivity_norm": nd,
            "domain_master_raw": obj.get("domain_master_raw", None),
            "join_status": join_status,
            "subjectivity": subj_val or "unknown",
            "cookie_practices_mentioned": cookie_tri,
            "mentions_ccpa": ccpa_tri,
            "has_opt_out_contacts": bool(has_opt_out),
            "has_access_delete_contacts": bool(has_access_delete),
            "opt_out_forms": safe_len(opt_out.get("form_urls")),
            "opt_out_emails": safe_len(opt_out.get("emails")),
            "opt_out_phones": safe_len(opt_out.get("phone_numbers")),
            "opt_out_urls": safe_len(opt_out.get("other_urls")),
            "ad_forms": safe_len(ad.get("form_urls")),
            "ad_emails": safe_len(ad.get("emails")),
            "ad_phones": safe_len(ad.get("phone_numbers")),
            "ad_urls": safe_len(ad.get("other_urls")),
            "extras_status": "ok",
        })
    else:
        err = pe.get("error") if isinstance(pe, dict) else None
        rows.append({
            "domain_subjectivity_raw": subj_dom_raw_l,
            "domain_subjectivity_norm": nd,
            "domain_master_raw": None,
            "join_status": join_status,
            "subjectivity": subj_val or "unknown",
            "cookie_practices_mentioned": "missing",
            "mentions_ccpa": "missing",
            "has_opt_out_contacts": None,
            "has_access_delete_contacts": None,
            "opt_out_forms": 0, "opt_out_emails": 0, "opt_out_phones": 0, "opt_out_urls": 0,
            "ad_forms": 0, "ad_emails": 0, "ad_phones": 0, "ad_urls": 0,
            "extras_status": f"error:{err}" if err else "missing",
        })

df = pd.DataFrame(rows)

total = len(df)
ok_count = (df["extras_status"] == "ok").sum()
err_count = total - ok_count
missing_ct = len(missing_in_master)

print("=== Subjectivity-Superset Analysis ===")
print(f"Total subjectivity domains (analyzed): {total}")
print(f"Joined OK with policy_extras          : {ok_count}")
print(f"Missing/error policy_extras           : {err_count}")
print(f"Subjectivity domains missing in master: {missing_ct}")
print()

def pct(n, d):
    return (100.0 * n / d) if d else 0.0

grouped = df.groupby("subjectivity", dropna=False)

summary_rows = []
for subj, g in grouped:
    n = len(g)
    cookie_true = (g["cookie_practices_mentioned"] == "true").sum()
    ccpa_true   = (g["mentions_ccpa"] == "true").sum()
    opt_out_any = g["has_opt_out_contacts"].fillna(False).sum()
    ad_any      = g["has_access_delete_contacts"].fillna(False).sum()

    ch = {
        "opt_out_forms_total": g["opt_out_forms"].sum(),
        "opt_out_emails_total": g["opt_out_emails"].sum(),
        "opt_out_phones_total": g["opt_out_phones"].sum(),
        "opt_out_urls_total": g["opt_out_urls"].sum(),
        "access_delete_forms_total": g["ad_forms"].sum(),
        "access_delete_emails_total": g["ad_emails"].sum(),
        "access_delete_phones_total": g["ad_phones"].sum(),
        "access_delete_urls_total": g["ad_urls"].sum(),
    }

    summary_rows.append({
        "subjectivity": subj,
        "n_domains": n,
        "cookie_practices_true": cookie_true,
        "cookie_practices_true_pct": round(pct(cookie_true, n), 2),
        "mentions_ccpa_true": ccpa_true,
        "mentions_ccpa_true_pct": round(pct(ccpa_true, n), 2),
        "has_opt_out_contacts_n": int(opt_out_any),
        "has_opt_out_contacts_pct": round(pct(opt_out_any, n), 2),
        "has_access_delete_contacts_n": int(ad_any),
        "has_access_delete_contacts_pct": round(pct(ad_any, n), 2),
        **ch
    })

summary_df = pd.DataFrame(summary_rows).sort_values("subjectivity")
print("=== Summary by Subjectivity (superset-restricted) ===")
print(summary_df.to_string(index=False))
print()

def tri_distribution(g: pd.DataFrame, col: str) -> Dict[str, Any]:
    counts = g[col].value_counts(dropna=False).to_dict()
    for k in ["true", "false", "unspecified", "missing"]:
        counts.setdefault(k, 0)
    n = len(g)
    return {
        f"{col}_true_n": counts["true"],
        f"{col}_false_n": counts["false"],
        f"{col}_unspecified_n": counts["unspecified"],
        f"{col}_missing_n": counts["missing"],
        f"{col}_true_pct": round(pct(counts["true"], n), 2),
        f"{col}_false_pct": round(pct(counts["false"], n), 2),
        f"{col}_unspecified_pct": round(pct(counts["unspecified"], n), 2),
        f"{col}_missing_pct": round(pct(counts["missing"], n), 2),
    }

tri_rows = []
for subj, g in grouped:
    row = {"subjectivity": subj, "n_domains": len(g)}
    row.update(tri_distribution(g, "cookie_practices_mentioned"))
    row.update(tri_distribution(g, "mentions_ccpa"))
    tri_rows.append(row)

tri_df = pd.DataFrame(tri_rows).sort_values("subjectivity")
print("=== Tri-State Distribution by Subjectivity (superset-restricted) ===")
print(tri_df.to_string(index=False))
print()

df.to_csv(OUT_DOMAIN_CSV, index=False)
summary_df.to_csv(OUT_SUMMARY_SUBJ_CSV, index=False)
tri_df.to_csv(OUT_TRI_SUBJ_CSV, index=False)

pd.DataFrame(missing_in_master).to_csv(OUT_MISSING_IN_MASTER, index=False)

print(f"✅ Wrote: {OUT_DOMAIN_CSV}")
print(f"✅ Wrote: {OUT_SUMMARY_SUBJ_CSV}")
print(f"✅ Wrote: {OUT_TRI_SUBJ_CSV}")
print(f"✅ Wrote: {OUT_MISSING_IN_MASTER}")

print("\n=== Domain-level (first 10 rows) ===")
print(df.head(10).to_string(index=False))

In [ ]:
"""
Full analysis of policy audits by CCPA subjectivity.

Inputs:
  - ccpa_policy_audit_all.json
      {
        "<domain or subdomain>": {
          "online_data_practices": str,
          "offline_data_practices": str,
          "rubric_assessment": {
              "completeness_score": int,
              "usability_score": int,
              "accuracy_score": int,
              "policy_contradiction": bool,
              "disclosure_map": {
                  "data_collected": bool,
                  "data_shared": bool,
                  "purpose_of_collection": bool,
                  "retention_period": bool,
                  "right_to_access": bool,
                  "right_to_delete": bool,
                  "opt_out": bool
              },
              "..._reason": str
          },
          "behavioral_claims": {
              "honors_gpc": True|False|"unspecified",
              "respects_dnt": True|False|"unspecified",
              "sets_cookies_before_consent": True|False|"unspecified",
              "sets_cookies_after_rejecting_consent": True|False|"unspecified",
              "deletes_cookies_on_rejection": True|False|"unspecified",
              "uses_tracking_only_after_consent": True|False|"unspecified",
              "sells_data": True|False|"unspecified",
              "shares_with_third_parties": True|False|"unspecified",
              "justifications": {...}   # ignored here
          }
        },
        ...
      }

  - policy_domains_ccpa_subjectivity.json
      { "<etld+1>": "subjected" | "not_subjected" }

Outputs (CSV):
  - joined_policy_subjectivity.csv
  - rubric_stats_by_subjectivity.csv            # NOW: raw per-website rubric values
  - disclosure_coverage_by_subjectivity.csv
  - behavioral_claims_by_subjectivity.csv
  - policy_contradictions_by_subjectivity.csv
  - universe_counts.csv

Also prints a structured summary for paper use.
"""

import json
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import tldextract
import numpy as np

POLICY_JSON = "ccpa_policy_audit_all_clear_pastversion.json"
SUBJECTIVITY_JSON = "policy_domains_ccpa_subjectivity.json"

ALIAS_FIXES = {

}

JOINED_CSV = "joined_policy_subjectivity.csv"
RUBRIC_CSV = "rubric_stats_by_subjectivity.csv"
DISCLOSURE_CSV = "disclosure_coverage_by_subjectivity.csv"
BEHAVIORAL_CSV = "behavioral_claims_by_subjectivity.csv"
CONTRADICT_CSV = "policy_contradictions_by_subjectivity.csv"
UNIVERSE_COUNTS_CSV = "universe_counts.csv"

def etld1(host: str) -> str:
    """Normalize a hostname/domain to registrable domain (eTLD+1)."""
    if not host:
        return ""
    h = str(host).strip().lower()
    ext = tldextract.extract(h)
    if not ext.domain or not ext.suffix:
        return h
    return f"{ext.domain}.{ext.suffix}"

def apply_alias(d: str) -> str:
    return ALIAS_FIXES.get(d, d)

def safe_get(d: dict, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def to_tfu(value):
    """
    Normalize behavioral-claims value into one of:
      'true', 'false', 'unspecified'
    """
    if isinstance(value, bool):
        return "true" if value else "false"
    return "unspecified"

p_policy = Path(POLICY_JSON)
p_subj = Path(SUBJECTIVITY_JSON)
if not p_policy.exists():
    raise FileNotFoundError(f"Missing {p_policy.resolve()}")
if not p_subj.exists():
    raise FileNotFoundError(f"Missing {p_subj.resolve()}")

with open(p_policy, "r") as f:
    policy_data = json.load(f)

with open(p_subj, "r") as f:
    subj_map_raw = json.load(f)

policy_rows = []
for raw_dom, rec in policy_data.items():
    parent = apply_alias(etld1(raw_dom))
    policy_rows.append({
        "raw_policy_domain": raw_dom,
        "parent_domain": parent,
        "record": rec
    })
policy_df = pd.DataFrame(policy_rows)

subj_rows = []
for dom, label in subj_map_raw.items():
    parent = apply_alias(etld1(dom))
    subj_rows.append({
        "parent_domain": parent,
        "subjectivity": str(label).strip().lower()
    })
subj_df = pd.DataFrame(subj_rows).drop_duplicates("parent_domain")

joined = policy_df.merge(subj_df, on="parent_domain", how="left")

def extract_rubric(rec):
    r = rec.get("rubric_assessment", {}) if isinstance(rec, dict) else {}
    return pd.Series({
        "completeness_score": r.get("completeness_score", np.nan),
        "usability_score": r.get("usability_score", np.nan),
        "accuracy_score": r.get("accuracy_score", np.nan),
        "policy_contradiction": r.get("policy_contradiction", False),

        "disclosure_data_collected": safe_get(r, "disclosure_map", "data_collected", default=False),
        "disclosure_data_shared": safe_get(r, "disclosure_map", "data_shared", default=False),
        "disclosure_purpose_of_collection": safe_get(r, "disclosure_map", "purpose_of_collection", default=False),
        "disclosure_retention_period": safe_get(r, "disclosure_map", "retention_period", default=False),
        "disclosure_right_to_access": safe_get(r, "disclosure_map", "right_to_access", default=False),
        "disclosure_right_to_delete": safe_get(r, "disclosure_map", "right_to_delete", default=False),
        "disclosure_opt_out": safe_get(r, "disclosure_map", "opt_out", default=False),
    })

def extract_behavioral(rec):
    b = rec.get("behavioral_claims", {}) if isinstance(rec, dict) else {}
    return pd.Series({
        "honors_gpc": to_tfu(b.get("honors_gpc", "unspecified")),
        "respects_dnt": to_tfu(b.get("respects_dnt", "unspecified")),
        "sets_cookies_before_consent": to_tfu(b.get("sets_cookies_before_consent", "unspecified")),
        "sets_cookies_after_rejecting_consent": to_tfu(b.get("sets_cookies_after_rejecting_consent", "unspecified")),
        "deletes_cookies_on_rejection": to_tfu(b.get("deletes_cookies_on_rejection", "unspecified")),
        "uses_tracking_only_after_consent": to_tfu(b.get("uses_tracking_only_after_consent", "unspecified")),
        "sells_data": to_tfu(b.get("sells_data", "unspecified")),
        "shares_with_third_parties": to_tfu(b.get("shares_with_third_parties", "unspecified")),
    })

rubric_df = joined["record"].apply(extract_rubric)
behav_df = joined["record"].apply(extract_behavioral)

data = pd.concat([joined.drop(columns=["record"]), rubric_df, behav_df], axis=1)

policy_raw = len(policy_df)
policy_parents = policy_df["parent_domain"].nunique()
subj_raw = len(subj_df)
subj_parents = subj_df["parent_domain"].nunique()
intersection_parents = pd.Series(sorted(set(policy_df["parent_domain"]) & set(subj_df["parent_domain"]))).nunique()
policy_only_parents = policy_parents - intersection_parents
subj_only_parents = subj_parents - intersection_parents

universe_counts = pd.DataFrame([{
    "policy_raw_entries": policy_raw,
    "policy_unique_parents": policy_parents,
    "subjectivity_raw_entries": subj_raw,
    "subjectivity_unique_parents": subj_parents,
    "intersection_parents": intersection_parents,
    "policy_only_parents": policy_only_parents,
    "subjectivity_only_parents": subj_only_parents
}])
universe_counts.to_csv(UNIVERSE_COUNTS_CSV, index=False)

data["subjectivity"] = data["subjectivity"].fillna("unlabeled")
inters_mask = data["subjectivity"].isin(["subjected", "not_subjected"])
data_inter = data.loc[inters_mask].copy()

group_sizes = data_inter["subjectivity"].value_counts().to_dict()

rubric_cols = ["completeness_score", "usability_score", "accuracy_score"]
rubric_raw = data_inter[["raw_policy_domain", "parent_domain", "subjectivity"] + rubric_cols]
rubric_raw.to_csv(RUBRIC_CSV, index=False)

disclosure_cols = [
    "disclosure_data_collected",
    "disclosure_data_shared",
    "disclosure_purpose_of_collection",
    "disclosure_retention_period",
    "disclosure_right_to_access",
    "disclosure_right_to_delete",
    "disclosure_opt_out",
]

disc_rows = []
for label, g in data_inter.groupby("subjectivity"):
    n = len(g)
    for col in disclosure_cols:
        true_count = int(g[col].fillna(False).astype(bool).sum())
        disc_rows.append({
            "subjectivity": label,
            "metric": col,
            "true_count": true_count,
            "group_n": n,
            "true_pct": (true_count / n) if n else np.nan
        })
disclosure_coverage = pd.DataFrame(disc_rows).sort_values(["subjectivity", "metric"])
disclosure_coverage.to_csv(DISCLOSURE_CSV, index=False)

behavioral_cols = [
    "honors_gpc",
    "respects_dnt",
    "sets_cookies_before_consent",
    "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection",
    "uses_tracking_only_after_consent",
    "sells_data",
    "shares_with_third_parties",
]

behav_rows = []
for label, g in data_inter.groupby("subjectivity"):
    n = len(g)
    for col in behavioral_cols:
        counts = g[col].fillna("unspecified").value_counts()
        t = int(counts.get("true", 0))
        f = int(counts.get("false", 0))
        u = int(counts.get("unspecified", 0))
        behav_rows.append({
            "subjectivity": label,
            "claim": col,
            "true_count": t,
            "false_count": f,
            "unspecified_count": u,
            "group_n": n,
            "true_pct": t / n if n else np.nan,
            "false_pct": f / n if n else np.nan,
            "unspecified_pct": u / n if n else np.nan,
        })
behavioral_claims = pd.DataFrame(behav_rows).sort_values(["subjectivity", "claim"])
behavioral_claims.to_csv(BEHAVIORAL_CSV, index=False)

contr_rows = []
for label, g in data_inter.groupby("subjectivity"):
    n = len(g)
    c_true = int(g["policy_contradiction"].fillna(False).astype(bool).sum())
    contr_rows.append({
        "subjectivity": label,
        "contradictions": c_true,
        "group_n": n,
        "rate": c_true / n if n else np.nan
    })
contr_stats = pd.DataFrame(contr_rows).sort_values("subjectivity")
contr_stats.to_csv(CONTRADICT_CSV, index=False)

data_out_cols = [
    "raw_policy_domain", "parent_domain", "subjectivity",
    "completeness_score", "usability_score", "accuracy_score",
    "disclosure_data_collected", "disclosure_data_shared",
    "disclosure_purpose_of_collection", "disclosure_retention_period",
    "disclosure_right_to_access", "disclosure_right_to_delete",
    "disclosure_opt_out",
    "honors_gpc", "respects_dnt",
    "sets_cookies_before_consent", "sets_cookies_after_rejecting_consent",
    "deletes_cookies_on_rejection", "uses_tracking_only_after_consent",
    "sells_data", "shares_with_third_parties",
    "policy_contradiction",
]
data[data_out_cols].to_csv(JOINED_CSV, index=False)

print("\n=== Domain Universe & Intersection (by eTLD+1) ===")
print(universe_counts.to_string(index=False))

print("\n=== Group Sizes (intersection only) ===")
for k in ["subjected", "not_subjected"]:
    print(f"{k:>15}: {group_sizes.get(k, 0)}")

print("\n=== Sample Raw Rubric Values (first 10 rows) ===")
with pd.option_context('display.max_columns', None):
    print(rubric_raw.head(10).to_string(index=False))

print("\n=== Disclosure Coverage by Subjectivity (counts & pct of group) ===")
for label in ["subjected", "not_subjected"]:
    sub = disclosure_coverage[disclosure_coverage["subjectivity"] == label]
    if sub.empty:
        continue
    print(f"\n[{label}] n={int(sub['group_n'].iloc[0])}")
    for _, row in sub.iterrows():
        metric = row["metric"].replace("disclosure_", "")
        print(f"  {metric:>26}: {int(row['true_count']):4d}  ({row['true_pct']*100:5.1f}%)")

print("\n=== Behavioral Claims by Subjectivity (counts & pct of group) ===")
for label in ["subjected", "not_subjected"]:
    sub = behavioral_claims[behavioral_claims["subjectivity"] == label]
    if sub.empty:
        continue
    n = int(sub["group_n"].iloc[0])
    print(f"\n[{label}] n={n}")
    for _, row in sub.iterrows():
        name = row["claim"]
        t, f, u = int(row["true_count"]), int(row["false_count"]), int(row["unspecified_count"])
        tp, fp, up = row["true_pct"]*100, row["false_pct"]*100, row["unspecified_pct"]*100
        print(f"  {name:>34}: T={t:3d} ({tp:5.1f}%) | F={f:3d} ({fp:5.1f}%) | U={u:3d} ({up:5.1f}%)")

print("\n=== Policy Contradictions by Subjectivity ===")
print(contr_stats.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nWrote:")
print(f"  - {JOINED_CSV}")
print(f"  - {RUBRIC_CSV}  (raw per-website values)")
print(f"  - {DISCLOSURE_CSV}")
print(f"  - {BEHAVIORAL_CSV}")
print(f"  - {CONTRADICT_CSV}")
print(f"  - {UNIVERSE_COUNTS_CSV}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from math import sqrt

SPINE_COLOR = 'gray'
font = {'family' : 'Times', 'weight' : 'normal', 'size' : 12}
matplotlib.rc('font', **font)

def latexify(fig_width=None, fig_height=None, columns=1):
    assert(columns in [1,2])
    if fig_width is None:
        fig_width = 3.39 if columns==1 else 6.9
    if fig_height is None:
        golden_mean = (sqrt(5)-1.0)/2.0
        fig_height = fig_width*golden_mean
    MAX_HEIGHT_INCHES = 8.0
    if fig_height > MAX_HEIGHT_INCHES:
        fig_height = MAX_HEIGHT_INCHES
    params = {
        'backend': 'ps',
        'axes.labelsize': 10,
        'axes.titlesize': 12,
        'font.size': 10,
        'legend.fontsize': 10,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'text.usetex': False,
        'figure.figsize': [fig_width, fig_height],
        'font.family': 'Times'
    }
    matplotlib.rcParams.update(params)
    return params

def format_axes(ax):
    for side in ['top','right','left','bottom']:
        ax.spines[side].set_color(SPINE_COLOR)
        ax.spines[side].set_linewidth(0.5)
    ax.tick_params(direction='in', color=SPINE_COLOR, length=3)
    return ax

latexify(columns=2)

DISCLOSURE_CSV = "disclosure_coverage_by_industry_from_joined.csv"
BEHAVIORAL_CSV = "behavioral_claims_by_industry_from_joined.csv"
UNIVERSE_CSV   = "universe_counts_industry_from_joined.csv"

disc = pd.read_csv(DISCLOSURE_CSV)
beh  = pd.read_csv(BEHAVIORAL_CSV)

try:
    uni = pd.read_csv(UNIVERSE_CSV)
    industry_order = uni.sort_values("n_policies", ascending=False)["industry"].tolist()
except Exception:

    disc_counts = disc.groupby("industry")["group_n"].max().sort_values(ascending=False)
    industry_order = disc_counts.index.tolist()

disc_pivot = (disc
              .assign(true_pct=lambda d: d["true_pct"].astype(float) * 100.0)
              .pivot(index="industry", columns="metric", values="true_pct")
             ).reindex(index=industry_order)

beh_pivot = (beh
             .assign(true_pct=lambda d: d["true_pct"].astype(float) * 100.0)
             .pivot(index="industry", columns="claim", values="true_pct")
            ).reindex(index=industry_order)

disc_pivot = disc_pivot.reindex(sorted(disc_pivot.columns), axis=1)
beh_pivot  = beh_pivot.reindex(sorted(beh_pivot.columns), axis=1)

disc_cols = list(disc_pivot.columns)
beh_cols  = list(beh_pivot.columns)

combined = pd.concat([disc_pivot, beh_pivot], axis=1)
combined = combined.fillna(0.0)

disc_pivot.round(1).to_csv("plottable_disclosure_heatmap.csv")
beh_pivot.round(1).to_csv("plottable_behavior_heatmap.csv")
combined.round(1).to_csv("plottable_combined_heatmap.csv")

rows, cols = combined.shape
colsep = len(disc_cols)

cell_w = 0.32
cell_h = 0.28
fig_w = min(18.0, max(6.9, cols * cell_w))
fig_h = min(8.0, max(3.5, rows * cell_h))
plt.figure(figsize=(fig_w, fig_h), dpi=300)

ax = plt.gca()
im = ax.imshow(combined.values, aspect='auto', interpolation='nearest', vmin=0, vmax=100, cmap='viridis')

ax.set_yticks(np.arange(rows))
ax.set_yticklabels(combined.index)
ax.set_xticks(np.arange(cols))
ax.set_xticklabels(list(combined.columns), rotation=60, ha='right')

ax.axvline(x=colsep - 0.5, color='k', linewidth=1.0)

ax.text((colsep-1)/2, -1.2, "Disclosures", ha='center', va='bottom', fontsize=11)
ax.text((colsep + cols - 1)/2, -1.2, "Behavioral Claims", ha='center', va='bottom', fontsize=11)

cbar = plt.colorbar(im, fraction=0.025, pad=0.02)
cbar.set_label('% of policies (coverage / “true” claim)')

ax.set_xticks(np.arange(-.5, cols, 1), minor=True)
ax.set_yticks(np.arange(-.5, rows, 1), minor=True)
ax.grid(which='minor', color='w', linestyle='-', linewidth=0.3, alpha=0.6)
ax.tick_params(which='minor', bottom=False, left=False)

format_axes(ax)
plt.tight_layout()
plt.savefig("industry_disclosure+behavior_heatmap.png", bbox_inches="tight", dpi=300)
plt.savefig("industry_disclosure+behavior_heatmap.pdf", bbox_inches="tight")
print("Wrote: industry_disclosure+behavior_heatmap.(png|pdf) and CSVs for the plotted matrices.")

def plot_single_heatmap(mat: pd.DataFrame, title: str, fname_prefix: str):
    r, c = mat.shape
    cell_w = 0.32
    cell_h = 0.28
    fig_w = min(18.0, max(6.9, c * cell_w))
    fig_h = min(8.0, max(3.5, r * cell_h))
    plt.figure(figsize=(fig_w, fig_h), dpi=300)
    ax = plt.gca()
    im = ax.imshow(mat.values, aspect='auto', interpolation='nearest', vmin=0, vmax=100, cmap='viridis')
    ax.set_yticks(np.arange(r))
    ax.set_yticklabels(mat.index)
    ax.set_xticks(np.arange(c))
    ax.set_xticklabels(list(mat.columns), rotation=60, ha='right')
    cbar = plt.colorbar(im, fraction=0.025, pad=0.02)
    cbar.set_label('% of policies')
    ax.set_title(title, pad=8)
    format_axes(ax)
    plt.tight_layout()
    plt.savefig(f"{fname_prefix}.png", bbox_inches="tight", dpi=300)
    plt.savefig(f"{fname_prefix}.pdf", bbox_inches="tight")
    print(f"Wrote: {fname_prefix}.(png|pdf)")

plot_single_heatmap(disc_pivot, "Disclosure coverage by industry", "industry_disclosure_heatmap")
plot_single_heatmap(beh_pivot,  "Behavioral claims (%% true) by industry", "industry_behavior_heatmap")

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import tldextract

POLICY_JSON = "ccpa_policy_audit_all_clear_pastversion.json"
SUBJECTIVITY_JSON = "policy_domains_ccpa_subjectivity.json"
OUT_CSV = "data_practices_by_subjectivity.csv"

ALIAS_FIXES = {

}

def etld1(host: str) -> str:
    if not host:
        return ""
    ext = tldextract.extract(str(host).strip().lower())
    if not ext.domain or not ext.suffix:
        return str(host).strip().lower()
    return f"{ext.domain}.{ext.suffix}"

def apply_alias(d: str) -> str:
    return ALIAS_FIXES.get(d, d)

def _mentioned(txt):
    """Return True if txt meaningfully mentions practices (not empty / not 'Not mentioned.')."""
    if txt is None:
        return False
    s = str(txt).strip().lower()
    return bool(s) and s not in {"not mentioned", "not mentioned."}

p_policy = Path(POLICY_JSON)
p_subj = Path(SUBJECTIVITY_JSON)
if not p_policy.exists():
    raise FileNotFoundError(f"Missing {p_policy.resolve()}")
if not p_subj.exists():
    raise FileNotFoundError(f"Missing {p_subj.resolve()}")

with open(p_policy, "r") as f:
    policy_data = json.load(f)

with open(p_subj, "r") as f:
    subj_map_raw = json.load(f)

policy_rows = []
for raw_dom, rec in policy_data.items():
    parent = apply_alias(etld1(raw_dom))
    policy_rows.append({
        "raw_domain": raw_dom,
        "parent_domain": parent,
        "online_data_practices": rec.get("online_data_practices"),
        "offline_data_practices": rec.get("offline_data_practices"),
    })
policy_df = pd.DataFrame(policy_rows)

subj_rows = []
for d, label in subj_map_raw.items():
    subj_rows.append({
        "parent_domain": apply_alias(etld1(d)),
        "subjectivity": str(label).strip().lower(),
    })
subj_df = pd.DataFrame(subj_rows).drop_duplicates("parent_domain")

joined = policy_df.merge(subj_df, on="parent_domain", how="left")
joined["subjectivity"] = joined["subjectivity"].fillna("unlabeled")

joined["online_mentioned"] = joined["online_data_practices"].apply(_mentioned)
joined["offline_mentioned"] = joined["offline_data_practices"].apply(_mentioned)

joined["online_len"] = joined["online_data_practices"].fillna("").map(lambda s: len(str(s)))
joined["offline_len"] = joined["offline_data_practices"].fillna("").map(lambda s: len(str(s)))

def pct(num, den):
    return (num / den) if den else np.nan

rows = []
for label, g in joined.groupby("subjectivity"):
    n = len(g)
    online_yes = int(g["online_mentioned"].sum())
    offline_yes = int(g["offline_mentioned"].sum())
    both_yes = int((g["online_mentioned"] & g["offline_mentioned"]).sum())
    neither = int((~g["online_mentioned"] & ~g["offline_mentioned"]).sum())

    rows.append({
        "subjectivity": label,
        "group_n": n,
        "online_yes": online_yes,
        "online_pct": pct(online_yes, n),
        "offline_yes": offline_yes,
        "offline_pct": pct(offline_yes, n),
        "both_yes": both_yes,
        "both_pct": pct(both_yes, n),
        "neither": neither,
        "neither_pct": pct(neither, n),

        "avg_online_len": g["online_len"].mean() if n else np.nan,
        "avg_offline_len": g["offline_len"].mean() if n else np.nan,
    })

out = pd.DataFrame(rows).sort_values("subjectivity")
out.to_csv(OUT_CSV, index=False)

print("\n=== Data Practices by Subjectivity ===")
def fmt(p):
    return f"{p*100:5.1f}%" if pd.notna(p) else "  n/a "
for _, r in out.iterrows():
    print(
        f"[{r['subjectivity']}] n={int(r['group_n'])} | "
        f"online: {int(r['online_yes'])} ({fmt(r['online_pct'])}) | "
        f"offline: {int(r['offline_yes'])} ({fmt(r['offline_pct'])}) | "
        f"both: {int(r['both_yes'])} ({fmt(r['both_pct'])}) | "
        f"neither: {int(r['neither'])} ({fmt(r['neither_pct'])})"
    )
print(f"\nWrote: {OUT_CSV}")

In [ ]:
import pandas as pd

CSV_PATH = "data_source-2.csv"

df = pd.read_csv(CSV_PATH)

if "CCPA required" not in df.columns:
    raise ValueError(f"'CCPA required' column not found! Available columns: {list(df.columns)}")

types = df["CCPA required"].dropna().astype(str).str.strip().unique()

print("=== Unique values in 'CCPA required' column ===")
for t in sorted(types):
    print(f"- {t}")

In [ ]:
import pandas as pd

df = pd.read_csv("cookie_data_final - merged_updated.csv")

targeting_cookies = ["IDE", "_fbp", "_ga", "_gid", "uuid2", "MUID"]

mask = df["cookie_name"].str.strip().isin(targeting_cookies)
found = df[mask]

print("Found Targeting/Advertising Cookies:")
print(found)

counts = found["cookie_name"].value_counts()
print("\nCounts per cookie:")
print(counts)